In [ ]:
# =============================================================================
# CELL 1 — Environment, reproducibility, project folders, and GPU information
# =============================================================================
# PURPOSE
# -------
# 1. Fix random seeds so that generated noise and later experiments are
#    reproducible.
# 2. Detect the available hardware, including the GPU.
# 3. Create an organized project structure for datasets, figures, tables,
#    and validation reports.
#
# IMPORTANT
# ---------
# Round 1 uses SciPy's solve_ivp function, which runs mainly on the CPU.
# Your 8 GB GPU will become important when the PINNs are trained in later
# experimental rounds.

from pathlib import Path
import os
import sys
import platform
import random
import json
import warnings

import numpy as np

# Reproducibility seed
SEED = 20260803

random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

# -------------------------------------------------------------------------
# Determine the project root.
# If this notebook is inside a folder called "notebooks", the project root
# is assumed to be its parent folder.
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# -------------------------------------------------------------------------
# Create the project folders.
# -------------------------------------------------------------------------
DATA_DENSE = PROJECT_ROOT / "data" / "dense"
DATA_SPARSE = PROJECT_ROOT / "data" / "sparse"
DATA_NOISY = PROJECT_ROOT / "data" / "noisy"

FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

for directory in [
    DATA_DENSE,
    DATA_SPARSE,
    DATA_NOISY,
    FIGURE_DIR,
    TABLE_DIR,
    REPORT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# Display the computational environment.
# -------------------------------------------------------------------------
print("Project root:", PROJECT_ROOT)
print("Python version:", sys.version.split()[0])
print("Operating system:", platform.platform())
print("Random seed:", SEED)

# -------------------------------------------------------------------------
# Detect PyTorch and GPU information.
# Round 1 does not require the GPU, but this confirms that the system is
# ready for the later PINN experiments.
# -------------------------------------------------------------------------
try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

        total_memory_gb = (
            torch.cuda.get_device_properties(0).total_memory / 1024**3
        )

        print(f"GPU memory: {total_memory_gb:.2f} GB")

except Exception as exc:
    print("PyTorch/GPU information could not be obtained:", exc)

In [ ]:
# =============================================================================
# CELL 2 — Scientific imports and publication-output settings
# =============================================================================
# PURPOSE
# -------
# This cell imports all scientific libraries needed for:
#
# 1. Solving the deterministic AMR differential equations;
# 2. Storing and processing simulation results;
# 3. Interpolating numerical trajectories;
# 4. Conducting parameter-sensitivity analysis;
# 5. Generating high-resolution publication figures;
# 6. Saving figures in PNG, PDF, and SVG formats.
#
# IMPORTANT
# ---------
# This cell does not solve the AMR model yet.
# It only prepares the scientific environment and defines a reusable
# figure-saving function.
#
# Therefore, a cell containing only imports and function definitions normally
# produces no output. Confirmation messages are included at the end of this
# version so that successful execution is visible.

# -------------------------------------------------------------------------
# Python utilities
# -------------------------------------------------------------------------
from dataclasses import dataclass, asdict, replace
from typing import Callable, Dict, Tuple, Iterable
from pathlib import Path

# -------------------------------------------------------------------------
# Numerical and tabular computing
# -------------------------------------------------------------------------
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Scientific plotting
# -------------------------------------------------------------------------
import matplotlib
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# Differential-equation solver and numerical interpolation
# -------------------------------------------------------------------------
import scipy
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.stats import linregress


# =============================================================================
# CHECK THAT CELL 1 HAS ALREADY BEEN EXECUTED
# =============================================================================
# Cell 1 creates FIGURE_DIR. This test prevents later errors caused by running
# the notebook cells in the wrong order.

if "FIGURE_DIR" not in globals():
    raise RuntimeError(
        "FIGURE_DIR has not been defined. "
        "Please run Cell 1 before running Cell 2."
    )

# Ensure that the figure folder exists.
FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# PUBLICATION-QUALITY MATPLOTLIB SETTINGS
# =============================================================================
# These settings are applied to all figures generated later in the notebook.
#
# The figures will use:
# - readable font sizes;
# - clear axis lines;
# - high-resolution output;
# - sufficiently thick curves;
# - framed legends;
# - editable vector-format export.
#
# Matplotlib's default color sequence is retained. Individual publication
# figures can later be refined according to the target journal's requirements.

plt.rcParams.update(
    {
        # Display resolution inside Jupyter Notebook
        "figure.dpi": 120,

        # Resolution used when raster figures are saved
        "savefig.dpi": 300,

        # Default figure size
        "figure.figsize": (8.6, 5.4),

        # Font settings
        "font.family": "sans-serif",
        "font.size": 11,

        # Axis-label and title sizes
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "axes.titleweight": "normal",

        # Tick-label sizes
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,

        # Axis-line properties
        "axes.linewidth": 1.0,

        # Curve properties
        "lines.linewidth": 2.0,

        # Legend properties
        "legend.fontsize": 10,
        "legend.frameon": True,
        "legend.framealpha": 1.0,

        # Saved-figure settings
        "savefig.bbox": "tight",

        # Prevent minus signs from being shown incorrectly
        "axes.unicode_minus": False,
    }
)


# =============================================================================
# REUSABLE FIGURE-SAVING FUNCTION
# =============================================================================

def save_figure(
    fig: plt.Figure,
    stem: str,
    close_after_saving: bool = False,
) -> None:
    """
    Save one publication figure in PNG, PDF, and SVG formats.

    Parameters
    ----------
    fig:
        Matplotlib figure object to be saved.

    stem:
        Output filename without a file extension.

        Example
        -------
        stem = "figure_total_population"

        This produces:
        - figure_total_population.png
        - figure_total_population.pdf
        - figure_total_population.svg

    close_after_saving:
        When True, close the figure after saving it. This is useful when many
        figures are generated inside a loop because it prevents unnecessary
        memory use.

        The default is False so that figures remain visible in Jupyter.

    Output formats
    --------------
    PNG:
        High-resolution 300-dpi raster image suitable for previews,
        presentations, and many manuscript-submission systems.

    PDF:
        Vector format suitable for LaTeX manuscripts and publication.

    SVG:
        Editable vector format suitable for Adobe Illustrator, Inkscape,
        PowerPoint, and further graphical refinement.
    """

    # ---------------------------------------------------------------------
    # Validate the inputs.
    # ---------------------------------------------------------------------
    if not isinstance(fig, plt.Figure):
        raise TypeError(
            "fig must be a Matplotlib Figure object."
        )

    if not isinstance(stem, str):
        raise TypeError(
            "stem must be a string."
        )

    if not stem.strip():
        raise ValueError(
            "stem cannot be empty."
        )

    # Remove spaces at the beginning and end.
    stem = stem.strip()

    # Replace internal spaces with underscores to create safe filenames.
    safe_stem = stem.replace(" ", "_")

    # ---------------------------------------------------------------------
    # Define the three output paths.
    # ---------------------------------------------------------------------
    png_path = FIGURE_DIR / f"{safe_stem}.png"
    pdf_path = FIGURE_DIR / f"{safe_stem}.pdf"
    svg_path = FIGURE_DIR / f"{safe_stem}.svg"

    # ---------------------------------------------------------------------
    # Save high-resolution PNG.
    # ---------------------------------------------------------------------
    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    # ---------------------------------------------------------------------
    # Save vector PDF.
    # ---------------------------------------------------------------------
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )

    # ---------------------------------------------------------------------
    # Save editable vector SVG.
    # ---------------------------------------------------------------------
    fig.savefig(
        svg_path,
        bbox_inches="tight",
        facecolor="white",
    )

    # Optionally close the figure to release memory.
    if close_after_saving:
        plt.close(fig)


# =============================================================================
# EXECUTION CONFIRMATION
# =============================================================================
# These print statements confirm that:
# - all imports worked;
# - Matplotlib was configured;
# - the saving function was defined;
# - the output folder is available.

print("Cell 2 completed successfully.")
print("-" * 60)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("SciPy version:", scipy.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Publication figure settings have been applied.")
print("The save_figure() function has been defined.")
print("Figure output folder:", FIGURE_DIR.resolve())

In [ ]:
# =============================================================================
# CELL 3 — Biological parameters and numerical configuration
# =============================================================================
# PURPOSE
# -------
# Store the biological and numerical parameters in validated dataclasses.
#
# WHY THIS IS IMPORTANT
# ---------------------
# Using validated parameter containers prevents accidental use of impossible
# values such as:
# - negative carrying capacity;
# - mutation probability greater than one;
# - negative death rates;
# - initial population above carrying capacity.
#
# The baseline mutation probability is initially set to 1e-4 because it
# produces a visible resistant trajectory during debugging. Later experiments
# will also examine 1e-5 and 1e-3.

@dataclass(frozen=True)
class AMRParameters:
    """Biological parameters of the sensitive-resistant AMR model."""

    carrying_capacity: float = 1000.0
    sensitive_birth_rate: float = 1.0
    resistance_fitness_cost: float = 0.10
    sensitive_death_rate: float = 0.10
    resistant_death_rate: float = 0.10
    mutation_probability: float = 1.0e-4
    sensitive_drug_effect: float = 1.0
    resistant_drug_effect: float = 0.10

    @property
    def resistant_birth_rate(self) -> float:
        """
        Calculate the resistant birth rate after accounting for fitness cost.

        r_R = r_S(1-c_R)
        """

        return self.sensitive_birth_rate * (
            1.0 - self.resistance_fitness_cost
        )

    def validate(self) -> None:
        """Check that all biological parameters are admissible."""

        if self.carrying_capacity <= 0:
            raise ValueError("carrying_capacity must be positive.")

        if self.sensitive_birth_rate <= 0:
            raise ValueError("sensitive_birth_rate must be positive.")

        if self.sensitive_death_rate < 0:
            raise ValueError("sensitive_death_rate cannot be negative.")

        if self.resistant_death_rate < 0:
            raise ValueError("resistant_death_rate cannot be negative.")

        bounded_parameters = {
            "resistance_fitness_cost": self.resistance_fitness_cost,
            "mutation_probability": self.mutation_probability,
            "sensitive_drug_effect": self.sensitive_drug_effect,
            "resistant_drug_effect": self.resistant_drug_effect,
        }

        for name, value in bounded_parameters.items():
            if not 0.0 <= value <= 1.0:
                raise ValueError(
                    f"{name} must lie between zero and one."
                )

        if self.resistant_birth_rate < 0:
            raise ValueError(
                "The calculated resistant birth rate cannot be negative."
            )

    def as_dictionary(self) -> Dict[str, float]:
        """Return all parameters as a dictionary."""

        result = asdict(self)
        result["resistant_birth_rate"] = self.resistant_birth_rate

        return result


@dataclass(frozen=True)
class SimulationConfig:
    """Numerical configuration for deterministic data generation."""

    initial_sensitive: float = 100.0
    initial_resistant: float = 0.0

    start_time: float = 0.0
    final_time: float = 20.0

    dense_points: int = 1001
    sparse_total_points: int = 21

    relative_tolerance: float = 1.0e-9
    absolute_tolerance: float = 1.0e-11

    solver_method: str = "DOP853"

    def validate(self, carrying_capacity: float) -> None:
        """Check that the numerical configuration is admissible."""

        if self.initial_sensitive < 0:
            raise ValueError(
                "Initial sensitive population cannot be negative."
            )

        if self.initial_resistant < 0:
            raise ValueError(
                "Initial resistant population cannot be negative."
            )

        initial_total = (
            self.initial_sensitive + self.initial_resistant
        )

        if initial_total > carrying_capacity:
            raise ValueError(
                "Initial total population exceeds carrying capacity."
            )

        if self.final_time <= self.start_time:
            raise ValueError(
                "final_time must be greater than start_time."
            )

        if self.dense_points < 2:
            raise ValueError(
                "dense_points must contain at least two points."
            )

        if self.sparse_total_points < 2:
            raise ValueError(
                "sparse_total_points must contain at least two points."
            )


# -------------------------------------------------------------------------
# Instantiate and validate the baseline configurations.
# -------------------------------------------------------------------------
PARAMETERS = AMRParameters()
CONFIG = SimulationConfig()

PARAMETERS.validate()
CONFIG.validate(PARAMETERS.carrying_capacity)

# -------------------------------------------------------------------------
# Display a parameter table.
# -------------------------------------------------------------------------
parameter_dictionary = PARAMETERS.as_dictionary()

parameter_table = pd.DataFrame(
    {
        "Parameter": list(parameter_dictionary.keys()),
        "Value": list(parameter_dictionary.values()),
    }
)

parameter_table

In [ ]:
# =============================================================================
# CELL 4 — Prescribed antibiotic-treatment schedules
# =============================================================================
# PURPOSE
# -------
# Define the external antibiotic input A(t) for four experiments:
#
# 1. Untreated control;
# 2. Early continuous treatment;
# 3. Late continuous treatment;
# 4. Pulsed treatment.
#
# These schedules are prescribed rather than optimized. This is deliberate:
# Control-PINNs and reinforcement-learning antibiotic policies already exist.
# The first paper focuses on state and parameter inference.

TreatmentFunction = Callable[
    [float | np.ndarray],
    float | np.ndarray,
]


@dataclass(frozen=True)
class TreatmentSchedule:
    """Named normalized antibiotic-exposure function."""

    name: str
    function: TreatmentFunction
    description: str

    def __call__(
        self,
        t: float | np.ndarray,
    ) -> float | np.ndarray:
        """
        Evaluate the treatment schedule and ensure 0 <= A(t) <= 1.
        """

        values = np.asarray(
            self.function(t),
            dtype=float,
        )

        if np.any(~np.isfinite(values)):
            raise ValueError(
                f"{self.name} produced non-finite values."
            )

        if np.any((values < 0.0) | (values > 1.0)):
            raise ValueError(
                f"{self.name} must remain inside [0,1]."
            )

        if np.isscalar(t):
            return float(values)

        return values


def untreated_schedule() -> TreatmentSchedule:
    """No antibiotic exposure."""

    return TreatmentSchedule(
        name="untreated",
        function=lambda t: np.zeros_like(
            np.asarray(t, dtype=float)
        ),
        description="No antibiotic exposure.",
    )


def early_continuous_schedule(
    start: float = 5.0,
) -> TreatmentSchedule:
    """Continuous antibiotic exposure beginning at t=start."""

    return TreatmentSchedule(
        name="early_continuous",
        function=lambda t: (
            np.asarray(t, dtype=float) >= start
        ).astype(float),
        description=(
            f"Continuous antibiotic exposure beginning at t={start}."
        ),
    )


def late_continuous_schedule(
    start: float = 10.0,
) -> TreatmentSchedule:
    """Continuous antibiotic exposure beginning later."""

    return TreatmentSchedule(
        name="late_continuous",
        function=lambda t: (
            np.asarray(t, dtype=float) >= start
        ).astype(float),
        description=(
            f"Continuous antibiotic exposure beginning at t={start}."
        ),
    )


def pulsed_schedule(
    first_on: float = 5.0,
    first_off: float = 8.0,
    second_on: float = 12.0,
    second_off: float = 15.0,
) -> TreatmentSchedule:
    """Two finite antibiotic pulses."""

    def exposure(
        t: float | np.ndarray,
    ) -> np.ndarray:

        t_array = np.asarray(t, dtype=float)

        first_pulse = (
            (t_array >= first_on)
            & (t_array < first_off)
        )

        second_pulse = (
            (t_array >= second_on)
            & (t_array < second_off)
        )

        return (
            first_pulse | second_pulse
        ).astype(float)

    return TreatmentSchedule(
        name="pulsed",
        function=exposure,
        description=(
            f"Exposure on [{first_on},{first_off}) and "
            f"[{second_on},{second_off})."
        ),
    )


# -------------------------------------------------------------------------
# Store all schedules in one dictionary.
# -------------------------------------------------------------------------
SCHEDULES = {
    schedule.name: schedule
    for schedule in [
        untreated_schedule(),
        early_continuous_schedule(),
        late_continuous_schedule(),
        pulsed_schedule(),
    ]
}

schedule_table = pd.DataFrame(
    {
        "Schedule": list(SCHEDULES.keys()),
        "Description": [
            schedule.description
            for schedule in SCHEDULES.values()
        ],
    }
)

schedule_table

In [ ]:
# =============================================================================
# CELL 5 — Deterministic AMR differential equations
# =============================================================================
# PURPOSE
# -------
# Implement the mean-field sensitive-resistant bacterial model.
#
# BIOLOGICAL INTERPRETATION
# -------------------------
# 1. Sensitive division without mutation increases S.
# 2. A mutated sensitive division creates a resistant offspring.
# 3. The sensitive parent remains present during the mutation event.
# 4. Resistant bacteria reproduce with a fitness cost.
# 5. Antibiotic exposure suppresses sensitive and resistant birth rates by
#    phenotype-specific amounts.
# 6. Both populations experience logistic crowding.
#
# No artificial clipping is performed inside the ODE because clipping would
# alter the mathematical system. Biological feasibility is tested separately.

def amr_rhs(
    t: float,
    state: np.ndarray,
    parameters: AMRParameters,
    schedule: TreatmentSchedule,
) -> np.ndarray:
    """
    Calculate dS/dt and dR/dt.

    Parameters
    ----------
    t:
        Current time.

    state:
        Array containing [S, R].

    parameters:
        Biological parameter object.

    schedule:
        Antibiotic-exposure function A(t).
    """

    sensitive, resistant = state

    total_population = sensitive + resistant

    antibiotic_exposure = float(schedule(t))

    # Drug-modified sensitive birth rate
    sensitive_birth_rate = (
        parameters.sensitive_birth_rate
        * (
            1.0
            - parameters.sensitive_drug_effect
            * antibiotic_exposure
        )
    )

    # Drug-modified resistant birth rate
    resistant_birth_rate = (
        parameters.resistant_birth_rate
        * (
            1.0
            - parameters.resistant_drug_effect
            * antibiotic_exposure
        )
    )

    # Logistic crowding factor
    crowding = (
        1.0
        - total_population
        / parameters.carrying_capacity
    )

    # Sensitive population equation
    d_sensitive = (
        sensitive_birth_rate
        * (1.0 - parameters.mutation_probability)
        * crowding
        * sensitive
        - parameters.sensitive_death_rate
        * sensitive
    )

    # Resistant population equation
    d_resistant = (
        sensitive_birth_rate
        * parameters.mutation_probability
        * crowding
        * sensitive
        + resistant_birth_rate
        * crowding
        * resistant
        - parameters.resistant_death_rate
        * resistant
    )

    return np.array(
        [d_sensitive, d_resistant],
        dtype=float,
    )


# -------------------------------------------------------------------------
# Inspect the initial derivatives for each schedule.
# At t=0 the treatment schedules are inactive, so the initial derivatives
# should be the same for all four experiments.
# -------------------------------------------------------------------------
initial_state = np.array(
    [
        CONFIG.initial_sensitive,
        CONFIG.initial_resistant,
    ],
    dtype=float,
)

for schedule_name, schedule in SCHEDULES.items():

    derivative = amr_rhs(
        CONFIG.start_time,
        initial_state,
        PARAMETERS,
        schedule,
    )

    print(
        f"{schedule_name:>18s}: "
        f"dS/dt={derivative[0]: .6f}, "
        f"dR/dt={derivative[1]: .6f}"
    )

In [ ]:
# =============================================================================
# CELL 6 — High-accuracy deterministic ODE solver
# =============================================================================
# PURPOSE
# -------
# Solve the AMR model using SciPy's adaptive DOP853 solver.
#
# OUTPUT
# ------
# A tidy dataframe containing:
# - time;
# - sensitive population;
# - resistant population;
# - total population;
# - resistant fraction;
# - antibiotic exposure;
# - treatment-schedule name.
#
# These dense trajectories will become the trusted ground truth for later PINN
# training and evaluation.

def solve_deterministic(
    parameters: AMRParameters,
    config: SimulationConfig,
    schedule: TreatmentSchedule,
) -> pd.DataFrame:
    """Solve the deterministic AMR model."""

    parameters.validate()
    config.validate(parameters.carrying_capacity)

    time_grid = np.linspace(
        config.start_time,
        config.final_time,
        config.dense_points,
        dtype=float,
    )

    solution = solve_ivp(
        fun=lambda t, state: amr_rhs(
            t,
            state,
            parameters,
            schedule,
        ),
        t_span=(
            config.start_time,
            config.final_time,
        ),
        y0=np.array(
            [
                config.initial_sensitive,
                config.initial_resistant,
            ],
            dtype=float,
        ),
        t_eval=time_grid,
        method=config.solver_method,
        rtol=config.relative_tolerance,
        atol=config.absolute_tolerance,
    )

    if not solution.success:
        raise RuntimeError(
            f"Integration failed for {schedule.name}: "
            f"{solution.message}"
        )

    sensitive = solution.y[0]
    resistant = solution.y[1]

    total = sensitive + resistant

    resistant_fraction = np.divide(
        resistant,
        total,
        out=np.zeros_like(resistant),
        where=total > 0,
    )

    antibiotic = np.asarray(
        schedule(solution.t),
        dtype=float,
    )

    results = pd.DataFrame(
        {
            "time": solution.t,
            "sensitive": sensitive,
            "resistant": resistant,
            "total": total,
            "resistant_fraction": resistant_fraction,
            "antibiotic": antibiotic,
            "schedule": schedule.name,
        }
    )

    return results

In [ ]:
# =============================================================================
# CELL 7 — Biological and numerical validation
# =============================================================================
# PURPOSE
# -------
# Explicitly test whether every deterministic trajectory satisfies:
#
#     S(t) >= 0
#     R(t) >= 0
#     S(t) + R(t) <= K
#     0 <= R/(S+R) <= 1
#
# It also checks:
# - absence of NaN and infinite values;
# - correct initial conditions.
#
# This follows the important lesson from PINN-Proj and KKT-hPINN:
# prediction accuracy and constraint satisfaction should be measured
# independently.

def validate_trajectory(
    data: pd.DataFrame,
    parameters: AMRParameters,
    config: SimulationConfig,
    tolerance: float = 1.0e-6,
) -> Dict[str, float | bool]:
    """Validate one deterministic AMR trajectory."""

    numeric_columns = [
        "time",
        "sensitive",
        "resistant",
        "total",
        "resistant_fraction",
        "antibiotic",
    ]

    numeric_values = data[
        numeric_columns
    ].to_numpy(dtype=float)

    finite_values = bool(
        np.all(np.isfinite(numeric_values))
    )

    minimum_sensitive = float(
        data["sensitive"].min()
    )

    minimum_resistant = float(
        data["resistant"].min()
    )

    maximum_total = float(
        data["total"].max()
    )

    minimum_fraction = float(
        data["resistant_fraction"].min()
    )

    maximum_fraction = float(
        data["resistant_fraction"].max()
    )

    initial_sensitive_error = abs(
        float(data.iloc[0]["sensitive"])
        - config.initial_sensitive
    )

    initial_resistant_error = abs(
        float(data.iloc[0]["resistant"])
        - config.initial_resistant
    )

    report = {
        "finite_values": finite_values,
        "minimum_sensitive": minimum_sensitive,
        "minimum_resistant": minimum_resistant,
        "maximum_total": maximum_total,
        "minimum_resistant_fraction": minimum_fraction,
        "maximum_resistant_fraction": maximum_fraction,
        "initial_sensitive_error": initial_sensitive_error,
        "initial_resistant_error": initial_resistant_error,
        "nonnegative_populations": (
            minimum_sensitive >= -tolerance
            and minimum_resistant >= -tolerance
        ),
        "capacity_respected": (
            maximum_total
            <= parameters.carrying_capacity
            + tolerance
        ),
        "fraction_in_unit_interval": (
            minimum_fraction >= -tolerance
            and maximum_fraction <= 1.0 + tolerance
        ),
        "initial_conditions_respected": (
            initial_sensitive_error <= tolerance
            and initial_resistant_error <= tolerance
        ),
    }

    report["all_checks_passed"] = bool(
        report["finite_values"]
        and report["nonnegative_populations"]
        and report["capacity_respected"]
        and report["fraction_in_unit_interval"]
        and report["initial_conditions_respected"]
    )

    return report

In [ ]:
# =============================================================================
# CELL 8 — Run all four deterministic treatment scenarios
# =============================================================================
# PURPOSE
# -------
# Generate the dense reference trajectories for:
# 1. untreated control;
# 2. early continuous treatment;
# 3. late continuous treatment;
# 4. pulsed treatment.
#
# Every trajectory is validated and saved as a CSV file.

TRAJECTORIES: Dict[str, pd.DataFrame] = {}

VALIDATION_REPORTS: Dict[
    str,
    Dict[str, float | bool],
] = {}

summary_rows = []

for schedule_name, schedule in SCHEDULES.items():

    trajectory = solve_deterministic(
        PARAMETERS,
        CONFIG,
        schedule,
    )

    validation = validate_trajectory(
        trajectory,
        PARAMETERS,
        CONFIG,
    )

    if not validation["all_checks_passed"]:
        raise RuntimeError(
            f"Validation failed for {schedule_name}."
        )

    TRAJECTORIES[schedule_name] = trajectory
    VALIDATION_REPORTS[schedule_name] = validation

    # Save the complete dense trajectory
    trajectory.to_csv(
        DATA_DENSE
        / f"{schedule_name}_dense.csv",
        index=False,
    )

    summary_rows.append(
        {
            "schedule": schedule_name,
            "final_sensitive": float(
                trajectory.iloc[-1]["sensitive"]
            ),
            "final_resistant": float(
                trajectory.iloc[-1]["resistant"]
            ),
            "final_total": float(
                trajectory.iloc[-1]["total"]
            ),
            "final_resistant_fraction": float(
                trajectory.iloc[-1][
                    "resistant_fraction"
                ]
            ),
            "peak_total": float(
                trajectory["total"].max()
            ),
            "minimum_total": float(
                trajectory["total"].min()
            ),
            "all_checks_passed": bool(
                validation["all_checks_passed"]
            ),
        }
    )


SUMMARY = pd.DataFrame(summary_rows)

SUMMARY.to_csv(
    TABLE_DIR
    / "round1_deterministic_summary.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "round1_validation_reports.json",
    "w",
) as file:

    json.dump(
        {
            "parameters": (
                PARAMETERS.as_dictionary()
            ),
            "simulation": asdict(CONFIG),
            "validation": VALIDATION_REPORTS,
        },
        file,
        indent=2,
    )

SUMMARY

In [ ]:
# =============================================================================
# CELL 9 — Sparse observations for the future inverse PINN
# =============================================================================
# PURPOSE
# -------
# Construct the central partially observed inverse problem:
#
# - total bacterial load measured at 21 time points;
# - resistant fraction measured at only three time points.
#
# This reflects the BINNs principle of learning biological dynamics from sparse
# measurements while creating an AMR-specific hidden-state problem.

def nearest_rows(
    data: pd.DataFrame,
    requested_times: Iterable[float],
) -> pd.DataFrame:
    """
    Select dataframe rows nearest to requested observation times.
    """

    available_times = data[
        "time"
    ].to_numpy(dtype=float)

    indices = [
        int(
            np.argmin(
                np.abs(
                    available_times
                    - requested_time
                )
            )
        )
        for requested_time in requested_times
    ]

    unique_indices = np.unique(indices)

    return (
        data.iloc[unique_indices]
        .copy()
        .reset_index(drop=True)
    )


SPARSE_DATASETS: Dict[
    str,
    Dict[str, pd.DataFrame],
] = {}


for schedule_name, trajectory in TRAJECTORIES.items():

    total_observation_times = np.linspace(
        CONFIG.start_time,
        CONFIG.final_time,
        CONFIG.sparse_total_points,
    )

    phenotype_observation_times = [
        7.0,
        12.0,
        18.0,
    ]

    total_observations = nearest_rows(
        trajectory,
        total_observation_times,
    )[
        [
            "time",
            "total",
            "antibiotic",
            "schedule",
        ]
    ]

    phenotype_observations = nearest_rows(
        trajectory,
        phenotype_observation_times,
    )[
        [
            "time",
            "resistant_fraction",
            "antibiotic",
            "schedule",
        ]
    ]

    total_observations.to_csv(
        DATA_SPARSE
        / f"{schedule_name}_total_load.csv",
        index=False,
    )

    phenotype_observations.to_csv(
        DATA_SPARSE
        / f"{schedule_name}_resistant_fraction.csv",
        index=False,
    )

    SPARSE_DATASETS[schedule_name] = {
        "total_load": total_observations,
        "resistant_fraction": (
            phenotype_observations
        ),
    }


print(
    "Early-treatment total-load observations:"
)

display(
    SPARSE_DATASETS[
        "early_continuous"
    ]["total_load"].head()
)

print(
    "\nEarly-treatment resistant-fraction observations:"
)

display(
    SPARSE_DATASETS[
        "early_continuous"
    ]["resistant_fraction"]
)

In [ ]:
# =============================================================================
# CELL 10 — Synthetic noisy observation generator
# =============================================================================
# PURPOSE
# -------
# Create reproducible noisy datasets for later robustness experiments.
#
# NOISE MODELS
# ------------
# Total bacterial load:
#     Multiplicative log-normal noise is used because bacterial counts remain
#     positive and measurement variability often scales with magnitude.
#
# Resistant fraction:
#     Noise is added on the logit scale and transformed back to [0,1].
#
# The dense ground-truth trajectory is never altered.

def add_measurement_noise(
    total_observations: pd.DataFrame,
    fraction_observations: pd.DataFrame,
    coefficient_of_variation: float,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Add reproducible noise to sparse observations."""

    if coefficient_of_variation < 0:
        raise ValueError(
            "coefficient_of_variation "
            "must be non-negative."
        )

    rng = np.random.default_rng(seed)

    noisy_total = total_observations.copy()
    noisy_fraction = (
        fraction_observations.copy()
    )

    # No-noise reference case
    if coefficient_of_variation == 0:

        noisy_total[
            "total_observed"
        ] = noisy_total["total"]

        noisy_fraction[
            "resistant_fraction_observed"
        ] = noisy_fraction[
            "resistant_fraction"
        ]

        return noisy_total, noisy_fraction

    # ---------------------------------------------------------------------
    # Convert coefficient of variation to log-normal parameters.
    # ---------------------------------------------------------------------
    sigma_log = np.sqrt(
        np.log(
            1.0
            + coefficient_of_variation**2
        )
    )

    mu_log = -0.5 * sigma_log**2

    multiplicative_noise = rng.lognormal(
        mean=mu_log,
        sigma=sigma_log,
        size=len(noisy_total),
    )

    noisy_total[
        "total_observed"
    ] = (
        noisy_total["total"]
        * multiplicative_noise
    )

    # ---------------------------------------------------------------------
    # Add noise to resistant fractions on the logit scale.
    # ---------------------------------------------------------------------
    epsilon = 1.0e-8

    fraction = np.clip(
        noisy_fraction[
            "resistant_fraction"
        ].to_numpy(dtype=float),
        epsilon,
        1.0 - epsilon,
    )

    logits = np.log(
        fraction / (1.0 - fraction)
    )

    noisy_logits = logits + rng.normal(
        loc=0.0,
        scale=coefficient_of_variation,
        size=len(fraction),
    )

    noisy_fraction[
        "resistant_fraction_observed"
    ] = 1.0 / (
        1.0 + np.exp(-noisy_logits)
    )

    return noisy_total, noisy_fraction


NOISE_LEVELS = [
    0.00,
    0.02,
    0.05,
    0.10,
]


for noise_level in NOISE_LEVELS:

    noisy_total, noisy_fraction = (
        add_measurement_noise(
            SPARSE_DATASETS[
                "early_continuous"
            ]["total_load"],
            SPARSE_DATASETS[
                "early_continuous"
            ]["resistant_fraction"],
            coefficient_of_variation=(
                noise_level
            ),
            seed=(
                SEED
                + int(noise_level * 1000)
            ),
        )
    )

    noise_label = (
        f"{int(noise_level * 100):02d}pct"
    )

    noisy_total.to_csv(
        DATA_NOISY
        / (
            "early_continuous_"
            f"total_{noise_label}.csv"
        ),
        index=False,
    )

    noisy_fraction.to_csv(
        DATA_NOISY
        / (
            "early_continuous_"
            f"fraction_{noise_label}.csv"
        ),
        index=False,
    )


print(
    "Noisy observation files created "
    "for noise levels:",
    NOISE_LEVELS,
)

In [ ]:
# =============================================================================
# CELL 11 — Publication figures: state trajectories
# =============================================================================
# PURPOSE
# -------
# Generate one high-resolution independent figure for each treatment schedule.
#
# Each figure displays:
# - sensitive bacteria S(t);
# - resistant bacteria R(t);
# - total bacterial population S(t)+R(t).
#
# Every figure is saved as PNG, PDF, and SVG.

for schedule_name, data in TRAJECTORIES.items():

    fig, ax = plt.subplots(
        figsize=(8.6, 5.4)
    )

    ax.plot(
        data["time"],
        data["sensitive"],
        label=r"Sensitive, $S(t)$",
    )

    ax.plot(
        data["time"],
        data["resistant"],
        label=r"Resistant, $R(t)$",
    )

    ax.plot(
        data["time"],
        data["total"],
        label=r"Total, $S(t)+R(t)$",
    )

    ax.set_xlabel("Time")

    ax.set_ylabel(
        "Bacterial population"
    )

    ax.set_title(
        "Deterministic AMR dynamics: "
        + schedule_name.replace("_", " ")
    )

    ax.set_xlim(
        CONFIG.start_time,
        CONFIG.final_time,
    )

    ax.set_ylim(bottom=0)

    ax.grid(alpha=0.25)

    ax.legend(loc="best")

    fig.tight_layout()

    save_figure(
        fig,
        f"figure_state_{schedule_name}",
    )

    plt.show()

In [ ]:
# =============================================================================
# CELL 12 — Publication figure: resistant fraction across treatments
# =============================================================================
# PURPOSE
# -------
# Compare the latent resistant composition:
#
#               q(t) = R(t) / [S(t)+R(t)]
#
# across all treatment schedules.
#
# This figure motivates the hidden-state inference problem because cultures
# with similar total load can have very different resistance compositions.

fig, ax = plt.subplots(
    figsize=(8.6, 5.4)
)

for schedule_name, data in TRAJECTORIES.items():

    ax.plot(
        data["time"],
        data["resistant_fraction"],
        label=schedule_name.replace(
            "_",
            " ",
        ),
    )

ax.set_xlabel("Time")

ax.set_ylabel(
    r"Resistant fraction, $R/(S+R)$"
)

ax.set_title(
    "Evolution of resistant composition "
    "under prescribed treatments"
)

ax.set_xlim(
    CONFIG.start_time,
    CONFIG.final_time,
)

ax.set_ylim(0, 1)

ax.grid(alpha=0.25)

ax.legend(loc="best")

fig.tight_layout()

save_figure(
    fig,
    "figure_resistant_fraction_comparison",
)

plt.show()

In [ ]:
# =============================================================================
# CELL 13 — Publication figure: total bacterial load across treatments
# =============================================================================
# PURPOSE
# -------
# Compare the main measurable quantity:
#
#               B(t) = S(t) + R(t)
#
# under all treatment schedules.
#
# Total load is the principal observation supplied to the future inverse PINN.

fig, ax = plt.subplots(
    figsize=(8.6, 5.4)
)

for schedule_name, data in TRAJECTORIES.items():

    ax.plot(
        data["time"],
        data["total"],
        label=schedule_name.replace(
            "_",
            " ",
        ),
    )

ax.set_xlabel("Time")

ax.set_ylabel(
    r"Total bacterial load, $B(t)$"
)

ax.set_title(
    "Total bacterial load under "
    "alternative antibiotic schedules"
)

ax.set_xlim(
    CONFIG.start_time,
    CONFIG.final_time,
)

ax.set_ylim(bottom=0)

ax.grid(alpha=0.25)

ax.legend(loc="best")

fig.tight_layout()

save_figure(
    fig,
    "figure_total_load_comparison",
)

plt.show()

In [ ]:
# =============================================================================
# CELL 14 — Publication figure: eco-evolutionary phase plane
# =============================================================================
# PURPOSE
# -------
# Plot resistant population R against sensitive population S.
#
# The phase plane shows how each treatment redirects the population through
# sensitive-resistant state space.
#
# Circular markers indicate initial states.
# Cross markers indicate final states.

fig, ax = plt.subplots(
    figsize=(7.2, 6.2)
)

for schedule_name, data in TRAJECTORIES.items():

    ax.plot(
        data["sensitive"],
        data["resistant"],
        label=schedule_name.replace(
            "_",
            " ",
        ),
    )

    # Initial state
    ax.scatter(
        data.iloc[0]["sensitive"],
        data.iloc[0]["resistant"],
        s=35,
        marker="o",
        zorder=3,
    )

    # Final state
    ax.scatter(
        data.iloc[-1]["sensitive"],
        data.iloc[-1]["resistant"],
        s=45,
        marker="x",
        zorder=3,
    )

ax.set_xlabel(
    r"Sensitive population, $S$"
)

ax.set_ylabel(
    r"Resistant population, $R$"
)

ax.set_title(
    "Eco-evolutionary phase trajectories"
)

ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

ax.grid(alpha=0.25)

ax.legend(loc="best")

fig.tight_layout()

save_figure(
    fig,
    "figure_phase_plane",
)

plt.show()

In [ ]:
# =============================================================================
# CELL 15 — Publication figures: observed load and hidden composition
# =============================================================================
# PURPOSE
# -------
# Show the partially observed inverse problem for early treatment.
#
# Figure 1:
# Dense total-load ground truth and 21 available total-load observations.
#
# Figure 2:
# Dense hidden resistant fraction and only three available phenotype
# observations.
#
# These figures directly illustrate the paper's central inference challenge.

early_data = TRAJECTORIES[
    "early_continuous"
]

early_total_observations = (
    SPARSE_DATASETS[
        "early_continuous"
    ]["total_load"]
)

early_fraction_observations = (
    SPARSE_DATASETS[
        "early_continuous"
    ]["resistant_fraction"]
)

# -------------------------------------------------------------------------
# Figure 1: total bacterial load
# -------------------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8.6, 5.4)
)

ax.plot(
    early_data["time"],
    early_data["total"],
    label="Dense ground truth",
)

ax.scatter(
    early_total_observations["time"],
    early_total_observations["total"],
    s=38,
    label="Sparse total-load observations",
    zorder=3,
)

ax.set_xlabel("Time")

ax.set_ylabel(
    r"Total bacterial load, $B(t)$"
)

ax.set_title(
    "Sparse observation of total bacterial load"
)

ax.set_xlim(
    CONFIG.start_time,
    CONFIG.final_time,
)

ax.set_ylim(bottom=0)

ax.grid(alpha=0.25)

ax.legend(loc="best")

fig.tight_layout()

save_figure(
    fig,
    "figure_sparse_total_load",
)

plt.show()


# -------------------------------------------------------------------------
# Figure 2: resistant fraction
# -------------------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8.6, 5.4)
)

ax.plot(
    early_data["time"],
    early_data["resistant_fraction"],
    label="Hidden resistant fraction",
)

ax.scatter(
    early_fraction_observations["time"],
    early_fraction_observations[
        "resistant_fraction"
    ],
    s=45,
    label="Available phenotype observations",
    zorder=3,
)

ax.set_xlabel("Time")

ax.set_ylabel(
    r"Resistant fraction, $R/(S+R)$"
)

ax.set_title(
    "Sparse phenotype information "
    "in the inverse problem"
)

ax.set_xlim(
    CONFIG.start_time,
    CONFIG.final_time,
)

ax.set_ylim(0, 1)

ax.grid(alpha=0.25)

ax.legend(loc="best")

fig.tight_layout()

save_figure(
    fig,
    "figure_sparse_resistant_fraction",
)

plt.show()

In [ ]:
# =============================================================================
# CELL 16 — Parameter sweep over mutation probability and fitness cost
# =============================================================================
# PURPOSE
# -------
# Examine how the planned inverse parameters influence resistant dynamics:
#
#     mutation probability, mu
#     resistance fitness cost, c_R
#
# This must be checked before PINN parameter inference.
# A parameter that produces almost no change in the observables will be
# difficult or impossible to estimate from sparse data.

mutation_grid = np.logspace(
    -5,
    -3,
    9,
)

fitness_cost_grid = np.linspace(
    0.0,
    0.20,
    9,
)

sweep_rows = []

for mutation_probability in mutation_grid:

    for fitness_cost in fitness_cost_grid:

        sweep_parameters = replace(
            PARAMETERS,
            mutation_probability=float(
                mutation_probability
            ),
            resistance_fitness_cost=float(
                fitness_cost
            ),
        )

        trajectory = solve_deterministic(
            sweep_parameters,
            CONFIG,
            SCHEDULES[
                "early_continuous"
            ],
        )

        validation = validate_trajectory(
            trajectory,
            sweep_parameters,
            CONFIG,
        )

        sweep_rows.append(
            {
                "mutation_probability": (
                    mutation_probability
                ),
                "fitness_cost": fitness_cost,
                "final_sensitive": (
                    trajectory.iloc[-1][
                        "sensitive"
                    ]
                ),
                "final_resistant": (
                    trajectory.iloc[-1][
                        "resistant"
                    ]
                ),
                "final_total": (
                    trajectory.iloc[-1][
                        "total"
                    ]
                ),
                "final_resistant_fraction": (
                    trajectory.iloc[-1][
                        "resistant_fraction"
                    ]
                ),
                "peak_resistant": (
                    trajectory[
                        "resistant"
                    ].max()
                ),
                "all_checks_passed": (
                    validation[
                        "all_checks_passed"
                    ]
                ),
            }
        )


SWEEP = pd.DataFrame(sweep_rows)

if not SWEEP[
    "all_checks_passed"
].all():

    raise RuntimeError(
        "At least one parameter-sweep "
        "simulation failed validation."
    )


SWEEP.to_csv(
    TABLE_DIR
    / "round1_parameter_sweep.csv",
    index=False,
)

SWEEP.head()

In [ ]:
# =============================================================================
# CELL 17 — Publication figure: parameter-response heat map
# =============================================================================
# PURPOSE
# -------
# Visualize how the final resistant fraction depends on:
#
# - mutation probability;
# - resistance fitness cost.
#
# The horizontal axis uses log10(mu) because mutation probabilities span
# several orders of magnitude.

heatmap_table = SWEEP.pivot(
    index="fitness_cost",
    columns="mutation_probability",
    values="final_resistant_fraction",
)

fig, ax = plt.subplots(
    figsize=(8.0, 6.2)
)

image = ax.imshow(
    heatmap_table.to_numpy(),
    origin="lower",
    aspect="auto",
    extent=[
        np.log10(
            heatmap_table.columns.min()
        ),
        np.log10(
            heatmap_table.columns.max()
        ),
        heatmap_table.index.min(),
        heatmap_table.index.max(),
    ],
)

ax.set_xlabel(
    r"$\log_{10}(\mu)$"
)

ax.set_ylabel(
    r"Resistance fitness cost, $c_R$"
)

ax.set_title(
    "Final resistant fraction under "
    "early continuous treatment"
)

colorbar = fig.colorbar(
    image,
    ax=ax,
)

colorbar.set_label(
    "Final resistant fraction"
)

fig.tight_layout()

save_figure(
    fig,
    "figure_parameter_heatmap",
)

plt.show()

In [ ]:
# =============================================================================
# CELL 18 — Local sensitivity analysis for inverse parameters
# =============================================================================
# PURPOSE
# -------
# Estimate finite-difference sensitivities of:
#
# - total bacterial load;
# - resistant fraction;
#
# with respect to:
#
# - mutation probability;
# - resistance fitness cost.
#
# These are preliminary identifiability diagnostics. Later rounds will include
# profile likelihood and bootstrap uncertainty analysis.

def observable_vector(
    parameters: AMRParameters,
    schedule: TreatmentSchedule,
    observation_times: np.ndarray,
) -> np.ndarray:
    """
    Generate one vector containing total-load and resistant-fraction values.
    """

    trajectory = solve_deterministic(
        parameters,
        CONFIG,
        schedule,
    )

    total_interpolator = interp1d(
        trajectory["time"],
        trajectory["total"],
        kind="cubic",
    )

    fraction_interpolator = interp1d(
        trajectory["time"],
        trajectory[
            "resistant_fraction"
        ],
        kind="cubic",
    )

    total_values = total_interpolator(
        observation_times
    )

    fraction_values = (
        fraction_interpolator(
            observation_times
        )
    )

    return np.concatenate(
        [
            total_values,
            fraction_values,
        ]
    )


def normalized_central_sensitivity(
    base_parameters: AMRParameters,
    parameter_name: str,
    relative_step: float = 0.01,
) -> np.ndarray:
    """
    Calculate a normalized central finite-difference sensitivity.

    S_ij = theta_j/y_i * dy_i/dtheta_j
    """

    base_value = getattr(
        base_parameters,
        parameter_name,
    )

    finite_difference_step = max(
        abs(base_value)
        * relative_step,
        1.0e-8,
    )

    lower_value = (
        base_value
        - finite_difference_step
    )

    upper_value = (
        base_value
        + finite_difference_step
    )

    bounded_names = {
        "mutation_probability",
        "resistance_fitness_cost",
        "sensitive_drug_effect",
        "resistant_drug_effect",
    }

    if parameter_name in bounded_names:

        lower_value = max(
            lower_value,
            1.0e-12,
        )

        upper_value = min(
            upper_value,
            1.0 - 1.0e-12,
        )

    lower_parameters = replace(
        base_parameters,
        **{
            parameter_name: lower_value
        },
    )

    upper_parameters = replace(
        base_parameters,
        **{
            parameter_name: upper_value
        },
    )

    observation_times = np.linspace(
        CONFIG.start_time,
        CONFIG.final_time,
        21,
    )

    lower_observables = observable_vector(
        lower_parameters,
        SCHEDULES[
            "early_continuous"
        ],
        observation_times,
    )

    upper_observables = observable_vector(
        upper_parameters,
        SCHEDULES[
            "early_continuous"
        ],
        observation_times,
    )

    base_observables = observable_vector(
        base_parameters,
        SCHEDULES[
            "early_continuous"
        ],
        observation_times,
    )

    derivative = (
        upper_observables
        - lower_observables
    ) / (
        upper_value
        - lower_value
    )

    normalized_sensitivity = (
        base_value
        * derivative
        / (
            np.abs(base_observables)
            + 1.0e-8
        )
    )

    return normalized_sensitivity


sensitivity_mutation = (
    normalized_central_sensitivity(
        PARAMETERS,
        "mutation_probability",
    )
)

sensitivity_fitness_cost = (
    normalized_central_sensitivity(
        PARAMETERS,
        "resistance_fitness_cost",
    )
)


sensitivity_summary = pd.DataFrame(
    {
        "parameter": [
            "mutation_probability",
            "resistance_fitness_cost",
        ],
        "mean_absolute_sensitivity": [
            np.mean(
                np.abs(
                    sensitivity_mutation
                )
            ),
            np.mean(
                np.abs(
                    sensitivity_fitness_cost
                )
            ),
        ],
        "maximum_absolute_sensitivity": [
            np.max(
                np.abs(
                    sensitivity_mutation
                )
            ),
            np.max(
                np.abs(
                    sensitivity_fitness_cost
                )
            ),
        ],
        "l2_sensitivity_norm": [
            np.linalg.norm(
                sensitivity_mutation
            ),
            np.linalg.norm(
                sensitivity_fitness_cost
            ),
        ],
    }
)

sensitivity_summary.to_csv(
    TABLE_DIR
    / "round1_local_sensitivity_summary.csv",
    index=False,
)

sensitivity_summary

In [ ]:
# =============================================================================
# CELL 19 — Publication figures: time-resolved parameter sensitivities
# =============================================================================
# PURPOSE
# -------
# Show when total load and resistant fraction are most informative about:
#
# - mutation probability;
# - resistance fitness cost.
#
# These figures can later guide the placement of phenotype measurements.

observation_times = np.linspace(
    CONFIG.start_time,
    CONFIG.final_time,
    21,
)

number_of_times = len(
    observation_times
)

sensitivity_figure_information = [
    (
        r"mutation probability, $\mu$",
        sensitivity_mutation,
        "figure_sensitivity_mutation",
    ),
    (
        r"resistance fitness cost, $c_R$",
        sensitivity_fitness_cost,
        "figure_sensitivity_fitness_cost",
    ),
]


for (
    parameter_label,
    sensitivity_values,
    output_stem,
) in sensitivity_figure_information:

    fig, ax = plt.subplots(
        figsize=(8.6, 5.4)
    )

    ax.plot(
        observation_times,
        sensitivity_values[
            :number_of_times
        ],
        label=(
            "Normalized sensitivity "
            "of total load"
        ),
    )

    ax.plot(
        observation_times,
        sensitivity_values[
            number_of_times:
        ],
        label=(
            "Normalized sensitivity "
            "of resistant fraction"
        ),
    )

    ax.axhline(
        0.0,
        linewidth=1.0,
    )

    ax.set_xlabel("Time")

    ax.set_ylabel(
        "Normalized sensitivity"
    )

    ax.set_title(
        "Observation sensitivity to "
        + parameter_label
    )

    ax.set_xlim(
        CONFIG.start_time,
        CONFIG.final_time,
    )

    ax.grid(alpha=0.25)

    ax.legend(loc="best")

    fig.tight_layout()

    save_figure(
        fig,
        output_stem,
    )

    plt.show()

In [ ]:
# =============================================================================
# CELL 20 — Numerical convergence study
# =============================================================================
# PURPOSE
# -------
# Verify that the selected ODE solver tolerances generate sufficiently accurate
# reference data.
#
# PROCEDURE
# ---------
# 1. Generate a highly accurate reference solution.
# 2. Solve the same model using several tolerance settings.
# 3. Interpolate each test solution onto the reference grid.
# 4. Calculate maximum absolute and relative L2 errors.
#
# This is necessary because PINN accuracy cannot be assessed against an
# unreliable numerical reference.

reference_config = replace(
    CONFIG,
    dense_points=2001,
    relative_tolerance=1.0e-12,
    absolute_tolerance=1.0e-14,
)

reference_solution = solve_deterministic(
    PARAMETERS,
    reference_config,
    SCHEDULES[
        "early_continuous"
    ],
)

convergence_settings = [
    (1.0e-6, 1.0e-8),
    (1.0e-8, 1.0e-10),
    (1.0e-9, 1.0e-11),
    (1.0e-10, 1.0e-12),
]

convergence_rows = []


for relative_tolerance, absolute_tolerance in convergence_settings:

    test_config = replace(
        CONFIG,
        dense_points=1001,
        relative_tolerance=(
            relative_tolerance
        ),
        absolute_tolerance=(
            absolute_tolerance
        ),
    )

    test_solution = solve_deterministic(
        PARAMETERS,
        test_config,
        SCHEDULES[
            "early_continuous"
        ],
    )

    sensitive_interpolated = interp1d(
        test_solution["time"],
        test_solution["sensitive"],
        kind="cubic",
    )(
        reference_solution["time"]
    )

    resistant_interpolated = interp1d(
        test_solution["time"],
        test_solution["resistant"],
        kind="cubic",
    )(
        reference_solution["time"]
    )

    sensitive_reference = (
        reference_solution[
            "sensitive"
        ].to_numpy()
    )

    resistant_reference = (
        reference_solution[
            "resistant"
        ].to_numpy()
    )

    sensitive_error = (
        sensitive_interpolated
        - sensitive_reference
    )

    resistant_error = (
        resistant_interpolated
        - resistant_reference
    )

    convergence_rows.append(
        {
            "rtol": relative_tolerance,
            "atol": absolute_tolerance,
            "max_abs_error_sensitive": (
                np.max(
                    np.abs(
                        sensitive_error
                    )
                )
            ),
            "max_abs_error_resistant": (
                np.max(
                    np.abs(
                        resistant_error
                    )
                )
            ),
            "relative_l2_sensitive": (
                np.linalg.norm(
                    sensitive_error
                )
                / np.linalg.norm(
                    sensitive_reference
                )
            ),
            "relative_l2_resistant": (
                np.linalg.norm(
                    resistant_error
                )
                / (
                    np.linalg.norm(
                        resistant_reference
                    )
                    + 1.0e-12
                )
            ),
        }
    )


CONVERGENCE = pd.DataFrame(
    convergence_rows
)

CONVERGENCE.to_csv(
    TABLE_DIR
    / "round1_solver_convergence.csv",
    index=False,
)

CONVERGENCE

In [ ]:
# =============================================================================
# CELL 21 — Publication figure: deterministic solver convergence
# =============================================================================
# PURPOSE
# -------
# Display relative L2 error as a function of solver tolerance.
#
# This figure documents that the selected reference trajectories are
# numerically stable and appropriate for later PINN comparisons.

fig, ax = plt.subplots(
    figsize=(8.0, 5.4)
)

ax.loglog(
    CONVERGENCE["rtol"],
    CONVERGENCE[
        "relative_l2_sensitive"
    ],
    marker="o",
    label="Sensitive population",
)

ax.loglog(
    CONVERGENCE["rtol"],
    CONVERGENCE[
        "relative_l2_resistant"
    ],
    marker="s",
    label="Resistant population",
)

ax.set_xlabel(
    "Relative solver tolerance"
)

ax.set_ylabel(
    r"Relative $L_2$ error"
)

ax.set_title(
    "Numerical convergence of "
    "the deterministic AMR solver"
)

ax.grid(alpha=0.25)

ax.legend(loc="best")

# Smaller tolerances appear toward the right after inversion.
ax.invert_xaxis()

fig.tight_layout()

save_figure(
    fig,
    "figure_solver_convergence",
)

plt.show()

In [ ]:
# =============================================================================
# CELL 22 — Analytical equilibrium consistency check
# =============================================================================
# PURPOSE
# -------
# Explain why the untreated population approaches approximately 900 rather
# than the carrying capacity K=1000.
#
# For a sensitive-dominated untreated population:
#
#     r_S(1-mu)(1-N*/K) - d_S = 0
#
# Therefore:
#
#     N* = K[1 - d_S/{r_S(1-mu)}]
#
# This check confirms that the numerical result has the expected biological
# equilibrium and is not a coding error.

analytical_equilibrium = (
    PARAMETERS.carrying_capacity
    * (
        1.0
        - PARAMETERS.sensitive_death_rate
        / (
            PARAMETERS.sensitive_birth_rate
            * (
                1.0
                - PARAMETERS.mutation_probability
            )
        )
    )
)

simulated_final_total = float(
    TRAJECTORIES[
        "untreated"
    ].iloc[-1]["total"]
)

relative_equilibrium_error = (
    abs(
        simulated_final_total
        - analytical_equilibrium
    )
    / analytical_equilibrium
)

equilibrium_report = pd.DataFrame(
    {
        "quantity": [
            (
                "analytical_sensitive_"
                "dominant_equilibrium"
            ),
            "simulated_final_total",
            "relative_error",
        ],
        "value": [
            analytical_equilibrium,
            simulated_final_total,
            relative_equilibrium_error,
        ],
    }
)

equilibrium_report.to_csv(
    TABLE_DIR
    / "round1_equilibrium_check.csv",
    index=False,
)

equilibrium_report

In [ ]:
# =============================================================================
# CELL 23 — Export a LaTeX results table
# =============================================================================
# PURPOSE
# -------
# Convert the deterministic outcome summary into LaTeX code that can be
# inserted directly into the paper.
#
# The generated file will be saved in:
#
#     outputs/tables/round1_deterministic_summary.tex

latex_summary = SUMMARY[
    [
        "schedule",
        "final_sensitive",
        "final_resistant",
        "final_total",
        "final_resistant_fraction",
    ]
].copy()

latex_summary[
    "schedule"
] = latex_summary[
    "schedule"
].str.replace(
    "_",
    " ",
)

latex_summary = latex_summary.rename(
    columns={
        "schedule": "Treatment",
        "final_sensitive": (
            r"Final $S$"
        ),
        "final_resistant": (
            r"Final $R$"
        ),
        "final_total": (
            r"Final $S+R$"
        ),
        "final_resistant_fraction": (
            r"Final $R/(S+R)$"
        ),
    }
)

latex_code = latex_summary.to_latex(
    index=False,
    float_format=lambda value: (
        f"{value:.4f}"
    ),
    escape=False,
    caption=(
        "Deterministic outcomes under "
        "the prescribed treatment schedules."
    ),
    label=(
        "tab:round1_deterministic_outcomes"
    ),
)

latex_path = (
    TABLE_DIR
    / "round1_deterministic_summary.tex"
)

latex_path.write_text(
    latex_code,
    encoding="utf-8",
)

print(latex_code)

In [ ]:
# =============================================================================
# CELL 24 — Final Round 1 reproducibility audit
# =============================================================================
# PURPOSE
# -------
# Confirm that:
#
# 1. every treatment trajectory passed all biological checks;
# 2. every parameter-sweep simulation passed;
# 3. all required tables and reports were generated;
# 4. a complete inventory of project files is available.
#
# Round 1 should be considered complete only if this cell executes without an
# exception.

assert SUMMARY[
    "all_checks_passed"
].all()

assert SWEEP[
    "all_checks_passed"
].all()


required_files = [
    TABLE_DIR
    / "round1_deterministic_summary.csv",

    TABLE_DIR
    / "round1_parameter_sweep.csv",

    TABLE_DIR
    / "round1_solver_convergence.csv",

    TABLE_DIR
    / "round1_local_sensitivity_summary.csv",

    TABLE_DIR
    / "round1_equilibrium_check.csv",

    TABLE_DIR
    / "round1_deterministic_summary.tex",

    REPORT_DIR
    / "round1_validation_reports.json",
]


missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required outputs "
        f"are missing: {missing_files}"
    )


# -------------------------------------------------------------------------
# Create a complete file inventory for reproducibility.
# -------------------------------------------------------------------------
file_inventory = sorted(
    str(
        path.relative_to(
            PROJECT_ROOT
        )
    )
    for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
)

inventory_path = (
    REPORT_DIR
    / "round1_file_inventory.txt"
)

inventory_path.write_text(
    "\n".join(file_inventory),
    encoding="utf-8",
)


print(
    "Round 1 completed successfully."
)

print(
    "Dense datasets:",
    len(
        list(
            DATA_DENSE.glob("*.csv")
        )
    ),
)

print(
    "Sparse datasets:",
    len(
        list(
            DATA_SPARSE.glob("*.csv")
        )
    ),
)

print(
    "Noisy datasets:",
    len(
        list(
            DATA_NOISY.glob("*.csv")
        )
    ),
)

print(
    "Figure files:",
    len(
        list(
            FIGURE_DIR.glob("*")
        )
    ),
)

print(
    "Table files:",
    len(
        list(
            TABLE_DIR.glob("*")
        )
    ),
)

print(
    "Report files:",
    len(
        list(
            REPORT_DIR.glob("*")
        )
    ),
)

print("\nNext computational round:")

print(
    "Implement the corrected Gillespie "
    "stochastic simulator."
)

print(
    "Compare stochastic ensemble means, "
    "variances, extinction, and resistant "
    "establishment with the deterministic "
    "reference trajectories."
)

Round 2

In [ ]:
# =============================================================================
# CELL 1 — Environment, folders, reproducibility, and simulation budgets
# =============================================================================
# PURPOSE
# -------
# This cell:
# 1. imports the packages needed for Round 2;
# 2. fixes all random seeds;
# 3. creates folders for stochastic data, figures, tables, and reports;
# 4. reports the available GPU;
# 5. selects development or manuscript-scale simulation budgets.
#
# IMPORTANT
# ---------
# The Gillespie direct algorithm is event-driven and sequential. It mainly
# uses the CPU. Your 8 GB GPU will become useful during PINN training.

from pathlib import Path
from dataclasses import dataclass, asdict, replace
from typing import Dict, Tuple, Optional

import math
import json
import random
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

# Reproducibility seed
SEED = 20260804

random.seed(SEED)
np.random.seed(SEED)

# -------------------------------------------------------------------------
# Simulation mode
# -------------------------------------------------------------------------
# First run the complete notebook using QUICK_MODE=True.
#
# After every cell runs successfully:
# 1. change QUICK_MODE to False;
# 2. restart the Jupyter kernel;
# 3. run all cells again;
# 4. use only the final-mode results in the manuscript.
# -------------------------------------------------------------------------
QUICK_MODE = False

SHORT_REPLICATES = 30 if QUICK_MODE else 500
OUTCOME_REPLICATES = 60 if QUICK_MODE else 1000
RESCUE_GRID_REPLICATES = 10 if QUICK_MODE else 300

# -------------------------------------------------------------------------
# Project directory
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ENSEMBLE_DIR = (
    PROJECT_ROOT
    / "data"
    / "stochastic"
    / "ensembles"
)

SUMMARY_DIR = (
    PROJECT_ROOT
    / "data"
    / "stochastic"
    / "summaries"
)

PINN_EXPORT_DIR = (
    PROJECT_ROOT
    / "data"
    / "stochastic"
    / "pinn_exports"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "reports"
)

for directory in [
    ENSEMBLE_DIR,
    SUMMARY_DIR,
    PINN_EXPORT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    REPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# -------------------------------------------------------------------------
# Environment report
# -------------------------------------------------------------------------
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Operating system:", platform.platform())
print("QUICK_MODE:", QUICK_MODE)
print("Short replicates per schedule:", SHORT_REPLICATES)
print("Outcome replicates per schedule:", OUTCOME_REPLICATES)
print("Rescue-grid replicates per case:", RESCUE_GRID_REPLICATES)

# GPU reporting
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

        memory_gb = (
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3
        )

        print(f"GPU memory: {memory_gb:.2f} GB")

except Exception as exc:
    print("GPU information unavailable:", exc)

In [ ]:
# =============================================================================
# CELL 2 — Publication figure settings and reusable helpers
# =============================================================================
# PURPOSE
# -------
# This cell:
# 1. applies consistent publication-oriented figure settings;
# 2. defines a function that saves every figure as PNG, PDF, and SVG.
#
# No scientific result is produced here.

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "figure.figsize": (8.6, 5.4),
        "font.family": "sans-serif",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 1.0,
        "lines.linewidth": 2.0,
        "legend.fontsize": 9.5,
        "legend.frameon": True,
        "legend.framealpha": 1.0,
        "savefig.bbox": "tight",
        "axes.unicode_minus": False,
    }
)


def save_figure(
    fig: plt.Figure,
    stem: str,
    close_after_saving: bool = False,
) -> None:
    """
    Save one figure in publication and editable formats.

    Parameters
    ----------
    fig:
        Matplotlib figure object.

    stem:
        Output filename without extension.

    close_after_saving:
        Close the figure after saving when True.

    Files produced
    --------------
    PNG:
        300-dpi raster image.

    PDF:
        Vector figure suitable for LaTeX.

    SVG:
        Editable vector figure.
    """

    safe_stem = stem.strip().replace(" ", "_")

    fig.savefig(
        FIGURE_DIR / f"{safe_stem}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        FIGURE_DIR / f"{safe_stem}.pdf",
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        FIGURE_DIR / f"{safe_stem}.svg",
        bbox_inches="tight",
        facecolor="white",
    )

    if close_after_saving:
        plt.close(fig)


print("Cell 2 completed successfully.")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Figure directory:", FIGURE_DIR.resolve())

In [ ]:
# =============================================================================
# CELL 3 — Biological parameters, stochastic configuration, and treatments
# =============================================================================
# PURPOSE
# -------
# This cell defines:
# 1. biological parameters;
# 2. stochastic simulation settings;
# 3. untreated, early, late, and pulsed antibiotic schedules;
# 4. treatment-switch and output-grid utilities.
#
# A piecewise treatment schedule has one more exposure level than switch times.
#
# Example
# -------
# switch_times = (5, 8)
# levels       = (0, 1, 0)
#
# This means:
# A(t)=0 before t=5;
# A(t)=1 for 5<=t<8;
# A(t)=0 after t=8.

@dataclass(frozen=True)
class AMRParameters:
    """Biological parameters of the sensitive-resistant model."""

    carrying_capacity: int = 1000

    sensitive_birth_rate: float = 1.0
    resistance_fitness_cost: float = 0.10

    sensitive_death_rate: float = 0.10
    resistant_death_rate: float = 0.10

    mutation_probability: float = 1.0e-4

    sensitive_drug_effect: float = 1.0
    resistant_drug_effect: float = 0.10

    @property
    def resistant_birth_rate(self) -> float:
        """
        Resistant birth rate after accounting for fitness cost.

        r_R = r_S(1-c_R)
        """

        return (
            self.sensitive_birth_rate
            * (
                1.0
                - self.resistance_fitness_cost
            )
        )

    def validate(self) -> None:
        """Check whether all biological parameters are admissible."""

        if self.carrying_capacity <= 0:
            raise ValueError(
                "carrying_capacity must be positive."
            )

        if self.sensitive_birth_rate <= 0:
            raise ValueError(
                "sensitive_birth_rate must be positive."
            )

        if self.sensitive_death_rate < 0:
            raise ValueError(
                "sensitive_death_rate cannot be negative."
            )

        if self.resistant_death_rate < 0:
            raise ValueError(
                "resistant_death_rate cannot be negative."
            )

        bounded_parameters = {
            "resistance_fitness_cost":
                self.resistance_fitness_cost,

            "mutation_probability":
                self.mutation_probability,

            "sensitive_drug_effect":
                self.sensitive_drug_effect,

            "resistant_drug_effect":
                self.resistant_drug_effect,
        }

        for name, value in bounded_parameters.items():

            if not 0.0 <= value <= 1.0:
                raise ValueError(
                    f"{name} must lie in [0,1]."
                )


@dataclass(frozen=True)
class StochasticConfig:
    """Simulation and outcome settings."""

    initial_sensitive: int = 100
    initial_resistant: int = 0

    # Short trajectory comparison with Round 1
    short_final_time: float = 20.0
    short_output_points: int = 401

    # Longer horizon for extinction and rescue
    outcome_final_time: float = 120.0
    outcome_output_points: int = 601

    # Operational outcome definitions
    establishment_threshold: int = 10
    dominance_fraction: float = 0.50

    # Protection against accidental infinite runs
    maximum_events_per_replicate: int = 2_000_000


@dataclass(frozen=True)
class PiecewiseTreatment:
    """Piecewise-constant antibiotic-exposure schedule."""

    name: str
    switch_times: Tuple[float, ...]
    levels: Tuple[float, ...]
    description: str

    def __post_init__(self) -> None:

        if (
            len(self.levels)
            != len(self.switch_times) + 1
        ):
            raise ValueError(
                "levels must contain one more "
                "value than switch_times."
            )

        if (
            tuple(sorted(self.switch_times))
            != self.switch_times
        ):
            raise ValueError(
                "switch_times must be ordered."
            )

        if any(
            level < 0.0 or level > 1.0
            for level in self.levels
        ):
            raise ValueError(
                "Treatment levels must lie in [0,1]."
            )

    def exposure(
        self,
        t: float | np.ndarray,
    ) -> float | np.ndarray:
        """Evaluate A(t)."""

        values = np.asarray(
            t,
            dtype=float,
        )

        indexes = np.searchsorted(
            np.asarray(self.switch_times),
            values,
            side="right",
        )

        output = np.asarray(
            self.levels,
            dtype=float,
        )[indexes]

        if np.isscalar(t):
            return float(output)

        return output

    def next_switch_after(
        self,
        t: float,
        tolerance: float = 1.0e-12,
    ) -> float:
        """Return the next deterministic switch after time t."""

        for switch_time in self.switch_times:

            if switch_time > t + tolerance:
                return float(switch_time)

        return math.inf

    @property
    def first_on_time(self) -> float:
        """Return the first time at which treatment becomes active."""

        if self.levels[0] > 0:
            return 0.0

        for switch_time, level in zip(
            self.switch_times,
            self.levels[1:],
        ):

            if level > 0:
                return float(switch_time)

        return math.inf

    @property
    def permanent_at_end(self) -> bool:
        """Return True when treatment remains active at the final horizon."""

        return bool(self.levels[-1] > 0)

    def active_intervals(
        self,
        final_time: float,
    ) -> list[tuple[float, float]]:
        """Return treatment-active intervals for plotting."""

        boundaries = [0.0]

        boundaries.extend(
            switch_time
            for switch_time in self.switch_times
            if switch_time < final_time
        )

        boundaries.append(final_time)

        intervals = []

        for left, right in zip(
            boundaries[:-1],
            boundaries[1:],
        ):

            midpoint = 0.5 * (left + right)

            if self.exposure(midpoint) > 0:
                intervals.append(
                    (left, right)
                )

        return intervals


PARAMETERS = AMRParameters()
PARAMETERS.validate()

CONFIG = StochasticConfig()

initial_total = (
    CONFIG.initial_sensitive
    + CONFIG.initial_resistant
)

if initial_total > PARAMETERS.carrying_capacity:
    raise ValueError(
        "Initial population exceeds carrying capacity."
    )


SCHEDULES = {
    "untreated":
        PiecewiseTreatment(
            name="untreated",
            switch_times=(),
            levels=(0.0,),
            description="No treatment.",
        ),

    "early_continuous":
        PiecewiseTreatment(
            name="early_continuous",
            switch_times=(5.0,),
            levels=(0.0, 1.0),
            description="Permanent treatment from t=5.",
        ),

    "late_continuous":
        PiecewiseTreatment(
            name="late_continuous",
            switch_times=(10.0,),
            levels=(0.0, 1.0),
            description="Permanent treatment from t=10.",
        ),

    "pulsed":
        PiecewiseTreatment(
            name="pulsed",
            switch_times=(
                5.0,
                8.0,
                12.0,
                15.0,
            ),
            levels=(
                0.0,
                1.0,
                0.0,
                1.0,
                0.0,
            ),
            description=(
                "Treatment on [5,8) and [12,15)."
            ),
        ),
}


def make_output_grid(
    final_time: float,
    points: int,
    schedule: PiecewiseTreatment,
) -> np.ndarray:
    """
    Create a regular time grid containing every treatment switch exactly.
    """

    regular_grid = np.linspace(
        0.0,
        final_time,
        points,
    )

    switch_grid = np.asarray(
        [
            switch_time
            for switch_time in schedule.switch_times
            if 0.0 <= switch_time <= final_time
        ],
        dtype=float,
    )

    return np.unique(
        np.concatenate(
            [
                regular_grid,
                switch_grid,
            ]
        )
    )


def shade_treatment(
    ax: plt.Axes,
    schedule: PiecewiseTreatment,
    final_time: float,
) -> None:
    """Shade all intervals during which treatment is active."""

    for left, right in schedule.active_intervals(
        final_time
    ):

        ax.axvspan(
            left,
            right,
            alpha=0.08,
        )


schedule_table = pd.DataFrame(
    {
        "schedule": list(SCHEDULES.keys()),

        "description": [
            schedule.description
            for schedule in SCHEDULES.values()
        ],

        "switch_times": [
            schedule.switch_times
            for schedule in SCHEDULES.values()
        ],

        "levels": [
            schedule.levels
            for schedule in SCHEDULES.values()
        ],
    }
)

display(schedule_table)

In [ ]:
# =============================================================================
# CELL 4 — Five stochastic events, propensities, mean drift, and verification
# =============================================================================
# PURPOSE
# -------
# Implement and verify the five stochastic events.
#
# EVENT ORDER
# -----------
# 0. Sensitive birth without mutation
# 1. Resistant mutant offspring from sensitive division
# 2. Sensitive death
# 3. Resistant birth
# 4. Resistant death
#
# The order matches the well-mixed DrugRes_WMP.py implementation.
#
# CORRECTION
# ----------
# All birth propensities use:
#
#     max(0, 1-(S+R)/K)
#
# This prevents invalid negative event rates.

STOICHIOMETRIC_MATRIX = np.array(
    [
        [1, 0, -1, 0, 0],
        [0, 1,  0, 1, -1],
    ],
    dtype=np.int64,
)

EVENT_NAMES = (
    "sensitive_birth",
    "mutant_resistant_birth",
    "sensitive_death",
    "resistant_birth",
    "resistant_death",
)


def calculate_propensities(
    state: np.ndarray,
    t: float,
    parameters: AMRParameters,
    schedule: PiecewiseTreatment,
) -> np.ndarray:
    """
    Calculate five non-negative Gillespie event propensities.
    """

    sensitive = float(state[0])
    resistant = float(state[1])

    if sensitive < 0 or resistant < 0:
        raise ValueError(
            "Stochastic population counts cannot be negative."
        )

    exposure = float(
        schedule.exposure(t)
    )

    total = sensitive + resistant

    crowding = max(
        0.0,
        1.0
        - total
        / parameters.carrying_capacity,
    )

    sensitive_birth_rate = max(
        0.0,
        parameters.sensitive_birth_rate
        * (
            1.0
            - parameters.sensitive_drug_effect
            * exposure
        ),
    )

    resistant_birth_rate = max(
        0.0,
        parameters.resistant_birth_rate
        * (
            1.0
            - parameters.resistant_drug_effect
            * exposure
        ),
    )

    propensities = np.array(
        [
            # Sensitive birth without mutation
            sensitive_birth_rate
            * (
                1.0
                - parameters.mutation_probability
            )
            * crowding
            * sensitive,

            # Resistant mutant offspring
            sensitive_birth_rate
            * parameters.mutation_probability
            * crowding
            * sensitive,

            # Sensitive death
            parameters.sensitive_death_rate
            * sensitive,

            # Resistant birth
            resistant_birth_rate
            * crowding
            * resistant,

            # Resistant death
            parameters.resistant_death_rate
            * resistant,
        ],
        dtype=float,
    )

    if np.any(
        ~np.isfinite(propensities)
    ):
        raise FloatingPointError(
            "Non-finite propensity detected."
        )

    if np.any(
        propensities < -1.0e-14
    ):
        raise FloatingPointError(
            "Negative propensity detected."
        )

    return np.maximum(
        propensities,
        0.0,
    )


def deterministic_rhs(
    t: float,
    state: np.ndarray,
    parameters: AMRParameters,
    schedule: PiecewiseTreatment,
) -> np.ndarray:
    """
    Calculate the deterministic mean-field equations independently.

    The adaptive ODE solver uses floating-point intermediate states.
    Consequently, this function does not call the integer-state stochastic
    validator.
    """

    sensitive, resistant = np.asarray(
        state,
        dtype=float,
    )

    exposure = float(
        schedule.exposure(t)
    )

    total = sensitive + resistant

    crowding = (
        1.0
        - total
        / parameters.carrying_capacity
    )

    sensitive_birth_rate = (
        parameters.sensitive_birth_rate
        * (
            1.0
            - parameters.sensitive_drug_effect
            * exposure
        )
    )

    resistant_birth_rate = (
        parameters.resistant_birth_rate
        * (
            1.0
            - parameters.resistant_drug_effect
            * exposure
        )
    )

    d_sensitive = (
        sensitive_birth_rate
        * (
            1.0
            - parameters.mutation_probability
        )
        * crowding
        * sensitive
        - parameters.sensitive_death_rate
        * sensitive
    )

    d_resistant = (
        sensitive_birth_rate
        * parameters.mutation_probability
        * crowding
        * sensitive
        + resistant_birth_rate
        * crowding
        * resistant
        - parameters.resistant_death_rate
        * resistant
    )

    return np.array(
        [
            d_sensitive,
            d_resistant,
        ],
        dtype=float,
    )


def solve_deterministic_reference(
    parameters: AMRParameters,
    schedule: PiecewiseTreatment,
    output_grid: np.ndarray,
) -> pd.DataFrame:
    """Solve the deterministic mean-field equations."""

    solution = solve_ivp(
        fun=lambda t, state: deterministic_rhs(
            t,
            state,
            parameters,
            schedule,
        ),

        t_span=(
            float(output_grid[0]),
            float(output_grid[-1]),
        ),

        y0=np.array(
            [
                CONFIG.initial_sensitive,
                CONFIG.initial_resistant,
            ],
            dtype=float,
        ),

        t_eval=output_grid,

        method="DOP853",

        rtol=1.0e-10,
        atol=1.0e-12,
    )

    if not solution.success:
        raise RuntimeError(
            solution.message
        )

    sensitive = solution.y[0]
    resistant = solution.y[1]

    total = sensitive + resistant

    resistant_fraction = np.divide(
        resistant,
        total,
        out=np.zeros_like(resistant),
        where=total > 0,
    )

    return pd.DataFrame(
        {
            "time": solution.t,
            "sensitive": sensitive,
            "resistant": resistant,
            "total": total,
            "resistant_fraction":
                resistant_fraction,
            "antibiotic":
                schedule.exposure(solution.t),
        }
    )


# -------------------------------------------------------------------------
# Mathematical consistency test
# -------------------------------------------------------------------------
# Compare:
#
#     V @ a(X,t)
#
# with independently written mean-field equations at random feasible states.
# -------------------------------------------------------------------------
test_rng = np.random.default_rng(
    SEED
)

maximum_drift_difference = 0.0
minimum_propensity = math.inf

for _ in range(500):

    sensitive = int(
        test_rng.integers(
            0,
            PARAMETERS.carrying_capacity + 1,
        )
    )

    resistant = int(
        test_rng.integers(
            0,
            PARAMETERS.carrying_capacity
            - sensitive
            + 1,
        )
    )

    state = np.array(
        [
            sensitive,
            resistant,
        ],
        dtype=np.int64,
    )

    schedule = SCHEDULES[
        test_rng.choice(
            list(SCHEDULES.keys())
        )
    ]

    test_time = float(
        test_rng.uniform(
            0.0,
            CONFIG.short_final_time,
        )
    )

    propensities = calculate_propensities(
        state,
        test_time,
        PARAMETERS,
        schedule,
    )

    matrix_drift = (
        STOICHIOMETRIC_MATRIX
        @ propensities
    )

    direct_drift = deterministic_rhs(
        test_time,
        state,
        PARAMETERS,
        schedule,
    )

    maximum_drift_difference = max(
        maximum_drift_difference,
        float(
            np.max(
                np.abs(
                    matrix_drift
                    - direct_drift
                )
            )
        ),
    )

    minimum_propensity = min(
        minimum_propensity,
        float(
            propensities.min()
        ),
    )


# Births must stop at carrying capacity.
capacity_state = np.array(
    [
        900,
        100,
    ],
    dtype=np.int64,
)

capacity_propensities = (
    calculate_propensities(
        capacity_state,
        0.0,
        PARAMETERS,
        SCHEDULES["untreated"],
    )
)

assert maximum_drift_difference < 1.0e-12
assert minimum_propensity >= 0.0

assert np.all(
    capacity_propensities[
        [0, 1, 3]
    ] == 0.0
)

print("All propensity and drift tests passed.")
print(
    "Maximum drift difference:",
    maximum_drift_difference,
)
print(
    "Birth propensities at capacity:",
    capacity_propensities[
        [0, 1, 3]
    ],
)

In [ ]:
# =============================================================================
# CELL 5 — Corrected single-replicate Gillespie direct simulator
# =============================================================================
# Corrections relative to the audited repository:
# - accepts a seed;
# - records complete regularly sampled trajectories;
# - interrupts waiting times at every treatment switch and final horizon;
# - handles zero total propensity;
# - never loses a final-time replicate;
# - separately records emergence, establishment, dominance, and extinction.
# Establishment is operationally defined as first R>=10, rather than requiring
# R to equal an exact equilibrium value.

def simulate_gillespie_replicate(parameters,schedule,output_grid,seed,
                                  establishment_threshold=CONFIG.establishment_threshold,
                                  dominance_fraction=CONFIG.dominance_fraction,
                                  maximum_events=CONFIG.maximum_events_per_replicate,
                                  keep_event_log=False):
    grid=np.asarray(output_grid,dtype=float)
    if grid.ndim!=1 or len(grid)<2 or abs(grid[0])>1e-14 or np.any(np.diff(grid)<=0):
        raise ValueError('output_grid must be strictly increasing and start at zero.')
    rng=np.random.default_rng(seed)
    state=np.array([CONFIG.initial_sensitive,CONFIG.initial_resistant],dtype=np.int64)
    t=float(grid[0]); final=float(grid[-1]); index=0
    sampled_S=np.empty(len(grid),dtype=np.int32); sampled_R=np.empty(len(grid),dtype=np.int32)
    while index<len(grid) and grid[index]<=t+1e-14:
        sampled_S[index],sampled_R[index]=state; index+=1

    emergence=0.0 if state[1]>0 else math.nan
    establishment=0.0 if state[1]>=establishment_threshold else math.nan
    dominance=0.0 if state.sum()>0 and state[1]/state.sum()>=dominance_fraction else math.nan
    S_extinction=0.0 if state[0]==0 else math.nan
    total_extinction=0.0 if state.sum()==0 else math.nan
    first_on=schedule.first_on_time
    state_at_treatment=(int(state[0]),int(state[1])) if first_on==0 else None
    event_count=0

    if keep_event_log:
        log_t=[t]; log_S=[int(state[0])]; log_R=[int(state[1])]; log_event=[-1]

    while t<final-1e-14:
        if event_count>=maximum_events:
            raise RuntimeError('maximum_events reached.')
        boundary=min(schedule.next_switch_after(t),final)
        prop=calculate_propensities(state,t,parameters,schedule)
        a0=float(prop.sum())

        if a0<=0:
            while index<len(grid) and grid[index]<=boundary+1e-14:
                sampled_S[index],sampled_R[index]=state; index+=1
            t=boundary
            if state_at_treatment is None and math.isfinite(first_on) and t>=first_on-1e-12:
                state_at_treatment=(int(state[0]),int(state[1]))
            if keep_event_log:
                log_t.append(t); log_S.append(int(state[0])); log_R.append(int(state[1])); log_event.append(-2)
            if t>=final-1e-14: break
            continue

        proposed=t+float(rng.exponential(1.0/a0))
        if proposed>=boundary-1e-14:
            while index<len(grid) and grid[index]<=boundary+1e-14:
                sampled_S[index],sampled_R[index]=state; index+=1
            t=boundary
            if state_at_treatment is None and math.isfinite(first_on) and t>=first_on-1e-12:
                state_at_treatment=(int(state[0]),int(state[1]))
            if keep_event_log:
                log_t.append(t); log_S.append(int(state[0])); log_R.append(int(state[1])); log_event.append(-2)
            if t>=final-1e-14: break
            continue

        while index<len(grid) and grid[index]<proposed-1e-14:
            sampled_S[index],sampled_R[index]=state; index+=1

        draw=float(rng.random()*a0)
        event_index=min(int(np.searchsorted(np.cumsum(prop),draw,side='right')),4)
        state=state+STOICHIOMETRIC_MATRIX[:,event_index]
        t=proposed; event_count+=1
        if np.any(state<0) or state.sum()>parameters.carrying_capacity:
            raise RuntimeError('Invalid state generated.')

        if math.isnan(emergence) and state[1]>0: emergence=t
        if math.isnan(establishment) and state[1]>=establishment_threshold: establishment=t
        if math.isnan(dominance) and state.sum()>0 and state[1]/state.sum()>=dominance_fraction: dominance=t
        if math.isnan(S_extinction) and state[0]==0: S_extinction=t
        if math.isnan(total_extinction) and state.sum()==0: total_extinction=t
        if keep_event_log:
            log_t.append(t); log_S.append(int(state[0])); log_R.append(int(state[1])); log_event.append(event_index)
        if state.sum()==0:
            while index<len(grid):
                sampled_S[index]=0; sampled_R[index]=0; index+=1
            break

    while index<len(grid):
        sampled_S[index],sampled_R[index]=state; index+=1
    if state_at_treatment is None:
        state_at_treatment=(-1,-1)

    final_S=int(sampled_S[-1]); final_R=int(sampled_R[-1])
    established=bool(np.isfinite(establishment)); survived=bool(final_S+final_R>0)
    rescue=bool(schedule.permanent_at_end and final_S==0 and final_R>0 and established)
    output={
        'time':grid.copy(),'sensitive':sampled_S,'resistant':sampled_R,'event_count':event_count,
        'emergence_time':emergence,'establishment_time':establishment,'dominance_time':dominance,
        'sensitive_extinction_time':S_extinction,'total_extinction_time':total_extinction,
        'sensitive_at_treatment':state_at_treatment[0],'resistant_at_treatment':state_at_treatment[1],
        'final_sensitive':final_S,'final_resistant':final_R,
        'mutant_present_at_treatment':bool(state_at_treatment[1]>0),'established':established,
        'resistant_dominance_reached':bool(np.isfinite(dominance)),'survived':survived,
        'extinct':not survived,'evolutionary_rescue':rescue,
    }
    if keep_event_log:
        output['event_log']=pd.DataFrame({'time':log_t,'sensitive':log_S,'resistant':log_R,'event_id':log_event})
    return output

In [ ]:
# =============================================================================
# OPTIONAL TEST — Run one Gillespie replicate
# =============================================================================
# This cell checks that the function defined in Cell 5 can generate one complete
# stochastic AMR trajectory.

test_schedule = SCHEDULES["early_continuous"]

test_output_grid = make_output_grid(
    final_time=CONFIG.short_final_time,
    points=CONFIG.short_output_points,
    schedule=test_schedule,
)

test_result = simulate_gillespie_replicate(
    parameters=PARAMETERS,
    schedule=test_schedule,
    output_grid=test_output_grid,
    seed=SEED + 1,
    keep_event_log=True,
)

print("Single Gillespie replicate completed successfully.")
print("-" * 60)
print("Number of recorded output times:", len(test_result["time"]))
print("Number of stochastic events:", test_result["event_count"])
print("Final sensitive population:", test_result["final_sensitive"])
print("Final resistant population:", test_result["final_resistant"])
print("Resistance emergence time:", test_result["emergence_time"])
print("Resistance establishment time:", test_result["establishment_time"])
print("Sensitive extinction time:", test_result["sensitive_extinction_time"])
print("Total extinction time:", test_result["total_extinction_time"])
print("Resistant population established:", test_result["established"])
print("Population survived:", test_result["survived"])
print("Population became extinct:", test_result["extinct"])
print("Evolutionary rescue occurred:", test_result["evolutionary_rescue"])

display(test_result["event_log"].head(10))

In [ ]:
# =============================================================================
# CELL 6 — Unit tests and event-level publication figure
# =============================================================================
# The same seed must reproduce the same path. Treatment switches must be present
# exactly in the output grid, and every population must remain integer,
# non-negative, and below capacity.

test_schedule=SCHEDULES['pulsed']
test_grid=make_output_grid(CONFIG.short_final_time,CONFIG.short_output_points,test_schedule)
a=simulate_gillespie_replicate(PARAMETERS,test_schedule,test_grid,12345)
b=simulate_gillespie_replicate(PARAMETERS,test_schedule,test_grid,12345)
c=simulate_gillespie_replicate(PARAMETERS,test_schedule,test_grid,54321)
assert np.array_equal(a['sensitive'],b['sensitive']) and np.array_equal(a['resistant'],b['resistant'])
assert not (np.array_equal(a['sensitive'],c['sensitive']) and np.array_equal(a['resistant'],c['resistant']))
assert np.all(a['sensitive']>=0) and np.all(a['resistant']>=0)
assert np.all(a['sensitive']+a['resistant']<=PARAMETERS.carrying_capacity)
for switch in test_schedule.switch_times: assert np.any(np.isclose(test_grid,switch))
print('All single-replicate tests passed.')

example_schedule=SCHEDULES['early_continuous']
example_grid=make_output_grid(CONFIG.short_final_time,CONFIG.short_output_points,example_schedule)
example=simulate_gillespie_replicate(PARAMETERS,example_schedule,example_grid,SEED+777,keep_event_log=True)
log=example['event_log'].copy(); log['total']=log['sensitive']+log['resistant']
fig,ax=plt.subplots(figsize=(8.8,5.5))
ax.step(log['time'],log['sensitive'],where='post',label=r'Sensitive, $S(t)$')
ax.step(log['time'],log['resistant'],where='post',label=r'Resistant, $R(t)$')
ax.step(log['time'],log['total'],where='post',label=r'Total, $S(t)+R(t)$')
shade_treatment(ax,example_schedule,CONFIG.short_final_time)
ax.set(xlabel='Time',ylabel='Integer bacterial count',title='Exact event-level Gillespie trajectory',xlim=(0,CONFIG.short_final_time),ylim=(0,None))
ax.grid(alpha=0.25); ax.legend(loc='best'); fig.tight_layout(); save_figure(fig,'round2_event_level_example'); plt.show()
log.to_csv(SUMMARY_DIR/'round2_event_level_example.csv',index=False)
print('Example event count:',example['event_count'])
print('Emergence time:',example['emergence_time'])
print('Establishment time:',example['establishment_time'])

In [ ]:
# =============================================================================
# CELL 7 — Ensemble runner and ensemble-statistics functions
# =============================================================================
# Independent child seeds are generated using SeedSequence. Full event logs are
# not retained for all replicates; regularly sampled arrays and replicate-level
# outcome summaries are saved.

def run_gillespie_ensemble(parameters,schedule,output_grid,number_of_replicates,base_seed,label,show_progress=True):
    children=np.random.SeedSequence(base_seed).spawn(number_of_replicates)
    seeds=[int(child.generate_state(1,dtype=np.uint32)[0]) for child in children]
    S=np.empty((number_of_replicates,len(output_grid)),dtype=np.int32)
    R=np.empty_like(S); rows=[]; start=time.perf_counter(); step=max(1,number_of_replicates//10)
    for i,seed in enumerate(seeds):
        result=simulate_gillespie_replicate(parameters,schedule,output_grid,seed)
        S[i]=result['sensitive']; R[i]=result['resistant']
        rows.append({'replicate':i,'seed':seed,'schedule':schedule.name,
                     **{key:result[key] for key in [
                         'event_count','emergence_time','establishment_time','dominance_time',
                         'sensitive_extinction_time','total_extinction_time','sensitive_at_treatment',
                         'resistant_at_treatment','final_sensitive','final_resistant',
                         'mutant_present_at_treatment','established','resistant_dominance_reached',
                         'survived','extinct','evolutionary_rescue']}})
        if show_progress and ((i+1)%step==0 or i+1==number_of_replicates):
            print(f'{label}: {i+1}/{number_of_replicates}')
    summary=pd.DataFrame(rows); elapsed=time.perf_counter()-start
    np.savez_compressed(ENSEMBLE_DIR/f'{label}.npz',time=output_grid,sensitive=S,resistant=R,seeds=np.asarray(seeds,dtype=np.uint32))
    summary.to_csv(SUMMARY_DIR/f'{label}_replicate_summary.csv',index=False)
    return {'label':label,'schedule':schedule,'time':output_grid,'sensitive':S,'resistant':R,'summary':summary,'elapsed_seconds':elapsed,'parameters':parameters}

def calculate_ensemble_statistics(ensemble):
    S=ensemble['sensitive'].astype(float); R=ensemble['resistant'].astype(float); N=S+R
    output={'time':ensemble['time']}
    for prefix,values in [('sensitive',S),('resistant',R),('total',N)]:
        mean=values.mean(axis=0); variance=values.var(axis=0,ddof=1)
        output.update({
            f'{prefix}_mean':mean,f'{prefix}_median':np.median(values,axis=0),
            f'{prefix}_q05':np.quantile(values,0.05,axis=0),f'{prefix}_q25':np.quantile(values,0.25,axis=0),
            f'{prefix}_q75':np.quantile(values,0.75,axis=0),f'{prefix}_q95':np.quantile(values,0.95,axis=0),
            f'{prefix}_variance':variance,
            f'{prefix}_fano':np.divide(variance,mean,out=np.zeros_like(variance),where=mean>0),
        })
    return pd.DataFrame(output)

In [ ]:
# =============================================================================
# OPTIONAL TEST — Verify the Cell 7 ensemble functions
# =============================================================================
# PURPOSE
# -------
# Run a very small three-replicate ensemble to confirm that:
# - independent seeds are generated;
# - every trajectory is completed;
# - replicate summaries are created;
# - ensemble statistics are calculated;
# - NPZ and CSV files are saved correctly.
#
# This is only a function test, not a scientific experiment.

test_schedule = SCHEDULES["early_continuous"]

test_grid = make_output_grid(
    final_time=CONFIG.short_final_time,
    points=101,
    schedule=test_schedule,
)

test_ensemble = run_gillespie_ensemble(
    parameters=PARAMETERS,
    schedule=test_schedule,
    output_grid=test_grid,
    number_of_replicates=3,
    base_seed=SEED + 700,
    label="test_cell7_ensemble",
    show_progress=True,
)

test_statistics = calculate_ensemble_statistics(
    test_ensemble
)

print("\nCell 7 ensemble test completed successfully.")
print("-" * 65)

print(
    "Sensitive-array shape:",
    test_ensemble["sensitive"].shape,
)

print(
    "Resistant-array shape:",
    test_ensemble["resistant"].shape,
)

print(
    "Number of summary rows:",
    len(test_ensemble["summary"]),
)

print(
    "Elapsed simulation time:",
    f"{test_ensemble['elapsed_seconds']:.3f} seconds",
)

print("\nReplicate-level summary:")
display(
    test_ensemble["summary"][
        [
            "replicate",
            "seed",
            "event_count",
            "final_sensitive",
            "final_resistant",
            "established",
            "survived",
            "extinct",
            "evolutionary_rescue",
        ]
    ]
)

print("\nFirst five ensemble-statistics rows:")
display(
    test_statistics.head()
)

In [ ]:
# =============================================================================
# CELL 8 — Short-horizon ensembles for untreated, early, late, and pulsed cases
# =============================================================================
# These t<=20 ensembles match Round 1 and support deterministic-versus-stochastic
# comparisons and later stochastic-data robustness tests.

SHORT_ENSEMBLES={}; SHORT_STATISTICS={}
for schedule_index,(name,schedule) in enumerate(SCHEDULES.items()):
    grid=make_output_grid(CONFIG.short_final_time,CONFIG.short_output_points,schedule)
    ensemble=run_gillespie_ensemble(PARAMETERS,schedule,grid,SHORT_REPLICATES,SEED+10_000*schedule_index,f'short_{name}')
    statistics=calculate_ensemble_statistics(ensemble)
    statistics.to_csv(SUMMARY_DIR/f'short_{name}_time_statistics.csv',index=False)
    SHORT_ENSEMBLES[name]=ensemble; SHORT_STATISTICS[name]=statistics

display(pd.DataFrame({
    'schedule':list(SHORT_ENSEMBLES),
    'replicates':[x['sensitive'].shape[0] for x in SHORT_ENSEMBLES.values()],
    'time_points':[x['sensitive'].shape[1] for x in SHORT_ENSEMBLES.values()],
    'elapsed_seconds':[x['elapsed_seconds'] for x in SHORT_ENSEMBLES.values()],
}))

In [ ]:
# =============================================================================
# CELL 9 — Ensemble audit, statistics, and deterministic comparison figures
# =============================================================================
# All stochastic states are audited separately from prediction error. For each
# schedule, the ODE solution is compared with the stochastic mean and 5–95%
# interval. Nonlinear dynamics mean exact equality is not expected.

audit_rows=[]; metric_rows=[]
for name,ensemble in SHORT_ENSEMBLES.items():
    S,R=ensemble['sensitive'],ensemble['resistant']; summary=ensemble['summary']
    checks={
        'nonnegative':bool(np.all(S>=0) and np.all(R>=0)),
        'capacity_respected':bool(np.all(S+R<=PARAMETERS.carrying_capacity)),
        'complete_accounting':bool(np.all(summary['survived'].astype(int)+summary['extinct'].astype(int)==1)),
        'complete_rows':bool(len(summary)==SHORT_REPLICATES),
    }
    assert all(checks.values())
    audit_rows.append({'schedule':name,'replicates':len(summary),'minimum_sensitive':int(S.min()),'minimum_resistant':int(R.min()),'maximum_total':int((S+R).max()),**checks,'all_checks_passed':all(checks.values())})

    stats=SHORT_STATISTICS[name]; deterministic=solve_deterministic_reference(PARAMETERS,ensemble['schedule'],ensemble['time'])
    for variable in ['sensitive','resistant','total']:
        mean=stats[f'{variable}_mean'].to_numpy(); exact=deterministic[variable].to_numpy(); error=mean-exact
        metric_rows.append({'schedule':name,'variable':variable,
                            'relative_l2_mean_vs_ode':np.linalg.norm(error)/(np.linalg.norm(exact)+1e-12),
                            'rmse_mean_vs_ode':float(np.sqrt(np.mean(error**2))),
                            'fraction_ode_inside_5_95_interval':float(np.mean((exact>=stats[f'{variable}_q05'])&(exact<=stats[f'{variable}_q95'])))})

    for variable,label in [('total','Total bacterial population'),('resistant','Resistant bacterial population')]:
        fig,ax=plt.subplots(figsize=(8.8,5.5))
        ax.fill_between(stats['time'],stats[f'{variable}_q05'],stats[f'{variable}_q95'],alpha=0.18,label='Stochastic 5–95% interval')
        ax.plot(stats['time'],stats[f'{variable}_mean'],label='Stochastic ensemble mean')
        ax.plot(deterministic['time'],deterministic[variable],linestyle='--',label='Deterministic mean-field solution')
        shade_treatment(ax,ensemble['schedule'],CONFIG.short_final_time)
        ax.set(xlabel='Time',ylabel=label,title=f'Deterministic and stochastic {variable}: {name.replace("_"," ")}',xlim=(0,CONFIG.short_final_time),ylim=(0,None))
        ax.grid(alpha=0.25); ax.legend(loc='best'); fig.tight_layout(); save_figure(fig,f'round2_{variable}_comparison_{name}'); plt.show()

SHORT_AUDIT=pd.DataFrame(audit_rows); SHORT_AUDIT.to_csv(TABLE_DIR/'round2_short_ensemble_audit.csv',index=False)
COMPARISON_METRICS=pd.DataFrame(metric_rows); COMPARISON_METRICS.to_csv(TABLE_DIR/'round2_deterministic_stochastic_metrics.csv',index=False)
display(SHORT_AUDIT); display(COMPARISON_METRICS)

In [ ]:
# =============================================================================
# CELL 10 — Demographic variability and Fano-factor figures
# =============================================================================
# Fano factor = variance/mean. Values above one indicate overdispersion relative
# to a Poisson-like count process.

for variable,label in [('sensitive','Sensitive-population Fano factor'),('resistant','Resistant-population Fano factor')]:
    fig,ax=plt.subplots(figsize=(8.8,5.5))
    for name,stats in SHORT_STATISTICS.items():
        ax.plot(stats['time'],stats[f'{variable}_fano'],label=name.replace('_',' '))
    ax.set(xlabel='Time',ylabel=label,title=f'Demographic variability of the {variable} population',xlim=(0,CONFIG.short_final_time),ylim=(0,None))
    ax.grid(alpha=0.25); ax.legend(loc='best'); fig.tight_layout(); save_figure(fig,f'round2_{variable}_fano_comparison'); plt.show()

In [ ]:
# =============================================================================
# CELL 11 — Extended-horizon ensembles for extinction and rescue
# =============================================================================
# t=20 is often too short for complete extinction. Permanent early and late
# treatment schedules are therefore simulated to t=120. Rescue is defined only
# for permanent treatment: final S=0, final R>0, and establishment was reached.

OUTCOME_ENSEMBLES={}
for schedule_index,name in enumerate(['early_continuous','late_continuous']):
    schedule=SCHEDULES[name]
    grid=make_output_grid(CONFIG.outcome_final_time,CONFIG.outcome_output_points,schedule)
    OUTCOME_ENSEMBLES[name]=run_gillespie_ensemble(PARAMETERS,schedule,grid,OUTCOME_REPLICATES,SEED+200_000+10_000*schedule_index,f'outcome_{name}')
print('Extended-outcome ensembles completed.')

In [ ]:
# =============================================================================
# CELL 12 — Outcome probabilities, Wilson intervals, and probability figures
# =============================================================================
# Wilson intervals remain well behaved when probabilities are close to zero or
# one. Separate figures are generated for each outcome.

def wilson_interval(successes,trials,z=1.959963984540054):
    if trials<=0: return math.nan,math.nan,math.nan
    p=successes/trials; denominator=1+z*z/trials
    center=(p+z*z/(2*trials))/denominator
    half=z*math.sqrt(p*(1-p)/trials+z*z/(4*trials*trials))/denominator
    return p,max(0,center-half),min(1,center+half)

OUTCOME_COLUMNS={
    'mutant_present_at_treatment':'Resistance present at treatment',
    'established':'Resistant establishment',
    'resistant_dominance_reached':'Resistant dominance',
    'extinct':'Population extinction','survived':'Population survival',
    'evolutionary_rescue':'Evolutionary rescue',
}
rows=[]
for name,ensemble in OUTCOME_ENSEMBLES.items():
    summary=ensemble['summary']; n=len(summary)
    for column,label in OUTCOME_COLUMNS.items():
        successes=int(summary[column].sum()); p,lower,upper=wilson_interval(successes,n)
        rows.append({'schedule':name,'outcome':label,'successes':successes,'replicates':n,'probability':p,'ci95_lower':lower,'ci95_upper':upper})
OUTCOME_PROBABILITIES=pd.DataFrame(rows); OUTCOME_PROBABILITIES.to_csv(TABLE_DIR/'round2_outcome_probabilities.csv',index=False)
display(OUTCOME_PROBABILITIES)

for label in OUTCOME_COLUMNS.values():
    subset=OUTCOME_PROBABILITIES[OUTCOME_PROBABILITIES['outcome']==label]
    x=np.arange(len(subset)); lower=subset['probability']-subset['ci95_lower']; upper=subset['ci95_upper']-subset['probability']
    fig,ax=plt.subplots(figsize=(7.4,5.2))
    ax.errorbar(x,subset['probability'],yerr=np.vstack([lower,upper]),fmt='o',capsize=5,markersize=7)
    ax.set_xticks(x); ax.set_xticklabels(subset['schedule'].str.replace('_',' '))
    ax.set(ylabel='Estimated probability',title=f'{label} under continuous treatment',ylim=(0,1))
    ax.grid(axis='y',alpha=0.25); fig.tight_layout(); save_figure(fig,'round2_probability_'+label.lower().replace(' ','_')); plt.show()

In [ ]:
# =============================================================================
# CELL 13 — Cumulative-incidence figures for event times
# =============================================================================
# Replicates without an event are treated as right-censored at the final time.
# The plotted quantity is the fraction of all replicates that experienced the
# event by each time.

def cumulative_incidence(event_times,evaluation_times):
    event_times=np.asarray(event_times,dtype=float); finite=np.isfinite(event_times)
    return np.array([np.mean(finite&(event_times<=t)) for t in evaluation_times])

EVENT_COLUMNS={
    'emergence_time':'Resistance emergence',
    'establishment_time':'Resistant establishment',
    'sensitive_extinction_time':'Sensitive-lineage extinction',
    'total_extinction_time':'Total-population extinction',
}
evaluation_times=np.linspace(0,CONFIG.outcome_final_time,500)
for column,title in EVENT_COLUMNS.items():
    fig,ax=plt.subplots(figsize=(8.6,5.4))
    for name,ensemble in OUTCOME_ENSEMBLES.items():
        ax.step(evaluation_times,cumulative_incidence(ensemble['summary'][column],evaluation_times),where='post',label=name.replace('_',' '))
    ax.set(xlabel='Time',ylabel='Cumulative incidence',title=title,xlim=(0,CONFIG.outcome_final_time),ylim=(0,1))
    ax.grid(alpha=0.25); ax.legend(loc='best'); fig.tight_layout(); save_figure(fig,'round2_cumulative_'+column.replace('_time','')); plt.show()

In [ ]:
# =============================================================================
# CELL 14 — Terminal-distribution publication figures
# =============================================================================
# Violin plots show the distribution shape, mean, and median instead of only a
# point estimate.

names=list(OUTCOME_ENSEMBLES)
final_R=[OUTCOME_ENSEMBLES[name]['summary']['final_resistant'].to_numpy() for name in names]
fig,ax=plt.subplots(figsize=(7.8,5.5))
ax.violinplot(final_R,positions=np.arange(1,len(names)+1),showmeans=True,showmedians=True)
ax.set_xticks(np.arange(1,len(names)+1)); ax.set_xticklabels([x.replace('_',' ') for x in names])
ax.set(ylabel='Final resistant population',title='Distribution of terminal resistant population',ylim=(0,None))
ax.grid(axis='y',alpha=0.25); fig.tight_layout(); save_figure(fig,'round2_terminal_resistant_violin'); plt.show()

fractions=[]
for name in names:
    summary=OUTCOME_ENSEMBLES[name]['summary']; total=(summary['sensitive_at_treatment']+summary['resistant_at_treatment']).to_numpy(float)
    R=summary['resistant_at_treatment'].to_numpy(float)
    fractions.append(np.divide(R,total,out=np.zeros_like(R),where=total>0))
fig,ax=plt.subplots(figsize=(7.8,5.5))
ax.violinplot(fractions,positions=np.arange(1,len(names)+1),showmeans=True,showmedians=True)
ax.set_xticks(np.arange(1,len(names)+1)); ax.set_xticklabels([x.replace('_',' ') for x in names])
ax.set(ylabel='Resistant fraction at treatment',title='Pre-treatment resistance composition',ylim=(0,1))
ax.grid(axis='y',alpha=0.25); fig.tight_layout(); save_figure(fig,'round2_resistant_fraction_at_treatment_violin'); plt.show()

In [ ]:
# =============================================================================
# CELL 15 — Mutation-by-treatment-time rescue experiment and heat maps
# =============================================================================
# This experiment tests how rescue depends jointly on mutation probability and
# permanent-treatment timing. Only t=0, treatment time, and t=120 are sampled to
# reduce memory; first-passage outcomes are still checked after every event.

MUTATION_GRID=np.array([1e-5,3e-5,1e-4,3e-4,1e-3])
TREATMENT_TIME_GRID=np.array([2.5,5.0,7.5,10.0,12.5])
rows=[]; case=0; total_cases=len(MUTATION_GRID)*len(TREATMENT_TIME_GRID)
for mu in MUTATION_GRID:
    parameters=replace(PARAMETERS,mutation_probability=float(mu))
    for treatment_time in TREATMENT_TIME_GRID:
        case+=1
        schedule=PiecewiseTreatment(f'continuous_T{treatment_time:g}',(float(treatment_time),),(0.0,1.0),f'Permanent treatment from t={treatment_time}.')
        grid=np.array([0.0,float(treatment_time),CONFIG.outcome_final_time])
        children=np.random.SeedSequence(SEED+1_000_000+int(round(-math.log10(mu)*1000))+int(round(treatment_time*100))).spawn(RESCUE_GRID_REPLICATES)
        rescue=extinction=established=present=0
        for child in children:
            result=simulate_gillespie_replicate(parameters,schedule,grid,int(child.generate_state(1,dtype=np.uint32)[0]))
            rescue+=int(result['evolutionary_rescue']); extinction+=int(result['extinct']); established+=int(result['established']); present+=int(result['mutant_present_at_treatment'])
        p,lower,upper=wilson_interval(rescue,RESCUE_GRID_REPLICATES)
        rows.append({'mutation_probability':mu,'treatment_time':treatment_time,'replicates':RESCUE_GRID_REPLICATES,
                     'rescue_probability':p,'rescue_ci95_lower':lower,'rescue_ci95_upper':upper,
                     'extinction_probability':extinction/RESCUE_GRID_REPLICATES,
                     'establishment_probability':established/RESCUE_GRID_REPLICATES,
                     'mutant_at_treatment_probability':present/RESCUE_GRID_REPLICATES})
        print(f'Grid {case}/{total_cases}: mu={mu:.1e}, T={treatment_time:g}, P_rescue={p:.3f}')
RESCUE_GRID_RESULTS=pd.DataFrame(rows); RESCUE_GRID_RESULTS.to_csv(TABLE_DIR/'round2_rescue_parameter_grid.csv',index=False)

def probability_heatmap(column,title,label,stem):
    pivot=RESCUE_GRID_RESULTS.pivot(index='mutation_probability',columns='treatment_time',values=column)
    fig,ax=plt.subplots(figsize=(8.0,5.8))
    image=ax.imshow(pivot.to_numpy(),origin='lower',aspect='auto',extent=[pivot.columns.min(),pivot.columns.max(),np.log10(pivot.index.min()),np.log10(pivot.index.max())],vmin=0,vmax=1)
    ax.set(xlabel='Permanent-treatment introduction time',ylabel=r'$\log_{10}(\mu)$',title=title)
    colorbar=fig.colorbar(image,ax=ax); colorbar.set_label(label)
    fig.tight_layout(); save_figure(fig,stem); plt.show()

probability_heatmap('rescue_probability','Evolutionary rescue across mutation and treatment timing','Rescue probability','round2_rescue_probability_heatmap')
probability_heatmap('extinction_probability','Population extinction across mutation and treatment timing','Extinction probability','round2_extinction_probability_heatmap')
probability_heatmap('mutant_at_treatment_probability','Resistance present when treatment begins','Probability','round2_mutant_at_treatment_heatmap')

In [ ]:
# =============================================================================
# CELL 16 — Monte Carlo convergence of rescue probability
# =============================================================================
# The cumulative estimate and Wilson interval are plotted as replicate count
# grows. Stable estimates and narrowing intervals support the selected budget.

indicators=OUTCOME_ENSEMBLES['early_continuous']['summary']['evolutionary_rescue'].astype(int).to_numpy()
sample_sizes=np.unique(np.linspace(10,len(indicators),min(60,max(2,len(indicators)-9)),dtype=int))
rows=[]
for n in sample_sizes:
    p,lower,upper=wilson_interval(int(indicators[:n].sum()),int(n))
    rows.append({'sample_size':n,'probability':p,'ci95_lower':lower,'ci95_upper':upper})
MONTE_CARLO_CONVERGENCE=pd.DataFrame(rows); MONTE_CARLO_CONVERGENCE.to_csv(TABLE_DIR/'round2_monte_carlo_convergence.csv',index=False)
fig,ax=plt.subplots(figsize=(8.4,5.4))
ax.fill_between(MONTE_CARLO_CONVERGENCE['sample_size'],MONTE_CARLO_CONVERGENCE['ci95_lower'],MONTE_CARLO_CONVERGENCE['ci95_upper'],alpha=0.18,label='95% Wilson interval')
ax.plot(MONTE_CARLO_CONVERGENCE['sample_size'],MONTE_CARLO_CONVERGENCE['probability'],label='Cumulative rescue estimate')
ax.set(xlabel='Number of stochastic replicates',ylabel='Estimated rescue probability',title='Monte Carlo convergence of evolutionary-rescue probability',ylim=(0,1))
ax.grid(alpha=0.25); ax.legend(loc='best'); fig.tight_layout(); save_figure(fig,'round2_rescue_monte_carlo_convergence'); plt.show()

In [ ]:
# =============================================================================
# CELL 17 — Export a stochastic sparse-data benchmark for later PINN tests
# =============================================================================
# The main PINN experiments remain based on deterministic ground truth. This
# export is a separate robustness benchmark containing 20 stochastic cultures,
# 21 total-load measurements, and phenotype measurements at t={7,12,18}.

number_to_export=min(20,SHORT_REPLICATES)
total_times=np.linspace(0,CONFIG.short_final_time,21)
phenotype_times=np.array([7.0,12.0,18.0])
rows=[]
for name,ensemble in SHORT_ENSEMBLES.items():
    grid=ensemble['time']; schedule=ensemble['schedule']
    total_indexes=[int(np.argmin(np.abs(grid-t))) for t in total_times]
    phenotype_indexes={int(np.argmin(np.abs(grid-t))) for t in phenotype_times}
    for replicate in range(number_to_export):
        S=ensemble['sensitive'][replicate]; R=ensemble['resistant'][replicate]
        for index in total_indexes:
            total=int(S[index]+R[index]); fraction=float(R[index]/total) if total>0 else 0.0
            rows.append({'schedule':name,'replicate':replicate,'time':float(grid[index]),
                         'sensitive_hidden':int(S[index]),'resistant_hidden':int(R[index]),
                         'total_observed':total,
                         'resistant_fraction_observed':fraction if index in phenotype_indexes else math.nan,
                         'phenotype_observed':bool(index in phenotype_indexes),
                         'antibiotic':float(schedule.exposure(grid[index]))})
STOCHASTIC_PINN_EXPORT=pd.DataFrame(rows)
STOCHASTIC_PINN_EXPORT.to_csv(PINN_EXPORT_DIR/'round2_stochastic_sparse_pinn_benchmark.csv',index=False)
display(STOCHASTIC_PINN_EXPORT.head(25))

In [ ]:
# =============================================================================
# CELL 18 — LaTeX export, manifest, file inventory, and final audit
# =============================================================================
# QUICK_MODE results are for code verification. Set QUICK_MODE=False and rerun
# before inserting probabilities into the manuscript.

latex=OUTCOME_PROBABILITIES.copy(); latex['schedule']=latex['schedule'].str.replace('_',' ')
latex['Probability [95\\% CI]']=latex.apply(lambda row:f"{row['probability']:.3f} [{row['ci95_lower']:.3f}, {row['ci95_upper']:.3f}]",axis=1)
latex_code=latex[['schedule','outcome','Probability [95\\% CI]']].rename(columns={'schedule':'Treatment','outcome':'Outcome'}).to_latex(index=False,escape=False,caption='Stochastic AMR outcomes under continuous treatment with 95\\% Wilson intervals.',label='tab:stochastic_outcomes')
(TABLE_DIR/'round2_stochastic_outcomes.tex').write_text(latex_code,encoding='utf-8')

manifest={
    'seed':SEED,'quick_mode':QUICK_MODE,'short_replicates':SHORT_REPLICATES,
    'outcome_replicates':OUTCOME_REPLICATES,'rescue_grid_replicates':RESCUE_GRID_REPLICATES,
    'parameters':asdict(PARAMETERS),'resistant_birth_rate':PARAMETERS.resistant_birth_rate,
    'config':asdict(CONFIG),
    'schedules':{name:{'switch_times':s.switch_times,'levels':s.levels,'description':s.description} for name,s in SCHEDULES.items()},
    'outcome_definitions':{
        'emergence':'first R>=1','establishment':f'first R>={CONFIG.establishment_threshold}',
        'dominance':f'first R/(S+R)>={CONFIG.dominance_fraction}','extinction':'S=R=0',
        'evolutionary_rescue':'permanent treatment, final S=0, final R>0, establishment reached',
    },
}
with open(REPORT_DIR/'round2_manifest.json','w') as file: json.dump(manifest,file,indent=2)

required=[TABLE_DIR/'round2_short_ensemble_audit.csv',TABLE_DIR/'round2_deterministic_stochastic_metrics.csv',TABLE_DIR/'round2_outcome_probabilities.csv',TABLE_DIR/'round2_rescue_parameter_grid.csv',TABLE_DIR/'round2_monte_carlo_convergence.csv',TABLE_DIR/'round2_stochastic_outcomes.tex',REPORT_DIR/'round2_manifest.json',PINN_EXPORT_DIR/'round2_stochastic_sparse_pinn_benchmark.csv']
missing=[str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError(missing)
assert SHORT_AUDIT['all_checks_passed'].all()
assert len(RESCUE_GRID_RESULTS)==len(MUTATION_GRID)*len(TREATMENT_TIME_GRID)

inventory=sorted(str(path.relative_to(PROJECT_ROOT)) for path in PROJECT_ROOT.rglob('*') if path.is_file())
(REPORT_DIR/'round2_file_inventory.txt').write_text('\n'.join(inventory),encoding='utf-8')
print('Round 2 completed successfully.')
print('QUICK_MODE:',QUICK_MODE)
print('Ensemble files:',len(list(ENSEMBLE_DIR.glob('*.npz'))))
print('Summary CSV files:',len(list(SUMMARY_DIR.glob('*.csv'))))
print('PINN benchmark files:',len(list(PINN_EXPORT_DIR.glob('*.csv'))))
print('Figure files:',len(list(FIGURE_DIR.glob('*'))))
print('Table files:',len(list(TABLE_DIR.glob('*'))))
print('Report files:',len(list(REPORT_DIR.glob('*'))))
if QUICK_MODE:
    print("\nPreliminary run completed. Set QUICK_MODE=False for manuscript-scale results.")
else:
    print("\nManuscript-scale final run completed successfully.")

Round 3

In [ ]:
# =============================================================================
# ROUND 3 — CELL 1
# Environment, reproducibility, GPU-aware folders, and execution budgets
# FINAL MANUSCRIPT-SCALE OBSERVATION-REGIME CONFIGURATION
# =============================================================================
# WHAT THIS CELL DOES
# -------------------
# 1. Configures Round 3 for the final O1–O4 observation-regime experiment.
# 2. Uses manuscript-scale training budgets.
# 3. Activates only RUN_OBSERVATION_REGIME_STUDY.
# 4. Keeps every other expensive Round 3 experiment disabled.
# 5. Creates the Round 3 final-output folders.
# 6. Performs configuration checks before training begins.
#
# SCIENTIFIC PURPOSE
# ------------------
# This experiment evaluates how the amount of phenotype information affects:
#
#     - reconstruction of the hidden sensitive population S(t);
#     - reconstruction of the hidden resistant population R(t);
#     - estimation of mutation probability mu;
#     - estimation of resistance fitness cost c_R;
#     - physics consistency;
#     - biological feasibility.
#
# The observation regimes are:
#
#     O1: total bacterial load only;
#     O2: total load + one resistant-fraction observation;
#     O3: total load + three resistant-fraction observations;
#     O4: fully phenotype-resolved observations.
#
# EXECUTION SEQUENCE
# ------------------
# 1. Run this corrected Cell 1.
# 2. Restart the kernel.
# 3. Run Round 3 Cells 1–13 sequentially.
# 4. Skip Cells 14–17.
# 5. Run Cell 18.
#
# IMPORTANT
# ---------
# QUICK_MODE=False is used because this is the final manuscript-scale run.
# Four hard-constrained PINNs will be trained, one for each regime O1–O4.

from pathlib import Path
import os
import sys
import platform
import random
import json
import math
import time
import copy
import gc
import warnings

from dataclasses import dataclass, asdict, replace
from typing import Dict, List, Tuple, Optional, Iterable, Any

import numpy as np
import pandas as pd


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 20260805

random.seed(SEED)
np.random.seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)


# =============================================================================
# SINGLE EXECUTION-MODE SWITCH
# =============================================================================
# True:
#     Reduced structural verification only.
#     Results must not be used in the manuscript.
#
# False:
#     Manuscript-scale training and evaluation.
#
# The current cell is configured for the final O1–O4 experiment.

QUICK_MODE: bool = False

if not isinstance(QUICK_MODE, bool):
    raise TypeError(
        "QUICK_MODE must be a Boolean value: either True or False."
    )

RUN_LABEL = "quick" if QUICK_MODE else "final"


# =============================================================================
# CURRENT EXPERIMENT
# =============================================================================

CURRENT_EXPERIMENT = "observation_regime_study"


# =============================================================================
# PROJECT AND OUTPUT FOLDERS
# =============================================================================
# The notebook may be stored:
#
# 1. directly inside the project root; or
# 2. inside a folder named 'notebooks'.

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# Quick and final outputs are isolated:
#
# project/
# └── round3/
#     ├── quick/
#     └── final/
#
# Because QUICK_MODE=False, the current outputs are written to:
#
# project/round3/final/

ROUND3_BASE_ROOT = PROJECT_ROOT / "round3"
ROUND3_ROOT = ROUND3_BASE_ROOT / RUN_LABEL

MODEL_DIR = ROUND3_ROOT / "models"
PREDICTION_DIR = ROUND3_ROOT / "predictions"
HISTORY_DIR = ROUND3_ROOT / "histories"
FIGURE_DIR_R3 = ROUND3_ROOT / "figures"
TABLE_DIR_R3 = ROUND3_ROOT / "tables"
REPORT_DIR_R3 = ROUND3_ROOT / "reports"
CACHE_DIR_R3 = ROUND3_ROOT / "cache"

ROUND3_DIRECTORIES = [
    MODEL_DIR,
    PREDICTION_DIR,
    HISTORY_DIR,
    FIGURE_DIR_R3,
    TABLE_DIR_R3,
    REPORT_DIR_R3,
    CACHE_DIR_R3,
]

for directory in ROUND3_DIRECTORIES:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# TRAINING BUDGETS
# =============================================================================
# One O1–O4 model is trained at a time.
#
# The final configuration uses:
#
#     2,000 data-pretraining epochs;
#     up to 12,000 Adam epochs;
#     up to 1,000 L-BFGS iterations;
#     2,048 collocation points per treatment interval.
#
# Early stopping may terminate Adam before 12,000 epochs when the validation
# score does not improve for the specified patience period.

if QUICK_MODE:

    # -------------------------------------------------------------------------
    # Preliminary structural verification
    # -------------------------------------------------------------------------

    PRETRAIN_EPOCHS = 250
    ADAM_EPOCHS = 750
    LBFGS_MAX_ITER = 35

    COLLOCATION_POINTS_PER_INTERVAL = 256

    EARLY_STOPPING_PATIENCE = 300
    LOG_EVERY = 25

else:

    # -------------------------------------------------------------------------
    # Final manuscript-scale configuration
    # -------------------------------------------------------------------------

    PRETRAIN_EPOCHS = 2000
    ADAM_EPOCHS = 12000
    LBFGS_MAX_ITER = 1000

    COLLOCATION_POINTS_PER_INTERVAL = 2048

    EARLY_STOPPING_PATIENCE = 2500
    LOG_EVERY = 100


# =============================================================================
# EXPERIMENT-SUITE SWITCHES
# =============================================================================
# Only the O1–O4 observation-regime study is active.
#
# The primary single-model experiment and architecture ablation have already
# been completed and must not be retrained during this run.

RUN_SINGLE_MODEL_DEMO = False

RUN_PRIMARY_ABLATION = False

RUN_OBSERVATION_REGIME_STUDY = True

RUN_NOISE_ROBUSTNESS_STUDY = False

RUN_MULTI_SCHEDULE_STUDY = False

RUN_PROFILE_LIKELIHOOD = False

RUN_STOCHASTIC_BENCHMARK = False

RUN_STOCHASTIC_TRANSFER_VALIDATION = False


# =============================================================================
# EXPERIMENT-SWITCH INVENTORY
# =============================================================================

EXPERIMENT_SWITCHES = {
    "single_model_demo": RUN_SINGLE_MODEL_DEMO,
    "primary_ablation": RUN_PRIMARY_ABLATION,
    "observation_regime_study": RUN_OBSERVATION_REGIME_STUDY,
    "noise_robustness_study": RUN_NOISE_ROBUSTNESS_STUDY,
    "multi_schedule_study": RUN_MULTI_SCHEDULE_STUDY,
    "profile_likelihood": RUN_PROFILE_LIKELIHOOD,
    "stochastic_benchmark": RUN_STOCHASTIC_BENCHMARK,
    "stochastic_transfer_validation": (
        RUN_STOCHASTIC_TRANSFER_VALIDATION
    ),
}

active_experiments = [
    experiment_name
    for experiment_name, enabled in EXPERIMENT_SWITCHES.items()
    if enabled
]


# =============================================================================
# OBSERVATION-REGIME DEFINITIONS
# =============================================================================
# These labels are repeated here for configuration reporting. Cell 18 creates
# the actual datasets using build_observation_dataset().

OBSERVATION_REGIMES = {
    "O1": {
        "description": "Total bacterial load only",
        "phenotype_information": "No resistant-fraction measurement",
    },
    "O2": {
        "description": "Total load plus one phenotype measurement",
        "phenotype_information": "One resistant-fraction measurement",
    },
    "O3": {
        "description": "Total load plus three phenotype measurements",
        "phenotype_information": "Three resistant-fraction measurements",
    },
    "O4": {
        "description": "Fully phenotype-resolved observations",
        "phenotype_information": "Dense resistant-fraction measurements",
    },
}


# =============================================================================
# CONFIGURATION VALIDATION
# =============================================================================
# These checks prevent an accidental rerun of another expensive experiment.

if QUICK_MODE:
    warnings.warn(
        "QUICK_MODE=True. Results from this run are preliminary and must not "
        "be used as final manuscript results."
    )

if not active_experiments:
    raise RuntimeError(
        "No Round 3 experiment suite is active."
    )

if len(active_experiments) != 1:
    raise RuntimeError(
        "Exactly one Round 3 experiment suite must be active for this run. "
        f"Currently active: {active_experiments}"
    )

if active_experiments[0] != CURRENT_EXPERIMENT:
    raise RuntimeError(
        "The active experiment does not match CURRENT_EXPERIMENT. "
        f"Expected: {CURRENT_EXPERIMENT}; "
        f"active: {active_experiments}"
    )

if RUN_SINGLE_MODEL_DEMO:
    raise RuntimeError(
        "RUN_SINGLE_MODEL_DEMO must remain False during the dedicated "
        "observation-regime experiment."
    )

if RUN_PRIMARY_ABLATION:
    raise RuntimeError(
        "RUN_PRIMARY_ABLATION must remain False. The ablation experiment "
        "has already been completed and must not be retrained here."
    )

if not RUN_OBSERVATION_REGIME_STUDY:
    raise RuntimeError(
        "RUN_OBSERVATION_REGIME_STUDY must be True."
    )

if len(OBSERVATION_REGIMES) != 4:
    raise RuntimeError(
        "The observation-regime experiment must contain O1, O2, O3, and O4."
    )

if list(OBSERVATION_REGIMES.keys()) != ["O1", "O2", "O3", "O4"]:
    raise RuntimeError(
        "Observation regimes must be ordered as O1, O2, O3, and O4."
    )


# =============================================================================
# PRINT EXECUTION CONFIGURATION
# =============================================================================

print("=" * 82)
print("ROUND 3 EXECUTION CONFIGURATION")
print("=" * 82)

print("\nProject root:")
print(" ", PROJECT_ROOT.resolve())

print("\nRound 3 base root:")
print(" ", ROUND3_BASE_ROOT.resolve())

print("\nCurrent output root:")
print(" ", ROUND3_ROOT.resolve())

print("\nEnvironment")
print("  Python:", sys.version.split()[0])
print("  Operating system:", platform.platform())
print("  Random seed:", SEED)

print("\nExecution mode")
print("  QUICK_MODE:", QUICK_MODE)
print("  Run label:", RUN_LABEL)
print("  Current experiment:", CURRENT_EXPERIMENT)

print("\nTraining budgets")
print("  Pretraining epochs:", PRETRAIN_EPOCHS)
print("  Adam epochs:", ADAM_EPOCHS)
print("  L-BFGS maximum iterations:", LBFGS_MAX_ITER)
print(
    "  Collocation points per treatment interval:",
    COLLOCATION_POINTS_PER_INTERVAL,
)
print(
    "  Early-stopping patience:",
    EARLY_STOPPING_PATIENCE,
)
print(
    "  Logging interval:",
    LOG_EVERY,
)

print("\nExperiment switches")

for experiment_name, enabled in EXPERIMENT_SWITCHES.items():
    print(
        f"  {experiment_name:35s}: {enabled}"
    )

print("\nActive experiment suites:")
print(" ", active_experiments)

print("\nObservation regimes")

for regime_name, regime_information in OBSERVATION_REGIMES.items():

    print(
        f"  {regime_name}: "
        f"{regime_information['description']}"
    )

print("\nExpected experiment")
print(
    "  Final manuscript-scale O1–O4 observation-regime study"
)

print("\nRequired execution sequence")
print("  1. Restart the kernel.")
print("  2. Run Round 3 Cells 1–13 sequentially.")
print("  3. Skip Cells 14–17.")
print("  4. Run Cell 18.")
print("  5. Inspect and save the O1–O4 metrics.")
print("  6. Run the observation-regime plotting cell, when available.")

print("\nExpected Cell 18 model labels")
print("  round3_observation_O1")
print("  round3_observation_O2")
print("  round3_observation_O3")
print("  round3_observation_O4")

print("\nExpected output table")
print(
    " ",
    TABLE_DIR_R3 / "round3_observation_regime_metrics.csv",
)

print("=" * 82)


# =============================================================================
# FINAL CONFIGURATION ASSERTIONS
# =============================================================================
# These assertions verify that Cell 1 is configured exactly for the final
# observation-regime experiment.

assert QUICK_MODE is False, (
    "This Cell 1 is configured for manuscript-scale execution. "
    "QUICK_MODE must be False."
)

assert RUN_SINGLE_MODEL_DEMO is False, (
    "RUN_SINGLE_MODEL_DEMO must be False during the dedicated "
    "observation-regime run."
)

assert RUN_PRIMARY_ABLATION is False, (
    "RUN_PRIMARY_ABLATION must be False because the architecture ablation "
    "has already been completed."
)

assert RUN_OBSERVATION_REGIME_STUDY is True, (
    "RUN_OBSERVATION_REGIME_STUDY must be True."
)

assert active_experiments == ["observation_regime_study"], (
    "Only observation_regime_study should be active. "
    f"Currently active: {active_experiments}"
)

assert RUN_LABEL == "final", (
    "The observation-regime manuscript run must use RUN_LABEL='final'."
)

assert PRETRAIN_EPOCHS == 2000
assert ADAM_EPOCHS == 12000
assert LBFGS_MAX_ITER == 1000
assert COLLOCATION_POINTS_PER_INTERVAL == 2048

print(
    "\nCell 1 completed successfully: final manuscript-scale "
    "O1–O4 observation-regime configuration is active."
)

In [ ]:
# ROUND 3 — CELL 2
# Scientific imports, PyTorch precision, deterministic settings, and figures
# =============================================================================
# Why double precision is used
# ----------------------------
# PINNs differentiate network outputs with respect to time and optimize small
# mutation probabilities. Float64 substantially improves residual and parameter
# stability. The network and collocation sets are small enough that an 8 GB GPU
# can use float64 without difficulty.
#
# Figure policy
# -------------
# Every manuscript figure is saved as:
# - PNG at 400 dpi for convenient viewing;
# - PDF for vector-quality publication;
# - SVG for editing in Inkscape/Illustrator.

import scipy
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares, minimize_scalar
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, LogLocator
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore', category=UserWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
# Float64 is selected explicitly rather than through torch.set_default_dtype so
# external libraries and integer tensors are not affected unexpectedly.

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 400,
    'font.family': 'sans-serif',
    'font.size': 10.5,
    'axes.labelsize': 11.5,
    'axes.titlesize': 12.5,
    'legend.fontsize': 9.0,
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.0,
    'legend.frameon': True,
    'legend.framealpha': 1.0,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})


def save_figure_r3(fig: plt.Figure, stem: str, close: bool = False) -> None:
    """Save a Round 3 figure in high-resolution raster and vector formats."""
    clean = stem.strip().replace(' ', '_')
    fig.savefig(FIGURE_DIR_R3 / f'{clean}.png', dpi=400,
                bbox_inches='tight', facecolor='white')
    fig.savefig(FIGURE_DIR_R3 / f'{clean}.pdf',
                bbox_inches='tight', facecolor='white')
    fig.savefig(FIGURE_DIR_R3 / f'{clean}.svg',
                bbox_inches='tight', facecolor='white')
    if close:
        plt.close(fig)


def clear_gpu_memory() -> None:
    """Release unused Python and CUDA memory between sequential experiments."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('PyTorch:', torch.__version__)
print('SciPy:', scipy.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Device:', DEVICE)
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name)
    print(f'GPU memory: {properties.total_memory / 1024**3:.2f} GB')
print('Tensor dtype:', DTYPE)

In [ ]:
# ROUND 3 — CELL 3
# Repository provenance, novelty boundary, and reproducibility manifest
# =============================================================================
# This cell documents exactly how each audited repository influenced the code.
# It is important for discussions with a supervisor or reviewer because it
# separates reusable published ideas from the proposed AMR-specific synthesis.

REPOSITORY_PROVENANCE = {
    'DrugRes_StructPop': {
        'used': [
            'sensitive/resistant stochastic reaction structure',
            'logistic birth, death, mutation during sensitive division',
            'treatment-switch handling and evolutionary-rescue terminology',
        ],
        'not_claimed_as_novel': 'The AMR population model and Gillespie method.',
        'round3_role': 'Provides the deterministic mean-field equations and later stochastic transfer test.',
    },
    'BINNs': {
        'used': [
            'sparse biological inverse-learning philosophy',
            'explicit non-negative output mappings as a baseline',
            'separation of data-fit and mechanistic residual losses',
            'careful output scaling and validation tracking',
        ],
        'not_claimed_as_novel': 'Learning biological dynamics from sparse data.',
        'round3_role': 'Motivates sparse total-load and phenotype-fraction inference.',
    },
    'PINN-Proj': {
        'used': [
            'ordinary versus soft versus exact-constraint comparisons',
            'separate reporting of prediction and constraint errors',
            'L-BFGS refinement after first-order optimization',
        ],
        'not_claimed_as_novel': 'Differentiable projection for conservation constraints.',
        'round3_role': 'Motivates the controlled constraint ablation suite.',
    },
    'Control-PINNs': {
        'used': [
            'treatment represented explicitly as an external system input',
            'hierarchical state representations as a conceptual reference',
        ],
        'excluded': 'No adjoint, stationarity equation, or treatment optimization is used in Paper 1.',
        'round3_role': 'Confirms that prescribed-schedule inference is distinct from optimal control.',
    },
    'evo_dm': {
        'used': [
            'fitness-cost interpretation',
            'evaluation across multiple treatment environments',
        ],
        'excluded': 'No reinforcement-learning or drug-selection policy is trained in Round 3.',
        'round3_role': 'Motivates schedule diversity without broadening Paper 1 into control.',
    },
    'KKT_hPINN': {
        'used': [
            'hard versus soft feasibility comparison',
            'training through an exact differentiable constraint mapping',
            'constraint audits independent of predictive accuracy',
        ],
        'not_claimed_as_novel': 'Hard linear equality constraints using KKT projection.',
        'round3_role': 'Motivates exact feasibility but cannot directly encode the AMR feasible set.',
    },
}

NOVELTY_BOUNDARY = {
    'proposed_contribution': [
        'latent separation of sensitive and resistant populations from total load plus sparse phenotype fractions',
        'coupled total-abundance sigmoid and phenotype softmax representation',
        'exact non-negativity, phenotype simplex, carrying capacity, initial state, and treatment-interface continuity',
        'identifiability-aware inference of mutation probability and resistance fitness cost',
        'transfer of deterministic inverse estimates to independent Gillespie outcome validation',
    ],
    'not_claimed': [
        'first PINN', 'first AMR model', 'first hard-constrained neural network',
        'first Gillespie AMR simulator', 'first antibiotic-control or reinforcement-learning framework',
    ],
}

with open(REPORT_DIR_R3 / 'round3_repository_provenance.json', 'w', encoding='utf-8') as file:
    json.dump({'repositories': REPOSITORY_PROVENANCE,
               'novelty_boundary': NOVELTY_BOUNDARY}, file, indent=2)

pd.DataFrame([
    {'repository': name,
     'round3_role': details.get('round3_role', ''),
     'excluded_or_not_novel': details.get('excluded', details.get('not_claimed_as_novel', ''))}
    for name, details in REPOSITORY_PROVENANCE.items()
])

In [ ]:
# ROUND 3 — CELL 4
# Eco-evolutionary AMR parameters, treatment schedules, and deterministic solver
# =============================================================================
# The state is normalized by carrying capacity during PINN training:
#     s = S/K, r = R/K.
# This keeps both states and ODE residuals near order one and improves PINN
# optimization. Dimensional bacterial counts are restored for all reports.

@dataclass(frozen=True)
class AMRParametersR3:
    carrying_capacity: float = 1000.0
    sensitive_birth_rate: float = 1.0
    resistance_fitness_cost: float = 0.10
    sensitive_death_rate: float = 0.10
    resistant_death_rate: float = 0.10
    mutation_probability: float = 1.0e-4
    sensitive_drug_effect: float = 1.0
    resistant_drug_effect: float = 0.10

    @property
    def resistant_birth_rate(self) -> float:
        return self.sensitive_birth_rate * (1.0 - self.resistance_fitness_cost)

    def validate(self) -> None:
        if self.carrying_capacity <= 0 or self.sensitive_birth_rate <= 0:
            raise ValueError('Capacity and sensitive birth rate must be positive.')
        for name, value in {
            'resistance_fitness_cost': self.resistance_fitness_cost,
            'mutation_probability': self.mutation_probability,
            'sensitive_drug_effect': self.sensitive_drug_effect,
            'resistant_drug_effect': self.resistant_drug_effect,
        }.items():
            if not 0 <= value <= 1:
                raise ValueError(f'{name} must lie in [0,1].')


@dataclass(frozen=True)
class ScheduleR3:
    name: str
    switch_times: Tuple[float, ...]
    levels: Tuple[float, ...]
    description: str

    def __post_init__(self):
        if len(self.levels) != len(self.switch_times) + 1:
            raise ValueError('A piecewise schedule requires one more level than switches.')
        if tuple(sorted(self.switch_times)) != self.switch_times:
            raise ValueError('switch_times must be strictly ordered.')
        if any((level < 0 or level > 1) for level in self.levels):
            raise ValueError('Exposure levels must remain in [0,1].')

    def exposure_numpy(self, t: np.ndarray | float) -> np.ndarray | float:
        arr = np.asarray(t, dtype=float)
        indices = np.searchsorted(np.asarray(self.switch_times), arr, side='right')
        values = np.asarray(self.levels, dtype=float)[indices]
        return float(values) if np.isscalar(t) else values

    def exposure_torch(self, t: torch.Tensor) -> torch.Tensor:
        result = torch.full_like(t, float(self.levels[0]))
        for switch, level in zip(self.switch_times, self.levels[1:]):
            result = torch.where(t >= float(switch),
                                 torch.full_like(t, float(level)), result)
        return result

    def intervals(self, start: float, final: float) -> List[Tuple[float, float, float]]:
        boundaries = [start] + [x for x in self.switch_times if start < x < final] + [final]
        intervals = []
        for left, right in zip(boundaries[:-1], boundaries[1:]):
            midpoint = 0.5 * (left + right)
            intervals.append((left, right, float(self.exposure_numpy(midpoint))))
        return intervals


TRUE_PARAMETERS = AMRParametersR3()
TRUE_PARAMETERS.validate()
INITIAL_S = 100.0
INITIAL_R = 0.0
T_START = 0.0
T_FINAL = 20.0
DENSE_POINTS = 1001

SCHEDULES_R3 = {
    'untreated': ScheduleR3('untreated', (), (0.0,), 'No antibiotic exposure.'),
    'early_continuous': ScheduleR3('early_continuous', (5.0,), (0.0, 1.0),
                                   'Continuous exposure beginning at t=5.'),
    'late_continuous': ScheduleR3('late_continuous', (10.0,), (0.0, 1.0),
                                  'Continuous exposure beginning at t=10.'),
    'pulsed': ScheduleR3('pulsed', (5.0, 8.0, 12.0, 15.0),
                         (0.0, 1.0, 0.0, 1.0, 0.0),
                         'Exposure on [5,8) and [12,15).'),
}


def normalized_amr_rhs_numpy(t: float, state: np.ndarray,
                             parameters: AMRParametersR3,
                             schedule: ScheduleR3) -> np.ndarray:
    """Mean-field ODE for normalized sensitive and resistant abundances."""
    s, r = np.asarray(state, dtype=float)
    exposure = float(schedule.exposure_numpy(t))
    b_s = parameters.sensitive_birth_rate * (1.0 - parameters.sensitive_drug_effect * exposure)
    b_r = parameters.resistant_birth_rate * (1.0 - parameters.resistant_drug_effect * exposure)
    crowding = 1.0 - s - r
    ds = b_s * (1.0 - parameters.mutation_probability) * crowding * s \
         - parameters.sensitive_death_rate * s
    dr = b_s * parameters.mutation_probability * crowding * s \
         + b_r * crowding * r - parameters.resistant_death_rate * r
    return np.array([ds, dr], dtype=float)


def solve_reference_r3(parameters: AMRParametersR3,
                       schedule: ScheduleR3,
                       times: Optional[np.ndarray] = None) -> pd.DataFrame:
    """Solve each treatment interval separately to respect coefficient jumps."""
    if times is None:
        times = np.linspace(T_START, T_FINAL, DENSE_POINTS)
    times = np.asarray(times, dtype=float)
    current_state = np.array([INITIAL_S / parameters.carrying_capacity,
                              INITIAL_R / parameters.carrying_capacity], dtype=float)
    out_s = np.empty_like(times)
    out_r = np.empty_like(times)

    for interval_index, (left, right, _) in enumerate(schedule.intervals(T_START, T_FINAL)):
        is_last = interval_index == len(schedule.intervals(T_START, T_FINAL)) - 1
        mask = (times >= left - 1e-12) & ((times <= right + 1e-12) if is_last else (times < right - 1e-12))
        interval_times = times[mask]
        eval_times = np.unique(np.concatenate(([left], interval_times, [right])))
        sol = solve_ivp(
            lambda t, y: normalized_amr_rhs_numpy(t, y, parameters, schedule),
            (left, right), current_state, t_eval=eval_times,
            method='DOP853', rtol=1e-10, atol=1e-12,
        )
        if not sol.success:
            raise RuntimeError(sol.message)
        if interval_times.size:
            indices = np.searchsorted(eval_times, interval_times)
            out_s[mask] = sol.y[0, indices]
            out_r[mask] = sol.y[1, indices]
        current_state = sol.y[:, -1]

    S = parameters.carrying_capacity * out_s
    R = parameters.carrying_capacity * out_r
    B = S + R
    q = np.divide(R, B, out=np.zeros_like(R), where=B > 0)
    return pd.DataFrame({
        'time': times, 'sensitive': S, 'resistant': R, 'total': B,
        'resistant_fraction': q,
        'antibiotic': schedule.exposure_numpy(times),
        'schedule': schedule.name,
    })

print(pd.DataFrame([asdict(TRUE_PARAMETERS) | {'resistant_birth_rate': TRUE_PARAMETERS.resistant_birth_rate}]))
print(pd.DataFrame([{'schedule': s.name, 'description': s.description}
                    for s in SCHEDULES_R3.values()]))

In [ ]:
# ROUND 3 — CELL 5
# Load Round 1 data, verify it, and construct O1–O4 observation regimes
# =============================================================================
# Data loading strategy
# ---------------------
# The cell first searches for the dense CSV files produced in Round 1. When a
# file is unavailable, it regenerates the same high-accuracy deterministic
# trajectory. Therefore, Round 3 remains reproducible even after a kernel restart.
#
# Observation regimes
# -------------------
# O1: 21 total-load values, no phenotype fraction.
# O2: 21 total-load values, one phenotype fraction at t=12.
# O3: 21 total-load values, phenotype fractions at t=7,12,18.
# O4: 21 total-load values and phenotype fractions at all 21 times.

@dataclass
class ObservationDataset:
    schedule_name: str
    regime: str
    noise_level: float
    total_time: np.ndarray
    total_observed: np.ndarray
    fraction_time: np.ndarray
    fraction_observed: np.ndarray
    total_train_mask: np.ndarray
    total_validation_mask: np.ndarray
    fraction_train_mask: np.ndarray
    fraction_validation_mask: np.ndarray

    def summary(self) -> Dict[str, Any]:
        return {
            'schedule': self.schedule_name,
            'regime': self.regime,
            'noise_level': self.noise_level,
            'total_points': len(self.total_time),
            'fraction_points': len(self.fraction_time),
            'total_train': int(self.total_train_mask.sum()),
            'total_validation': int(self.total_validation_mask.sum()),
            'fraction_train': int(self.fraction_train_mask.sum()),
            'fraction_validation': int(self.fraction_validation_mask.sum()),
        }


def nearest_values(frame: pd.DataFrame, requested_times: np.ndarray,
                   column: str) -> np.ndarray:
    grid = frame['time'].to_numpy(float)
    values = frame[column].to_numpy(float)
    indices = [int(np.argmin(np.abs(grid - time_value))) for time_value in requested_times]
    return values[np.asarray(indices, dtype=int)]


def load_or_generate_dense(schedule_name: str) -> pd.DataFrame:
    candidates = [
        PROJECT_ROOT / 'data' / 'dense' / f'{schedule_name}_dense.csv',
        PROJECT_ROOT / 'AMR_PINN_Round1_Comprehensive' / 'data' / 'dense' / f'{schedule_name}_dense.csv',
    ]
    for path in candidates:
        if path.exists():
            frame = pd.read_csv(path)
            required = {'time','sensitive','resistant','total','resistant_fraction','antibiotic'}
            if required.issubset(frame.columns):
                return frame
    frame = solve_reference_r3(TRUE_PARAMETERS, SCHEDULES_R3[schedule_name])
    fallback_dir = CACHE_DIR_R3 / 'dense_reference'
    fallback_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(fallback_dir / f'{schedule_name}_dense.csv', index=False)
    return frame


DENSE_REFERENCE = {name: load_or_generate_dense(name) for name in SCHEDULES_R3}


def build_observation_dataset(schedule_name: str, regime: str,
                              noise_level: float = 0.0,
                              seed: int = SEED) -> ObservationDataset:
    if regime not in {'O1','O2','O3','O4'}:
        raise ValueError('regime must be O1, O2, O3, or O4.')
    if noise_level < 0:
        raise ValueError('noise_level cannot be negative.')

    dense = DENSE_REFERENCE[schedule_name]
    total_time = np.linspace(T_START, T_FINAL, 21)
    true_total = nearest_values(dense, total_time, 'total')

    fraction_times_by_regime = {
        'O1': np.array([], dtype=float),
        'O2': np.array([12.0], dtype=float),
        'O3': np.array([7.0, 12.0, 18.0], dtype=float),
        'O4': total_time.copy(),
    }
    fraction_time = fraction_times_by_regime[regime]
    true_fraction = nearest_values(dense, fraction_time, 'resistant_fraction') \
                    if len(fraction_time) else np.array([], dtype=float)

    rng = np.random.default_rng(seed)
    if noise_level == 0:
        total_observed = true_total.copy()
        fraction_observed = true_fraction.copy()
    else:
        sigma_log = np.sqrt(np.log1p(noise_level**2))
        total_observed = true_total * rng.lognormal(-0.5*sigma_log**2,
                                                    sigma_log, len(true_total))
        eps = 1e-8
        clipped = np.clip(true_fraction, eps, 1-eps)
        logits = np.log(clipped / (1-clipped))
        fraction_observed = 1 / (1 + np.exp(-(logits + rng.normal(0, noise_level, len(logits)))))

    # Hold out every fifth total observation for validation, but always retain
    # the initial point in training. With only one phenotype measurement (O2),
    # it is retained for training and validation is based on total load.
    total_validation_mask = (np.arange(len(total_time)) % 5 == 4)
    total_validation_mask[0] = False
    total_train_mask = ~total_validation_mask

    fraction_validation_mask = np.zeros(len(fraction_time), dtype=bool)
    if len(fraction_time) >= 3:
        fraction_validation_mask[-1] = True
    fraction_train_mask = ~fraction_validation_mask

    return ObservationDataset(
        schedule_name=schedule_name, regime=regime, noise_level=noise_level,
        total_time=total_time, total_observed=total_observed,
        fraction_time=fraction_time, fraction_observed=fraction_observed,
        total_train_mask=total_train_mask,
        total_validation_mask=total_validation_mask,
        fraction_train_mask=fraction_train_mask,
        fraction_validation_mask=fraction_validation_mask,
    )

OBSERVATION_EXAMPLES = {
    regime: build_observation_dataset('early_continuous', regime, 0.0, SEED)
    for regime in ['O1','O2','O3','O4']
}

summary = pd.DataFrame([dataset.summary() for dataset in OBSERVATION_EXAMPLES.values()])
summary.to_csv(TABLE_DIR_R3 / 'round3_observation_regime_definitions.csv', index=False)
summary

In [ ]:
# ROUND 3 — CELL 6
# Publication figure: observation design and hidden-state ambiguity
# =============================================================================
# This figure communicates the inverse problem before presenting any model:
# total bacterial load is observed frequently, while phenotype composition is
# observed sparsely. The shaded region shows how much of the total population is
# resistant even though only the total curve is continuously visible.

example_dense = DENSE_REFERENCE['early_continuous']
example_obs = OBSERVATION_EXAMPLES['O3']

fig, axes = plt.subplots(2, 2, figsize=(11.2, 7.4), constrained_layout=True)

ax = axes[0,0]
ax.plot(example_dense['time'], example_dense['sensitive'], label='Hidden sensitive $S(t)$')
ax.plot(example_dense['time'], example_dense['resistant'], label='Hidden resistant $R(t)$')
ax.fill_between(example_dense['time'], 0, example_dense['resistant'], alpha=0.18)
ax.axvline(5.0, linestyle='--', linewidth=1.4, label='Treatment begins')
ax.set(xlabel='Time', ylabel='Bacterial abundance', title='A. Hidden phenotype trajectories')
ax.grid(alpha=0.22); ax.legend(loc='best')

ax = axes[0,1]
ax.plot(example_dense['time'], example_dense['total'], label='True total $B(t)$')
ax.scatter(example_obs.total_time[example_obs.total_train_mask],
           example_obs.total_observed[example_obs.total_train_mask], s=38,
           label='Training total-load measurements', zorder=4)
ax.scatter(example_obs.total_time[example_obs.total_validation_mask],
           example_obs.total_observed[example_obs.total_validation_mask], s=55,
           marker='D', label='Withheld validation measurements', zorder=5)
ax.axvline(5.0, linestyle='--', linewidth=1.4)
ax.set(xlabel='Time', ylabel='Total load', title='B. Aggregated observations')
ax.grid(alpha=0.22); ax.legend(loc='best')

ax = axes[1,0]
ax.plot(example_dense['time'], example_dense['resistant_fraction'],
        label='True resistant fraction $q(t)$')
ax.scatter(example_obs.fraction_time[example_obs.fraction_train_mask],
           example_obs.fraction_observed[example_obs.fraction_train_mask],
           s=55, label='Sparse phenotype measurements', zorder=5)
if example_obs.fraction_validation_mask.any():
    ax.scatter(example_obs.fraction_time[example_obs.fraction_validation_mask],
               example_obs.fraction_observed[example_obs.fraction_validation_mask],
               s=60, marker='D', label='Withheld phenotype value', zorder=5)
ax.set(xlabel='Time', ylabel='Resistant fraction', ylim=(-0.03,1.03),
       title='C. Sparse phenotype information')
ax.grid(alpha=0.22); ax.legend(loc='best')

ax = axes[1,1]
regimes = ['O1','O2','O3','O4']
counts = [len(OBSERVATION_EXAMPLES[r].fraction_time) for r in regimes]
ax.bar(regimes, counts)
for x, count in enumerate(counts):
    ax.text(x, count + 0.35, str(count), ha='center', va='bottom')
ax.set(xlabel='Observation regime', ylabel='Phenotype-fraction measurements',
       title='D. Progressively informative regimes')
ax.set_ylim(0, max(counts)+4); ax.grid(axis='y', alpha=0.22)

save_figure_r3(fig, 'round3_observation_design')
plt.show()

In [ ]:
# ROUND 3 — CELL 7
# Tensor conversion, treatment-interface collocation, and reproducible sampling
# =============================================================================
# Collocation points are sampled separately inside every treatment interval.
# Points exactly at a coefficient discontinuity are excluded. This prevents the
# network from being penalized for a derivative jump that is physically expected
# when treatment changes instantaneously.


def tensor(data: np.ndarray | List[float], requires_grad: bool = False) -> torch.Tensor:
    value = torch.as_tensor(np.asarray(data), dtype=DTYPE, device=DEVICE)
    if value.ndim == 1:
        value = value[:, None]
    value.requires_grad_(requires_grad)
    return value


def sample_interface_aware_collocation(schedule: ScheduleR3,
                                       points_per_interval: int,
                                       seed: int) -> torch.Tensor:
    rng = np.random.default_rng(seed)
    samples = []
    margin = 1e-6
    for left, right, _ in schedule.intervals(T_START, T_FINAL):
        if right - left <= 2*margin:
            continue
        # Latin-hypercube-like stratification in each interval gives more even
        # temporal coverage than independent uniform samples.
        bins = (np.arange(points_per_interval) + rng.random(points_per_interval)) \
               / points_per_interval
        points = left + margin + bins * (right-left-2*margin)
        rng.shuffle(points)
        samples.append(points)
    merged = np.concatenate(samples)
    return tensor(merged, requires_grad=True)


def observation_to_tensors(dataset: ObservationDataset, split: str = 'train') -> Dict[str, torch.Tensor]:
    if split not in {'train','validation','all'}:
        raise ValueError('split must be train, validation, or all.')
    if split == 'train':
        total_mask = dataset.total_train_mask
        fraction_mask = dataset.fraction_train_mask
    elif split == 'validation':
        total_mask = dataset.total_validation_mask
        fraction_mask = dataset.fraction_validation_mask
    else:
        total_mask = np.ones(len(dataset.total_time), dtype=bool)
        fraction_mask = np.ones(len(dataset.fraction_time), dtype=bool)

    return {
        'total_time': tensor(dataset.total_time[total_mask]),
        'total_observed': tensor(dataset.total_observed[total_mask] / TRUE_PARAMETERS.carrying_capacity),
        'fraction_time': tensor(dataset.fraction_time[fraction_mask]),
        'fraction_observed': tensor(dataset.fraction_observed[fraction_mask]),
    }

# Verify that no sampled point is placed directly on a treatment switch.
check_points = sample_interface_aware_collocation(SCHEDULES_R3['pulsed'], 20, SEED).detach().cpu().numpy().ravel()
for switch in SCHEDULES_R3['pulsed'].switch_times:
    assert np.min(np.abs(check_points-switch)) > 1e-7
print('Interface-aware collocation test passed with', len(check_points), 'points.')

In [ ]:
# ROUND 3 — CELL 8
# Neural building blocks and bounded inverse parameters
# =============================================================================
# The unknown Tier-I parameters are:
#     mutation probability mu in [1e-6, 1e-2] on a logarithmic scale;
#     resistance fitness cost c_R in [0, 0.40].
# Parameter transforms make every optimizer iterate biologically admissible.

class TanhMLP(nn.Module):
    """Four-layer tanh MLP with Xavier initialization."""
    def __init__(self, input_dim: int, output_dim: int,
                 width: int = 64, depth: int = 4):
        super().__init__()
        layers: List[nn.Module] = []
        previous = input_dim
        for _ in range(depth):
            linear = nn.Linear(previous, width)
            nn.init.xavier_normal_(linear.weight)
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.Tanh()]
            previous = width
        output = nn.Linear(previous, output_dim)
        nn.init.xavier_normal_(output.weight, gain=0.5)
        nn.init.zeros_(output.bias)
        layers.append(output)
        # Construct all trainable layers directly in float64. Calling model.to(DEVICE)
        # changes the device but not the dtype, so this explicit conversion prevents
        # Float/Double mismatches during automatic differentiation.
        self.network = nn.Sequential(*layers).to(dtype=DTYPE)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def inverse_sigmoid(value: float) -> float:
    value = float(np.clip(value, 1e-8, 1-1e-8))
    return math.log(value/(1-value))


class TierIInverseParameters(nn.Module):
    """Bounded trainable mutation probability and resistance fitness cost."""
    def __init__(self, initial_mu: float = 3e-4, initial_cost: float = 0.15,
                 trainable: bool = True):
        super().__init__()
        self.log10_mu_min = -6.0
        self.log10_mu_max = -2.0
        mu_position = (math.log10(initial_mu)-self.log10_mu_min) \
                      / (self.log10_mu_max-self.log10_mu_min)
        cost_position = initial_cost / 0.40
        self.raw_mu = nn.Parameter(torch.tensor(inverse_sigmoid(mu_position), dtype=DTYPE),
                                   requires_grad=trainable)
        self.raw_cost = nn.Parameter(torch.tensor(inverse_sigmoid(cost_position), dtype=DTYPE),
                                     requires_grad=trainable)

    def mutation_probability(self) -> torch.Tensor:
        log10_mu = self.log10_mu_min + (self.log10_mu_max-self.log10_mu_min) \
                   * torch.sigmoid(self.raw_mu)
        return torch.pow(torch.tensor(10.0, dtype=DTYPE, device=self.raw_mu.device), log10_mu)

    def resistance_fitness_cost(self) -> torch.Tensor:
        return 0.40 * torch.sigmoid(self.raw_cost)

    def as_floats(self) -> Dict[str, float]:
        return {
            'mutation_probability': float(self.mutation_probability().detach().cpu()),
            'resistance_fitness_cost': float(self.resistance_fitness_cost().detach().cpu()),
        }

In [ ]:
# ROUND 3 — CELL 9
# Treatment-interface state network and all architectural ablations
# =============================================================================
# Available model kinds
# ---------------------
# data_only : unconstrained state network, no ODE residual.
# vanilla   : unconstrained PINN with exact IC/interface continuity only.
# soft      : vanilla PINN plus penalties for negativity and capacity excess.
# positive  : softplus S and R, but total abundance is not bounded by K.
# simplex   : positive total and exact phenotype fractions, but total is unbounded.
# hard      : proposed bounded total + phenotype softmax; exact feasible set.
#
# Every model uses one subnetwork per treatment interval. The state at the end
# of one interval becomes the exact anchor of the next interval. This gives
# continuous S and R while allowing their derivatives to respond immediately
# to the changed treatment coefficient.

class InterfaceStateNetwork(nn.Module):
    def __init__(self, schedule: ScheduleR3, model_kind: str,
                 initial_s: float, initial_r: float,
                 width: int = 64, depth: int = 4, tau_h: float = 0.35):
        super().__init__()
        allowed = {'data_only','vanilla','soft','positive','simplex','hard'}
        if model_kind not in allowed:
            raise ValueError(f'Unknown model_kind {model_kind}.')
        self.schedule = schedule
        self.model_kind = model_kind
        self.initial_s = float(initial_s)
        self.initial_r = float(initial_r)
        self.tau_h = float(tau_h)
        self.intervals = schedule.intervals(T_START, T_FINAL)
        output_dim = 3 if model_kind in {'simplex','hard'} else 2
        self.segment_networks = nn.ModuleList([
            TanhMLP(1, output_dim, width=width, depth=depth)
            for _ in self.intervals
        ])

    def _scaled_local_time(self, t: torch.Tensor, left: float, right: float) -> torch.Tensor:
        return 2.0 * (t-left)/(right-left) - 1.0

    def _transform(self, raw: torch.Tensor, local_elapsed: torch.Tensor,
                   anchor: torch.Tensor) -> torch.Tensor:
        h = 1.0 - torch.exp(-torch.clamp(local_elapsed, min=0.0)/self.tau_h)
        anchor = anchor.expand(raw.shape[0], -1)

        if self.model_kind in {'data_only','vanilla','soft'}:
            candidate = raw[:, :2]
            return (1-h)*anchor + h*candidate

        if self.model_kind == 'positive':
            candidate = F.softplus(raw[:, :2])
            return (1-h)*anchor + h*candidate

        anchor_total = torch.clamp(anchor.sum(dim=1, keepdim=True), min=1e-12)
        anchor_fraction = anchor / anchor_total
        candidate_fraction = torch.softmax(raw[:, 1:3], dim=1)
        fraction = (1-h)*anchor_fraction + h*candidate_fraction

        if self.model_kind == 'simplex':
            candidate_total = F.softplus(raw[:, 0:1])
        else:  # proposed hard model
            candidate_total = torch.sigmoid(raw[:, 0:1])
        total = (1-h)*anchor_total + h*candidate_total
        return total * fraction

    def _anchors(self) -> List[torch.Tensor]:
        device = next(self.parameters()).device
        anchor = torch.tensor([[self.initial_s, self.initial_r]], dtype=DTYPE, device=device)
        anchors = [anchor]
        for index in range(1, len(self.intervals)):
            previous_left, previous_right, _ = self.intervals[index-1]
            switch_tensor = torch.tensor([[previous_right]], dtype=DTYPE, device=device)
            scaled = self._scaled_local_time(switch_tensor, previous_left, previous_right)
            raw = self.segment_networks[index-1](scaled)
            anchor = self._transform(raw, switch_tensor-previous_left, anchor)
            anchors.append(anchor)
        return anchors

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        if t.ndim == 1:
            t = t[:, None]
        anchors = self._anchors()
        output = torch.zeros((len(t),2), dtype=t.dtype, device=t.device)
        for index, (left, right, _) in enumerate(self.intervals):
            is_last = index == len(self.intervals)-1
            mask = (t[:,0] >= left-1e-12) & ((t[:,0] <= right+1e-12) if is_last else (t[:,0] < right-1e-12))
            if mask.any():
                local_t = t[mask]
                scaled = self._scaled_local_time(local_t, left, right)
                raw = self.segment_networks[index](scaled)
                output[mask] = self._transform(raw, local_t-left, anchors[index])
        return output


class AMRInverseModel(nn.Module):
    """One or more experiment-specific state networks sharing Tier-I parameters."""
    def __init__(self, schedules: Dict[str, ScheduleR3], model_kind: str,
                 width: int = 64, depth: int = 4,
                 parameter_initialization: Tuple[float,float] = (3e-4,0.15)):
        super().__init__()
        self.model_kind = model_kind
        self.physics_enabled = model_kind != 'data_only'
        self.state_networks = nn.ModuleDict({
            name: InterfaceStateNetwork(schedule, model_kind,
                                        INITIAL_S/TRUE_PARAMETERS.carrying_capacity,
                                        INITIAL_R/TRUE_PARAMETERS.carrying_capacity,
                                        width=width, depth=depth)
            for name, schedule in schedules.items()
        })
        self.inverse_parameters = TierIInverseParameters(
            parameter_initialization[0], parameter_initialization[1],
            trainable=self.physics_enabled,
        )

    def forward(self, t: torch.Tensor, schedule_name: str) -> torch.Tensor:
        return self.state_networks[schedule_name](t)

    def inferred_parameters(self) -> Dict[str,float]:
        if not self.physics_enabled:
            return {'mutation_probability': np.nan,
                    'resistance_fitness_cost': np.nan}
        return self.inverse_parameters.as_floats()

# Architectural guarantees are verified before training.
for kind in ['data_only','vanilla','soft','positive','simplex','hard']:
    trial = AMRInverseModel({'early_continuous': SCHEDULES_R3['early_continuous']},
                            kind, width=16, depth=2).to(DEVICE)
    t_test = tensor(np.linspace(0,20,101))
    with torch.no_grad():
        state = trial(t_test, 'early_continuous').cpu().numpy()
    assert np.max(np.abs(state[0]-np.array([0.1,0.0]))) < 1e-12
    if kind in {'positive','simplex','hard'}:
        assert state.min() >= -1e-12
    if kind == 'hard':
        assert state.sum(axis=1).max() <= 1+1e-12
    del trial
clear_gpu_memory()
print('All untrained architecture guarantee tests passed.')

In [ ]:
# ROUND 3 — CELL 10
# Physics residuals, observation likelihoods, and soft-constraint penalties
# =============================================================================
# Total-load data are compared on a log scale because bacterial abundance spans
# a wide range. Phenotype fractions are compared directly in [0,1].
# The proposed hard model receives no positivity or capacity penalties because
# those properties are exact consequences of its output mapping.

@dataclass
class LossWeights:
    total_data: float = 10.0
    fraction_data: float = 5.0
    physics: float = 1.0
    soft_constraint: float = 10.0
    weight_decay: float = 1e-8

LOSS_WEIGHTS = LossWeights()


def physics_residual(model: AMRInverseModel, t: torch.Tensor,
                     schedule_name: str) -> Tuple[torch.Tensor,torch.Tensor]:
    if not t.requires_grad:
        t = t.clone().detach().requires_grad_(True)
    state = model(t, schedule_name)
    s = state[:,0:1]
    r = state[:,1:2]
    ds_dt = torch.autograd.grad(s, t, torch.ones_like(s), create_graph=True,
                                retain_graph=True)[0]
    dr_dt = torch.autograd.grad(r, t, torch.ones_like(r), create_graph=True,
                                retain_graph=True)[0]

    mu = model.inverse_parameters.mutation_probability()
    cost = model.inverse_parameters.resistance_fitness_cost()
    schedule = SCHEDULES_R3[schedule_name]
    exposure = schedule.exposure_torch(t)
    b_s = TRUE_PARAMETERS.sensitive_birth_rate * \
          (1.0-TRUE_PARAMETERS.sensitive_drug_effect*exposure)
    b_r = TRUE_PARAMETERS.sensitive_birth_rate * (1.0-cost) * \
          (1.0-TRUE_PARAMETERS.resistant_drug_effect*exposure)
    crowding = 1.0-s-r
    rhs_s = b_s*(1.0-mu)*crowding*s - TRUE_PARAMETERS.sensitive_death_rate*s
    rhs_r = b_s*mu*crowding*s + b_r*crowding*r - TRUE_PARAMETERS.resistant_death_rate*r
    return ds_dt-rhs_s, dr_dt-rhs_r


def data_loss_for_dataset(model: AMRInverseModel, dataset: ObservationDataset,
                          split: str = 'train') -> Tuple[torch.Tensor,torch.Tensor]:
    values = observation_to_tensors(dataset, split)
    eps = torch.tensor(1e-10, dtype=DTYPE, device=DEVICE)

    if len(values['total_time']):
        state_total = model(values['total_time'], dataset.schedule_name)
        predicted_total = state_total.sum(dim=1, keepdim=True)
        # Unconstrained baselines can temporarily predict a non-positive total.
        # A lower clamp keeps the logarithmic likelihood finite; feasibility is
        # still measured separately and the soft baseline receives explicit
        # negativity/capacity penalties. The proposed hard model never uses it.
        predicted_total_for_log = torch.clamp(predicted_total, min=float(eps))
        total_loss = torch.mean((torch.log(predicted_total_for_log)
                                 - torch.log(values['total_observed']+eps))**2)
    else:
        total_loss = torch.zeros((), dtype=DTYPE, device=DEVICE)

    if len(values['fraction_time']):
        state_fraction = model(values['fraction_time'], dataset.schedule_name)
        denominator = torch.clamp(state_fraction.sum(dim=1, keepdim=True), min=float(eps))
        predicted_fraction = state_fraction[:,1:2] / denominator
        # Clipping is used only in the observation operator for invalid baseline
        # states. Raw S/R predictions remain unchanged for constraint auditing.
        predicted_fraction = torch.clamp(predicted_fraction, min=0.0, max=1.0)
        fraction_loss = torch.mean((predicted_fraction-values['fraction_observed'])**2)
    else:
        fraction_loss = torch.zeros((), dtype=DTYPE, device=DEVICE)
    return total_loss, fraction_loss


def regularization_loss(model: nn.Module) -> torch.Tensor:
    value = torch.zeros((), dtype=DTYPE, device=DEVICE)
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.ndim >= 2:
            value = value + torch.mean(parameter**2)
    return value


def soft_constraint_loss(model: AMRInverseModel,
                         collocation: Dict[str,torch.Tensor]) -> torch.Tensor:
    if model.model_kind != 'soft' or not collocation:
        # During data-only pretraining no collocation points are supplied, so
        # the soft-constraint term is intentionally inactive.
        return torch.zeros((), dtype=DTYPE, device=DEVICE)
    penalties = []
    for schedule_name, times in collocation.items():
        state = model(times, schedule_name)
        negative = F.relu(-state)
        capacity = F.relu(state.sum(dim=1,keepdim=True)-1.0)
        penalties.append(torch.mean(negative**2)+torch.mean(capacity**2))
    return torch.stack(penalties).mean()


def complete_loss(model: AMRInverseModel,
                  datasets: Dict[str,ObservationDataset],
                  collocation: Dict[str,torch.Tensor],
                  physics_weight: float,
                  split: str = 'train') -> Tuple[torch.Tensor,Dict[str,float]]:
    total_terms, fraction_terms, physics_terms = [], [], []
    for schedule_name, dataset in datasets.items():
        load_loss, fraction_loss = data_loss_for_dataset(model, dataset, split)
        total_terms.append(load_loss)
        fraction_terms.append(fraction_loss)
        if model.physics_enabled and schedule_name in collocation:
            residual_s, residual_r = physics_residual(model, collocation[schedule_name], schedule_name)
            physics_terms.append(torch.mean(residual_s**2)+torch.mean(residual_r**2))

    load_loss = torch.stack(total_terms).mean()
    fraction_loss = torch.stack(fraction_terms).mean()
    phys_loss = torch.stack(physics_terms).mean() if physics_terms \
                else torch.zeros((),dtype=DTYPE,device=DEVICE)
    constraint_loss = soft_constraint_loss(model, collocation)
    reg_loss = regularization_loss(model)

    total = LOSS_WEIGHTS.total_data*load_loss \
            + LOSS_WEIGHTS.fraction_data*fraction_loss \
            + physics_weight*LOSS_WEIGHTS.physics*phys_loss \
            + LOSS_WEIGHTS.soft_constraint*constraint_loss \
            + LOSS_WEIGHTS.weight_decay*reg_loss
    components = {
        'total': float(total.detach().cpu()),
        'load': float(load_loss.detach().cpu()),
        'fraction': float(fraction_loss.detach().cpu()),
        'physics': float(phys_loss.detach().cpu()),
        'constraint': float(constraint_loss.detach().cpu()),
        'regularization': float(reg_loss.detach().cpu()),
    }
    return total, components

In [ ]:
# ROUND 3 — CELL 11
# Three-stage optimizer, early stopping, checkpointing, and training history
# =============================================================================
# Stage I  : data pretraining.
# Stage II : joint AdamW optimization with periodically resampled collocation.
# Stage III: full-batch L-BFGS refinement using fixed collocation points.
#
# Memory policy for 8 GB GPU
# --------------------------
# Only one model is trained at a time. The best state_dict is moved to CPU, and
# CUDA cache is cleared after each experiment. No mixed precision is used because
# float64 derivatives and very small mutation probabilities are more stable.

@dataclass
class TrainingResult:
    label: str
    model: AMRInverseModel
    history: pd.DataFrame
    elapsed_seconds: float
    best_validation_score: float


def make_collocation_bundle(datasets: Dict[str,ObservationDataset], seed: int,
                            points_per_interval: Optional[int] = None) -> Dict[str,torch.Tensor]:
    count = points_per_interval or COLLOCATION_POINTS_PER_INTERVAL
    return {
        name: sample_interface_aware_collocation(SCHEDULES_R3[name], count,
                                                 seed+1009*index)
        for index, name in enumerate(datasets)
    }


def validation_score(model: AMRInverseModel,
                     datasets: Dict[str,ObservationDataset],
                     seed: int) -> float:
    collocation = make_collocation_bundle(datasets, seed,
                                          max(32, COLLOCATION_POINTS_PER_INTERVAL//4))
    _, components = complete_loss(model, datasets, collocation,
                                  physics_weight=1.0 if model.physics_enabled else 0.0,
                                  split='validation')
    # When a validation split has no phenotype point, fraction loss is zero.
    return components['load'] + components['fraction'] + 0.1*components['physics']


def train_inverse_model(model: AMRInverseModel,
                        datasets: Dict[str,ObservationDataset],
                        label: str,
                        seed: int = SEED) -> TrainingResult:
    model = model.to(device=DEVICE, dtype=DTYPE)
    start_time = time.time()
    history_rows: List[Dict[str,Any]] = []
    best_state = copy.deepcopy(model.state_dict())
    best_score = np.inf
    stale_epochs = 0

    # ----------------------------- Stage I -----------------------------
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.0)
    empty_collocation: Dict[str,torch.Tensor] = {}
    for epoch in range(PRETRAIN_EPOCHS):
        optimizer.zero_grad(set_to_none=True)
        total, components = complete_loss(model, datasets, empty_collocation,
                                          physics_weight=0.0, split='train')
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        if epoch % LOG_EVERY == 0 or epoch == PRETRAIN_EPOCHS-1:
            score = validation_score(model, datasets, seed+epoch)
            history_rows.append({'stage':'pretrain','epoch':epoch,
                                 'validation_score':score, **components,
                                 **model.inferred_parameters()})
            if score < best_score:
                best_score = score
                best_state = copy.deepcopy(model.state_dict())

    # ----------------------------- Stage II ----------------------------
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=max(50, ADAM_EPOCHS//20),
        min_lr=1e-5,
    )
    for epoch in range(ADAM_EPOCHS):
        collocation = make_collocation_bundle(datasets, seed+10_000+epoch)
        optimizer.zero_grad(set_to_none=True)
        total, components = complete_loss(model, datasets, collocation,
                                          physics_weight=1.0, split='train')
        if not torch.isfinite(total):
            raise FloatingPointError(f'Non-finite loss in {label} at Adam epoch {epoch}.')
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == ADAM_EPOCHS-1:
            score = validation_score(model, datasets, seed+20_000+epoch)
            scheduler.step(score)
            row = {'stage':'adam','epoch':epoch,
                   'validation_score':score,
                   'learning_rate':optimizer.param_groups[0]['lr'],
                   **components, **model.inferred_parameters()}
            history_rows.append(row)
            if score < best_score*(1-1e-6):
                best_score = score
                best_state = copy.deepcopy(model.state_dict())
                stale_epochs = 0
            else:
                stale_epochs += LOG_EVERY
            if epoch % max(LOG_EVERY, ADAM_EPOCHS//10) == 0:
                print(f'{label:28s} | Adam {epoch:6d}/{ADAM_EPOCHS} | '
                      f'loss={components["total"]:.3e} | val={score:.3e} | '
                      f'mu={model.inferred_parameters()["mutation_probability"]:.3e} | '
                      f'c={model.inferred_parameters()["resistance_fitness_cost"]:.4f}')
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                print(f'{label}: Adam early stopping at epoch {epoch}.')
                break

    model.load_state_dict(best_state)

    # ----------------------------- Stage III ---------------------------
    fixed_collocation = make_collocation_bundle(datasets, seed+90_000,
                                                 COLLOCATION_POINTS_PER_INTERVAL)
    if LBFGS_MAX_ITER > 0:
        optimizer_lbfgs = torch.optim.LBFGS(
            model.parameters(), lr=0.8, max_iter=LBFGS_MAX_ITER,
            max_eval=max(LBFGS_MAX_ITER*2, 10), tolerance_grad=1e-10,
            tolerance_change=1e-12, history_size=50, line_search_fn='strong_wolfe',
        )
        closure_count = {'value':0}
        def closure():
            optimizer_lbfgs.zero_grad(set_to_none=True)
            total, _ = complete_loss(model, datasets, fixed_collocation,
                                     physics_weight=1.0 if model.physics_enabled else 0.0,
                                     split='train')
            total.backward()
            closure_count['value'] += 1
            return total
        optimizer_lbfgs.step(closure)
        total, components = complete_loss(model, datasets, fixed_collocation,
                                          physics_weight=1.0 if model.physics_enabled else 0.0,
                                          split='train')
        score = validation_score(model, datasets, seed+99_000)
        history_rows.append({'stage':'lbfgs','epoch':closure_count['value'],
                             'validation_score':score, **components,
                             **model.inferred_parameters()})
        if score < best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    elapsed = time.time()-start_time
    history = pd.DataFrame(history_rows)
    history.to_csv(HISTORY_DIR/f'{label}_history.csv', index=False)
    torch.save({'model_state': model.state_dict(),
                'model_kind': model.model_kind,
                'schedules': list(datasets),
                'inferred_parameters': model.inferred_parameters(),
                'training_budget': {
                    'quick_mode': QUICK_MODE,
                    'pretrain_epochs': PRETRAIN_EPOCHS,
                    'adam_epochs': ADAM_EPOCHS,
                    'lbfgs_max_iter': LBFGS_MAX_ITER,
                    'collocation_points_per_interval': COLLOCATION_POINTS_PER_INTERVAL,
                }}, MODEL_DIR/f'{label}.pt')
    print(f'{label} completed in {elapsed/60:.2f} min; best validation score={best_score:.3e}.')
    return TrainingResult(label, model, history, elapsed, best_score)

In [ ]:
# ROUND 3 — CELL 12
# Dense evaluation, state/parameter metrics, constraint audit, and inference time
# =============================================================================


def predict_dense(model: AMRInverseModel, schedule_name: str,
                  dense: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    dense = DENSE_REFERENCE[schedule_name] if dense is None else dense
    t_values = dense['time'].to_numpy(float)
    t_tensor = tensor(t_values)
    model.eval()
    with torch.no_grad():
        prediction = model(t_tensor, schedule_name).detach().cpu().numpy()
    S = TRUE_PARAMETERS.carrying_capacity*prediction[:,0]
    R = TRUE_PARAMETERS.carrying_capacity*prediction[:,1]
    B = S+R
    q = np.divide(R,B,out=np.zeros_like(R),where=B>0)
    frame = dense.copy()
    frame['predicted_sensitive'] = S
    frame['predicted_resistant'] = R
    frame['predicted_total'] = B
    frame['predicted_resistant_fraction'] = q
    return frame


def relative_l2(predicted: np.ndarray, true: np.ndarray) -> float:
    return float(np.linalg.norm(predicted-true)/(np.linalg.norm(true)+1e-12))


def residual_rmse_dense(model: AMRInverseModel, schedule_name: str) -> float:
    schedule = SCHEDULES_R3[schedule_name]
    points = []
    for left,right,_ in schedule.intervals(T_START,T_FINAL):
        points.append(np.linspace(left+1e-5,right-1e-5,150))
    t_res = tensor(np.concatenate(points), requires_grad=True)
    if not model.physics_enabled:
        return np.nan
    residual_s,residual_r = physics_residual(model,t_res,schedule_name)
    value = torch.sqrt(torch.mean(residual_s**2+residual_r**2))
    return float(value.detach().cpu())


def inference_milliseconds(model: AMRInverseModel, schedule_name: str,
                           n_points: int = 1001, repeats: int = 20) -> float:
    t_eval = tensor(np.linspace(T_START,T_FINAL,n_points))
    with torch.no_grad():
        _ = model(t_eval,schedule_name)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        start=time.perf_counter()
        for _ in range(repeats):
            _=model(t_eval,schedule_name)
        if torch.cuda.is_available(): torch.cuda.synchronize()
    return 1000*(time.perf_counter()-start)/repeats


def evaluate_trained_model(result: TrainingResult,
                           schedule_name: str) -> Tuple[Dict[str,Any],pd.DataFrame]:
    frame = predict_dense(result.model,schedule_name)
    metrics: Dict[str,Any] = {
        'label':result.label, 'model_kind':result.model.model_kind,
        'schedule':schedule_name, 'training_seconds':result.elapsed_seconds,
        'best_validation_score':result.best_validation_score,
    }
    pairs = {
        'sensitive':('sensitive','predicted_sensitive'),
        'resistant':('resistant','predicted_resistant'),
        'total':('total','predicted_total'),
        'fraction':('resistant_fraction','predicted_resistant_fraction'),
    }
    for name,(true_col,pred_col) in pairs.items():
        true=frame[true_col].to_numpy(float); pred=frame[pred_col].to_numpy(float)
        metrics[f'rmse_{name}']=float(np.sqrt(mean_squared_error(true,pred)))
        metrics[f'mae_{name}']=float(mean_absolute_error(true,pred))
        metrics[f'r2_{name}']=float(r2_score(true,pred))
        metrics[f'relative_l2_{name}']=relative_l2(pred,true)

    S=frame['predicted_sensitive'].to_numpy(float)
    R=frame['predicted_resistant'].to_numpy(float)
    metrics.update({
        'maximum_negative_violation':float(max(0,-S.min(),-R.min())),
        'maximum_capacity_violation':float(max(0,(S+R).max()-TRUE_PARAMETERS.carrying_capacity)),
        'fraction_points_negative':float(np.mean((S<0)|(R<0))),
        'fraction_points_over_capacity':float(np.mean(S+R>TRUE_PARAMETERS.carrying_capacity+1e-8)),
        'physics_residual_rmse':residual_rmse_dense(result.model,schedule_name),
        'inference_ms_1001_points':inference_milliseconds(result.model,schedule_name),
        **result.model.inferred_parameters(),
    })
    if result.model.physics_enabled:
        metrics['relative_error_mu']=abs(metrics['mutation_probability']-TRUE_PARAMETERS.mutation_probability)/TRUE_PARAMETERS.mutation_probability
        metrics['relative_error_cost']=abs(metrics['resistance_fitness_cost']-TRUE_PARAMETERS.resistance_fitness_cost)/TRUE_PARAMETERS.resistance_fitness_cost
    else:
        metrics['relative_error_mu']=np.nan; metrics['relative_error_cost']=np.nan
    frame.to_csv(PREDICTION_DIR/f'{result.label}_{schedule_name}_predictions.csv',index=False)
    return metrics,frame

In [ ]:
# ROUND 3 — CELL 13
# Classical ODE nonlinear least-squares baseline with multiple starting points
# =============================================================================
# This baseline asks how well conventional parameter fitting performs when the
# states are generated by numerical integration rather than a neural network.
# It is essential because a PINN should not be compared only with other PINNs.


def ode_prediction_for_parameters(schedule_name: str, times: np.ndarray,
                                  log10_mu: float, cost: float) -> pd.DataFrame:
    parameters = replace(TRUE_PARAMETERS,
                         mutation_probability=10.0**log10_mu,
                         resistance_fitness_cost=float(cost))
    return solve_reference_r3(parameters,SCHEDULES_R3[schedule_name],times)


def ode_residual_vector(x: np.ndarray,
                       datasets: Dict[str,ObservationDataset]) -> np.ndarray:
    log10_mu,cost=x
    residuals=[]
    for schedule_name,dataset in datasets.items():
        all_times=np.unique(np.concatenate([dataset.total_time,dataset.fraction_time]))
        pred=ode_prediction_for_parameters(schedule_name,all_times,log10_mu,cost)
        pred_total=nearest_values(pred,dataset.total_time,'total')
        residuals.extend(np.log(pred_total+1e-8)-np.log(dataset.total_observed+1e-8))
        if len(dataset.fraction_time):
            pred_fraction=nearest_values(pred,dataset.fraction_time,'resistant_fraction')
            residuals.extend(2.0*(pred_fraction-dataset.fraction_observed))
    return np.asarray(residuals,float)


def fit_ode_baseline(datasets: Dict[str,ObservationDataset],
                     label: str, seed: int=SEED,
                     starts: int=12) -> Dict[str,Any]:
    rng=np.random.default_rng(seed)
    initial_points=[np.array([-4.0,0.10])]
    initial_points += [np.array([rng.uniform(-5.8,-2.2),rng.uniform(0.005,0.38)])
                       for _ in range(starts-1)]
    best=None
    start_time=time.time()
    for initial in initial_points:
        fit=least_squares(lambda x: ode_residual_vector(x,datasets),initial,
                          bounds=([-6.0,0.0],[-2.0,0.40]),
                          xtol=1e-11,ftol=1e-11,gtol=1e-11,max_nfev=400)
        if best is None or np.sum(fit.fun**2)<np.sum(best.fun**2):
            best=fit
    elapsed=time.time()-start_time
    result={
        'label':label,'model_kind':'ode_fit',
        'mutation_probability':10.0**best.x[0],
        'resistance_fitness_cost':best.x[1],
        'objective':float(np.sum(best.fun**2)),
        'nfev':int(best.nfev),'training_seconds':elapsed,
        'relative_error_mu':abs(10.0**best.x[0]-TRUE_PARAMETERS.mutation_probability)/TRUE_PARAMETERS.mutation_probability,
        'relative_error_cost':abs(best.x[1]-TRUE_PARAMETERS.resistance_fitness_cost)/TRUE_PARAMETERS.resistance_fitness_cost,
    }
    with open(REPORT_DIR_R3/f'{label}_ode_fit.json','w') as file:
        json.dump(result,file,indent=2)
    return result

print('ODE baseline functions are ready.')

In [ ]:
# ROUND 3 — CELL 14
# Single proposed-model demonstration: early treatment, O3, noise-free data
# =============================================================================
# This is the first cell to perform neural training. It should be run before the
# larger experiment suites. In QUICK_MODE it verifies the complete pipeline. In
# final mode it produces the principal proposed-model reconstruction.

PRIMARY_DATASET = build_observation_dataset('early_continuous','O3',0.0,SEED)
PRIMARY_BUNDLE = {'early_continuous':PRIMARY_DATASET}
PRIMARY_HARD_RESULT = None
PRIMARY_HARD_METRICS = None
PRIMARY_HARD_PREDICTION = None

if RUN_SINGLE_MODEL_DEMO:
    primary_model = AMRInverseModel(
        {'early_continuous':SCHEDULES_R3['early_continuous']},
        model_kind='hard', width=64, depth=4,
        parameter_initialization=(3e-4,0.15),
    )
    PRIMARY_HARD_RESULT = train_inverse_model(primary_model,PRIMARY_BUNDLE,
                                               'round3_primary_hard_O3',SEED)
    PRIMARY_HARD_METRICS,PRIMARY_HARD_PREDICTION = evaluate_trained_model(
        PRIMARY_HARD_RESULT,'early_continuous')
    pd.DataFrame([PRIMARY_HARD_METRICS]).to_csv(
        TABLE_DIR_R3/'round3_primary_hard_metrics.csv',index=False)
    display(pd.DataFrame([PRIMARY_HARD_METRICS]).T.rename(columns={0:'value'}))
else:
    print('RUN_SINGLE_MODEL_DEMO=False; no model was trained in this cell.')

In [ ]:
# ROUND 3 — CELL 15
# Publication figure: proposed-model reconstruction, errors, and parameters
# =============================================================================

if PRIMARY_HARD_RESULT is not None:
    frame=PRIMARY_HARD_PREDICTION
    obs=PRIMARY_DATASET
    fig,axes=plt.subplots(2,2,figsize=(11.4,7.6),constrained_layout=True)

    LEGEND_STYLE = dict(fontsize=9, framealpha=0.95, borderaxespad=0.6)

    # --- A: peaks mid-left, empty in the upper-right as the curves decay ---
    ax=axes[0,0]
    ax.plot(frame.time,frame.sensitive,label='Reference sensitive')
    ax.plot(frame.time,frame.predicted_sensitive,linestyle='--',label='PINN sensitive')
    ax.plot(frame.time,frame.resistant,label='Reference resistant')
    ax.plot(frame.time,frame.predicted_resistant,linestyle='--',label='PINN resistant')
    ax.axvline(5,linestyle=':',linewidth=1.5)
    ax.set(xlabel='Time',ylabel='Bacterial abundance',title='A. Hidden-state reconstruction')
    ax.set_ylim(top=ax.get_ylim()[1]*1.18)  # headroom so legend clears the peak
    ax.grid(alpha=.22); ax.legend(loc='upper right', ncol=2, **LEGEND_STYLE)

    # --- B: same shape as A, empty upper-right ---
    ax=axes[0,1]
    ax.plot(frame.time,frame.total,label='Reference total')
    ax.plot(frame.time,frame.predicted_total,linestyle='--',label='PINN total')
    ax.scatter(obs.total_time[obs.total_train_mask],obs.total_observed[obs.total_train_mask],s=30,label='Training data')
    ax.scatter(obs.total_time[obs.total_validation_mask],obs.total_observed[obs.total_validation_mask],s=45,marker='D',label='Validation data')
    ax.set(xlabel='Time',ylabel='Total bacterial load',title='B. Aggregated-load fit')
    ax.set_ylim(top=ax.get_ylim()[1]*1.18)
    ax.grid(alpha=.22); ax.legend(loc='upper right', **LEGEND_STYLE)

    # --- C: fraction stays near 0 for most of the range, top half is empty ---
    ax=axes[1,0]
    ax.plot(frame.time,frame.resistant_fraction,label='Reference fraction')
    ax.plot(frame.time,frame.predicted_resistant_fraction,linestyle='--',label='PINN fraction')
    ax.scatter(obs.fraction_time[obs.fraction_train_mask],obs.fraction_observed[obs.fraction_train_mask],s=45,label='Training phenotype')
    if obs.fraction_validation_mask.any():
        ax.scatter(obs.fraction_time[obs.fraction_validation_mask],obs.fraction_observed[obs.fraction_validation_mask],s=55,marker='D',label='Validation phenotype')
    ax.set(xlabel='Time',ylabel='Resistant fraction',ylim=(-.03,1.03),title='C. Latent phenotype composition')
    ax.grid(alpha=.22); ax.legend(loc='upper left', **LEGEND_STYLE)

    # --- D: log scale — push ylim way up so legend/text sit in headroom the curves never reach ---
    ax=axes[1,1]
    error_s=np.abs(frame.predicted_sensitive-frame.sensitive)
    error_r=np.abs(frame.predicted_resistant-frame.resistant)
    ax.semilogy(frame.time,error_s+1e-10,label='$|S_\theta-S^*|$')
    ax.semilogy(frame.time,error_r+1e-10,label='$|R_\theta-R^*|$')
    ax.set(xlabel='Time',ylabel='Absolute error',title='D. Pointwise reconstruction error')
    ax.set_ylim(1e-10, 1e4)  # headroom above the curves (which top out ~1e1)
    ax.grid(alpha=.22,which='both')
    ax.legend(loc='upper left', **LEGEND_STYLE)
    text=(f"$\\hat{{\\mu}}$={PRIMARY_HARD_METRICS['mutation_probability']:.2e}\n"
          f"$\\hat{{c}}_R$={PRIMARY_HARD_METRICS['resistance_fitness_cost']:.3f}\n"
          f"$V_K$={PRIMARY_HARD_METRICS['maximum_capacity_violation']:.1e}")
    ax.text(.98,.98,text,transform=ax.transAxes,ha='right',va='top',fontsize=9,
            bbox=dict(boxstyle='round,pad=.4',facecolor='white',alpha=.95))

    save_figure_r3(fig,'round3_primary_hard_reconstruction')
    plt.show()
else:
    print('Run Cell 14 with RUN_SINGLE_MODEL_DEMO=True before plotting.')

In [ ]:
print("RUN_PRIMARY_ABLATION:", RUN_PRIMARY_ABLATION)
print("RUN_SINGLE_MODEL_DEMO:", RUN_SINGLE_MODEL_DEMO)
print("PRIMARY_DATASET exists:", "PRIMARY_DATASET" in globals())
print("PRIMARY_BUNDLE exists:", "PRIMARY_BUNDLE" in globals())
print("ABLATION_METRICS exists:", "ABLATION_METRICS" in globals())
print("ABLATION_PREDICTIONS exists:", "ABLATION_PREDICTIONS" in globals())

In [ ]:
# =============================================================================
# TEMPORARY SWITCH OVERRIDE FOR CORRECTED CELL 16
# =============================================================================

RUN_SINGLE_MODEL_DEMO = False
RUN_PRIMARY_ABLATION = True
RUN_OBSERVATION_REGIME_STUDY = False
RUN_NOISE_ROBUSTNESS_STUDY = False
RUN_MULTI_SCHEDULE_STUDY = False
RUN_PROFILE_LIKELIHOOD = False
RUN_STOCHASTIC_BENCHMARK = False
RUN_STOCHASTIC_TRANSFER_VALIDATION = False

CURRENT_EXPERIMENT = "primary_ablation"

EXPERIMENT_SWITCHES = {
    "single_model_demo": RUN_SINGLE_MODEL_DEMO,
    "primary_ablation": RUN_PRIMARY_ABLATION,
    "observation_regime_study": RUN_OBSERVATION_REGIME_STUDY,
    "noise_robustness_study": RUN_NOISE_ROBUSTNESS_STUDY,
    "multi_schedule_study": RUN_MULTI_SCHEDULE_STUDY,
    "profile_likelihood": RUN_PROFILE_LIKELIHOOD,
    "stochastic_benchmark": RUN_STOCHASTIC_BENCHMARK,
    "stochastic_transfer_validation": (
        RUN_STOCHASTIC_TRANSFER_VALIDATION
    ),
}

active_experiments = [
    name
    for name, enabled in EXPERIMENT_SWITCHES.items()
    if enabled
]

print("QUICK_MODE:", QUICK_MODE)
print("RUN_PRIMARY_ABLATION:", RUN_PRIMARY_ABLATION)
print(
    "RUN_OBSERVATION_REGIME_STUDY:",
    RUN_OBSERVATION_REGIME_STUDY,
)
print("Active experiments:", active_experiments)
print("PRIMARY_DATASET exists:", "PRIMARY_DATASET" in globals())
print("PRIMARY_BUNDLE exists:", "PRIMARY_BUNDLE" in globals())

assert QUICK_MODE is False
assert RUN_PRIMARY_ABLATION is True
assert RUN_OBSERVATION_REGIME_STUDY is False
assert active_experiments == ["primary_ablation"]
assert "PRIMARY_DATASET" in globals()
assert "PRIMARY_BUNDLE" in globals()

print(
    "\nTemporary primary-ablation configuration is active. "
    "You may now run corrected Cell 16."
)

In [ ]:
# =============================================================================
# ROUND 3 — CELL 16 (CORRECTED FAIR-COMPARISON VERSION)
# Architecture ablation: six neural models + matched ODE benchmark
# =============================================================================
# Corrections:
# 1. ODE fitting uses only the same TRAINING observations used by the PINNs.
# 2. Held-out validation observations are not used during ODE optimization.
# 3. Dense hidden S/R trajectories are used only after fitting for evaluation.
# 4. The ODE optimizer does not start from the exact true parameter pair.
# 5. ODE physics-residual RMSE is reported as N/A rather than zero.
# 6. A common held-out observation score is calculated for every method.
# 7. Existing neural results can be reused, avoiding several hours of retraining.
#
# Current recommendation:
# Keep REUSE_EXISTING_NEURAL_RESULTS=True because the six neural models have
# already been trained successfully. This cell will then refit only the fair ODE
# benchmark and rebuild the comparison tables.
# =============================================================================

from datetime import datetime
from pathlib import Path
import copy
import json
import math
import shutil
import time

import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


REUSE_EXISTING_NEURAL_RESULTS = True
SCHEDULE_NAME = "early_continuous"
MODEL_KINDS = ["data_only", "vanilla", "soft", "positive", "simplex", "hard"]


# -----------------------------------------------------------------------------
# Preconditions
# -----------------------------------------------------------------------------
required_objects = [
    "PRIMARY_BUNDLE", "PRIMARY_DATASET", "TRUE_PARAMETERS", "SCHEDULES_R3",
    "DENSE_REFERENCE", "LOSS_WEIGHTS", "AMRInverseModel", "train_inverse_model",
    "evaluate_trained_model", "solve_reference_r3", "nearest_values",
    "relative_l2", "clear_gpu_memory",
]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(f"Run Round 3 Cells 1–14 first. Missing: {missing}")


# -----------------------------------------------------------------------------
# Common held-out observation score for all methods
# -----------------------------------------------------------------------------
def common_validation_metrics(frame, dataset):
    """Evaluate every method on the same held-out observations only."""
    total_mask = np.asarray(dataset.total_validation_mask, dtype=bool)
    frac_mask = np.asarray(dataset.fraction_validation_mask, dtype=bool)

    if total_mask.any():
        times = dataset.total_time[total_mask]
        observed = dataset.total_observed[total_mask]
        predicted = nearest_values(frame, times, "predicted_total")
        total_log_mse = float(np.mean(
            (np.log(np.clip(predicted, 1e-10, None))
             - np.log(np.clip(observed, 1e-10, None)))**2
        ))
    else:
        total_log_mse = 0.0

    if frac_mask.any():
        times = dataset.fraction_time[frac_mask]
        observed = dataset.fraction_observed[frac_mask]
        predicted = nearest_values(frame, times, "predicted_resistant_fraction")
        fraction_mse = float(np.mean(
            (np.clip(predicted, 0.0, 1.0) - observed)**2
        ))
    else:
        fraction_mse = 0.0

    return {
        "heldout_total_log_mse": total_log_mse,
        "heldout_fraction_mse": fraction_mse,
        "heldout_observation_score": total_log_mse + fraction_mse,
    }


# -----------------------------------------------------------------------------
# Fair ODE objective: use TRAINING masks only
# -----------------------------------------------------------------------------
def fair_ode_residual_vector(x, datasets):
    log10_mu, cost = x
    blocks = []

    for schedule_name, dataset in datasets.items():
        total_mask = np.asarray(dataset.total_train_mask, dtype=bool)
        frac_mask = np.asarray(dataset.fraction_train_mask, dtype=bool)

        total_times = dataset.total_time[total_mask]
        total_obs = dataset.total_observed[total_mask]
        frac_times = dataset.fraction_time[frac_mask]
        frac_obs = dataset.fraction_observed[frac_mask]

        all_times = np.unique(np.concatenate([total_times, frac_times]))
        parameters = replace(
            TRUE_PARAMETERS,
            mutation_probability=10.0**float(log10_mu),
            resistance_fitness_cost=float(cost),
        )
        pred = solve_reference_r3(
            parameters,
            SCHEDULES_R3[schedule_name],
            all_times,
        )

        if total_times.size:
            pred_total = nearest_values(pred, total_times, "total")
            residual = (
                np.log(np.clip(pred_total, 1e-10, None))
                - np.log(np.clip(total_obs, 1e-10, None))
            )
            scale = math.sqrt(
                float(LOSS_WEIGHTS.total_data) / max(len(residual), 1)
            )
            blocks.append(scale * residual)

        if frac_times.size:
            pred_fraction = nearest_values(
                pred, frac_times, "resistant_fraction"
            )
            residual = np.clip(pred_fraction, 0.0, 1.0) - frac_obs
            scale = math.sqrt(
                float(LOSS_WEIGHTS.fraction_data) / max(len(residual), 1)
            )
            blocks.append(scale * residual)

    return np.concatenate(blocks).astype(float)


def fit_fair_ode_baseline(datasets, label, seed, starts):
    """
    Fit the ODE from the same non-oracle initialization used by the PINNs,
    followed by random multi-starts. The exact true pair is not inserted.
    """
    rng = np.random.default_rng(seed)
    lower = np.array([-6.0, 0.0])
    upper = np.array([-2.0, 0.40])

    initial_points = [np.array([np.log10(3e-4), 0.15])]
    initial_points += [
        np.array([rng.uniform(-5.8, -2.2), rng.uniform(0.005, 0.38)])
        for _ in range(starts - 1)
    ]

    true_pair = np.array([
        np.log10(TRUE_PARAMETERS.mutation_probability),
        TRUE_PARAMETERS.resistance_fitness_cost,
    ])
    assert not np.allclose(initial_points[0], true_pair, atol=1e-14, rtol=0.0)

    best = None
    audit = []
    start_time = time.time()

    for start_index, initial in enumerate(initial_points):
        fit = least_squares(
            lambda z: fair_ode_residual_vector(z, datasets),
            x0=initial,
            bounds=(lower, upper),
            xtol=1e-11,
            ftol=1e-11,
            gtol=1e-11,
            max_nfev=500 if QUICK_MODE else 1500,
        )
        objective = float(np.sum(fit.fun**2))
        audit.append({
            "start_index": start_index,
            "initial_mu": float(10.0**initial[0]),
            "initial_cost": float(initial[1]),
            "fitted_mu": float(10.0**fit.x[0]),
            "fitted_cost": float(fit.x[1]),
            "objective": objective,
            "nfev": int(fit.nfev),
            "success": bool(fit.success),
        })
        if best is None or objective < float(np.sum(best.fun**2)):
            best = fit

    elapsed = time.time() - start_time
    result = {
        "label": label,
        "model_kind": "ode_fit",
        "method_class": "matched_mechanistic_benchmark",
        "schedule": SCHEDULE_NAME,
        "training_seconds": float(elapsed),
        "best_validation_score": np.nan,
        "mutation_probability": float(10.0**best.x[0]),
        "resistance_fitness_cost": float(best.x[1]),
        "relative_error_mu": float(
            abs(10.0**best.x[0] - TRUE_PARAMETERS.mutation_probability)
            / TRUE_PARAMETERS.mutation_probability
        ),
        "relative_error_cost": float(
            abs(best.x[1] - TRUE_PARAMETERS.resistance_fitness_cost)
            / TRUE_PARAMETERS.resistance_fitness_cost
        ),
        "objective": float(np.sum(best.fun**2)),
        "nfev": int(best.nfev),
        "fit_scope": "training observations only",
        "hidden_states_used_for_fit": False,
        "exact_true_initialization_used": False,
        "multistart_count": int(len(initial_points)),
    }
    return result, pd.DataFrame(audit)


# -----------------------------------------------------------------------------
# Dense post-fit evaluation of the ODE benchmark
# -----------------------------------------------------------------------------
def evaluate_fair_ode(ode_metrics):
    fitted = replace(
        TRUE_PARAMETERS,
        mutation_probability=ode_metrics["mutation_probability"],
        resistance_fitness_cost=ode_metrics["resistance_fitness_cost"],
    )

    true = DENSE_REFERENCE[SCHEDULE_NAME].copy()
    times = true.time.to_numpy(float)
    pred = solve_reference_r3(fitted, SCHEDULES_R3[SCHEDULE_NAME], times)

    frame = true.copy()
    frame["predicted_sensitive"] = pred.sensitive.to_numpy(float)
    frame["predicted_resistant"] = pred.resistant.to_numpy(float)
    frame["predicted_total"] = pred.total.to_numpy(float)
    frame["predicted_resistant_fraction"] = (
        pred.resistant_fraction.to_numpy(float)
    )

    pairs = {
        "sensitive": ("sensitive", "predicted_sensitive"),
        "resistant": ("resistant", "predicted_resistant"),
        "total": ("total", "predicted_total"),
        "fraction": ("resistant_fraction", "predicted_resistant_fraction"),
    }
    for name, (true_col, pred_col) in pairs.items():
        y_true = frame[true_col].to_numpy(float)
        y_pred = frame[pred_col].to_numpy(float)
        ode_metrics[f"rmse_{name}"] = float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        )
        ode_metrics[f"mae_{name}"] = float(
            mean_absolute_error(y_true, y_pred)
        )
        ode_metrics[f"r2_{name}"] = float(r2_score(y_true, y_pred))
        ode_metrics[f"relative_l2_{name}"] = relative_l2(y_pred, y_true)

    S = frame.predicted_sensitive.to_numpy(float)
    R = frame.predicted_resistant.to_numpy(float)
    total = S + R
    K = float(TRUE_PARAMETERS.carrying_capacity)

    ode_metrics.update({
        "maximum_negative_violation": float(max(0.0, -S.min(), -R.min())),
        "maximum_capacity_violation": float(max(0.0, total.max() - K)),
        "fraction_points_negative": float(np.mean((S < 0) | (R < 0))),
        "fraction_points_over_capacity": float(np.mean(total > K + 1e-8)),
        # N/A: the neural autograd residual metric is not directly comparable
        # with a trajectory obtained by direct numerical integration.
        "physics_residual_rmse": np.nan,
        "constraint_interpretation": (
            "empirical mechanistic-trajectory audit; not a neural guarantee"
        ),
    })
    ode_metrics.update(common_validation_metrics(frame, PRIMARY_DATASET))

    # Comparable prediction time at 1001 requested points
    eval_times = np.linspace(T_START, T_FINAL, 1001)
    repeats = 5 if QUICK_MODE else 10
    _ = solve_reference_r3(fitted, SCHEDULES_R3[SCHEDULE_NAME], eval_times)
    start = time.perf_counter()
    for _ in range(repeats):
        _ = solve_reference_r3(
            fitted, SCHEDULES_R3[SCHEDULE_NAME], eval_times
        )
    ode_metrics["inference_ms_1001_points"] = float(
        1000.0 * (time.perf_counter() - start) / repeats
    )
    return ode_metrics, frame


# =============================================================================
# Execute the corrected ablation
# =============================================================================
if RUN_PRIMARY_ABLATION:

    previous_metrics = (
        ABLATION_METRICS.copy()
        if "ABLATION_METRICS" in globals()
        and isinstance(ABLATION_METRICS, pd.DataFrame)
        else None
    )
    previous_predictions = (
        copy.copy(ABLATION_PREDICTIONS)
        if "ABLATION_PREDICTIONS" in globals()
        and isinstance(ABLATION_PREDICTIONS, dict)
        else {}
    )

    ABLATION_RESULTS = {}
    ABLATION_METRICS_ROWS = []
    ABLATION_PREDICTIONS = {}

    combined_path = (
        TABLE_DIR_R3 / "round3_architecture_ablation_metrics.csv"
    )
    neural_path = (
        TABLE_DIR_R3 / "round3_architecture_ablation_neural_metrics.csv"
    )
    ode_path = (
        TABLE_DIR_R3 / "round3_matched_ode_benchmark_metrics.csv"
    )
    multistart_path = (
        TABLE_DIR_R3 / "round3_matched_ode_multistart_audit.csv"
    )

    # -------------------------------------------------------------------------
    # Reuse or retrain neural models
    # -------------------------------------------------------------------------
    candidate = previous_metrics
    if candidate is None and combined_path.exists():
        candidate = pd.read_csv(combined_path)

    can_reuse = False
    if REUSE_EXISTING_NEURAL_RESULTS and candidate is not None:
        candidate = candidate[candidate.model_kind.isin(MODEL_KINDS)].copy()
        if candidate.model_kind.nunique() == len(MODEL_KINDS):
            can_reuse = True
            for kind in MODEL_KINDS:
                if kind in previous_predictions:
                    pred = previous_predictions[kind].copy()
                else:
                    pred_path = PREDICTION_DIR / (
                        f"round3_ablation_{kind}_O3_"
                        f"{SCHEDULE_NAME}_predictions.csv"
                    )
                    if not pred_path.exists():
                        can_reuse = False
                        break
                    pred = pd.read_csv(pred_path)
                ABLATION_PREDICTIONS[kind] = pred

    if can_reuse:
        print(
            "Reusing completed neural results; refitting only the corrected "
            "ODE benchmark."
        )
        for _, series in candidate.iterrows():
            row = series.to_dict()
            kind = row["model_kind"]
            row["method_class"] = (
                "proposed_neural_method"
                if kind == "hard"
                else "neural_baseline"
            )
            row["fit_scope"] = "training observations only"
            row["hidden_states_used_for_fit"] = False
            row.update(
                common_validation_metrics(
                    ABLATION_PREDICTIONS[kind],
                    PRIMARY_DATASET,
                )
            )
            ABLATION_METRICS_ROWS.append(row)
    else:
        if REUSE_EXISTING_NEURAL_RESULTS:
            print(
                "Reusable neural files were incomplete; retraining all six "
                "neural models."
            )
        for index, kind in enumerate(MODEL_KINDS):
            label = f"round3_ablation_{kind}_O3"
            model = AMRInverseModel(
                {SCHEDULE_NAME: SCHEDULES_R3[SCHEDULE_NAME]},
                model_kind=kind,
                width=64,
                depth=4,
                parameter_initialization=(3e-4, 0.15),
            )
            result = train_inverse_model(
                model, PRIMARY_BUNDLE, label, SEED + 1000 * index
            )
            metrics, pred = evaluate_trained_model(
                result, SCHEDULE_NAME
            )
            metrics["method_class"] = (
                "proposed_neural_method"
                if kind == "hard"
                else "neural_baseline"
            )
            metrics["fit_scope"] = "training observations only"
            metrics["hidden_states_used_for_fit"] = False
            metrics.update(
                common_validation_metrics(pred, PRIMARY_DATASET)
            )
            ABLATION_RESULTS[kind] = result
            ABLATION_METRICS_ROWS.append(metrics)
            ABLATION_PREDICTIONS[kind] = pred
            result.model.to("cpu")
            clear_gpu_memory()

    # -------------------------------------------------------------------------
    # Corrected ODE benchmark
    # -------------------------------------------------------------------------
    ode_metrics, multistart_audit = fit_fair_ode_baseline(
        PRIMARY_BUNDLE,
        label="round3_ablation_ode_fit_fair_O3",
        seed=SEED + 880_000,
        starts=8 if QUICK_MODE else 24,
    )
    ode_metrics, ode_pred = evaluate_fair_ode(ode_metrics)

    ode_pred.to_csv(
        PREDICTION_DIR
        / "round3_ablation_ode_fit_fair_O3_predictions.csv",
        index=False,
    )
    multistart_audit.to_csv(multistart_path, index=False)
    ABLATION_PREDICTIONS["ode_fit"] = ode_pred
    ABLATION_METRICS_ROWS.append(ode_metrics)

    # -------------------------------------------------------------------------
    # Save separate and combined tables
    # -------------------------------------------------------------------------
    ABLATION_METRICS = pd.DataFrame(ABLATION_METRICS_ROWS)
    order_map = {
        "ode_fit": 0, "data_only": 1, "vanilla": 2, "soft": 3,
        "positive": 4, "simplex": 5, "hard": 6,
    }
    ABLATION_METRICS["display_order"] = (
        ABLATION_METRICS.model_kind.map(order_map)
    )
    ABLATION_METRICS = (
        ABLATION_METRICS
        .sort_values("display_order")
        .reset_index(drop=True)
    )

    if combined_path.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        shutil.copy2(
            combined_path,
            TABLE_DIR_R3
            / f"round3_architecture_ablation_metrics_before_fair_fix_{stamp}.csv",
        )

    neural_table = ABLATION_METRICS[
        ABLATION_METRICS.model_kind.isin(MODEL_KINDS)
    ].copy()
    ode_table = ABLATION_METRICS[
        ABLATION_METRICS.model_kind == "ode_fit"
    ].copy()

    ABLATION_METRICS.to_csv(combined_path, index=False)
    neural_table.to_csv(neural_path, index=False)
    ode_table.to_csv(ode_path, index=False)

    fairness_audit = {
        "ode_fit_scope": "training observations only",
        "validation_points_used_during_ode_fit": False,
        "dense_hidden_states_used_during_ode_fit": False,
        "dense_hidden_states_used_for_postfit_evaluation": True,
        "exact_true_initialization_used": False,
        "shared_non_oracle_initialization": {
            "mutation_probability": 3e-4,
            "resistance_fitness_cost": 0.15,
        },
        "ode_physics_residual_rmse": "N/A",
        "common_validation_metric": (
            "heldout total log-MSE + heldout resistant-fraction MSE"
        ),
    }
    with open(
        REPORT_DIR_R3 / "round3_ablation_fairness_audit.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(fairness_audit, file, indent=2)

    summary_columns = [
        "method_class", "model_kind",
        "rmse_sensitive", "rmse_resistant",
        "heldout_observation_score",
        "relative_error_mu", "relative_error_cost",
        "maximum_negative_violation",
        "maximum_capacity_violation",
        "physics_residual_rmse",
        "training_seconds",
    ]
    summary_columns = [
        c for c in summary_columns if c in ABLATION_METRICS.columns
    ]

    display(ABLATION_METRICS[summary_columns])

    print("\nFair-comparison Cell 16 completed successfully.")
    print("ODE fitting used training observations only.")
    print("The exact true parameter pair was not used as an initialization.")
    print("Dense hidden states were used only for post-fit evaluation.")
    print("ODE and neural results were saved separately:")
    print(" ", neural_path)
    print(" ", ode_path)
    print("Combined table:")
    print(" ", combined_path)

else:
    print(
        "Set RUN_PRIMARY_ABLATION=True in Cell 1 and rerun this cell."
    )

In [ ]:
# =============================================================================
# LOAD COMPLETED ABLATION RESULTS FOR PLOTTING
# =============================================================================
# Plotting should not depend on RUN_PRIMARY_ABLATION because the expensive
# training has already been completed and saved by Cell 16.

ABLATION_TABLE_PATH = (
    TABLE_DIR_R3
    / "round3_architecture_ablation_metrics.csv"
)

if (
    "ABLATION_METRICS" not in globals()
    or not isinstance(ABLATION_METRICS, pd.DataFrame)
    or ABLATION_METRICS.empty
):
    if ABLATION_TABLE_PATH.exists():
        ABLATION_METRICS = pd.read_csv(
            ABLATION_TABLE_PATH
        )

        print(
            "Loaded completed ablation metrics from:"
        )
        print(" ", ABLATION_TABLE_PATH)

    else:
        raise FileNotFoundError(
            "The completed Cell 16 ablation table was not found:\n"
            f"  {ABLATION_TABLE_PATH}\n"
            "Run corrected Cell 16 once before generating the figures."
        )

print(
    f"Available ablation methods: "
    f"{len(ABLATION_METRICS)}"
)

In [ ]:
# ROUND 3 — CELL 17
# FINAL PUBLICATION-READY ABLATION FIGURES
# =============================================================================
# This cell does not retrain any model.
#
# It:
#   1. Loads the completed Cell 16 ablation table when needed.
#   2. Generates a neural-only state-error heatmap.
#   3. Generates a two-panel accuracy-feasibility comparison without
#      overlapping zero-violation points.
#   4. Generates a clearer parameter-recovery figure with the matched ODE
#      benchmark visually separated from the neural methods.
#   5. Uses explicit decimal values instead of computer-style "e" notation.
# =============================================================================

from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter
from types import SimpleNamespace


# =============================================================================
# STANDALONE ROUND-3 PATH / HELPER RECOVERY
# =============================================================================
# This makes Cell 17 safe to run even after a Jupyter kernel restart.
# If the original Round-3 setup variables already exist, they are reused.
# Otherwise, the canonical project folders are reconstructed from Path.home().
# NO TRAINING IS PERFORMED HERE.

ROUND3_ROOT = Path(
    globals().get(
        "ROUND3_ROOT",
        Path.home() / "round3" / "final",
    )
)

TABLE_DIR_R3 = Path(
    globals().get(
        "TABLE_DIR_R3",
        ROUND3_ROOT / "tables",
    )
)

FIGURE_DIR_R3 = Path(
    globals().get(
        "FIGURE_DIR_R3",
        ROUND3_ROOT / "figures",
    )
)

TABLE_DIR_R3.mkdir(parents=True, exist_ok=True)
FIGURE_DIR_R3.mkdir(parents=True, exist_ok=True)

# Recover the two data-generating parameters used throughout the final
# Round-3/Round-4 manuscript experiments if the original parameter container
# is not currently present in memory.
if "TRUE_PARAMETERS" not in globals():
    TRUE_PARAMETERS = SimpleNamespace(
        mutation_probability=1.0e-4,
        resistance_fitness_cost=0.10,
    )

# Recover the figure-saving helper if the notebook setup cell has not been run
# in the current kernel.  The same basename is written in vector and raster
# formats for manuscript and presentation use.
if "save_figure_r3" not in globals():

    def save_figure_r3(fig, stem):
        stem = str(stem)

        pdf_path = FIGURE_DIR_R3 / f"{stem}.pdf"
        png_path = FIGURE_DIR_R3 / f"{stem}.png"
        svg_path = FIGURE_DIR_R3 / f"{stem}.svg"

        fig.savefig(
            pdf_path,
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            png_path,
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            svg_path,
            bbox_inches="tight",
            facecolor="white",
        )

        print("Saved figure files:")
        print(" ", pdf_path.resolve())
        print(" ", png_path.resolve())
        print(" ", svg_path.resolve())


print("Round-3 Cell 17 paths:")
print("  Tables :", TABLE_DIR_R3.resolve())
print("  Figures:", FIGURE_DIR_R3.resolve())


# =============================================================================
# LOAD COMPLETED ABLATION RESULTS
# =============================================================================

ABLATION_TABLE_PATH = (
    TABLE_DIR_R3
    / "round3_architecture_ablation_metrics.csv"
)

if (
    "ABLATION_METRICS" not in globals()
    or not isinstance(ABLATION_METRICS, pd.DataFrame)
    or ABLATION_METRICS.empty
):
    if ABLATION_TABLE_PATH.exists():
        ABLATION_METRICS = pd.read_csv(
            ABLATION_TABLE_PATH
        )

        print("Loaded completed ablation metrics from:")
        print(" ", ABLATION_TABLE_PATH)

    else:
        raise FileNotFoundError(
            "The completed Cell 16 ablation table was not found:\n"
            f"  {ABLATION_TABLE_PATH}\n"
            "Run corrected Cell 16 once before generating these figures."
        )


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def explicit_decimal(value, significant_digits=3):
    """
    Format a number without computer-style e notation.

    Examples
    --------
    154.0       -> 154
    1.538       -> 1.54
    0.517       -> 0.517
    0.00466     -> 0.00466
    0.000705    -> 0.000705
    3.081e-7    -> 0.000000308
    """
    if pd.isna(value):
        return "N/A"

    value = float(value)

    if value == 0.0:
        return "0"

    absolute_value = abs(value)

    if absolute_value >= 100:
        decimal_places = 0
    elif absolute_value >= 10:
        decimal_places = 1
    elif absolute_value >= 1:
        decimal_places = 2
    else:
        exponent = math.floor(math.log10(absolute_value))
        decimal_places = max(
            significant_digits,
            -exponent + significant_digits - 1,
        )
        decimal_places = min(decimal_places, 12)

    formatted = f"{value:.{decimal_places}f}"

    if "." in formatted:
        formatted = formatted.rstrip("0").rstrip(".")

    return formatted


def set_black_frame(axis):
    """Give every panel a complete black frame."""
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)


def add_panel_label(axis, label):
    axis.text(
        -0.10,
        1.05,
        label,
        transform=axis.transAxes,
        fontsize=12,
        fontweight="bold",
        ha="left",
        va="bottom",
    )


# =============================================================================
# PREPARE AND ORDER THE METRICS
# =============================================================================

metrics = ABLATION_METRICS.copy()

display_name_map = {
    "ode_fit": "Matched ODE benchmark",
    "data_only": "Data-only NN",
    "vanilla": "Vanilla PINN",
    "soft": "Soft-constrained PINN",
    "positive": "Positive-output PINN",
    "simplex": "Simplex-only PINN",
    "hard": "Proposed hard PINN",
}

method_order = [
    "ode_fit",
    "data_only",
    "vanilla",
    "soft",
    "positive",
    "simplex",
    "hard",
]

metrics["display_name"] = (
    metrics["model_kind"]
    .map(display_name_map)
    .fillna(metrics["model_kind"])
)

metrics["display_order"] = metrics["model_kind"].map(
    {name: index for index, name in enumerate(method_order)}
)

metrics = (
    metrics
    .sort_values("display_order")
    .reset_index(drop=True)
)

ode_metrics = (
    metrics[
        metrics["model_kind"] == "ode_fit"
    ]
    .copy()
    .reset_index(drop=True)
)

neural_metrics = (
    metrics[
        metrics["model_kind"] != "ode_fit"
    ]
    .copy()
    .reset_index(drop=True)
)

# Save transparent audit tables.
neural_metrics.to_csv(
    TABLE_DIR_R3
    / "round3_architecture_ablation_neural_only_metrics.csv",
    index=False,
)

ode_metrics.to_csv(
    TABLE_DIR_R3
    / "round3_matched_ode_benchmark_metrics.csv",
    index=False,
)

print(
    "Cell 17: generating final publication-ready figures from "
    f"{len(neural_metrics)} neural methods and "
    f"{len(ode_metrics)} matched ODE benchmark."
)


# =============================================================================
# DEFAULT MATPLOTLIB COLOR CYCLE
# =============================================================================
# The default cycle is used consistently rather than defining custom colors.

default_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

method_color = {
    "data_only": default_cycle[0],
    "vanilla": default_cycle[1],
    "soft": default_cycle[2],
    "positive": default_cycle[3],
    "simplex": default_cycle[4],
    "hard": default_cycle[5],
    "ode_fit": default_cycle[0],
}


# =============================================================================
# FIGURE A — NEURAL-ONLY STATE-ERROR HEATMAP
# =============================================================================

heat_columns = [
    "rmse_sensitive",
    "rmse_resistant",
    "rmse_total",
    "rmse_fraction",
]

heat_labels = [
    r"RMSE $S$",
    r"RMSE $R$",
    r"RMSE $S+R$",
    r"RMSE $q_R$",
]

heat_values = (
    neural_metrics[heat_columns]
    .astype(float)
    .to_numpy()
)

heat_log10 = np.log10(
    np.maximum(heat_values, 1e-12)
)

finite_heat = heat_log10[
    np.isfinite(heat_log10)
]

if finite_heat.size == 0:
    raise RuntimeError(
        "No finite neural RMSE values are available for the heatmap."
    )

heat_norm = mcolors.Normalize(
    vmin=float(np.nanmin(finite_heat)),
    vmax=float(np.nanmax(finite_heat)),
)

fig, ax = plt.subplots(
    figsize=(10.4, 5.5)
)

image = ax.imshow(
    heat_log10,
    aspect="auto",
    norm=heat_norm,
    interpolation="nearest",
)

ax.set_xticks(
    np.arange(len(heat_labels))
)

ax.set_xticklabels(
    heat_labels,
    fontsize=11,
)

ax.set_yticks(
    np.arange(len(neural_metrics))
)

ax.set_yticklabels(
    neural_metrics["display_name"],
    fontsize=10.8,
)

ax.set_title(
    "Neural architecture ablation: state-reconstruction accuracy",
    fontsize=15,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel(
    "Reconstruction metric",
    fontsize=11.5,
)

ax.set_ylabel(
    "Inference method",
    fontsize=11.5,
)

# Add cell boundaries.
ax.set_xticks(
    np.arange(-0.5, len(heat_labels), 1),
    minor=True,
)

ax.set_yticks(
    np.arange(-0.5, len(neural_metrics), 1),
    minor=True,
)

ax.grid(
    which="minor",
    linewidth=1.2,
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False,
)

# Add explicit decimal values.
for row_index in range(heat_values.shape[0]):
    for column_index in range(heat_values.shape[1]):

        original_value = heat_values[
            row_index,
            column_index,
        ]

        normalized_value = heat_norm(
            heat_log10[
                row_index,
                column_index,
            ]
        )

        text_color = (
            "white"
            if normalized_value < 0.42
            else "black"
        )

        ax.text(
            column_index,
            row_index,
            explicit_decimal(original_value),
            ha="center",
            va="center",
            fontsize=9.3,
            fontweight="semibold",
            color=text_color,
        )

colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.038,
    pad=0.025,
)

colorbar.set_label(
    r"$\log_{10}(\mathrm{RMSE})$ used for colour only",
    fontsize=10.5,
)

set_black_frame(ax)

fig.tight_layout()

save_figure_r3(
    fig,
    "round3_ablation_error_heatmap_neural_only_final",
)

plt.show()


# =============================================================================
# MATCHED ODE BENCHMARK — FORMATTED SEPARATE TABLE
# =============================================================================

if not ode_metrics.empty:

    ode_row = ode_metrics.iloc[0]

    ode_display = pd.DataFrame({
        "Method": [
            "Matched ODE benchmark"
        ],
        "RMSE S": [
            explicit_decimal(
                ode_row["rmse_sensitive"]
            )
        ],
        "RMSE R": [
            explicit_decimal(
                ode_row["rmse_resistant"]
            )
        ],
        "RMSE total": [
            explicit_decimal(
                ode_row["rmse_total"]
            )
        ],
        "RMSE fraction": [
            explicit_decimal(
                ode_row["rmse_fraction"]
            )
        ],
        "Estimated mu": [
            explicit_decimal(
                ode_row["mutation_probability"]
            )
        ],
        "Estimated c_R": [
            explicit_decimal(
                ode_row["resistance_fitness_cost"]
            )
        ],
    })

    print(
        "\nMatched ODE benchmark "
        "(reported separately from the neural architecture ablation):"
    )

    display(ode_display)


# =============================================================================
# =============================================================================
# FIGURE B — FINAL PUBLICATION ACCURACY–FEASIBILITY COMPARISON
# =============================================================================
# Scientific purpose
# ------------------
# Panel A shows neural methods that exhibit nonzero state-feasibility
# violations on the evaluation grid. The plotted vertical quantity is the
# maximum negative-state violation, V_neg, because the completed ablation has
# zero carrying-capacity violation for every method.
#
# Panel B shows neural methods for which no violation was observed on the
# evaluation grid. Marker SHAPE encodes the architectural guarantee:
#   square   -> positivity only
#   triangle -> positivity + simplex
#   star     -> full feasible set
#
# Important interpretation
# ------------------------
# Zero observed violation on a finite evaluation grid is not equivalent to a
# guarantee by construction. The proposed hard PINN is the only architecture
# in this comparison that guarantees the complete feasible set by design.
#
# No model is retrained in this section.
# =============================================================================

required_feasibility_columns = [
    "rmse_resistant",
    "maximum_negative_violation",
    "maximum_capacity_violation",
]

missing_feasibility_columns = [
    column
    for column in required_feasibility_columns
    if column not in neural_metrics.columns
]

if missing_feasibility_columns:
    raise KeyError(
        "Missing feasibility columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_feasibility_columns
        )
    )

# -----------------------------------------------------------------------------
# Classify methods by whether ANY feasibility violation was observed.
# -----------------------------------------------------------------------------

neural_metrics["total_feasibility_violation"] = (
    neural_metrics["maximum_negative_violation"].astype(float)
    + neural_metrics["maximum_capacity_violation"].astype(float)
)

violation_tolerance = 1e-12

nonzero_violation = (
    neural_metrics[
        neural_metrics["total_feasibility_violation"] > violation_tolerance
    ]
    .copy()
    .sort_values("rmse_resistant")
    .reset_index(drop=True)
)

zero_violation = (
    neural_metrics[
        neural_metrics["total_feasibility_violation"] <= violation_tolerance
    ]
    .copy()
    .sort_values(
        "rmse_resistant",
        ascending=False,
    )
    .reset_index(drop=True)
)

if nonzero_violation.empty:
    raise RuntimeError(
        "No nonzero-violation neural methods are available for Panel A."
    )

if zero_violation.empty:
    raise RuntimeError(
        "No zero-observed-violation neural methods are available for Panel B."
    )

# In the completed ablation, Panel A violations are negativity violations.
if np.any(
    nonzero_violation["maximum_negative_violation"].to_numpy(float)
    <= violation_tolerance
):
    raise RuntimeError(
        "At least one nonzero-violation method has no positive negativity "
        "violation. Revisit the Panel A metric before plotting."
    )

# -----------------------------------------------------------------------------
# Publication layout
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.6, 5.2),
    gridspec_kw={
        "width_ratios": [1.08, 1.0],
    },
)

# =============================================================================
# PANEL A — METHODS WITH OBSERVED CONSTRAINT VIOLATION
# =============================================================================

axis = axes[0]

nonzero_x_values = (
    nonzero_violation["rmse_resistant"]
    .to_numpy(float)
)

nonzero_y_values = (
    nonzero_violation["maximum_negative_violation"]
    .to_numpy(float)
)

axis.set_xscale("log")
axis.set_yscale("log")

axis.set_xlim(
    max(np.min(nonzero_x_values) / 1.80, 1e-12),
    np.max(nonzero_x_values) * 1.28,
)

axis.set_ylim(
    max(np.min(nonzero_y_values) / 1.55, 1e-12),
    np.max(nonzero_y_values) * 1.55,
)

# Hand-tuned offsets to avoid clipping and overlap
annotation_offsets = {
    "data_only": (-12, 8),   # move inside from right edge
    "vanilla": (10, 10),     # move above, not below axis
    "soft": (10, 8),
}

for _, row in nonzero_violation.iterrows():

    model_kind = row["model_kind"]
    rmse_value = float(row["rmse_resistant"])
    violation_value = float(row["maximum_negative_violation"])

    axis.scatter(
        rmse_value,
        violation_value,
        s=118,
        marker="o",
        color=method_color.get(model_kind, None),
        edgecolor="black",
        linewidth=0.85,
        zorder=4,
    )

    offset_x, offset_y = annotation_offsets.get(
        model_kind,
        (9, 7),
    )

    axis.annotate(
        row["display_name"],
        (rmse_value, violation_value),
        xytext=(offset_x, offset_y),
        textcoords="offset points",
        fontsize=9.0,
        ha=("right" if offset_x < 0 else "left"),
        va="center",
        annotation_clip=False,
        clip_on=False,
        zorder=5,
    )

axis.set_xlabel(
    r"Resistant-state error, $\mathrm{RMSE}_R$",
    fontsize=11.1,
    labelpad=6,
)

axis.set_ylabel(
    r"Observed feasibility violation",
    fontsize=11.1,
    labelpad=7,
)

axis.set_title(
    "Methods with nonzero violation",
    fontsize=12.0,
    fontweight="semibold",
    pad=10,
)

axis.grid(
    alpha=0.20,
    which="major",
    linestyle="--",
    linewidth=0.75,
)

axis.grid(
    alpha=0.11,
    which="minor",
    linestyle="--",
    linewidth=0.55,
)

axis.tick_params(
    axis="both",
    which="major",
    labelsize=9.6,
    width=0.9,
    length=4.2,
    direction="out",
)

axis.tick_params(
    axis="both",
    which="minor",
    width=0.7,
    length=2.5,
    direction="out",
)

set_black_frame(axis)
add_panel_label(axis, "A")

# =============================================================================
# PANEL B — METHODS WITH ZERO OBSERVED CONSTRAINT VIOLATION
# =============================================================================

axis = axes[1]

zero_x_values = (
    zero_violation["rmse_resistant"]
    .to_numpy(float)
)

axis.set_xscale("log")

# Publication-safe horizontal limits.
# Extra right-side headroom keeps the rightmost square and its numeric
# annotation comfortably inside the plotting frame after PDF export.
axis.set_xlim(
    max(np.min(zero_x_values) / 1.25, 1e-12),
    np.max(zero_x_values) * 1.28,
)

zero_y = np.arange(len(zero_violation))

marker_map = {
    "positive": "s",
    "simplex": "^",
    "hard": "*",
}

guarantee_text = {
    "positive": "Positivity only",
    "simplex": "Positivity + simplex",
    "hard": "Full feasible set",
}

marker_size = {
    "positive": 170,
    "simplex": 190,
    "hard": 285,
}

# Explicit method-specific numeric-label placement.
#
# The positive-output PINN is the rightmost point. Its value is therefore
# forced to the LEFT of the square. The other two values are placed to the
# RIGHT of their markers. This deterministic placement is more robust than
# threshold-based logic and prevents labels from leaving the axes after
# raster/PDF export or journal down-scaling.
value_label_style = {
    "positive": {
        "xytext": (-12, 0),
        "ha": "right",
    },
    "simplex": {
        "xytext": (10, 0),
        "ha": "left",
    },
    "hard": {
        "xytext": (10, 0),
        "ha": "left",
    },
}

for row_index, row in zero_violation.iterrows():

    model_kind = row["model_kind"]
    rmse_value = float(row["rmse_resistant"])

    axis.scatter(
        rmse_value,
        row_index,
        s=marker_size.get(model_kind, 170),
        marker=marker_map.get(model_kind, "o"),
        color=method_color.get(model_kind, None),
        edgecolor="black",
        linewidth=0.9,
        zorder=4,
    )

    label_style = value_label_style.get(
        model_kind,
        {
            "xytext": (10, 0),
            "ha": "left",
        },
    )

    axis.annotate(
        explicit_decimal(rmse_value),
        (rmse_value, row_index),
        xytext=label_style["xytext"],
        textcoords="offset points",
        fontsize=9.1,
        ha=label_style["ha"],
        va="center",
        annotation_clip=True,
        clip_on=True,
        zorder=5,
    )

axis.set_yticks(zero_y)
axis.set_yticklabels(
    zero_violation["display_name"],
    fontsize=9.7,
)

axis.set_xlabel(
    r"Resistant-state error, $\mathrm{RMSE}_R$",
    fontsize=11.1,
    labelpad=6,
)

axis.set_title(
    "Zero observed violation on the evaluation grid",
    fontsize=12.0,
    fontweight="semibold",
    pad=10,
)

axis.grid(
    alpha=0.18,
    axis="x",
    which="major",
    linestyle="--",
    linewidth=0.75,
)

axis.grid(
    alpha=0.09,
    axis="x",
    which="minor",
    linestyle="--",
    linewidth=0.55,
)

axis.set_ylim(
    len(zero_violation) - 0.52,
    -0.48,
)

axis.tick_params(
    axis="x",
    which="major",
    labelsize=9.6,
    width=0.9,
    length=4.2,
    direction="out",
)

axis.tick_params(
    axis="x",
    which="minor",
    width=0.7,
    length=2.5,
    direction="out",
)

axis.tick_params(
    axis="y",
    which="major",
    width=0.9,
    length=4.0,
    direction="out",
)

set_black_frame(axis)
add_panel_label(axis, "B")

# -----------------------------------------------------------------------------
# Guarantee-by-construction legend
# -----------------------------------------------------------------------------

legend_handles = [
    Line2D(
        [0],
        [0],
        marker=marker_map["positive"],
        linestyle="None",
        markerfacecolor=method_color["positive"],
        markeredgecolor="black",
        markeredgewidth=0.9,
        markersize=8.0,
        label=guarantee_text["positive"],
    ),
    Line2D(
        [0],
        [0],
        marker=marker_map["simplex"],
        linestyle="None",
        markerfacecolor=method_color["simplex"],
        markeredgecolor="black",
        markeredgewidth=0.9,
        markersize=8.4,
        label=guarantee_text["simplex"],
    ),
    Line2D(
        [0],
        [0],
        marker=marker_map["hard"],
        linestyle="None",
        markerfacecolor=method_color["hard"],
        markeredgecolor="black",
        markeredgewidth=0.9,
        markersize=11.0,
        label=guarantee_text["hard"],
    ),
]

legend = axis.legend(
    handles=legend_handles,
    loc="lower right",
    frameon=True,
    fancybox=False,
    framealpha=1.0,
    edgecolor="black",
    title="Architectural guarantee",
    fontsize=8.7,
    title_fontsize=9.0,
    borderpad=0.65,
    labelspacing=0.55,
    handletextpad=0.70,
)

legend.get_frame().set_linewidth(0.8)
legend.get_title().set_fontweight("semibold")

# =============================================================================
# FINAL LAYOUT AND SAVE
# =============================================================================

fig.suptitle(
    "Accuracy-feasibility comparison across neural inference methods",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)

fig.text(
    0.5,
    0.03,
    "Zero observed violation does not imply the same architectural guarantee; "
    "only the proposed hard PINN guarantees the complete feasible set by construction.",
    ha="center",
    va="center",
    fontsize=10.0,
)

fig.subplots_adjust(
    left=0.085,
    right=0.985,
    bottom=0.16,
    top=0.83,
    wspace=0.34,
)

save_figure_r3(
    fig,
    "round3_ablation_accuracy_feasibility_final_v2",
)

plt.show()


# =============================================================================
# FIGURE C — EVOLUTIONARY-PARAMETER RECOVERY
# =============================================================================
# The matched ODE benchmark is retained because it is a meaningful parameter-
# estimation baseline. It is visually separated from the neural methods.
# =============================================================================

parameter_metrics = (
    metrics[
        metrics["mutation_probability"].notna()
        &
        metrics["resistance_fitness_cost"].notna()
    ]
    .copy()
)

parameter_order = [
    "ode_fit",
    "vanilla",
    "soft",
    "positive",
    "simplex",
    "hard",
]

parameter_metrics["parameter_order"] = (
    parameter_metrics["model_kind"].map(
        {
            name: index
            for index, name in enumerate(
                parameter_order
            )
        }
    )
)

parameter_metrics = (
    parameter_metrics
    .sort_values("parameter_order")
    .reset_index(drop=True)
)

parameter_y = np.arange(
    len(parameter_metrics)
)

true_mutation = float(
    TRUE_PARAMETERS.mutation_probability
)

true_cost = float(
    TRUE_PARAMETERS.resistance_fitness_cost
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.8, 5.8),
    sharey=True,
)

parameter_marker = {
    "ode_fit": "D",
    "vanilla": "o",
    "soft": "o",
    "positive": "o",
    "simplex": "o",
    "hard": "*",
}

parameter_size = {
    "ode_fit": 130,
    "vanilla": 105,
    "soft": 105,
    "positive": 105,
    "simplex": 105,
    "hard": 180,
}

# -----------------------------------------------------------------------------
# Panel A: mutation probability
# -----------------------------------------------------------------------------

axis = axes[0]

mutation_estimates = (
    parameter_metrics["mutation_probability"].to_numpy(float)
)

mutation_min = min(
    float(np.min(mutation_estimates)),
    true_mutation,
)

mutation_max = max(
    float(np.max(mutation_estimates)),
    true_mutation,
)

mutation_mid = math.sqrt(
    mutation_min * mutation_max
)

axis.set_xscale("log")
axis.set_xlim(
    mutation_min / 1.75,
    mutation_max * 1.85,
)

for row_index, row in parameter_metrics.iterrows():

    model_kind = row["model_kind"]
    estimate = float(
        row["mutation_probability"]
    )

    axis.scatter(
        estimate,
        row_index,
        s=parameter_size[model_kind],
        marker=parameter_marker[model_kind],
        facecolors=(
            "none"
            if model_kind == "ode_fit"
            else None
        ),
        edgecolor="black",
        linewidth=1.0,
        zorder=3,
    )

    if estimate > mutation_mid:
        label_offset = (-8, 0)
        label_alignment = "right"
    else:
        label_offset = (8, 0)
        label_alignment = "left"

    axis.annotate(
        explicit_decimal(estimate),
        (
            estimate,
            row_index,
        ),
        xytext=label_offset,
        textcoords="offset points",
        fontsize=8.7,
        ha=label_alignment,
        va="center",
        annotation_clip=True,
        clip_on=True,
    )

axis.axvline(
    true_mutation,
    linestyle="--",
    linewidth=1.6,
    color="black",
    label=(
        "True value = "
        + explicit_decimal(
            true_mutation
        )
    ),
)

axis.set_yticks(
    parameter_y
)

axis.set_yticklabels(
    parameter_metrics["display_name"],
    fontsize=10,
)

axis.set_ylim(
    len(parameter_metrics) - 0.50,
    -0.85,
)

axis.text(
    true_mutation,
    -0.58,
    "True value = " + explicit_decimal(true_mutation),
    fontsize=8.8,
    ha="center",
    va="center",
)

axis.set_xlabel(
    r"Estimated mutation probability, $\hat{\mu}$",
    fontsize=11,
)

axis.set_title(
    "Mutation-probability recovery",
    fontsize=12.2,
    fontweight="semibold",
    pad=10,
)

axis.grid(
    alpha=0.24,
    which="both",
    linestyle="--",
)

set_black_frame(axis)
add_panel_label(axis, "A")

# -----------------------------------------------------------------------------
# Panel B: resistance fitness cost
# -----------------------------------------------------------------------------

axis = axes[1]

cost_estimates = (
    parameter_metrics["resistance_fitness_cost"].to_numpy(float)
)

cost_min = min(
    float(np.min(cost_estimates)),
    true_cost,
)

cost_max = max(
    float(np.max(cost_estimates)),
    true_cost,
)

cost_span = max(
    cost_max - cost_min,
    0.02,
)

cost_mid = 0.5 * (cost_min + cost_max)

axis.set_xlim(
    max(0.0, cost_min - (0.08 * cost_span + 0.004)),
    cost_max + (0.18 * cost_span + 0.008),
)

for row_index, row in parameter_metrics.iterrows():

    model_kind = row["model_kind"]
    estimate = float(
        row["resistance_fitness_cost"]
    )

    axis.scatter(
        estimate,
        row_index,
        s=parameter_size[model_kind],
        marker=parameter_marker[model_kind],
        facecolors=(
            "none"
            if model_kind == "ode_fit"
            else None
        ),
        edgecolor="black",
        linewidth=1.0,
        zorder=3,
    )

    if estimate > cost_mid:
        label_offset = (-8, 0)
        label_alignment = "right"
    else:
        label_offset = (8, 0)
        label_alignment = "left"

    axis.annotate(
        explicit_decimal(estimate),
        (
            estimate,
            row_index,
        ),
        xytext=label_offset,
        textcoords="offset points",
        fontsize=8.7,
        ha=label_alignment,
        va="center",
        annotation_clip=True,
        clip_on=True,
    )

axis.axvline(
    true_cost,
    linestyle="--",
    linewidth=1.6,
    color="black",
    label=(
        "True value = "
        + explicit_decimal(
            true_cost
        )
    ),
)

axis.set_ylim(
    len(parameter_metrics) - 0.50,
    -0.85,
)

axis.text(
    true_cost,
    -0.58,
    "True value = " + explicit_decimal(true_cost),
    fontsize=8.8,
    ha="center",
    va="center",
)

axis.set_xlabel(
    r"Estimated resistance fitness cost, $\hat{c}_R$",
    fontsize=11,
)

axis.set_title(
    "Resistance-fitness-cost recovery",
    fontsize=12.2,
    fontweight="semibold",
    pad=10,
)

axis.grid(
    alpha=0.24,
    linestyle="--",
)

set_black_frame(axis)
add_panel_label(axis, "B")

# Separate the ODE benchmark row from neural methods.
for axis in axes:
    axis.axhline(
        0.5,
        color="black",
        linewidth=1.0,
        linestyle=":",
        alpha=0.75,
    )

benchmark_handles = [
    Line2D(
        [0],
        [0],
        marker="D",
        linestyle="None",
        markerfacecolor="none",
        markeredgecolor="black",
        markersize=8,
        label="Matched ODE benchmark",
    ),
    Line2D(
        [0],
        [0],
        marker="*",
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=11,
        label="Proposed hard PINN",
    ),
    Line2D(
        [0],
        [0],
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="True parameter value",
    ),
]

fig.legend(
    handles=benchmark_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.045),
    ncol=3,
    frameon=True,
    fontsize=8.7,
)

fig.suptitle(
    "Evolutionary-parameter recovery across inference methods",
    fontsize=15,
    fontweight="bold",
    y=1.01,
)

fig.text(
    0.5,
    0.005,
    "The hollow diamond denotes the matched mechanistic ODE benchmark; "
    "the rows below the dotted separator are neural inference architectures.",
    ha="center",
    va="bottom",
    fontsize=9.1,
)

fig.tight_layout(
    rect=[0.0, 0.12, 1.0, 0.98],
    w_pad=2.5,
)

save_figure_r3(
    fig,
    "round3_ablation_parameter_recovery_final_v2",
)

plt.show()


# =============================================================================
# FINAL CONFIRMATION
# =============================================================================

print(
    "\nCell 17 completed successfully."
)

print(
    "Generated final publication figures:"
)

print(
    " ",
    FIGURE_DIR_R3
    / "round3_ablation_error_heatmap_neural_only_final",
)

print(
    " ",
    FIGURE_DIR_R3
    / "round3_ablation_accuracy_feasibility_final_v2",
)

print(
    " ",
    FIGURE_DIR_R3
    / "round3_ablation_parameter_recovery_final_v2",
)

In [ ]:
# =============================================================================
# ROUND 3 — CELL 18 (FIVE-SEED, PAIRED, RESUME-SAFE VERSION)
# Observation-regime experiment: O1, O2, O3, and O4 with the hard model
# =============================================================================
# Five replicate seeds are used. Within each replicate, O1–O4 use the same
# dataset seed and the same training seed, so phenotype information is the
# controlled change. Results are saved after every completed model.
# =============================================================================

import gc
import random
from typing import Dict, Any

import numpy as np
import pandas as pd
import torch


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
REGIMES = ["O1", "O2", "O3", "O4"]
SEED_IDS = [1, 2, 3, 4, 5]

FORCE_RERUN = False
KEEP_RESULTS_IN_MEMORY = True

RAW_RESULTS_PATH = (
    TABLE_DIR_R3 / "round3_observation_regime_all_seeds_metrics.csv"
)
SUMMARY_RESULTS_PATH = (
    TABLE_DIR_R3 / "round3_observation_regime_summary_mean_std.csv"
)
MANUSCRIPT_TABLE_PATH = (
    TABLE_DIR_R3 / "round3_observation_regime_manuscript_mean_std.csv"
)

OBSERVATION_STUDY_RESULTS: Dict[str, Any] = {}
OBSERVATION_STUDY_ROWS = []


# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------
def set_reproducible_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# Save one unique row per regime and seed
# -----------------------------------------------------------------------------
def save_raw_rows(rows) -> pd.DataFrame:
    frame = pd.DataFrame(rows)

    if frame.empty:
        return frame

    frame = (
        frame
        .drop_duplicates(subset=["regime", "seed_id"], keep="last")
        .copy()
    )

    order_map = {regime: i for i, regime in enumerate(REGIMES)}
    frame["_order"] = frame["regime"].map(order_map)

    frame = (
        frame
        .sort_values(["_order", "seed_number"])
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    frame.to_csv(RAW_RESULTS_PATH, index=False)
    return frame


# -----------------------------------------------------------------------------
# Build numeric mean/SD and manuscript-ready "mean ± SD" tables
# -----------------------------------------------------------------------------
def build_mean_sd_tables(all_runs: pd.DataFrame):
    group_columns = ["regime", "phenotype_points"]

    excluded_numeric = {
        "seed_number",
        "dataset_seed",
        "training_seed",
        "phenotype_points",
    }

    numeric_metrics = [
        column
        for column in all_runs.select_dtypes(include=[np.number]).columns
        if column not in excluded_numeric
    ]

    grouped = all_runs.groupby(
        group_columns,
        sort=False,
        dropna=False,
    )

    summary = grouped[numeric_metrics].agg(["mean", "std"]).reset_index()

    summary.columns = [
        column
        if isinstance(column, str)
        else "_".join(
            str(part) for part in column if str(part) not in {"", "None"}
        )
        for column in summary.columns
    ]

    counts = grouped.size().rename("n_seeds").reset_index()

    summary = counts.merge(
        summary,
        on=group_columns,
        how="left",
        validate="one_to_one",
    )

    order_map = {regime: i for i, regime in enumerate(REGIMES)}
    summary["_order"] = summary["regime"].map(order_map)

    summary = (
        summary
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    def format_mean_sd(mean_value, std_value, metric):
        if pd.isna(mean_value):
            return "N/A"

        if pd.isna(std_value):
            return f"{mean_value:.4g} ± N/A"

        if metric.endswith("_percent"):
            return f"{mean_value:.1f} ± {std_value:.1f}"

        if metric == "training_minutes":
            return f"{mean_value:.2f} ± {std_value:.2f}"

        magnitude = abs(float(mean_value))

        if (
            metric == "mutation_probability"
            or (magnitude != 0.0 and magnitude < 1e-3)
            or magnitude >= 1e3
        ):
            return f"{mean_value:.3e} ± {std_value:.2e}"

        return f"{mean_value:.4f} ± {std_value:.4f}"

    manuscript_metrics = [
        "rmse_sensitive",
        "rmse_resistant",
        "rmse_total",
        "rmse_fraction",
        "relative_l2_sensitive",
        "relative_l2_resistant",
        "mutation_probability",
        "mu_error_percent",
        "resistance_fitness_cost",
        "cost_error_percent",
        "physics_residual_rmse",
        "fraction_points_negative",
        "fraction_points_over_capacity",
        "training_minutes",
    ]

    manuscript_rows = []

    for _, row in summary.iterrows():
        output_row = {
            "regime": row["regime"],
            "phenotype_points": int(row["phenotype_points"]),
            "n_seeds": int(row["n_seeds"]),
        }

        for metric in manuscript_metrics:
            mean_column = f"{metric}_mean"
            std_column = f"{metric}_std"

            if mean_column in summary.columns:
                output_row[metric] = format_mean_sd(
                    row[mean_column],
                    row[std_column],
                    metric,
                )

        manuscript_rows.append(output_row)

    manuscript = pd.DataFrame(manuscript_rows)

    return summary, manuscript


# =============================================================================
# Run the five-seed study
# =============================================================================
if RUN_OBSERVATION_REGIME_STUDY:

    if QUICK_MODE:
        print(
            "WARNING: QUICK_MODE=True. These are preliminary runs, "
            "not final manuscript results."
        )

    # -------------------------------------------------------------------------
    # Resume from the new five-seed raw-results file
    # -------------------------------------------------------------------------
    if RAW_RESULTS_PATH.exists() and not FORCE_RERUN:
        existing = pd.read_csv(RAW_RESULTS_PATH)

        required_columns = {"regime", "seed_id", "seed_number"}

        if required_columns.issubset(existing.columns):
            OBSERVATION_STUDY_ROWS = existing.to_dict("records")
            print(
                f"Resuming from {len(existing)} completed regime-seed runs."
            )
        else:
            print(
                "Existing file has an incompatible schema and will not be "
                "used for resume."
            )

    completed_pairs = {
        (str(row["regime"]), str(row["seed_id"]))
        for row in OBSERVATION_STUDY_ROWS
        if "regime" in row and "seed_id" in row
    }

    total_runs = len(REGIMES) * len(SEED_IDS)

    print(
        f"Running {total_runs} models: "
        f"{len(REGIMES)} regimes × {len(SEED_IDS)} seeds."
    )

    # -------------------------------------------------------------------------
    # Paired design: the same seeds are reused across O1–O4
    # -------------------------------------------------------------------------
    for seed_number in SEED_IDS:

        seed_id = f"seed{seed_number}"

        dataset_seed = int(SEED + 10_000 * seed_number)
        training_seed = int(SEED + 100_000 + 10_000 * seed_number)

        print("\n" + "=" * 78)
        print(
            f"{seed_id}: dataset_seed={dataset_seed}, "
            f"training_seed={training_seed}"
        )
        print("=" * 78)

        for regime in REGIMES:

            pair = (regime, seed_id)
            label = f"round3_observation_{regime}_{seed_id}"

            if pair in completed_pairs and not FORCE_RERUN:
                print(f"Skipping completed run: {label}")
                continue

            print(f"\nTraining {label}")

            try:
                # Use the same dataset randomness across O1–O4 for this seed.
                set_reproducible_seed(dataset_seed)

                dataset = build_observation_dataset(
                    "early_continuous",
                    regime,
                    0.0,
                    dataset_seed,
                )

                bundle = {"early_continuous": dataset}

                # Use the same network/training randomness across O1–O4.
                set_reproducible_seed(training_seed)

                model = AMRInverseModel(
                    {
                        "early_continuous":
                        SCHEDULES_R3["early_continuous"]
                    },
                    "hard",
                    width=64,
                    depth=4,
                    parameter_initialization=(3e-4, 0.15),
                )

                result = train_inverse_model(
                    model,
                    bundle,
                    label,
                    training_seed,
                )

                metrics, prediction = evaluate_trained_model(
                    result,
                    "early_continuous",
                )

                relative_error_mu = metrics.get(
                    "relative_error_mu",
                    np.nan,
                )
                relative_error_cost = metrics.get(
                    "relative_error_cost",
                    np.nan,
                )
                training_seconds = metrics.get(
                    "training_seconds",
                    np.nan,
                )

                metrics.update({
                    "regime": regime,
                    "phenotype_points": int(len(dataset.fraction_time)),
                    "seed_id": seed_id,
                    "seed_number": int(seed_number),
                    "dataset_seed": int(dataset_seed),
                    "training_seed": int(training_seed),
                    "mu_error_percent": (
                        100.0 * float(relative_error_mu)
                        if pd.notna(relative_error_mu)
                        else np.nan
                    ),
                    "cost_error_percent": (
                        100.0 * float(relative_error_cost)
                        if pd.notna(relative_error_cost)
                        else np.nan
                    ),
                    "training_minutes": (
                        float(training_seconds) / 60.0
                        if pd.notna(training_seconds)
                        else np.nan
                    ),
                })

                # Save each dense prediction for later polished plotting.
                prediction_path = (
                    PREDICTION_DIR
                    / f"{label}_early_continuous_predictions.csv"
                )
                prediction.to_csv(prediction_path, index=False)

                if KEEP_RESULTS_IN_MEMORY:
                    OBSERVATION_STUDY_RESULTS[label] = result

                OBSERVATION_STUDY_ROWS.append(metrics)
                completed_pairs.add(pair)

                current_raw = save_raw_rows(
                    OBSERVATION_STUDY_ROWS
                )

                print(
                    f"Saved {label}: "
                    f"{len(current_raw)}/{total_runs} runs completed."
                )

                result.model.to("cpu")

                del model, dataset, bundle, prediction
                gc.collect()
                clear_gpu_memory()

            except Exception:
                save_raw_rows(OBSERVATION_STUDY_ROWS)
                clear_gpu_memory()

                print(
                    f"\nThe run failed at {label}. All completed runs "
                    f"remain saved in:\n  {RAW_RESULTS_PATH}"
                )
                raise

    # -------------------------------------------------------------------------
    # Final tables
    # -------------------------------------------------------------------------
    OBSERVATION_STUDY_ALL_RUNS = save_raw_rows(
        OBSERVATION_STUDY_ROWS
    )

    (
        OBSERVATION_STUDY_SUMMARY,
        OBSERVATION_STUDY_MANUSCRIPT,
    ) = build_mean_sd_tables(
        OBSERVATION_STUDY_ALL_RUNS
    )

    OBSERVATION_STUDY_SUMMARY.to_csv(
        SUMMARY_RESULTS_PATH,
        index=False,
    )

    OBSERVATION_STUDY_MANUSCRIPT.to_csv(
        MANUSCRIPT_TABLE_PATH,
        index=False,
    )

    # Backward-compatible name.
    OBSERVATION_STUDY = OBSERVATION_STUDY_ALL_RUNS

    print("\n" + "=" * 78)
    print("FIVE-SEED OBSERVATION-REGIME STUDY COMPLETED")
    print("=" * 78)

    print("\nRaw per-seed table:")
    print(" ", RAW_RESULTS_PATH)

    print("\nNumeric mean/SD table:")
    print(" ", SUMMARY_RESULTS_PATH)

    print("\nManuscript-ready mean ± SD table:")
    print(" ", MANUSCRIPT_TABLE_PATH)

    print("\nPer-seed results:")
    display(OBSERVATION_STUDY_ALL_RUNS)

    print("\nMean ± sample SD results:")
    display(OBSERVATION_STUDY_MANUSCRIPT)

    completed_counts = (
        OBSERVATION_STUDY_ALL_RUNS
        .groupby("regime")["seed_id"]
        .nunique()
        .reindex(REGIMES, fill_value=0)
    )

    if (completed_counts == len(SEED_IDS)).all():
        print(
            "\nAll O1–O4 regimes contain five paired seeds. "
            "SD is the sample standard deviation (ddof=1)."
        )
    else:
        print(
            "\nWARNING: The study is incomplete. Completed seeds:\n"
            + completed_counts.to_string()
        )

else:
    print(
        "Set RUN_OBSERVATION_REGIME_STUDY=True in Cell 1 "
        "to run the five-seed O1–O4 study."
    )

In [ ]:
# ROUND 3 — CELL 19 (FINAL REFINED VERSION)
# OVERLAP-FREE, MORE POLISHED PUBLICATION FIGURES
# =============================================================================
# This revision specifically fixes:
# 1. title / legend overlap,
# 2. crowded x-axis labels,
# 3. figure-bottom note overlap,
# 4. paired-seed legend collision,
# 5. overly dense panel spacing.
#
# It does NOT retrain any model. It only reads the saved five-seed results.
# =============================================================================

from pathlib import Path
import re
import warnings
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter


# =============================================================================
# STANDALONE ROUND-3 PATH / HELPER RECOVERY
# =============================================================================
# Cell 19 is a post-processing cell. It does not retrain any model.
# If the Round-3 setup variables are already present, they are reused.
# Otherwise, the canonical final-result folders are reconstructed so that
# the saved five-seed tables can be read directly after a kernel restart.
# =============================================================================

ROUND3_ROOT = Path(
    globals().get(
        "ROUND3_ROOT",
        Path.home() / "round3" / "final",
    )
)

TABLE_DIR_R3 = Path(
    globals().get(
        "TABLE_DIR_R3",
        ROUND3_ROOT / "tables",
    )
)

FIGURE_DIR_R3 = Path(
    globals().get(
        "FIGURE_DIR_R3",
        ROUND3_ROOT / "figures",
    )
)

TABLE_DIR_R3.mkdir(parents=True, exist_ok=True)
FIGURE_DIR_R3.mkdir(parents=True, exist_ok=True)

if "TRUE_PARAMETERS" not in globals():
    TRUE_PARAMETERS = SimpleNamespace(
        mutation_probability=1.0e-4,
        resistance_fitness_cost=0.10,
    )

if "save_figure_r3" not in globals():

    def save_figure_r3(fig, stem):
        """Save one Round-3 publication figure as PDF, PNG, and SVG."""
        stem = str(stem)

        pdf_path = FIGURE_DIR_R3 / f"{stem}.pdf"
        png_path = FIGURE_DIR_R3 / f"{stem}.png"
        svg_path = FIGURE_DIR_R3 / f"{stem}.svg"

        fig.savefig(
            pdf_path,
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            png_path,
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            svg_path,
            bbox_inches="tight",
            facecolor="white",
        )

        print("Saved figure files:")
        print(" ", pdf_path.resolve())
        print(" ", png_path.resolve())
        print(" ", svg_path.resolve())


print("Round-3 Cell 19 paths:")
print("  Tables :", TABLE_DIR_R3.resolve())
print("  Figures:", FIGURE_DIR_R3.resolve())


# =============================================================================
# CONFIGURATION
# =============================================================================

REGIME_ORDER = ["O1", "O2", "O3", "O4"]

REGIME_DESCRIPTIONS = {
    "O1": "Total load only",
    "O2": "Total load + 1 phenotype measurement",
    "O3": "Total load + 3 phenotype measurements",
    "O4": "Fully phenotype-resolved observations",
}

EXPECTED_PHENOTYPE_POINTS = {
    "O1": 0,
    "O2": 1,
    "O3": 3,
    "O4": 21,
}

RAW_RESULTS_PATH = TABLE_DIR_R3 / "round3_observation_regime_all_seeds_metrics.csv"
FINAL_TABLE_PATH = TABLE_DIR_R3 / "round3_observation_regime_final_manuscript_table_v5.csv"
NUMERIC_SUMMARY_PATH = TABLE_DIR_R3 / "round3_observation_regime_final_numeric_summary_v5.csv"
FEASIBILITY_SUMMARY_PATH = TABLE_DIR_R3 / "round3_observation_regime_feasibility_summary_v5.csv"
EFFECT_SIZE_PATH = TABLE_DIR_R3 / "round3_observation_regime_effect_sizes_v5.csv"

GENERATE_PAIRED_SEED_FIGURE = True
GENERATE_EFFECT_SIZE_FIGURE = True


# =============================================================================
# PUBLICATION STYLE
# =============================================================================

PUBLICATION_RC = {
    "font.family": "sans-serif",
    "font.size": 10.5,
    "axes.titlesize": 11.7,
    "axes.labelsize": 11.0,
    "xtick.labelsize": 9.1,
    "ytick.labelsize": 9.4,
    "legend.fontsize": 8.5,
    "axes.linewidth": 1.0,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "grid.alpha": 0.18,
    "grid.linestyle": "--",
    "grid.linewidth": 0.65,
}

DEFAULT_COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

REGIME_COLORS = {
    regime: DEFAULT_COLORS[index % len(DEFAULT_COLORS)]
    for index, regime in enumerate(REGIME_ORDER)
}

REGIME_MARKERS = {
    "O1": "o",
    "O2": "s",
    "O3": "^",
    "O4": "D",
}


# =============================================================================
# LOAD RAW FIVE-SEED RESULTS
# =============================================================================

if (
    "OBSERVATION_STUDY_ALL_RUNS" in globals()
    and isinstance(OBSERVATION_STUDY_ALL_RUNS, pd.DataFrame)
    and not OBSERVATION_STUDY_ALL_RUNS.empty
):
    observation_runs = OBSERVATION_STUDY_ALL_RUNS.copy()
    print("Using OBSERVATION_STUDY_ALL_RUNS from memory.")
elif RAW_RESULTS_PATH.exists():
    observation_runs = pd.read_csv(RAW_RESULTS_PATH)
    print("Loaded five-seed results from:")
    print(" ", RAW_RESULTS_PATH)
else:
    raise FileNotFoundError(
        "Five-seed raw-results table not found. Expected:\n"
        f"  {RAW_RESULTS_PATH}\n"
        "Run the five-seed Cell 18 first."
    )

required_columns = [
    "regime",
    "seed_id",
    "phenotype_points",
    "rmse_sensitive",
    "rmse_resistant",
    "rmse_total",
    "rmse_fraction",
    "relative_l2_sensitive",
    "relative_l2_resistant",
    "mutation_probability",
    "mu_error_percent",
    "resistance_fitness_cost",
    "cost_error_percent",
    "physics_residual_rmse",
    "fraction_points_negative",
    "fraction_points_over_capacity",
    "training_minutes",
]

missing_columns = [column for column in required_columns if column not in observation_runs.columns]
if missing_columns:
    raise KeyError(
        "The five-seed table is missing required columns:\n"
        + "\n".join(f"  - {column}" for column in missing_columns)
    )

observation_runs = observation_runs[
    observation_runs["regime"].astype(str).isin(REGIME_ORDER)
].copy()

observation_runs = (
    observation_runs
    .drop_duplicates(subset=["regime", "seed_id"], keep="last")
    .reset_index(drop=True)
)

observation_runs["regime"] = pd.Categorical(
    observation_runs["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

observation_runs = (
    observation_runs
    .sort_values(["regime", "seed_id"])
    .reset_index(drop=True)
)

seed_counts = (
    observation_runs
    .groupby("regime", observed=True)["seed_id"]
    .nunique()
    .reindex(REGIME_ORDER, fill_value=0)
)

print("\nCompleted seed counts:")
print(seed_counts.to_string())

if not (seed_counts == 5).all():
    warnings.warn(
        "At least one regime does not contain five unique seeds. "
        "Figures will use the available completed runs."
    )


# =============================================================================
# CORRECT SUMMARY TABLES
# =============================================================================

metrics_for_summary = [
    "rmse_sensitive",
    "rmse_resistant",
    "rmse_total",
    "rmse_fraction",
    "relative_l2_sensitive",
    "relative_l2_resistant",
    "mutation_probability",
    "mu_error_percent",
    "resistance_fitness_cost",
    "cost_error_percent",
    "physics_residual_rmse",
    "fraction_points_negative",
    "fraction_points_over_capacity",
    "training_minutes",
]

summary_mean = observation_runs.groupby("regime", observed=True)[metrics_for_summary].mean()
summary_std = observation_runs.groupby("regime", observed=True)[metrics_for_summary].std(ddof=1)
summary_count = observation_runs.groupby("regime", observed=True).size().rename("n_seeds")
phenotype_points = observation_runs.groupby("regime", observed=True)["phenotype_points"].first().rename("phenotype_points")

numeric_summary = pd.concat(
    [
        phenotype_points,
        summary_count,
        summary_mean.add_prefix("mean_"),
        summary_std.add_prefix("std_"),
    ],
    axis=1,
).reset_index()

numeric_summary["regime"] = pd.Categorical(
    numeric_summary["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

numeric_summary = numeric_summary.sort_values("regime").reset_index(drop=True)
numeric_summary.to_csv(NUMERIC_SUMMARY_PATH, index=False)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def seed_sort_key(seed_id):
    match = re.search(r"(\d+)$", str(seed_id))
    if match:
        return int(match.group(1))
    return str(seed_id)


def compact_regime_labels():
    return [
        "O1\n(0 pts)",
        "O2\n(1 pt)",
        "O3\n(3 pts)",
        "O4\n(21 pts)",
    ]


def summary_value(regime, metric, statistic="mean"):
    row = numeric_summary[numeric_summary["regime"].astype(str) == regime]
    if row.empty:
        return np.nan
    column = f"{statistic}_{metric}"
    return float(row.iloc[0][column])


def raw_values(regime, metric):
    return (
        observation_runs.loc[
            observation_runs["regime"].astype(str) == regime,
            metric,
        ]
        .astype(float)
        .dropna()
        .to_numpy()
    )


def format_value(value, metric):
    if pd.isna(value):
        return "N/A"
    value = float(value)

    if metric in {"mutation_probability", "rmse_fraction", "physics_residual_rmse"}:
        if abs(value) < 0.001:
            return f"{value:.6f}".rstrip("0").rstrip(".")
        return f"{value:.4f}".rstrip("0").rstrip(".")

    if metric in {"mu_error_percent", "cost_error_percent"}:
        return f"{value:.1f}%"

    if metric == "training_minutes":
        return f"{value:.1f}"

    if abs(value) >= 100:
        return f"{value:.0f}"

    if abs(value) >= 1:
        return f"{value:.2f}"

    return f"{value:.3f}".rstrip("0").rstrip(".")



def format_annotation_value(value, metric):
    """
    Format values shown inside figure annotation boxes.

    Resistant-fraction RMSE is shown in scientific notation for consistent
    journal-scale readability; other metrics retain the compact decimal style.
    """
    if pd.isna(value):
        return "N/A"

    value = float(value)

    if metric == "rmse_fraction" and value != 0.0:
        exponent = int(np.floor(np.log10(abs(value))))
        mantissa = value / (10.0 ** exponent)
        return rf"${mantissa:.2f}\times10^{{{exponent}}}$"

    return format_value(value, metric)


def format_mean_sd(mean_value, sd_value, metric):
    if pd.isna(mean_value):
        return "N/A"
    if pd.isna(sd_value):
        return f"{format_value(mean_value, metric)} ± N/A"

    if metric in {"mutation_probability", "rmse_fraction", "physics_residual_rmse"}:
        return f"{format_value(mean_value, metric)} ± {format_value(sd_value, metric)}"

    if metric in {"mu_error_percent", "cost_error_percent"}:
        return f"{mean_value:.1f} ± {sd_value:.1f}"

    if metric == "training_minutes":
        return f"{mean_value:.2f} ± {sd_value:.2f}"

    return f"{mean_value:.4f} ± {sd_value:.4f}"


def set_black_frame(axis):
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)


def add_panel_label(axis, label):
    axis.text(
        -0.09,
        1.04,
        label,
        transform=axis.transAxes,
        fontsize=12.5,
        fontweight="bold",
        ha="left",
        va="bottom",
    )


def add_footer_note(figure, text, y=0.045):
    figure.text(
        0.5,
        y,
        text,
        ha="center",
        va="center",
        fontsize=8.6,
        color="black",
    )


def annotation_layout_for_regime(regime_index, total_regimes):
    # Place numeric labels slightly to the side of the markers so they do not
    # sit on the mean-connecting line or on top of the error bars.
    if total_regimes <= 4:
        layout_map = {
            0: dict(offset=(12, 8), ha="left"),
            1: dict(offset=(12, 8), ha="left"),
            2: dict(offset=(-12, 8), ha="right"),
            3: dict(offset=(-12, 8), ha="right"),
        }
        return layout_map.get(regime_index, dict(offset=(10, 8), ha="left"))

    return dict(offset=(10, 8), ha="left")


def annotate_mean_value(axis, x_value, y_value, metric, regime_index=None, total_regimes=4):
    """Add a compact publication-style label to the across-seed mean."""
    layout = annotation_layout_for_regime(regime_index, total_regimes)

    # Final manuscript micro-adjustment:
    # In the resistant-fraction panel, place the O3 mean label on the
    # RIGHT side and ABOVE the connecting line for maximum clarity.
    vertical_alignment = "bottom"
    if metric == "rmse_fraction" and regime_index == 2:
        layout = dict(offset=(14, 10), ha="left")
        vertical_alignment = "bottom"

    axis.annotate(
        format_annotation_value(y_value, metric),
        (x_value, y_value),
        xytext=layout["offset"],
        textcoords="offset points",
        ha=layout["ha"],
        va=vertical_alignment,
        fontsize=8.3,
        fontweight="semibold",
        annotation_clip=True,
        clip_on=True,
        bbox=dict(
            boxstyle="round,pad=0.14",
            facecolor="white",
            edgecolor="0.40",
            linewidth=0.60,
            alpha=0.94,
        ),
        zorder=7,
    )


def plot_distribution_panel(
    axis,
    metric,
    ylabel,
    title,
    *,
    log_scale=False,
    true_value=None,
    annotate_means=True,
    lower_is_better=False,
    nonnegative_metric=False,
):
    # Retained for compatibility with existing calls; interpretation belongs
    # in the Results and Discussion rather than inside the figure panels.
    _ = lower_is_better

    x = np.arange(len(REGIME_ORDER), dtype=float)

    means = np.array([summary_value(regime, metric, "mean") for regime in REGIME_ORDER])
    standard_deviations = np.array([summary_value(regime, metric, "std") for regime in REGIME_ORDER])

    for regime_index, regime in enumerate(REGIME_ORDER):
        values = raw_values(regime, metric)
        if len(values) == 0:
            continue

        jitter = np.array([0.0]) if len(values) == 1 else np.linspace(-0.12, 0.12, len(values))

        axis.scatter(
            np.full(len(values), x[regime_index]) + jitter,
            values,
            s=28,
            marker=REGIME_MARKERS[regime],
            facecolor=REGIME_COLORS[regime],
            edgecolor="black",
            linewidth=0.35,
            alpha=0.38,
            zorder=2,
        )

    axis.plot(
        x,
        means,
        color="black",
        linewidth=1.35,
        alpha=0.78,
        zorder=3,
    )

    for regime_index, regime in enumerate(REGIME_ORDER):

        mean_value = float(means[regime_index])
        sd_value = float(standard_deviations[regime_index])

        # RMSE and related error metrics are nonnegative by definition.
        # For those panels, retain the full +1 SD upper bar but truncate only
        # the visual lower extent at zero. This avoids displaying an impossible
        # negative RMSE while preserving the reported mean and sample SD.
        if nonnegative_metric and not log_scale:
            lower_error = min(sd_value, max(mean_value, 0.0))
            upper_error = sd_value

            y_error = np.array(
                [
                    [lower_error],
                    [upper_error],
                ],
                dtype=float,
            )

        else:
            y_error = sd_value

        axis.errorbar(
            x[regime_index],
            means[regime_index],
            yerr=y_error,
            fmt=REGIME_MARKERS[regime],
            markersize=7.8,
            markerfacecolor=REGIME_COLORS[regime],
            markeredgecolor="black",
            markeredgewidth=0.85,
            ecolor="black",
            elinewidth=1.15,
            capsize=4.0,
            capthick=1.0,
            zorder=5,
        )

    if true_value is not None:
        axis.axhline(
            true_value,
            linestyle="--",
            linewidth=1.35,
            color="black",
            zorder=1,
            label="True value = " + format_value(true_value, metric),
        )

    if log_scale:
        axis.set_yscale("log")
        axis.yaxis.set_major_locator(LogLocator(base=10))
        axis.yaxis.set_minor_formatter(NullFormatter())

    elif nonnegative_metric:
        # RMSE cannot be negative. Keep the automatically determined upper
        # limit but anchor the visible axis at the physically valid boundary.
        _, current_upper = axis.get_ylim()
        axis.set_ylim(
            bottom=0.0,
            top=current_upper,
        )

    axis.set_xticks(x)
    axis.set_xticklabels(compact_regime_labels())
    axis.set_xlim(-0.45, len(REGIME_ORDER) - 0.55)

    axis.set_ylabel(ylabel)
    axis.set_title(title, loc="left", fontweight="semibold", pad=7)
    axis.grid(True, axis="y")
    axis.set_axisbelow(True)
    set_black_frame(axis)

    if annotate_means:
        for regime_index, mean_value in enumerate(means):
            if pd.isna(mean_value):
                continue
            annotate_mean_value(axis, x[regime_index], mean_value, metric, regime_index=regime_index, total_regimes=len(REGIME_ORDER))

    return means, standard_deviations


# =============================================================================
# FIGURE 1 — FINAL MANUSCRIPT STATE-RECONSTRUCTION FIGURE
# =============================================================================
# Main O1--O4 state-observability figure.
#
# Publication decisions:
#   - no large in-figure title;
#   - no explanatory footer inside the artwork;
#   - concise, neutral panel titles;
#   - individual seed results retained;
#   - across-seed mean ± sample SD retained;
#   - log scale retained for resistant-fraction RMSE;
#   - compact mean-value labels;
#   - scientific notation used consistently in Panel D;
#   - RMSE axes remain physically valid (no negative displayed RMSE).
# =============================================================================

with plt.rc_context(PUBLICATION_RC):

    figure, axes = plt.subplots(
        2,
        2,
        figsize=(12.4, 8.35),
    )

    state_panels = [
        (
            "rmse_sensitive",
            axes[0, 0],
            r"Sensitive-state RMSE",
            r"Sensitive state, $S(t)$",
            False,
        ),
        (
            "rmse_resistant",
            axes[0, 1],
            r"Resistant-state RMSE",
            r"Resistant state, $R(t)$",
            False,
        ),
        (
            "rmse_total",
            axes[1, 0],
            r"Total-population RMSE",
            r"Total population, $S(t)+R(t)$",
            False,
        ),
        (
            "rmse_fraction",
            axes[1, 1],
            r"Resistant-fraction RMSE",
            r"Resistant fraction, $q_R(t)$",
            True,
        ),
    ]

    for panel_label, (
        metric,
        axis,
        ylabel,
        title,
        use_log,
    ) in zip(
        ["A", "B", "C", "D"],
        state_panels,
    ):

        plot_distribution_panel(
            axis,
            metric,
            ylabel,
            title,
            log_scale=use_log,
            annotate_means=True,
            lower_is_better=True,
            nonnegative_metric=True,
        )

        add_panel_label(
            axis,
            panel_label,
        )

        axis.set_title(
            title,
            loc="left",
            fontsize=11.8,
            fontweight="semibold",
            pad=8,
        )

        axis.tick_params(
            axis="both",
            which="major",
            direction="out",
            width=0.85,
            length=3.8,
        )

    figure.align_ylabels(
        axes[:, 0]
    )
    figure.align_ylabels(
        axes[:, 1]
    )

    # No figure-level title and no explanatory footer.
    # These belong in the manuscript caption / Results text.

    figure.subplots_adjust(
        left=0.085,
        right=0.985,
        bottom=0.085,
        top=0.955,
        wspace=0.27,
        hspace=0.38,
    )

    save_figure_r3(
        figure,
        "round3_observation_regime_state_distributions_v5",
    )

    plt.show()


# =============================================================================
# FIGURE 2 — PARAMETER RECOVERY AND IDENTIFIABILITY
# =============================================================================

with plt.rc_context(PUBLICATION_RC):

    figure, axes = plt.subplots(2, 2, figsize=(12.4, 9.1))

    parameter_panels = [
        ("mutation_probability", axes[0, 0], r"Estimated mutation probability, $\hat{\mu}$", "Mutation-probability recovery", True, float(TRUE_PARAMETERS.mutation_probability), False),
        ("resistance_fitness_cost", axes[0, 1], r"Estimated resistance fitness cost, $\hat{c}_R$", "Resistance-fitness-cost recovery", False, float(TRUE_PARAMETERS.resistance_fitness_cost), False),
        ("mu_error_percent", axes[1, 0], r"Relative error in $\mu$ (%)", "Mutation-probability identifiability", True, None, True),
        ("cost_error_percent", axes[1, 1], r"Relative error in $c_R$ (%)", "Fitness-cost identifiability", False, None, True),
    ]

    for panel_label, (metric, axis, ylabel, title, use_log, true_value, lower_is_better) in zip(
        ["A", "B", "C", "D"], parameter_panels
    ):
        plot_distribution_panel(
            axis,
            metric,
            ylabel,
            title,
            log_scale=use_log,
            true_value=true_value,
            annotate_means=True,
            lower_is_better=lower_is_better,
        )
        if true_value is not None:
            axis.legend(loc="upper right", frameon=True, fontsize=8.2)
        add_panel_label(axis, panel_label)

    figure.suptitle(
        "State observability and evolutionary-parameter identifiability are distinct",
        fontsize=15.7,
        fontweight="bold",
        y=0.975,
    )

    add_footer_note(
        figure,
        "Even when hidden-state reconstruction improves, parameter recovery can remain substantially harder.",
        y=0.038,
    )

    figure.subplots_adjust(
        left=0.10,
        right=0.985,
        bottom=0.11,
        top=0.90,
        wspace=0.25,
        hspace=0.35,
    )

    save_figure_r3(figure, "round3_observation_regime_parameter_distributions_v5")
    plt.show()


# =============================================================================
# FIGURE 3 — PHYSICS CONSISTENCY AND COMPUTATIONAL COST
# =============================================================================

with plt.rc_context(PUBLICATION_RC):

    figure, axes = plt.subplots(1, 2, figsize=(12.4, 5.2))

    plot_distribution_panel(
        axes[0],
        "physics_residual_rmse",
        "Physics-residual RMSE",
        "Dynamical consistency",
        log_scale=True,
        annotate_means=True,
        lower_is_better=True,
    )

    plot_distribution_panel(
        axes[1],
        "training_minutes",
        "Training time (min)",
        "Computational cost",
        log_scale=False,
        annotate_means=True,
        lower_is_better=False,
    )

    add_panel_label(axes[0], "A")
    add_panel_label(axes[1], "B")

    figure.suptitle(
        "Physics consistency and computational cost",
        fontsize=15.5,
        fontweight="bold",
        y=0.96,
    )

    add_footer_note(
        figure,
        "O3 is computationally cheapest on average, while O4 attains the best resistant-state reconstruction.",
        y=0.06,
    )

    figure.subplots_adjust(
        left=0.09,
        right=0.985,
        bottom=0.18,
        top=0.84,
        wspace=0.26,
    )

    save_figure_r3(figure, "round3_observation_regime_physics_cost_v5")
    plt.show()


# =============================================================================
# FIGURE 4 — INFORMATION GAIN RELATIVE TO O1
# =============================================================================

if GENERATE_EFFECT_SIZE_FIGURE:

    baseline_metrics = {
        metric: summary_value("O1", metric, "mean")
        for metric in ["rmse_resistant", "rmse_fraction", "mu_error_percent", "cost_error_percent"]
    }

    effect_rows = []
    for regime in REGIME_ORDER:
        row = {"regime": regime, "phenotype_points": EXPECTED_PHENOTYPE_POINTS[regime]}
        for metric, baseline in baseline_metrics.items():
            current = summary_value(regime, metric, "mean")
            row[f"{metric}_improvement_percent"] = 100.0 * (baseline - current) / baseline
        effect_rows.append(row)

    effect_sizes = pd.DataFrame(effect_rows)
    effect_sizes.to_csv(EFFECT_SIZE_PATH, index=False)

    with plt.rc_context(PUBLICATION_RC):

        figure, axes = plt.subplots(1, 2, figsize=(12.4, 5.2))
        x = np.arange(len(REGIME_ORDER))
        width = 0.34

        resistant_improvement = effect_sizes["rmse_resistant_improvement_percent"].to_numpy(float)
        fraction_improvement = effect_sizes["rmse_fraction_improvement_percent"].to_numpy(float)

        axes[0].bar(x - width / 2, resistant_improvement, width, label=r"RMSE $R$ reduction")
        axes[0].bar(x + width / 2, fraction_improvement, width, label=r"RMSE $q_R$ reduction")
        axes[0].axhline(0.0, color="black", linewidth=1.0)
        axes[0].set_ylabel("Improvement relative to O1 (%)")
        axes[0].set_title("Gain in resistant-state observability", loc="left", fontweight="semibold")
        axes[0].legend(loc="upper left", frameon=True)

        mu_improvement = effect_sizes["mu_error_percent_improvement_percent"].to_numpy(float)
        cost_improvement = effect_sizes["cost_error_percent_improvement_percent"].to_numpy(float)

        axes[1].bar(x - width / 2, mu_improvement, width, label=r"Error reduction in $\mu$")
        axes[1].bar(x + width / 2, cost_improvement, width, label=r"Error reduction in $c_R$")
        axes[1].axhline(0.0, color="black", linewidth=1.0)
        axes[1].set_ylabel("Improvement relative to O1 (%)")
        axes[1].set_title("Gain in parameter identifiability", loc="left", fontweight="semibold")
        axes[1].legend(loc="upper left", frameon=True)

        for panel_label, axis in zip(["A", "B"], axes):
            axis.set_xticks(x)
            axis.set_xticklabels(compact_regime_labels())
            axis.grid(True, axis="y")
            axis.set_axisbelow(True)
            set_black_frame(axis)
            add_panel_label(axis, panel_label)

        figure.suptitle(
            "Information gain from additional phenotype observations",
            fontsize=15.5,
            fontweight="bold",
            y=0.96,
        )

        add_footer_note(
            figure,
            "O4 provides the largest overall gain; O2 can help some quantities while degrading others.",
            y=0.065,
        )

        figure.subplots_adjust(
            left=0.09,
            right=0.985,
            bottom=0.18,
            top=0.84,
            wspace=0.28,
        )

        save_figure_r3(figure, "round3_observation_regime_information_gain_v5")
        plt.show()


# =============================================================================
# FIGURE 5 — PAIRED-SEED STABILITY
# =============================================================================

if GENERATE_PAIRED_SEED_FIGURE:

    with plt.rc_context(PUBLICATION_RC):

        figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.5))
        x = np.arange(len(REGIME_ORDER))

        seed_ids = sorted(observation_runs["seed_id"].astype(str).unique(), key=seed_sort_key)

        seed_colors = {
            seed_id: DEFAULT_COLORS[index % len(DEFAULT_COLORS)]
            for index, seed_id in enumerate(seed_ids)
        }

        for seed_id in seed_ids:
            seed_frame = observation_runs[observation_runs["seed_id"].astype(str) == seed_id].copy()
            seed_frame["regime"] = pd.Categorical(seed_frame["regime"], categories=REGIME_ORDER, ordered=True)
            seed_frame = seed_frame.sort_values("regime").reset_index(drop=True)

            if len(seed_frame) != len(REGIME_ORDER):
                continue

            axes[0].plot(
                x,
                seed_frame["rmse_resistant"].to_numpy(float),
                marker="o",
                markersize=4.8,
                linewidth=1.05,
                alpha=0.50,
                color=seed_colors[seed_id],
                label=seed_id,
            )

            axes[1].plot(
                x,
                seed_frame["mu_error_percent"].to_numpy(float),
                marker="o",
                markersize=4.8,
                linewidth=1.05,
                alpha=0.50,
                color=seed_colors[seed_id],
                label=seed_id,
            )

        resistant_mean = np.array([summary_value(regime, "rmse_resistant", "mean") for regime in REGIME_ORDER])
        mu_error_mean = np.array([summary_value(regime, "mu_error_percent", "mean") for regime in REGIME_ORDER])

        axes[0].plot(
            x,
            resistant_mean,
            color="black",
            marker="D",
            markersize=6.8,
            linewidth=2.25,
            label="Mean",
            zorder=6,
        )

        axes[1].plot(
            x,
            mu_error_mean,
            color="black",
            marker="D",
            markersize=6.8,
            linewidth=2.25,
            label="Mean",
            zorder=6,
        )

        axes[0].set_ylabel(r"Resistant-state RMSE")
        axes[0].set_title("Seed-level resistant-state reconstruction", loc="left", fontweight="semibold")

        axes[1].set_yscale("log")
        axes[1].set_ylabel(r"Relative error in $\mu$ (%)")
        axes[1].set_title("Seed-level mutation-identification error", loc="left", fontweight="semibold")

        for panel_label, axis in zip(["A", "B"], axes):
            axis.set_xticks(x)
            axis.set_xticklabels(compact_regime_labels())
            axis.set_xlim(-0.25, len(REGIME_ORDER) - 0.75)
            axis.grid(True, axis="y")
            axis.set_axisbelow(True)
            set_black_frame(axis)
            add_panel_label(axis, panel_label)

        legend_handles, legend_labels = axes[1].get_legend_handles_labels()

        figure.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.90),
            ncol=3,
            frameon=True,
            columnspacing=1.2,
            handlelength=2.0,
        )

        figure.suptitle(
            "Paired-seed stability across observation regimes",
            fontsize=15.7,
            fontweight="bold",
            y=0.975,
        )

        add_footer_note(
            figure,
            "Thin colored lines show paired runs for each seed; the black line shows the across-seed mean.",
            y=0.065,
        )

        figure.subplots_adjust(
            left=0.08,
            right=0.985,
            bottom=0.18,
            top=0.77,
            wspace=0.28,
        )

        save_figure_r3(figure, "round3_observation_regime_paired_seed_stability_v5")
        plt.show()


# =============================================================================
# MANUSCRIPT TABLE
# =============================================================================

table_metrics = [
    "rmse_sensitive",
    "rmse_resistant",
    "rmse_total",
    "rmse_fraction",
    "mutation_probability",
    "mu_error_percent",
    "resistance_fitness_cost",
    "cost_error_percent",
    "physics_residual_rmse",
    "training_minutes",
]

table_rows = []
for regime in REGIME_ORDER:
    row = {
        "Regime": regime,
        "Phenotype information": REGIME_DESCRIPTIONS[regime],
        "Phenotype points": EXPECTED_PHENOTYPE_POINTS[regime],
        "Seeds": int(seed_counts.loc[regime]),
    }
    for metric in table_metrics:
        row[metric] = format_mean_sd(
            summary_value(regime, metric, "mean"),
            summary_value(regime, metric, "std"),
            metric,
        )
    table_rows.append(row)

final_manuscript_table = (
    pd.DataFrame(table_rows)
    .rename(
        columns={
            "rmse_sensitive": "RMSE S",
            "rmse_resistant": "RMSE R",
            "rmse_total": "RMSE total",
            "rmse_fraction": "RMSE fraction",
            "mutation_probability": "Estimated mu",
            "mu_error_percent": "Mu error (%)",
            "resistance_fitness_cost": "Estimated c_R",
            "cost_error_percent": "c_R error (%)",
            "physics_residual_rmse": "Physics RMSE",
            "training_minutes": "Training time (min)",
        }
    )
)

final_manuscript_table.to_csv(FINAL_TABLE_PATH, index=False)


# =============================================================================
# FEASIBILITY TABLE
# =============================================================================

feasibility_rows = []
for regime in REGIME_ORDER:
    feasibility_rows.append({
        "Regime": regime,
        "Negative-point fraction": format_mean_sd(
            summary_value(regime, "fraction_points_negative", "mean"),
            summary_value(regime, "fraction_points_negative", "std"),
            "fraction_points_negative",
        ),
        "Over-capacity fraction": format_mean_sd(
            summary_value(regime, "fraction_points_over_capacity", "mean"),
            summary_value(regime, "fraction_points_over_capacity", "std"),
            "fraction_points_over_capacity",
        ),
    })

feasibility_summary = pd.DataFrame(feasibility_rows)
feasibility_summary.to_csv(FEASIBILITY_SUMMARY_PATH, index=False)


# =============================================================================
# FINAL REPORT
# =============================================================================

o1_rmse_r = summary_value("O1", "rmse_resistant", "mean")
o4_rmse_r = summary_value("O4", "rmse_resistant", "mean")
o1_rmse_fraction = summary_value("O1", "rmse_fraction", "mean")
o4_rmse_fraction = summary_value("O4", "rmse_fraction", "mean")

resistant_reduction_percent = 100.0 * (o1_rmse_r - o4_rmse_r) / o1_rmse_r
fraction_reduction_percent = 100.0 * (o1_rmse_fraction - o4_rmse_fraction) / o1_rmse_fraction

print("\n" + "=" * 78)
print("REFINED OBSERVATION-REGIME FIGURES COMPLETED")
print("=" * 78)

print("\nGenerated figures:")
print("  round3_observation_regime_state_distributions_v5")
print("  round3_observation_regime_parameter_distributions_v5")
print("  round3_observation_regime_physics_cost_v5")
if GENERATE_EFFECT_SIZE_FIGURE:
    print("  round3_observation_regime_information_gain_v5")
if GENERATE_PAIRED_SEED_FIGURE:
    print("  round3_observation_regime_paired_seed_stability_v5")

print("\nGenerated tables:")
print(" ", FINAL_TABLE_PATH)
print(" ", NUMERIC_SUMMARY_PATH)
print(" ", FEASIBILITY_SUMMARY_PATH)
if GENERATE_EFFECT_SIZE_FIGURE:
    print(" ", EFFECT_SIZE_PATH)

print("\nKey O4 improvements relative to O1:")
print(f"  Resistant-state RMSE reduction: {resistant_reduction_percent:.1f}%")
print(f"  Resistant-fraction RMSE reduction: {fraction_reduction_percent:.1f}%")

print("\nFinal manuscript-ready table:")
display(final_manuscript_table)

print("\nFeasibility summary:")
display(feasibility_summary)

print("\nNo retraining was performed in this cell.")
print("Main manuscript state figure updated: round3_observation_regime_state_distributions_v5")

Round 4

In [ ]:
# =============================================================================
# ROUND 4 — CELL 1
# Final primary multi-schedule experiment configuration for an 8 GB GPU
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# Round 4 tests whether combining several prescribed antibiotic-treatment
# schedules improves practical identifiability of the shared eco-evolutionary
# parameters:
#
#     mutation probability, mu,
#     resistance fitness cost, c_R.
#
# Every treatment schedule has its own hard-constrained neural state model:
#
#     S_j(t), R_j(t),
#
# while the biological parameters mu and c_R are shared across all schedules
# included in the same experimental design.
#
# The antibiotic treatment is prescribed and is NOT optimized in Round 4.
#
# FINAL PRIMARY EXPERIMENT
# ------------------------
# This configuration activates the five-seed M1–M5 study:
#
#     M1: early-continuous schedule, O3 observations
#     M2: early + late schedules, O3 observations
#     M3: early + late + pulsed schedules, O3 observations
#     M4: untreated + early + late + pulsed schedules, O3 observations
#     M5: all four schedules, O4 observations
#
# Number of primary models:
#
#     5 experimental designs × 5 paired seeds = 25 trained models
#
# EXECUTION POLICY FOR AN 8 GB GPU
# ---------------------------------
# 1. Train only one design-seed model at a time.
# 2. Use float64 for stable derivatives and small mutation probabilities.
# 3. Move completed models to CPU before starting the next model.
# 4. Clear unused Python and CUDA memory after every completed run.
# 5. Save each completed design-seed result immediately.
# 6. Resume from saved results after interruption.
# 7. Activate only one expensive Round 4 suite at a time.
#
# IMPORTANT EXECUTION INSTRUCTION
# -------------------------------
# After replacing the earlier quick-mode Cell 1 with this final version:
#
#     1. Save the notebook.
#     2. Restart the kernel.
#     3. Run Cells 1–14 sequentially.
#     4. Skip Cell 15 because the structural smoke test is disabled.
#     5. Skip Cell 16 because the single-design demonstration is disabled.
#     6. Run Cell 17 to train the 25 primary models.
#     7. Run Cells 18–20 after Cell 17 finishes.
#
# Do not activate the optional ODE, identifiability, profile, transfer, or
# plotting-only suites during this primary training run.
# =============================================================================

from pathlib import Path

import os
import sys
import platform
import random
import json
import math
import time
import copy
import gc
import warnings

from dataclasses import dataclass, asdict, replace, field
from typing import (
    Dict,
    List,
    Tuple,
    Optional,
    Iterable,
    Any,
    Sequence,
)

import numpy as np
import pandas as pd


# =============================================================================
# GLOBAL REPRODUCIBILITY SEED
# =============================================================================

SEED = 20260806

random.seed(SEED)
np.random.seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)


# =============================================================================
# EXECUTION MODE
# =============================================================================
# False activates the full manuscript-scale budgets and five paired seeds.

QUICK_MODE: bool = False

if not isinstance(QUICK_MODE, bool):
    raise TypeError(
        "QUICK_MODE must be either True or False."
    )

RUN_LABEL = (
    "quick"
    if QUICK_MODE
    else "final"
)


# =============================================================================
# PROJECT AND OUTPUT FOLDERS
# =============================================================================

PROJECT_ROOT = Path.cwd()

# When the notebook is stored in a project/notebooks folder, use the parent
# directory as the project root.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ROUND4_BASE_ROOT = (
    PROJECT_ROOT
    / "round4"
)

ROUND4_ROOT = (
    ROUND4_BASE_ROOT
    / RUN_LABEL
)

MODEL_DIR_R4 = (
    ROUND4_ROOT
    / "models"
)

PREDICTION_DIR_R4 = (
    ROUND4_ROOT
    / "predictions"
)

HISTORY_DIR_R4 = (
    ROUND4_ROOT
    / "histories"
)

FIGURE_DIR_R4 = (
    ROUND4_ROOT
    / "figures"
)

TABLE_DIR_R4 = (
    ROUND4_ROOT
    / "tables"
)

REPORT_DIR_R4 = (
    ROUND4_ROOT
    / "reports"
)

CACHE_DIR_R4 = (
    ROUND4_ROOT
    / "cache"
)

ROUND4_DIRECTORIES = [
    MODEL_DIR_R4,
    PREDICTION_DIR_R4,
    HISTORY_DIR_R4,
    FIGURE_DIR_R4,
    TABLE_DIR_R4,
    REPORT_DIR_R4,
    CACHE_DIR_R4,
]

for directory in ROUND4_DIRECTORIES:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# GPU-AWARE TRAINING BUDGETS
# =============================================================================
# The quick budgets remain defined for future debugging, but this final Cell 1
# uses the manuscript-scale branch because QUICK_MODE=False.

if QUICK_MODE:

    # -------------------------------------------------------------------------
    # Structural verification only
    # -------------------------------------------------------------------------

    PRETRAIN_EPOCHS_R4 = 15

    PHYSICS_WARMUP_EPOCHS_R4 = 15

    ADAM_EPOCHS_R4 = 30

    LBFGS_MAX_ITER_R4 = 0

    COLLOCATION_POINTS_PER_INTERVAL_R4 = 32

    VALIDATION_POINTS_PER_INTERVAL_R4 = 16

    LBFGS_POINTS_PER_INTERVAL_R4 = 24

    EARLY_STOPPING_PATIENCE_R4 = 30

    LOG_EVERY_R4 = 5

    RESAMPLE_EVERY_R4 = 5

    PRIMARY_SEED_IDS_R4 = [1]

else:

    # -------------------------------------------------------------------------
    # Final manuscript-scale experiment for an 8 GB GPU
    # -------------------------------------------------------------------------

    PRETRAIN_EPOCHS_R4 = 1500

    PHYSICS_WARMUP_EPOCHS_R4 = 1000

    ADAM_EPOCHS_R4 = 9000

    LBFGS_MAX_ITER_R4 = 300

    COLLOCATION_POINTS_PER_INTERVAL_R4 = 768

    VALIDATION_POINTS_PER_INTERVAL_R4 = 128

    LBFGS_POINTS_PER_INTERVAL_R4 = 256

    EARLY_STOPPING_PATIENCE_R4 = 1800

    LOG_EVERY_R4 = 100

    RESAMPLE_EVERY_R4 = 20

    PRIMARY_SEED_IDS_R4 = [
        1,
        2,
        3,
        4,
        5,
    ]


# =============================================================================
# NEURAL-NETWORK AND OPTIMIZER SETTINGS
# =============================================================================

NETWORK_WIDTH_R4 = 64

NETWORK_DEPTH_R4 = 4

# State-network learning rate
STATE_LEARNING_RATE_R4 = 8.0e-4

# Shared biological-parameter learning rate
PARAMETER_LEARNING_RATE_R4 = 2.0e-3

# Protects against occasional unstable gradient steps
GRADIENT_CLIP_NORM_R4 = 10.0


# =============================================================================
# PRIMARY EXPERIMENT SWITCHES
# =============================================================================
# Only the final five-seed M1–M5 primary experiment is activated here.

RUN_STRUCTURAL_SMOKE_TEST = False

RUN_SINGLE_DESIGN_DEMO = False

RUN_PRIMARY_MULTI_SCHEDULE_STUDY = True

RUN_MATCHED_ODE_BASELINES = False

RUN_IDENTIFIABILITY_GEOMETRY = False

RUN_LEAVE_ONE_SCHEDULE_OUT = False

RUN_PROFILE_SURFACES = False

RUN_PARAMETER_TRANSFER_TEST = False

RUN_PLOTTING_ONLY = False


# =============================================================================
# EXPERIMENT-SWITCH REGISTRY
# =============================================================================

EXPERIMENT_SWITCHES_R4 = {
    "structural_smoke_test":
        RUN_STRUCTURAL_SMOKE_TEST,

    "single_design_demo":
        RUN_SINGLE_DESIGN_DEMO,

    "primary_multi_schedule_study":
        RUN_PRIMARY_MULTI_SCHEDULE_STUDY,

    "matched_ode_baselines":
        RUN_MATCHED_ODE_BASELINES,

    "identifiability_geometry":
        RUN_IDENTIFIABILITY_GEOMETRY,

    "leave_one_schedule_out":
        RUN_LEAVE_ONE_SCHEDULE_OUT,

    "profile_surfaces":
        RUN_PROFILE_SURFACES,

    "parameter_transfer_test":
        RUN_PARAMETER_TRANSFER_TEST,

    "plotting_only":
        RUN_PLOTTING_ONLY,
}

active_experiments_r4 = [
    experiment_name
    for experiment_name, enabled
    in EXPERIMENT_SWITCHES_R4.items()
    if enabled
]


# =============================================================================
# CONFIGURATION VALIDATION
# =============================================================================

if len(active_experiments_r4) == 0:
    raise RuntimeError(
        "No Round 4 experiment is active. "
        "Enable exactly one experiment switch."
    )

if len(active_experiments_r4) > 1:
    raise RuntimeError(
        "More than one Round 4 experiment is active:\n"
        f"  {active_experiments_r4}\n"
        "Activate only one expensive suite at a time on the 8 GB GPU."
    )

CURRENT_EXPERIMENT_R4 = (
    active_experiments_r4[0]
)

if QUICK_MODE and RUN_PRIMARY_MULTI_SCHEDULE_STUDY:
    warnings.warn(
        "The primary multi-schedule study is active while QUICK_MODE=True. "
        "This would produce only preliminary debugging results."
    )

if (
    not QUICK_MODE
    and RUN_STRUCTURAL_SMOKE_TEST
):
    warnings.warn(
        "The structural smoke test is active while QUICK_MODE=False. "
        "The smoke test should normally use QUICK_MODE=True."
    )

if (
    RUN_PRIMARY_MULTI_SCHEDULE_STUDY
    and QUICK_MODE
):
    raise RuntimeError(
        "The final primary study requires QUICK_MODE=False."
    )

if (
    RUN_PRIMARY_MULTI_SCHEDULE_STUDY
    and PRIMARY_SEED_IDS_R4
    != [1, 2, 3, 4, 5]
):
    raise RuntimeError(
        "The final primary experiment must use five paired seeds: "
        "[1, 2, 3, 4, 5]."
    )


# =============================================================================
# EXPECTED PRIMARY-STUDY SIZE
# =============================================================================

PRIMARY_DESIGN_IDS_R4 = [
    "M1",
    "M2",
    "M3",
    "M4",
    "M5",
]

EXPECTED_PRIMARY_MODELS_R4 = (
    len(PRIMARY_DESIGN_IDS_R4)
    * len(PRIMARY_SEED_IDS_R4)
)


# =============================================================================
# RESUME-SAFE PRIMARY RESULTS PATH
# =============================================================================

PRIMARY_RESULTS_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_primary_multischedule_all_seeds_metrics.csv"
)

PRIMARY_SUMMARY_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_primary_multischedule_summary_mean_std.csv"
)

PRIMARY_MANUSCRIPT_TABLE_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_primary_multischedule_manuscript_table.csv"
)


# =============================================================================
# COMPATIBILITY ALIASES
# =============================================================================
# Some reusable utilities use the generic directory names below. They are
# mapped to the active Round 4 final-output folders.

MODEL_DIR = MODEL_DIR_R4

PREDICTION_DIR = PREDICTION_DIR_R4

HISTORY_DIR = HISTORY_DIR_R4

FIGURE_DIR = FIGURE_DIR_R4

TABLE_DIR = TABLE_DIR_R4

REPORT_DIR = REPORT_DIR_R4

CACHE_DIR = CACHE_DIR_R4


# =============================================================================
# PRINT AND VERIFY THE FINAL CONFIGURATION
# =============================================================================

print("=" * 78)

print(
    "ROUND 4 EXECUTION CONFIGURATION"
)

print("=" * 78)

print(
    "Project root:",
    PROJECT_ROOT.resolve(),
)

print(
    "Round 4 base root:",
    ROUND4_BASE_ROOT.resolve(),
)

print(
    "Output root:",
    ROUND4_ROOT.resolve(),
)

print(
    "Python:",
    sys.version.split()[0],
)

print(
    "Operating system:",
    platform.platform(),
)

print(
    "Random seed:",
    SEED,
)

print(
    "QUICK_MODE:",
    QUICK_MODE,
)

print(
    "Run label:",
    RUN_LABEL,
)

print(
    "Current experiment:",
    CURRENT_EXPERIMENT_R4,
)

print(
    "Primary designs:",
    PRIMARY_DESIGN_IDS_R4,
)

print(
    "Primary seeds:",
    PRIMARY_SEED_IDS_R4,
)

print(
    "Expected primary models:",
    EXPECTED_PRIMARY_MODELS_R4,
)

print(
    "Pretraining epochs:",
    PRETRAIN_EPOCHS_R4,
)

print(
    "Physics warm-up epochs:",
    PHYSICS_WARMUP_EPOCHS_R4,
)

print(
    "Joint Adam epochs:",
    ADAM_EPOCHS_R4,
)

print(
    "L-BFGS maximum iterations:",
    LBFGS_MAX_ITER_R4,
)

print(
    "Collocation points per interval:",
    COLLOCATION_POINTS_PER_INTERVAL_R4,
)

print(
    "Validation points per interval:",
    VALIDATION_POINTS_PER_INTERVAL_R4,
)

print(
    "L-BFGS points per interval:",
    LBFGS_POINTS_PER_INTERVAL_R4,
)

print(
    "Early-stopping patience:",
    EARLY_STOPPING_PATIENCE_R4,
)

print(
    "Logging interval:",
    LOG_EVERY_R4,
)

print(
    "Collocation-resampling interval:",
    RESAMPLE_EVERY_R4,
)

print(
    "Network width:",
    NETWORK_WIDTH_R4,
)

print(
    "Network depth:",
    NETWORK_DEPTH_R4,
)

print(
    "State learning rate:",
    STATE_LEARNING_RATE_R4,
)

print(
    "Parameter learning rate:",
    PARAMETER_LEARNING_RATE_R4,
)

print(
    "Gradient clipping norm:",
    GRADIENT_CLIP_NORM_R4,
)

print(
    "Active suites:",
    active_experiments_r4,
)

print(
    "Resume-safe primary table:",
    PRIMARY_RESULTS_PATH_R4.resolve(),
)

print("=" * 78)


# =============================================================================
# FINAL SAFETY ASSERTIONS
# =============================================================================

assert QUICK_MODE is False

assert RUN_STRUCTURAL_SMOKE_TEST is False

assert RUN_SINGLE_DESIGN_DEMO is False

assert RUN_PRIMARY_MULTI_SCHEDULE_STUDY is True

assert RUN_MATCHED_ODE_BASELINES is False

assert RUN_IDENTIFIABILITY_GEOMETRY is False

assert RUN_LEAVE_ONE_SCHEDULE_OUT is False

assert RUN_PROFILE_SURFACES is False

assert RUN_PARAMETER_TRANSFER_TEST is False

assert RUN_PLOTTING_ONLY is False

assert active_experiments_r4 == [
    "primary_multi_schedule_study"
]

assert PRIMARY_SEED_IDS_R4 == [
    1,
    2,
    3,
    4,
    5,
]

assert RUN_LABEL == "final"

assert ROUND4_ROOT.name == "final"


print(
    "\nCell 1 completed successfully."
)

print(
    "Final manuscript-scale M1–M5 primary configuration is active."
)

print(
    "Restarted kernel confirmed: run Cells 2–14 sequentially, "
    "skip Cells 15–16, and then run Cell 17."
)

In [ ]:

# =============================================================================
# ROUND 4 — CELL 2
# Scientific imports, PyTorch precision, deterministic settings, and figures
# =============================================================================
# Why double precision is used
# ----------------------------
# PINNs differentiate network outputs with respect to time and optimize small
# mutation probabilities. Float64 substantially improves residual and parameter
# stability. The network and collocation sets are small enough that an 8 GB GPU
# can use float64 without difficulty.
#
# Figure policy
# -------------
# Every manuscript figure is saved as:
# - PNG at 400 dpi for convenient viewing;
# - PDF for vector-quality publication;
# - SVG for editing in Inkscape/Illustrator.

import scipy
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares, minimize_scalar
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, LogLocator
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

import torch
from IPython.display import display
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore', category=UserWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
# Float64 is selected explicitly rather than through torch.set_default_dtype so
# external libraries and integer tensors are not affected unexpectedly.

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 400,
    'font.family': 'sans-serif',
    'font.size': 10.5,
    'axes.labelsize': 11.5,
    'axes.titlesize': 12.5,
    'legend.fontsize': 9.0,
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.0,
    'legend.frameon': True,
    'legend.framealpha': 1.0,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})


def save_figure_r4(fig: plt.Figure, stem: str, close: bool = False) -> None:
    """Save a Round 4 figure in high-resolution raster and vector formats."""
    clean = stem.strip().replace(' ', '_')
    fig.savefig(FIGURE_DIR_R4 / f'{clean}.png', dpi=400,
                bbox_inches='tight', facecolor='white')
    fig.savefig(FIGURE_DIR_R4 / f'{clean}.pdf',
                bbox_inches='tight', facecolor='white')
    fig.savefig(FIGURE_DIR_R4 / f'{clean}.svg',
                bbox_inches='tight', facecolor='white')
    if close:
        plt.close(fig)


def clear_gpu_memory() -> None:
    """Release unused Python and CUDA memory between sequential experiments."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('PyTorch:', torch.__version__)
print('SciPy:', scipy.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Device:', DEVICE)
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name)
    print(f'GPU memory: {properties.total_memory / 1024**3:.2f} GB')
print('Tensor dtype:', DTYPE)

In [ ]:
# =============================================================================
# ROUND 4 — CELL 3
# Six-repository provenance, scientific boundaries, and Round 4 hypotheses
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# The implementation deliberately reuses verified ideas without claiming those
# ideas as new. The proposed contribution is their AMR-specific synthesis:
# exact pointwise biological feasibility plus shared-parameter inference across
# heterogeneous prescribed treatment environments.

REPOSITORY_PROVENANCE_R4 = {
    "DrugRes_StructPop": {
        "used": [
            "sensitive/resistant birth, death, mutation, and crowding structure",
            "piecewise treatment changes applied exactly at switch times",
            "evolutionary-rescue interpretation and later stochastic validation",
        ],
        "not_claimed_as_novel": "The AMR population process and Gillespie method.",
        "round4_role": (
            "Supplies the deterministic mean-field biology and motivates "
            "schedule-specific treatment interfaces."
        ),
    },
    "BINNs": {
        "used": [
            "inverse biological learning from sparse observations",
            "separation of observation loss and mechanistic residual loss",
            "explicit normalization and held-out validation",
        ],
        "not_claimed_as_novel": "Sparse biological inverse learning.",
        "round4_role": (
            "Motivates joint inference from total burden and sparse phenotype fractions."
        ),
    },
    "PINN-Proj": {
        "used": [
            "hard constraints implemented through differentiable output mappings",
            "separate predictive and feasibility audits",
            "first-order optimization followed by optional L-BFGS refinement",
        ],
        "not_claimed_as_novel": "Projection-based exact conservation in PINNs.",
        "round4_role": (
            "Motivates exact feasibility and controlled comparisons, but no "
            "integral projection is copied into the AMR state mapping."
        ),
    },
    "Control-PINNs": {
        "used": [
            "treatment represented explicitly as an input to the dynamics",
            "one model containing multiple logically coupled components",
        ],
        "excluded": [
            "adjoint equations",
            "stationarity residuals",
            "learned or optimized treatment controls",
        ],
        "round4_role": (
            "Clarifies that Round 4 is multi-environment inverse inference, "
            "not antibiotic optimal control."
        ),
    },
    "evo_dm": {
        "used": [
            "schedule/environment diversity as evolutionary excitation",
            "fitness-cost interpretation across drug environments",
        ],
        "excluded": [
            "DQN training",
            "MDP policy optimization",
            "genotype-level policy claims",
        ],
        "round4_role": (
            "Motivates heterogeneous prescribed schedules as complementary "
            "experiments without importing the RL architecture."
        ),
    },
    "KKT_hPINN": {
        "used": [
            "training through an exact differentiable feasibility layer",
            "hard versus soft constraint reasoning",
            "independent reporting of feasibility residuals",
        ],
        "not_claimed_as_novel": "KKT projection for affine equality constraints.",
        "round4_role": (
            "Motivates exact enforcement, while the AMR inequalities and "
            "simplex/capacity structure require a different nonlinear mapping."
        ),
    },
}

ROUND4_HYPOTHESES = {
    "H1": (
        "Joint inference across heterogeneous schedules increases the weakest "
        "parameter-sensitivity direction relative to early treatment alone."
    ),
    "H2": (
        "Shared-parameter multi-schedule fitting reduces relative error in mu "
        "and c_R across paired seeds."
    ),
    "H3": (
        "The hard output mapping preserves S>=0, R>=0, and S+R<=K for every "
        "schedule, design, seed, and dense evaluation point."
    ),
    "H4": (
        "Leave-one-schedule-out and parameter-transfer tests reveal which "
        "treatment environments contribute complementary information."
    ),
}

NOVELTY_BOUNDARY_R4 = {
    "proposed_round4_contribution": [
        "shared eco-evolutionary parameter inference across prescribed treatment schedules",
        "schedule-specific interface networks with one shared bounded parameter module",
        "paired-seed comparison of nested schedule designs",
        "sensitivity-SVD, profile geometry, leave-one-out, and transfer analyses",
        "exact pointwise positivity, phenotype simplex, capacity, initial state, and interface continuity",
    ],
    "not_claimed": [
        "first PINN",
        "first AMR model",
        "first hard-constrained neural network",
        "first multi-task neural network",
        "first antibiotic RL or optimal-control method",
    ],
}

with open(
    REPORT_DIR_R4 / "round4_repository_provenance.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "repositories": REPOSITORY_PROVENANCE_R4,
            "hypotheses": ROUND4_HYPOTHESES,
            "novelty_boundary": NOVELTY_BOUNDARY_R4,
        },
        file,
        indent=2,
    )

provenance_table_r4 = pd.DataFrame([
    {
        "repository": name,
        "round4_role": details.get("round4_role", ""),
        "excluded_or_not_novel": "; ".join(details.get(
            "excluded",
            [details.get("not_claimed_as_novel", "")]
        )),
    }
    for name, details in REPOSITORY_PROVENANCE_R4.items()
])

display(provenance_table_r4)

In [ ]:
# ROUND 4 — CELL 4
# Eco-evolutionary AMR parameters, treatment schedules, and deterministic solver
# =============================================================================
# The state is normalized by carrying capacity during PINN training:
#     s = S/K, r = R/K.
# This keeps both states and ODE residuals near order one and improves PINN
# optimization. Dimensional bacterial counts are restored for all reports.

@dataclass(frozen=True)
class AMRParametersR4:
    carrying_capacity: float = 1000.0
    sensitive_birth_rate: float = 1.0
    resistance_fitness_cost: float = 0.10
    sensitive_death_rate: float = 0.10
    resistant_death_rate: float = 0.10
    mutation_probability: float = 1.0e-4
    sensitive_drug_effect: float = 1.0
    resistant_drug_effect: float = 0.10

    @property
    def resistant_birth_rate(self) -> float:
        return self.sensitive_birth_rate * (1.0 - self.resistance_fitness_cost)

    def validate(self) -> None:
        if self.carrying_capacity <= 0 or self.sensitive_birth_rate <= 0:
            raise ValueError('Capacity and sensitive birth rate must be positive.')
        for name, value in {
            'resistance_fitness_cost': self.resistance_fitness_cost,
            'mutation_probability': self.mutation_probability,
            'sensitive_drug_effect': self.sensitive_drug_effect,
            'resistant_drug_effect': self.resistant_drug_effect,
        }.items():
            if not 0 <= value <= 1:
                raise ValueError(f'{name} must lie in [0,1].')


@dataclass(frozen=True)
class ScheduleR4:
    name: str
    switch_times: Tuple[float, ...]
    levels: Tuple[float, ...]
    description: str

    def __post_init__(self):
        if len(self.levels) != len(self.switch_times) + 1:
            raise ValueError('A piecewise schedule requires one more level than switches.')
        if tuple(sorted(self.switch_times)) != self.switch_times:
            raise ValueError('switch_times must be strictly ordered.')
        if any((level < 0 or level > 1) for level in self.levels):
            raise ValueError('Exposure levels must remain in [0,1].')

    def exposure_numpy(self, t: np.ndarray | float) -> np.ndarray | float:
        arr = np.asarray(t, dtype=float)
        indices = np.searchsorted(np.asarray(self.switch_times), arr, side='right')
        values = np.asarray(self.levels, dtype=float)[indices]
        return float(values) if np.isscalar(t) else values

    def exposure_torch(self, t: torch.Tensor) -> torch.Tensor:
        result = torch.full_like(t, float(self.levels[0]))
        for switch, level in zip(self.switch_times, self.levels[1:]):
            result = torch.where(t >= float(switch),
                                 torch.full_like(t, float(level)), result)
        return result

    def intervals(self, start: float, final: float) -> List[Tuple[float, float, float]]:
        boundaries = [start] + [x for x in self.switch_times if start < x < final] + [final]
        intervals = []
        for left, right in zip(boundaries[:-1], boundaries[1:]):
            midpoint = 0.5 * (left + right)
            intervals.append((left, right, float(self.exposure_numpy(midpoint))))
        return intervals


TRUE_PARAMETERS = AMRParametersR4()
TRUE_PARAMETERS.validate()
INITIAL_S = 100.0
INITIAL_R = 0.0
T_START = 0.0
T_FINAL = 20.0
DENSE_POINTS = 1001

SCHEDULES_R4 = {
    'untreated': ScheduleR4('untreated', (), (0.0,), 'No antibiotic exposure.'),
    'early_continuous': ScheduleR4('early_continuous', (5.0,), (0.0, 1.0),
                                   'Continuous exposure beginning at t=5.'),
    'late_continuous': ScheduleR4('late_continuous', (10.0,), (0.0, 1.0),
                                  'Continuous exposure beginning at t=10.'),
    'pulsed': ScheduleR4('pulsed', (5.0, 8.0, 12.0, 15.0),
                         (0.0, 1.0, 0.0, 1.0, 0.0),
                         'Exposure on [5,8) and [12,15).'),
}


def normalized_amr_rhs_numpy(t: float, state: np.ndarray,
                             parameters: AMRParametersR4,
                             schedule: ScheduleR4) -> np.ndarray:
    """Mean-field ODE for normalized sensitive and resistant abundances."""
    s, r = np.asarray(state, dtype=float)
    exposure = float(schedule.exposure_numpy(t))
    b_s = parameters.sensitive_birth_rate * (1.0 - parameters.sensitive_drug_effect * exposure)
    b_r = parameters.resistant_birth_rate * (1.0 - parameters.resistant_drug_effect * exposure)
    crowding = 1.0 - s - r
    ds = b_s * (1.0 - parameters.mutation_probability) * crowding * s \
         - parameters.sensitive_death_rate * s
    dr = b_s * parameters.mutation_probability * crowding * s \
         + b_r * crowding * r - parameters.resistant_death_rate * r
    return np.array([ds, dr], dtype=float)


def solve_reference_r4(parameters: AMRParametersR4,
                       schedule: ScheduleR4,
                       times: Optional[np.ndarray] = None) -> pd.DataFrame:
    """Solve each treatment interval separately to respect coefficient jumps."""
    if times is None:
        times = np.linspace(T_START, T_FINAL, DENSE_POINTS)
    times = np.asarray(times, dtype=float)
    current_state = np.array([INITIAL_S / parameters.carrying_capacity,
                              INITIAL_R / parameters.carrying_capacity], dtype=float)
    out_s = np.empty_like(times)
    out_r = np.empty_like(times)

    for interval_index, (left, right, _) in enumerate(schedule.intervals(T_START, T_FINAL)):
        is_last = interval_index == len(schedule.intervals(T_START, T_FINAL)) - 1
        mask = (times >= left - 1e-12) & ((times <= right + 1e-12) if is_last else (times < right - 1e-12))
        interval_times = times[mask]
        eval_times = np.unique(np.concatenate(([left], interval_times, [right])))
        sol = solve_ivp(
            lambda t, y: normalized_amr_rhs_numpy(t, y, parameters, schedule),
            (left, right), current_state, t_eval=eval_times,
            method='DOP853', rtol=1e-10, atol=1e-12,
        )
        if not sol.success:
            raise RuntimeError(sol.message)
        if interval_times.size:
            indices = np.searchsorted(eval_times, interval_times)
            out_s[mask] = sol.y[0, indices]
            out_r[mask] = sol.y[1, indices]
        current_state = sol.y[:, -1]

    S = parameters.carrying_capacity * out_s
    R = parameters.carrying_capacity * out_r
    B = S + R
    q = np.divide(R, B, out=np.zeros_like(R), where=B > 0)
    return pd.DataFrame({
        'time': times, 'sensitive': S, 'resistant': R, 'total': B,
        'resistant_fraction': q,
        'antibiotic': schedule.exposure_numpy(times),
        'schedule': schedule.name,
    })

print(pd.DataFrame([asdict(TRUE_PARAMETERS) | {'resistant_birth_rate': TRUE_PARAMETERS.resistant_birth_rate}]))
print(pd.DataFrame([{'schedule': s.name, 'description': s.description}
                    for s in SCHEDULES_R4.values()]))

In [ ]:
# ROUND 4 — CELL 5
# Load Round 1 data, verify it, and construct O1–O4 observation regimes
# =============================================================================
# Data loading strategy
# ---------------------
# The cell first searches for the dense CSV files produced in Round 1. When a
# file is unavailable, it regenerates the same high-accuracy deterministic
# trajectory. Therefore, Round 4 remains reproducible even after a kernel restart.
#
# Observation regimes
# -------------------
# O1: 21 total-load values, no phenotype fraction.
# O2: 21 total-load values, one phenotype fraction at t=12.
# O3: 21 total-load values, phenotype fractions at t=7,12,18.
# O4: 21 total-load values and phenotype fractions at all 21 times.

@dataclass
class ObservationDataset:
    schedule_name: str
    regime: str
    noise_level: float
    total_time: np.ndarray
    total_observed: np.ndarray
    fraction_time: np.ndarray
    fraction_observed: np.ndarray
    total_train_mask: np.ndarray
    total_validation_mask: np.ndarray
    fraction_train_mask: np.ndarray
    fraction_validation_mask: np.ndarray

    def summary(self) -> Dict[str, Any]:
        return {
            'schedule': self.schedule_name,
            'regime': self.regime,
            'noise_level': self.noise_level,
            'total_points': len(self.total_time),
            'fraction_points': len(self.fraction_time),
            'total_train': int(self.total_train_mask.sum()),
            'total_validation': int(self.total_validation_mask.sum()),
            'fraction_train': int(self.fraction_train_mask.sum()),
            'fraction_validation': int(self.fraction_validation_mask.sum()),
        }


def nearest_values(frame: pd.DataFrame, requested_times: np.ndarray,
                   column: str) -> np.ndarray:
    grid = frame['time'].to_numpy(float)
    values = frame[column].to_numpy(float)
    indices = [int(np.argmin(np.abs(grid - time_value))) for time_value in requested_times]
    return values[np.asarray(indices, dtype=int)]


def load_or_generate_dense(schedule_name: str) -> pd.DataFrame:
    candidates = [
        PROJECT_ROOT / 'data' / 'dense' / f'{schedule_name}_dense.csv',
        PROJECT_ROOT / 'AMR_PINN_Round1_Comprehensive' / 'data' / 'dense' / f'{schedule_name}_dense.csv',
    ]
    for path in candidates:
        if path.exists():
            frame = pd.read_csv(path)
            required = {'time','sensitive','resistant','total','resistant_fraction','antibiotic'}
            if required.issubset(frame.columns):
                return frame
    frame = solve_reference_r4(TRUE_PARAMETERS, SCHEDULES_R4[schedule_name])
    fallback_dir = CACHE_DIR_R4 / 'dense_reference'
    fallback_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(fallback_dir / f'{schedule_name}_dense.csv', index=False)
    return frame


DENSE_REFERENCE = {name: load_or_generate_dense(name) for name in SCHEDULES_R4}


def build_observation_dataset(schedule_name: str, regime: str,
                              noise_level: float = 0.0,
                              seed: int = SEED) -> ObservationDataset:
    if regime not in {'O1','O2','O3','O4'}:
        raise ValueError('regime must be O1, O2, O3, or O4.')
    if noise_level < 0:
        raise ValueError('noise_level cannot be negative.')

    dense = DENSE_REFERENCE[schedule_name]
    total_time = np.linspace(T_START, T_FINAL, 21)
    true_total = nearest_values(dense, total_time, 'total')

    fraction_times_by_regime = {
        'O1': np.array([], dtype=float),
        'O2': np.array([12.0], dtype=float),
        'O3': np.array([7.0, 12.0, 18.0], dtype=float),
        'O4': total_time.copy(),
    }
    fraction_time = fraction_times_by_regime[regime]
    true_fraction = nearest_values(dense, fraction_time, 'resistant_fraction') \
                    if len(fraction_time) else np.array([], dtype=float)

    rng = np.random.default_rng(seed)
    if noise_level == 0:
        total_observed = true_total.copy()
        fraction_observed = true_fraction.copy()
    else:
        sigma_log = np.sqrt(np.log1p(noise_level**2))
        total_observed = true_total * rng.lognormal(-0.5*sigma_log**2,
                                                    sigma_log, len(true_total))
        eps = 1e-8
        clipped = np.clip(true_fraction, eps, 1-eps)
        logits = np.log(clipped / (1-clipped))
        fraction_observed = 1 / (1 + np.exp(-(logits + rng.normal(0, noise_level, len(logits)))))

    # Hold out every fifth total observation for validation, but always retain
    # the initial point in training. With only one phenotype measurement (O2),
    # it is retained for training and validation is based on total load.
    total_validation_mask = (np.arange(len(total_time)) % 5 == 4)
    total_validation_mask[0] = False
    total_train_mask = ~total_validation_mask

    fraction_validation_mask = np.zeros(len(fraction_time), dtype=bool)
    if len(fraction_time) >= 3:
        fraction_validation_mask[-1] = True
    fraction_train_mask = ~fraction_validation_mask

    return ObservationDataset(
        schedule_name=schedule_name, regime=regime, noise_level=noise_level,
        total_time=total_time, total_observed=total_observed,
        fraction_time=fraction_time, fraction_observed=fraction_observed,
        total_train_mask=total_train_mask,
        total_validation_mask=total_validation_mask,
        fraction_train_mask=fraction_train_mask,
        fraction_validation_mask=fraction_validation_mask,
    )

OBSERVATION_EXAMPLES = {
    regime: build_observation_dataset('early_continuous', regime, 0.0, SEED)
    for regime in ['O1','O2','O3','O4']
}

summary = pd.DataFrame([dataset.summary() for dataset in OBSERVATION_EXAMPLES.values()])
summary.to_csv(TABLE_DIR_R4 / 'round4_observation_regime_definitions.csv', index=False)
summary


In [ ]:
# =============================================================================
# ROUND 4 — CELL 6
# Nested multi-schedule designs, schedule-aware observations,
# and publication-ready experimental-design figure
# =============================================================================
#
# SCIENTIFIC PURPOSE
# ------------------
# Round 4 evaluates whether treatment-environment diversity improves
# practical identifiability of the shared eco-evolutionary parameters.
#
# M1 : early-continuous treatment only, sparse O3 phenotype observations
# M2 : early + late continuous treatment, O3
# M3 : early + late + pulsed treatment, O3
# M4 : untreated + early + late + pulsed treatment, O3
# M5 : same four environments as M4, but phenotype-rich O4 observations
#
# M1--M4 isolate the effect of schedule diversity while keeping phenotype
# observation richness fixed.
#
# M5 is an upper-information benchmark and must NOT be interpreted as a
# direct sparse-data replacement for M1--M4.
#
# O3 resistant-fraction observation times are schedule aware and are chosen
# away from treatment-coefficient discontinuities.
#
# This cell:
#   1. defines the Round-4 nested designs;
#   2. constructs schedule-aware O3/O4 observation datasets;
#   3. constructs design bundles;
#   4. saves the design-definition table; and
#   5. generates the final publication-ready treatment-design figure.
#
# NO MODEL TRAINING IS PERFORMED IN THIS CELL.
# =============================================================================


# =============================================================================
# 1. ROUND-4 DESIGN CONTAINER
# =============================================================================

@dataclass(frozen=True)
class Round4Design:
    """
    Definition of one nested Round-4 experimental design.
    """

    design_id: str
    schedule_names: Tuple[str, ...]
    observation_regime: str
    description: str

    @property
    def schedule_count(self) -> int:
        return len(self.schedule_names)


# =============================================================================
# 2. CANONICAL SCHEDULE ORDER
# =============================================================================
# Used consistently for:
#   - seed offsets,
#   - design construction,
#   - table ordering,
#   - figure-panel ordering.
# =============================================================================

SCHEDULE_ORDER_R4 = (
    "untreated",
    "early_continuous",
    "late_continuous",
    "pulsed",
)


SCHEDULE_SEED_OFFSET_R4 = {
    schedule_name: index
    for index, schedule_name in enumerate(
        SCHEDULE_ORDER_R4
    )
}


# =============================================================================
# 3. SPARSE O3 RESISTANT-FRACTION OBSERVATION TIMES
# =============================================================================
# Three phenotype observations are used per schedule.
#
# These times intentionally avoid known treatment-switch discontinuities.
# =============================================================================

PHENOTYPE_TIMES_O3_R4 = {

    "untreated":
        np.array(
            [4.0, 10.0, 16.0],
            dtype=float,
        ),

    "early_continuous":
        np.array(
            [4.0, 8.0, 16.0],
            dtype=float,
        ),

    "late_continuous":
        np.array(
            [5.0, 12.0, 17.0],
            dtype=float,
        ),

    "pulsed":
        np.array(
            [4.0, 6.5, 13.5],
            dtype=float,
        ),
}


# Basic structural audit.
for schedule_name_r4, times_r4 in PHENOTYPE_TIMES_O3_R4.items():

    if schedule_name_r4 not in SCHEDULES_R4:
        raise KeyError(
            f"O3 observation schedule '{schedule_name_r4}' "
            "is not defined in SCHEDULES_R4."
        )

    if len(times_r4) != 3:
        raise ValueError(
            f"{schedule_name_r4} must contain exactly "
            "three O3 phenotype-observation times."
        )

    if np.any(times_r4 < T_START) or np.any(times_r4 > T_FINAL):
        raise ValueError(
            f"O3 observation times for {schedule_name_r4} "
            "fall outside the simulation interval."
        )


# =============================================================================
# 4. NESTED MULTI-SCHEDULE DESIGNS
# =============================================================================

ROUND4_DESIGNS = {

    "M1":
        Round4Design(
            design_id="M1",
            schedule_names=(
                "early_continuous",
            ),
            observation_regime="O3",
            description=(
                "Single early-continuous treatment environment "
                "with three resistant-fraction observations."
            ),
        ),

    "M2":
        Round4Design(
            design_id="M2",
            schedule_names=(
                "early_continuous",
                "late_continuous",
            ),
            observation_regime="O3",
            description=(
                "Early- and late-continuous treatment environments "
                "with shared inverse parameters."
            ),
        ),

    "M3":
        Round4Design(
            design_id="M3",
            schedule_names=(
                "early_continuous",
                "late_continuous",
                "pulsed",
            ),
            observation_regime="O3",
            description=(
                "Three treated environments with shared "
                "inverse parameters."
            ),
        ),

    "M4":
        Round4Design(
            design_id="M4",
            schedule_names=(
                "untreated",
                "early_continuous",
                "late_continuous",
                "pulsed",
            ),
            observation_regime="O3",
            description=(
                "Untreated plus three treatment environments "
                "with sparse phenotype observations."
            ),
        ),

    "M5":
        Round4Design(
            design_id="M5",
            schedule_names=(
                "untreated",
                "early_continuous",
                "late_continuous",
                "pulsed",
            ),
            observation_regime="O4",
            description=(
                "Four-environment phenotype-rich "
                "upper-information benchmark."
            ),
        ),
}


DESIGN_ORDER_R4 = [
    "M1",
    "M2",
    "M3",
    "M4",
    "M5",
]


# =============================================================================
# 5. ROUND-4 OBSERVATION-DATASET CONSTRUCTOR
# =============================================================================

def build_round4_dataset(
    schedule_name: str,
    regime: str,
    noise_level: float = 0.0,
    seed: int = SEED,
) -> ObservationDataset:
    """
    Construct a schedule-aware Round-4 observation dataset.

    O3
    --
    21 total-load observations
    + 3 schedule-aware resistant-fraction observations.

    O4
    --
    21 total-load observations
    + resistant-fraction observations at all 21 observation times.
    """

    # ---------------------------------------------------------------------
    # Input checks
    # ---------------------------------------------------------------------

    if schedule_name not in SCHEDULES_R4:
        raise KeyError(
            f"Unknown Round-4 schedule: {schedule_name}"
        )

    if regime not in {
        "O3",
        "O4",
    }:
        raise ValueError(
            "Round-4 primary designs use only O3 or O4 observations."
        )

    if noise_level < 0:
        raise ValueError(
            "noise_level cannot be negative."
        )


    # ---------------------------------------------------------------------
    # Dense reference trajectory
    # ---------------------------------------------------------------------

    dense = DENSE_REFERENCE[
        schedule_name
    ]


    # ---------------------------------------------------------------------
    # Total-load observations
    # ---------------------------------------------------------------------

    total_time = np.linspace(
        T_START,
        T_FINAL,
        21,
    )


    true_total = nearest_values(
        dense,
        total_time,
        "total",
    )


    # ---------------------------------------------------------------------
    # Resistant-fraction observations
    # ---------------------------------------------------------------------

    if regime == "O3":

        fraction_time = (
            PHENOTYPE_TIMES_O3_R4[
                schedule_name
            ]
            .copy()
        )

    else:

        # O4: phenotype-resolved information
        # at every observation time.
        fraction_time = (
            total_time.copy()
        )


    true_fraction = nearest_values(
        dense,
        fraction_time,
        "resistant_fraction",
    )


    # ---------------------------------------------------------------------
    # Observation noise
    # ---------------------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )


    if noise_level == 0.0:

        total_observed = (
            true_total.copy()
        )

        fraction_observed = (
            true_fraction.copy()
        )

    else:

        # -------------------------------------------------------------
        # Positive multiplicative noise for total load
        # -------------------------------------------------------------

        sigma_log = np.sqrt(
            np.log1p(
                noise_level ** 2
            )
        )


        total_observed = (
            true_total
            *
            rng.lognormal(
                -0.5 * sigma_log ** 2,
                sigma_log,
                len(
                    true_total
                ),
            )
        )


        # -------------------------------------------------------------
        # Logit-space noise for the resistant fraction
        # -------------------------------------------------------------

        eps = 1.0e-8


        clipped = np.clip(
            true_fraction,
            eps,
            1.0 - eps,
        )


        logits = np.log(
            clipped
            /
            (
                1.0
                -
                clipped
            )
        )


        fraction_observed = (
            1.0
            /
            (
                1.0
                +
                np.exp(
                    -(
                        logits
                        +
                        rng.normal(
                            0.0,
                            noise_level,
                            len(
                                logits
                            ),
                        )
                    )
                )
            )
        )


    # ---------------------------------------------------------------------
    # Total-load train / validation split
    # ---------------------------------------------------------------------
    # The same held-out indices are used for every schedule and seed.
    # ---------------------------------------------------------------------

    total_validation_mask = (
        np.arange(
            len(
                total_time
            )
        )
        % 5
        == 4
    )


    # Keep the initial observation in training.
    total_validation_mask[
        0
    ] = False


    total_train_mask = (
        ~total_validation_mask
    )


    # ---------------------------------------------------------------------
    # Resistant-fraction train / validation split
    # ---------------------------------------------------------------------

    if regime == "O3":

        # Three phenotype observations:
        # first two for training,
        # last one held out for validation.
        fraction_validation_mask = np.array(
            [
                False,
                False,
                True,
            ],
            dtype=bool,
        )

    else:

        # O4:
        # hold out every fifth phenotype observation.
        fraction_validation_mask = (
            np.arange(
                len(
                    fraction_time
                )
            )
            % 5
            == 4
        )


        # Preserve the initial phenotype observation
        # in the training set.
        fraction_validation_mask[
            0
        ] = False


    fraction_train_mask = (
        ~fraction_validation_mask
    )


    # ---------------------------------------------------------------------
    # Final dataset
    # ---------------------------------------------------------------------

    return ObservationDataset(

        schedule_name=
            schedule_name,

        regime=
            regime,

        noise_level=
            noise_level,

        total_time=
            total_time,

        total_observed=
            total_observed,

        fraction_time=
            fraction_time,

        fraction_observed=
            fraction_observed,

        total_train_mask=
            total_train_mask,

        total_validation_mask=
            total_validation_mask,

        fraction_train_mask=
            fraction_train_mask,

        fraction_validation_mask=
            fraction_validation_mask,
    )


# =============================================================================
# 6. MULTI-SCHEDULE DESIGN-BUNDLE CONSTRUCTOR
# =============================================================================

def build_design_bundle_r4(
    design_id: str,
    seed: int,
    noise_level: float = 0.0,
) -> Dict[str, ObservationDataset]:
    """
    Construct all schedule-specific observations belonging to one
    Round-4 nested design.
    """

    if design_id not in ROUND4_DESIGNS:
        raise KeyError(
            f"Unknown Round-4 design: {design_id}"
        )


    design = ROUND4_DESIGNS[
        design_id
    ]


    bundle = {}


    for schedule_name in design.schedule_names:

        schedule_seed = (
            seed
            +
            1009
            *
            SCHEDULE_SEED_OFFSET_R4[
                schedule_name
            ]
        )


        bundle[
            schedule_name
        ] = build_round4_dataset(

            schedule_name=
                schedule_name,

            regime=
                design.observation_regime,

            noise_level=
                noise_level,

            seed=
                schedule_seed,
        )


    return bundle


# =============================================================================
# 7. SAVE ROUND-4 DESIGN-DEFINITION TABLE
# =============================================================================

design_definition_rows = []


for design_id in DESIGN_ORDER_R4:

    design = ROUND4_DESIGNS[
        design_id
    ]


    design_definition_rows.append(
        {
            "design":
                design_id,

            "schedule_count":
                design.schedule_count,

            "schedules":
                ", ".join(
                    design.schedule_names
                ),

            "regime":
                design.observation_regime,

            "phenotype_points_per_schedule":
                (
                    3
                    if
                    design.observation_regime
                    == "O3"
                    else
                    21
                ),

            "description":
                design.description,
        }
    )


ROUND4_DESIGN_TABLE = pd.DataFrame(
    design_definition_rows
)


ROUND4_DESIGN_TABLE.to_csv(
    TABLE_DIR_R4
    /
    "round4_design_definitions.csv",
    index=False,
)


display(
    ROUND4_DESIGN_TABLE
)


# =============================================================================
# =============================================================================
# 8. FINAL PUBLICATION FIGURE
# =============================================================================
#
# Publication design:
#
#   - Four vertically aligned treatment environments.
#   - Shared time axis for direct schedule comparison.
#   - Treatment exposure shown by a step function.
#   - O3 resistant-fraction observation times shown in a dedicated
#     upper observation band INSIDE each panel.
#   - Short panel titles.
#   - Compact, borderless legend separated from panel (a).
#   - No internal figure title; the manuscript caption provides context.
#
# NO SCIENTIFIC VALUES ARE CHANGED.
# =============================================================================


# -----------------------------------------------------------------------------
# Canonical plotting order
# -----------------------------------------------------------------------------

schedule_order_plot_r4 = [
    "untreated",
    "early_continuous",
    "late_continuous",
    "pulsed",
]


# -----------------------------------------------------------------------------
# Concise publication panel titles
# -----------------------------------------------------------------------------

panel_titles_r4 = {
    "untreated":
        "(a) Untreated",

    "early_continuous":
        "(b) Early continuous",

    "late_continuous":
        "(c) Late continuous",

    "pulsed":
        "(d) Pulsed",
}


# -----------------------------------------------------------------------------
# Treatment-switch annotations
# -----------------------------------------------------------------------------
# Kept separate from the panel titles to avoid overly long headings.
# -----------------------------------------------------------------------------

switch_annotations_r4 = {

    "untreated":
        None,

    "early_continuous":
        r"$t_{\mathrm{on}}=5$",

    "late_continuous":
        r"$t_{\mathrm{on}}=10$",

    "pulsed":
        r"ON: $5\!-\!8,\;12\!-\!15$",
}


# -----------------------------------------------------------------------------
# Dense time axis used only for plotting
# -----------------------------------------------------------------------------

time_plot_r4 = np.linspace(
    T_START,
    T_FINAL,
    1201,
)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------
# Increased height is intentional:
# each panel now has adequate breathing room.
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=4,
    ncols=1,
    figsize=(9.0, 8.0),
    sharex=True,
)


# =============================================================================
# 9. PLOT EACH TREATMENT ENVIRONMENT
# =============================================================================

for axis, schedule_name in zip(
    axes,
    schedule_order_plot_r4,
):

    schedule = SCHEDULES_R4[
        schedule_name
    ]


    exposure = schedule.exposure_numpy(
        time_plot_r4
    )


    phenotype_times = (
        PHENOTYPE_TIMES_O3_R4[
            schedule_name
        ]
    )


    # -------------------------------------------------------------------------
    # Prescribed antibiotic exposure
    # -------------------------------------------------------------------------

    exposure_line = axis.step(
        time_plot_r4,
        exposure,
        where="post",
        linewidth=2.2,
        zorder=3,
        label=r"Exposure $A(t)$",
    )[0]


    exposure_color_r4 = (
        exposure_line.get_color()
    )


    # -------------------------------------------------------------------------
    # Very light treatment-ON shading
    # -------------------------------------------------------------------------
    # Provides quick visual separation between treated and untreated intervals
    # without obscuring the step function.
    # -------------------------------------------------------------------------

    axis.fill_between(
        time_plot_r4,
        0.0,
        exposure,
        step="post",
        color=exposure_color_r4,
        alpha=0.055,
        linewidth=0.0,
        zorder=0,
    )


    # -------------------------------------------------------------------------
    # O3 observation guides
    # -------------------------------------------------------------------------

    for sample_time in phenotype_times:

        axis.axvline(
            sample_time,
            ymin=0.0,
            ymax=0.91,
            linestyle="--",
            linewidth=0.95,
            color=exposure_color_r4,
            alpha=0.50,
            zorder=1,
        )


    # -------------------------------------------------------------------------
    # Dedicated phenotype-observation band
    # -------------------------------------------------------------------------
    #
    # y = 1.14 is deliberately above the exposure maximum A(t)=1,
    # but remains INSIDE the plotting area.
    #
    # This prevents the diamonds from colliding with panel titles.
    # The y-axis still displays only the physically meaningful ticks 0 and 1.
    # -------------------------------------------------------------------------

    observation_y_r4 = 1.14


    axis.scatter(
        phenotype_times,
        np.full(
            len(
                phenotype_times
            ),
            observation_y_r4,
        ),
        marker="D",
        s=52,
        facecolor=exposure_color_r4,
        edgecolor="black",
        linewidth=0.65,
        zorder=5,
        label=r"O3 $q_R$ observation",
    )


    # -------------------------------------------------------------------------
    # Panel limits
    # -------------------------------------------------------------------------
    # Additional headroom separates the observation markers from A(t)=1.
    # -------------------------------------------------------------------------

    axis.set_xlim(
        T_START,
        T_FINAL,
    )


    axis.set_ylim(
        -0.055,
        1.25,
    )


    axis.set_yticks(
        [0, 1]
    )


    # -------------------------------------------------------------------------
    # Panel title
    # -------------------------------------------------------------------------

    axis.set_title(
        panel_titles_r4[
            schedule_name
        ],
        loc="left",
        fontsize=12.0,
        fontweight="semibold",
        pad=10,
    )


    # -------------------------------------------------------------------------
        # -------------------------------------------------------------------------
    # Small treatment-timing annotation
    # -------------------------------------------------------------------------

    timing_text_r4 = (
        switch_annotations_r4[
            schedule_name
        ]
    )

    if timing_text_r4 is not None:

        axis.text(
            0.995,
            1.075,
            timing_text_r4,
            transform=axis.transAxes,
            ha="right",
            va="bottom",
            fontsize=9.3,
            clip_on=False,
        )


    # -------------------------------------------------------------------------
    # Grid
    # -------------------------------------------------------------------------

    axis.grid(
        axis="x",
        linewidth=0.55,
        alpha=0.13,
    )


    axis.set_axisbelow(
        True
    )


    # -------------------------------------------------------------------------
    # Tick appearance
    # -------------------------------------------------------------------------

    axis.tick_params(
        axis="both",
        which="major",
        labelsize=9.5,
        width=0.8,
        length=4.0,
        direction="out",
    )


    # -------------------------------------------------------------------------
    # Journal-style frame
    # -------------------------------------------------------------------------

    for spine in axis.spines.values():

        spine.set_linewidth(
            0.85
        )


# =============================================================================
# 10. SHARED AXIS LABELS
# =============================================================================

axes[-1].set_xlabel(
    "Time",
    fontsize=11.5,
    labelpad=7,
)


fig.supylabel(
    r"Antibiotic exposure, $A(t)$",
    fontsize=11.5,
    x=0.035,
)


# =============================================================================
# 11. CLEAN SHARED LEGEND
# =============================================================================
#
# Short labels are used here.
# The manuscript caption defines q_R explicitly as the resistant fraction.
# =============================================================================

handles_r4, labels_r4 = (
    axes[0]
    .get_legend_handles_labels()
)


legend_r4 = fig.legend(
    handles_r4,
    labels_r4,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.965),
    ncol=2,
    frameon=False,
    fontsize=10.5,
    handlelength=2.7,
    handletextpad=0.65,
    columnspacing=2.2,
)


# =============================================================================
# 12. FINAL PUBLICATION LAYOUT
# =============================================================================
#
# Large vertical separation is intentional.
#
# top=0.89:
#   reserves a dedicated band for the legend.
#
# hspace=0.68:
#   prevents panel titles from appearing attached to the preceding subplot.
#
# =============================================================================

fig.subplots_adjust(
    left=0.105,
    right=0.985,
    bottom=0.085,
    top=0.885,
    hspace=0.68,
)


# =============================================================================
# 13. SAVE FINAL FIGURE
# =============================================================================
#
# Same filename as before:
# downstream LaTeX does not need to change.
# =============================================================================

save_figure_r4(
    fig,
    "round4_schedule_and_observation_design",
)


plt.show()


print(
    "\n"
    + "=" * 84
)


print(
    "ROUND 4 CELL 6 PUBLICATION FIGURE UPDATED SUCCESSFULLY"
)


print(
    "=" * 84
)


print(
    "\nFigure:"
)


print(
    "  round4_schedule_and_observation_design"
)


print(
    "\nNO MODEL TRAINING WAS PERFORMED."
)

In [ ]:
# =============================================================================
# ROUND 4 — CELL 7
# Tensor conversion, treatment-interface collocation, and reproducible sampling
# =============================================================================
# Collocation points are sampled separately inside every treatment interval.
# Points exactly at a coefficient discontinuity are excluded. This prevents the
# network from being penalized for a derivative jump that is physically expected
# when treatment changes instantaneously.


def tensor(data: np.ndarray | List[float], requires_grad: bool = False) -> torch.Tensor:
    value = torch.as_tensor(np.asarray(data), dtype=DTYPE, device=DEVICE)
    if value.ndim == 1:
        value = value[:, None]
    value.requires_grad_(requires_grad)
    return value


def sample_interface_aware_collocation(schedule: ScheduleR4,
                                       points_per_interval: int,
                                       seed: int) -> torch.Tensor:
    rng = np.random.default_rng(seed)
    samples = []
    margin = 1e-6
    for left, right, _ in schedule.intervals(T_START, T_FINAL):
        if right - left <= 2*margin:
            continue
        # Latin-hypercube-like stratification in each interval gives more even
        # temporal coverage than independent uniform samples.
        bins = (np.arange(points_per_interval) + rng.random(points_per_interval)) \
               / points_per_interval
        points = left + margin + bins * (right-left-2*margin)
        rng.shuffle(points)
        samples.append(points)
    merged = np.concatenate(samples)
    return tensor(merged, requires_grad=True)


def observation_to_tensors(dataset: ObservationDataset, split: str = 'train') -> Dict[str, torch.Tensor]:
    if split not in {'train','validation','all'}:
        raise ValueError('split must be train, validation, or all.')
    if split == 'train':
        total_mask = dataset.total_train_mask
        fraction_mask = dataset.fraction_train_mask
    elif split == 'validation':
        total_mask = dataset.total_validation_mask
        fraction_mask = dataset.fraction_validation_mask
    else:
        total_mask = np.ones(len(dataset.total_time), dtype=bool)
        fraction_mask = np.ones(len(dataset.fraction_time), dtype=bool)

    return {
        'total_time': tensor(dataset.total_time[total_mask]),
        'total_observed': tensor(dataset.total_observed[total_mask] / TRUE_PARAMETERS.carrying_capacity),
        'fraction_time': tensor(dataset.fraction_time[fraction_mask]),
        'fraction_observed': tensor(dataset.fraction_observed[fraction_mask]),
    }

# Verify that no sampled point is placed directly on a treatment switch.
check_points = sample_interface_aware_collocation(SCHEDULES_R4['pulsed'], 20, SEED).detach().cpu().numpy().ravel()
for switch in SCHEDULES_R4['pulsed'].switch_times:
    assert np.min(np.abs(check_points-switch)) > 1e-7
print('Interface-aware collocation test passed with', len(check_points), 'points.')

In [ ]:
# =============================================================================
# ROUND 4 — CELL 8
# Neural building blocks and bounded inverse parameters
# =============================================================================
# The unknown Tier-I parameters are:
#     mutation probability mu in [1e-6, 1e-2] on a logarithmic scale;
#     resistance fitness cost c_R in [0, 0.40].
# Parameter transforms make every optimizer iterate biologically admissible.

class TanhMLP(nn.Module):
    """Four-layer tanh MLP with Xavier initialization."""
    def __init__(self, input_dim: int, output_dim: int,
                 width: int = 64, depth: int = 4):
        super().__init__()
        layers: List[nn.Module] = []
        previous = input_dim
        for _ in range(depth):
            linear = nn.Linear(previous, width)
            nn.init.xavier_normal_(linear.weight)
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.Tanh()]
            previous = width
        output = nn.Linear(previous, output_dim)
        nn.init.xavier_normal_(output.weight, gain=0.5)
        nn.init.zeros_(output.bias)
        layers.append(output)
        # Construct all trainable layers directly in float64. Calling model.to(DEVICE)
        # changes the device but not the dtype, so this explicit conversion prevents
        # Float/Double mismatches during automatic differentiation.
        self.network = nn.Sequential(*layers).to(dtype=DTYPE)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def inverse_sigmoid(value: float) -> float:
    value = float(np.clip(value, 1e-8, 1-1e-8))
    return math.log(value/(1-value))


class TierIInverseParameters(nn.Module):
    """Bounded trainable mutation probability and resistance fitness cost."""
    def __init__(self, initial_mu: float = 3e-4, initial_cost: float = 0.15,
                 trainable: bool = True):
        super().__init__()
        self.log10_mu_min = -6.0
        self.log10_mu_max = -2.0
        mu_position = (math.log10(initial_mu)-self.log10_mu_min) \
                      / (self.log10_mu_max-self.log10_mu_min)
        cost_position = initial_cost / 0.40
        self.raw_mu = nn.Parameter(torch.tensor(inverse_sigmoid(mu_position), dtype=DTYPE),
                                   requires_grad=trainable)
        self.raw_cost = nn.Parameter(torch.tensor(inverse_sigmoid(cost_position), dtype=DTYPE),
                                     requires_grad=trainable)

    def mutation_probability(self) -> torch.Tensor:
        log10_mu = self.log10_mu_min + (self.log10_mu_max-self.log10_mu_min) \
                   * torch.sigmoid(self.raw_mu)
        return torch.pow(torch.tensor(10.0, dtype=DTYPE, device=self.raw_mu.device), log10_mu)

    def resistance_fitness_cost(self) -> torch.Tensor:
        return 0.40 * torch.sigmoid(self.raw_cost)

    def as_floats(self) -> Dict[str, float]:
        return {
            'mutation_probability': float(self.mutation_probability().detach().cpu()),
            'resistance_fitness_cost': float(self.resistance_fitness_cost().detach().cpu()),
        }


In [ ]:
# =============================================================================
# ROUND 4 — CELL 9
# Treatment-interface state network and all architectural ablations
# =============================================================================
# Available model kinds
# ---------------------
# data_only : unconstrained state network, no ODE residual.
# vanilla   : unconstrained PINN with exact IC/interface continuity only.
# soft      : vanilla PINN plus penalties for negativity and capacity excess.
# positive  : softplus S and R, but total abundance is not bounded by K.
# simplex   : positive total and exact phenotype fractions, but total is unbounded.
# hard      : proposed bounded total + phenotype softmax; exact feasible set.
#
# Every model uses one subnetwork per treatment interval. The state at the end
# of one interval becomes the exact anchor of the next interval. This gives
# continuous S and R while allowing their derivatives to respond immediately
# to the changed treatment coefficient.

class InterfaceStateNetwork(nn.Module):
    def __init__(self, schedule: ScheduleR4, model_kind: str,
                 initial_s: float, initial_r: float,
                 width: int = 64, depth: int = 4, tau_h: float = 0.35):
        super().__init__()
        allowed = {'data_only','vanilla','soft','positive','simplex','hard'}
        if model_kind not in allowed:
            raise ValueError(f'Unknown model_kind {model_kind}.')
        self.schedule = schedule
        self.model_kind = model_kind
        self.initial_s = float(initial_s)
        self.initial_r = float(initial_r)
        self.tau_h = float(tau_h)
        self.intervals = schedule.intervals(T_START, T_FINAL)
        output_dim = 3 if model_kind in {'simplex','hard'} else 2
        self.segment_networks = nn.ModuleList([
            TanhMLP(1, output_dim, width=width, depth=depth)
            for _ in self.intervals
        ])

    def _scaled_local_time(self, t: torch.Tensor, left: float, right: float) -> torch.Tensor:
        return 2.0 * (t-left)/(right-left) - 1.0

    def _transform(self, raw: torch.Tensor, local_elapsed: torch.Tensor,
                   anchor: torch.Tensor) -> torch.Tensor:
        h = 1.0 - torch.exp(-torch.clamp(local_elapsed, min=0.0)/self.tau_h)
        anchor = anchor.expand(raw.shape[0], -1)

        if self.model_kind in {'data_only','vanilla','soft'}:
            candidate = raw[:, :2]
            return (1-h)*anchor + h*candidate

        if self.model_kind == 'positive':
            candidate = F.softplus(raw[:, :2])
            return (1-h)*anchor + h*candidate

        anchor_total = torch.clamp(anchor.sum(dim=1, keepdim=True), min=1e-12)
        anchor_fraction = anchor / anchor_total
        candidate_fraction = torch.softmax(raw[:, 1:3], dim=1)
        fraction = (1-h)*anchor_fraction + h*candidate_fraction

        if self.model_kind == 'simplex':
            candidate_total = F.softplus(raw[:, 0:1])
        else:  # proposed hard model
            candidate_total = torch.sigmoid(raw[:, 0:1])
        total = (1-h)*anchor_total + h*candidate_total
        return total * fraction

    def _anchors(self) -> List[torch.Tensor]:
        device = next(self.parameters()).device
        anchor = torch.tensor([[self.initial_s, self.initial_r]], dtype=DTYPE, device=device)
        anchors = [anchor]
        for index in range(1, len(self.intervals)):
            previous_left, previous_right, _ = self.intervals[index-1]
            switch_tensor = torch.tensor([[previous_right]], dtype=DTYPE, device=device)
            scaled = self._scaled_local_time(switch_tensor, previous_left, previous_right)
            raw = self.segment_networks[index-1](scaled)
            anchor = self._transform(raw, switch_tensor-previous_left, anchor)
            anchors.append(anchor)
        return anchors

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        if t.ndim == 1:
            t = t[:, None]
        anchors = self._anchors()
        output = torch.zeros((len(t),2), dtype=t.dtype, device=t.device)
        for index, (left, right, _) in enumerate(self.intervals):
            is_last = index == len(self.intervals)-1
            mask = (t[:,0] >= left-1e-12) & ((t[:,0] <= right+1e-12) if is_last else (t[:,0] < right-1e-12))
            if mask.any():
                local_t = t[mask]
                scaled = self._scaled_local_time(local_t, left, right)
                raw = self.segment_networks[index](scaled)
                output[mask] = self._transform(raw, local_t-left, anchors[index])
        return output


class AMRInverseModel(nn.Module):
    """One or more experiment-specific state networks sharing Tier-I parameters."""
    def __init__(self, schedules: Dict[str, ScheduleR4], model_kind: str,
                 width: int = 64, depth: int = 4,
                 parameter_initialization: Tuple[float,float] = (3e-4,0.15)):
        super().__init__()
        self.model_kind = model_kind
        self.physics_enabled = model_kind != 'data_only'
        self.state_networks = nn.ModuleDict({
            name: InterfaceStateNetwork(schedule, model_kind,
                                        INITIAL_S/TRUE_PARAMETERS.carrying_capacity,
                                        INITIAL_R/TRUE_PARAMETERS.carrying_capacity,
                                        width=width, depth=depth)
            for name, schedule in schedules.items()
        })
        self.inverse_parameters = TierIInverseParameters(
            parameter_initialization[0], parameter_initialization[1],
            trainable=self.physics_enabled,
        )

    def forward(self, t: torch.Tensor, schedule_name: str) -> torch.Tensor:
        return self.state_networks[schedule_name](t)

    def inferred_parameters(self) -> Dict[str,float]:
        if not self.physics_enabled:
            return {'mutation_probability': np.nan,
                    'resistance_fitness_cost': np.nan}
        return self.inverse_parameters.as_floats()

# Architectural guarantees are verified before training.
for kind in ['data_only','vanilla','soft','positive','simplex','hard']:
    trial = AMRInverseModel({'early_continuous': SCHEDULES_R4['early_continuous']},
                            kind, width=16, depth=2).to(DEVICE)
    t_test = tensor(np.linspace(0,20,101))
    with torch.no_grad():
        state = trial(t_test, 'early_continuous').cpu().numpy()
    assert np.max(np.abs(state[0]-np.array([0.1,0.0]))) < 1e-12
    if kind in {'positive','simplex','hard'}:
        assert state.min() >= -1e-12
    if kind == 'hard':
        assert state.sum(axis=1).max() <= 1+1e-12
    del trial
clear_gpu_memory()
print('All untrained architecture guarantee tests passed.')


In [ ]:
# =============================================================================
# ROUND 4 — CELL 10
# Physics residuals, observation likelihoods, and soft-constraint penalties
# =============================================================================
# Total-load data are compared on a log scale because bacterial abundance spans
# a wide range. Phenotype fractions are compared directly in [0,1].
# The proposed hard model receives no positivity or capacity penalties because
# those properties are exact consequences of its output mapping.

@dataclass
class LossWeights:
    total_data: float = 10.0
    fraction_data: float = 5.0
    physics: float = 1.0
    soft_constraint: float = 10.0
    weight_decay: float = 1e-8

LOSS_WEIGHTS = LossWeights()


def physics_residual(model: AMRInverseModel, t: torch.Tensor,
                     schedule_name: str) -> Tuple[torch.Tensor,torch.Tensor]:
    if not t.requires_grad:
        t = t.clone().detach().requires_grad_(True)
    state = model(t, schedule_name)
    s = state[:,0:1]
    r = state[:,1:2]
    ds_dt = torch.autograd.grad(s, t, torch.ones_like(s), create_graph=True,
                                retain_graph=True)[0]
    dr_dt = torch.autograd.grad(r, t, torch.ones_like(r), create_graph=True,
                                retain_graph=True)[0]

    mu = model.inverse_parameters.mutation_probability()
    cost = model.inverse_parameters.resistance_fitness_cost()
    schedule = SCHEDULES_R4[schedule_name]
    exposure = schedule.exposure_torch(t)
    b_s = TRUE_PARAMETERS.sensitive_birth_rate * \
          (1.0-TRUE_PARAMETERS.sensitive_drug_effect*exposure)
    b_r = TRUE_PARAMETERS.sensitive_birth_rate * (1.0-cost) * \
          (1.0-TRUE_PARAMETERS.resistant_drug_effect*exposure)
    crowding = 1.0-s-r
    rhs_s = b_s*(1.0-mu)*crowding*s - TRUE_PARAMETERS.sensitive_death_rate*s
    rhs_r = b_s*mu*crowding*s + b_r*crowding*r - TRUE_PARAMETERS.resistant_death_rate*r
    return ds_dt-rhs_s, dr_dt-rhs_r


def data_loss_for_dataset(model: AMRInverseModel, dataset: ObservationDataset,
                          split: str = 'train') -> Tuple[torch.Tensor,torch.Tensor]:
    values = observation_to_tensors(dataset, split)
    eps = torch.tensor(1e-10, dtype=DTYPE, device=DEVICE)

    if len(values['total_time']):
        state_total = model(values['total_time'], dataset.schedule_name)
        predicted_total = state_total.sum(dim=1, keepdim=True)
        # Unconstrained baselines can temporarily predict a non-positive total.
        # A lower clamp keeps the logarithmic likelihood finite; feasibility is
        # still measured separately and the soft baseline receives explicit
        # negativity/capacity penalties. The proposed hard model never uses it.
        predicted_total_for_log = torch.clamp(predicted_total, min=float(eps))
        total_loss = torch.mean((torch.log(predicted_total_for_log)
                                 - torch.log(values['total_observed']+eps))**2)
    else:
        total_loss = torch.zeros((), dtype=DTYPE, device=DEVICE)

    if len(values['fraction_time']):
        state_fraction = model(values['fraction_time'], dataset.schedule_name)
        denominator = torch.clamp(state_fraction.sum(dim=1, keepdim=True), min=float(eps))
        predicted_fraction = state_fraction[:,1:2] / denominator
        # Clipping is used only in the observation operator for invalid baseline
        # states. Raw S/R predictions remain unchanged for constraint auditing.
        predicted_fraction = torch.clamp(predicted_fraction, min=0.0, max=1.0)
        fraction_loss = torch.mean((predicted_fraction-values['fraction_observed'])**2)
    else:
        fraction_loss = torch.zeros((), dtype=DTYPE, device=DEVICE)
    return total_loss, fraction_loss


def regularization_loss(model: nn.Module) -> torch.Tensor:
    value = torch.zeros((), dtype=DTYPE, device=DEVICE)
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.ndim >= 2:
            value = value + torch.mean(parameter**2)
    return value


def soft_constraint_loss(model: AMRInverseModel,
                         collocation: Dict[str,torch.Tensor]) -> torch.Tensor:
    if model.model_kind != 'soft' or not collocation:
        # During data-only pretraining no collocation points are supplied, so
        # the soft-constraint term is intentionally inactive.
        return torch.zeros((), dtype=DTYPE, device=DEVICE)
    penalties = []
    for schedule_name, times in collocation.items():
        state = model(times, schedule_name)
        negative = F.relu(-state)
        capacity = F.relu(state.sum(dim=1,keepdim=True)-1.0)
        penalties.append(torch.mean(negative**2)+torch.mean(capacity**2))
    return torch.stack(penalties).mean()


def complete_loss(model: AMRInverseModel,
                  datasets: Dict[str,ObservationDataset],
                  collocation: Dict[str,torch.Tensor],
                  physics_weight: float,
                  split: str = 'train') -> Tuple[torch.Tensor,Dict[str,float]]:
    total_terms, fraction_terms, physics_terms = [], [], []
    for schedule_name, dataset in datasets.items():
        load_loss, fraction_loss = data_loss_for_dataset(model, dataset, split)
        total_terms.append(load_loss)
        fraction_terms.append(fraction_loss)
        if model.physics_enabled and schedule_name in collocation:
            residual_s, residual_r = physics_residual(model, collocation[schedule_name], schedule_name)
            physics_terms.append(torch.mean(residual_s**2)+torch.mean(residual_r**2))

    load_loss = torch.stack(total_terms).mean()
    fraction_loss = torch.stack(fraction_terms).mean()
    phys_loss = torch.stack(physics_terms).mean() if physics_terms \
                else torch.zeros((),dtype=DTYPE,device=DEVICE)
    constraint_loss = soft_constraint_loss(model, collocation)
    reg_loss = regularization_loss(model)

    total = LOSS_WEIGHTS.total_data*load_loss \
            + LOSS_WEIGHTS.fraction_data*fraction_loss \
            + physics_weight*LOSS_WEIGHTS.physics*phys_loss \
            + LOSS_WEIGHTS.soft_constraint*constraint_loss \
            + LOSS_WEIGHTS.weight_decay*reg_loss
    components = {
        'total': float(total.detach().cpu()),
        'load': float(load_loss.detach().cpu()),
        'fraction': float(fraction_loss.detach().cpu()),
        'physics': float(phys_loss.detach().cpu()),
        'constraint': float(constraint_loss.detach().cpu()),
        'regularization': float(reg_loss.detach().cpu()),
    }
    return total, components

In [ ]:
# =============================================================================
# ROUND 4 — CELL 11
# Four-stage shared-parameter optimizer with 8 GB GPU memory management
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# The optimization is staged to reduce compensation between flexible state
# networks and the two biological parameters.
#
# Stage I   : observation pretraining with mu and c_R frozen.
# Stage II  : physics warm-up with mu and c_R still frozen.
# Stage III : joint AdamW optimization with separate state/parameter learning rates.
# Stage IV  : optional full-batch L-BFGS refinement on reduced fixed collocation.
#
# The model is trained one design/seed at a time. The best checkpoint is copied
# to CPU, histories are saved immediately, and CUDA cache is explicitly cleared.

@dataclass
class TrainingResultR4:
    label: str
    design_id: str
    model: AMRInverseModel
    history: pd.DataFrame
    elapsed_seconds: float
    best_validation_score: float
    seed: int


def set_seed_r4(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cpu_state_dict_r4(model: nn.Module) -> Dict[str, torch.Tensor]:
    """Copy a model checkpoint to CPU so it does not consume GPU memory."""
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def state_parameters_r4(model: AMRInverseModel) -> List[nn.Parameter]:
    return [
        parameter
        for name, parameter in model.named_parameters()
        if not name.startswith("inverse_parameters.")
    ]


def parameter_parameters_r4(model: AMRInverseModel) -> List[nn.Parameter]:
    return list(model.inverse_parameters.parameters())


def set_inverse_trainable_r4(
    model: AMRInverseModel,
    trainable: bool,
) -> None:
    for parameter in model.inverse_parameters.parameters():
        parameter.requires_grad_(trainable)


def make_collocation_bundle_r4(
    datasets: Dict[str, ObservationDataset],
    seed: int,
    points_per_interval: Optional[int] = None,
) -> Dict[str, torch.Tensor]:
    count = (
        COLLOCATION_POINTS_PER_INTERVAL_R4
        if points_per_interval is None
        else int(points_per_interval)
    )
    return {
        schedule_name: sample_interface_aware_collocation(
            SCHEDULES_R4[schedule_name],
            count,
            seed + 1009 * schedule_index,
        )
        for schedule_index, schedule_name in enumerate(datasets)
    }


def validation_score_r4(
    model: AMRInverseModel,
    datasets: Dict[str, ObservationDataset],
    seed: int,
    physics_weight: float = 1.0,
) -> Tuple[float, Dict[str, float]]:
    collocation = make_collocation_bundle_r4(
        datasets,
        seed,
        VALIDATION_POINTS_PER_INTERVAL_R4,
    )
    total, components = complete_loss(
        model,
        datasets,
        collocation,
        physics_weight=physics_weight,
        split="validation",
    )
    score = (
        components["load"]
        + components["fraction"]
        + 0.10 * physics_weight * components["physics"]
    )
    del total, collocation
    clear_gpu_memory()
    return float(score), components


def append_history_row_r4(
    history_rows: List[Dict[str, Any]],
    model: AMRInverseModel,
    stage: str,
    epoch: int,
    score: float,
    components: Dict[str, float],
    learning_rate_state: float = np.nan,
    learning_rate_parameter: float = np.nan,
) -> None:
    history_rows.append({
        "stage": stage,
        "epoch": int(epoch),
        "validation_score": float(score),
        "learning_rate_state": learning_rate_state,
        "learning_rate_parameter": learning_rate_parameter,
        **components,
        **model.inferred_parameters(),
        "gpu_allocated_gb": (
            float(torch.cuda.memory_allocated() / 1024 ** 3)
            if torch.cuda.is_available()
            else 0.0
        ),
        "gpu_reserved_gb": (
            float(torch.cuda.memory_reserved() / 1024 ** 3)
            if torch.cuda.is_available()
            else 0.0
        ),
    })


def train_shared_parameter_model_r4(
    model: AMRInverseModel,
    datasets: Dict[str, ObservationDataset],
    design_id: str,
    label: str,
    seed: int,
) -> TrainingResultR4:
    """Train one nested multi-schedule design using one shared parameter pair."""
    set_seed_r4(seed)
    model = model.to(device=DEVICE, dtype=DTYPE)
    start_time = time.time()
    history_rows: List[Dict[str, Any]] = []

    # ------------------------------------------------------------------
    # Stage I: observation pretraining, inverse parameters frozen
    # ------------------------------------------------------------------
    set_inverse_trainable_r4(model, False)
    optimizer = torch.optim.AdamW(
        state_parameters_r4(model),
        lr=2.0e-3,
        weight_decay=0.0,
    )

    stage_best_score = np.inf
    stage_best_state = cpu_state_dict_r4(model)
    empty_collocation: Dict[str, torch.Tensor] = {}

    for epoch in range(PRETRAIN_EPOCHS_R4):
        optimizer.zero_grad(set_to_none=True)
        total, components = complete_loss(
            model,
            datasets,
            empty_collocation,
            physics_weight=0.0,
            split="train",
        )
        total.backward()
        torch.nn.utils.clip_grad_norm_(
            state_parameters_r4(model),
            max_norm=GRADIENT_CLIP_NORM_R4,
        )
        optimizer.step()

        if (
            epoch % LOG_EVERY_R4 == 0
            or epoch == PRETRAIN_EPOCHS_R4 - 1
        ):
            score, _ = validation_score_r4(
                model,
                datasets,
                seed + 10_000 + epoch,
                physics_weight=0.0,
            )
            append_history_row_r4(
                history_rows,
                model,
                "pretrain",
                epoch,
                score,
                components,
                optimizer.param_groups[0]["lr"],
                0.0,
            )
            if score < stage_best_score:
                stage_best_score = score
                stage_best_state = cpu_state_dict_r4(model)

    model.load_state_dict(stage_best_state)

    # ------------------------------------------------------------------
    # Stage II: physics warm-up, inverse parameters remain frozen
    # ------------------------------------------------------------------
    optimizer = torch.optim.AdamW(
        state_parameters_r4(model),
        lr=1.0e-3,
        weight_decay=0.0,
    )

    stage_best_score = np.inf
    stage_best_state = cpu_state_dict_r4(model)

    collocation = None
    for epoch in range(PHYSICS_WARMUP_EPOCHS_R4):
        if epoch % RESAMPLE_EVERY_R4 == 0 or collocation is None:
            if collocation is not None:
                del collocation
            collocation = make_collocation_bundle_r4(
                datasets,
                seed + 30_000 + epoch,
            )

        optimizer.zero_grad(set_to_none=True)
        total, components = complete_loss(
            model,
            datasets,
            collocation,
            physics_weight=1.0,
            split="train",
        )
        if not torch.isfinite(total):
            raise FloatingPointError(
                f"Non-finite warm-up loss in {label} at epoch {epoch}."
            )
        total.backward()
        torch.nn.utils.clip_grad_norm_(
            state_parameters_r4(model),
            max_norm=GRADIENT_CLIP_NORM_R4,
        )
        optimizer.step()

        if (
            epoch % LOG_EVERY_R4 == 0
            or epoch == PHYSICS_WARMUP_EPOCHS_R4 - 1
        ):
            score, _ = validation_score_r4(
                model,
                datasets,
                seed + 40_000 + epoch,
                physics_weight=1.0,
            )
            append_history_row_r4(
                history_rows,
                model,
                "physics_warmup",
                epoch,
                score,
                components,
                optimizer.param_groups[0]["lr"],
                0.0,
            )
            if score < stage_best_score:
                stage_best_score = score
                stage_best_state = cpu_state_dict_r4(model)

    if collocation is not None:
        del collocation
    model.load_state_dict(stage_best_state)
    clear_gpu_memory()

    # ------------------------------------------------------------------
    # Stage III: joint AdamW with separate parameter learning rate
    # ------------------------------------------------------------------
    set_inverse_trainable_r4(model, True)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": state_parameters_r4(model),
                "lr": STATE_LEARNING_RATE_R4,
            },
            {
                "params": parameter_parameters_r4(model),
                "lr": PARAMETER_LEARNING_RATE_R4,
            },
        ],
        weight_decay=0.0,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=max(50, ADAM_EPOCHS_R4 // 20),
        min_lr=1.0e-6,
    )

    best_score = np.inf
    best_state = cpu_state_dict_r4(model)
    stale_epochs = 0
    collocation = None

    for epoch in range(ADAM_EPOCHS_R4):
        if epoch % RESAMPLE_EVERY_R4 == 0 or collocation is None:
            if collocation is not None:
                del collocation
            collocation = make_collocation_bundle_r4(
                datasets,
                seed + 50_000 + epoch,
            )

        optimizer.zero_grad(set_to_none=True)
        total, components = complete_loss(
            model,
            datasets,
            collocation,
            physics_weight=1.0,
            split="train",
        )

        if not torch.isfinite(total):
            raise FloatingPointError(
                f"Non-finite joint loss in {label} at epoch {epoch}."
            )

        total.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP_NORM_R4,
        )
        optimizer.step()

        if epoch % LOG_EVERY_R4 == 0 or epoch == ADAM_EPOCHS_R4 - 1:
            score, _ = validation_score_r4(
                model,
                datasets,
                seed + 60_000 + epoch,
                physics_weight=1.0,
            )
            scheduler.step(score)

            append_history_row_r4(
                history_rows,
                model,
                "joint_adam",
                epoch,
                score,
                components,
                optimizer.param_groups[0]["lr"],
                optimizer.param_groups[1]["lr"],
            )

            inferred = model.inferred_parameters()
            print(
                f"{label:34s} | Adam {epoch:5d}/{ADAM_EPOCHS_R4} | "
                f"loss={components['total']:.3e} | val={score:.3e} | "
                f"mu={inferred['mutation_probability']:.3e} | "
                f"c={inferred['resistance_fitness_cost']:.4f}"
            )

            if score < best_score * (1.0 - 1.0e-6):
                best_score = score
                best_state = cpu_state_dict_r4(model)
                stale_epochs = 0
            else:
                stale_epochs += LOG_EVERY_R4

            if stale_epochs >= EARLY_STOPPING_PATIENCE_R4:
                print(
                    f"{label}: joint Adam early stopping at epoch {epoch}."
                )
                break

    if collocation is not None:
        del collocation
    model.load_state_dict(best_state)
    clear_gpu_memory()

    # ------------------------------------------------------------------
    # Stage IV: reduced fixed-collocation L-BFGS refinement
    # ------------------------------------------------------------------
    if LBFGS_MAX_ITER_R4 > 0:
        fixed_collocation = make_collocation_bundle_r4(
            datasets,
            seed + 90_000,
            LBFGS_POINTS_PER_INTERVAL_R4,
        )

        optimizer_lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=0.75,
            max_iter=LBFGS_MAX_ITER_R4,
            max_eval=max(2 * LBFGS_MAX_ITER_R4, 20),
            tolerance_grad=1.0e-10,
            tolerance_change=1.0e-12,
            history_size=40,
            line_search_fn="strong_wolfe",
        )

        closure_counter = {"value": 0}

        def closure():
            optimizer_lbfgs.zero_grad(set_to_none=True)
            total, _ = complete_loss(
                model,
                datasets,
                fixed_collocation,
                physics_weight=1.0,
                split="train",
            )
            total.backward()
            closure_counter["value"] += 1
            return total

        optimizer_lbfgs.step(closure)

        total, components = complete_loss(
            model,
            datasets,
            fixed_collocation,
            physics_weight=1.0,
            split="train",
        )

        score, _ = validation_score_r4(
            model,
            datasets,
            seed + 99_000,
            physics_weight=1.0,
        )

        append_history_row_r4(
            history_rows,
            model,
            "lbfgs",
            closure_counter["value"],
            score,
            components,
        )

        if score < best_score:
            best_score = score
            best_state = cpu_state_dict_r4(model)

        del fixed_collocation, total
        clear_gpu_memory()

    model.load_state_dict(best_state)

    elapsed = time.time() - start_time
    history = pd.DataFrame(history_rows)

    history_path = HISTORY_DIR_R4 / f"{label}_history.csv"
    model_path = MODEL_DIR_R4 / f"{label}.pt"

    history.to_csv(history_path, index=False)

    checkpoint = {
        "model_state": cpu_state_dict_r4(model),
        "model_kind": model.model_kind,
        "design_id": design_id,
        "schedules": list(datasets),
        "inferred_parameters": model.inferred_parameters(),
        "seed": seed,
        "training_budget": {
            "quick_mode": QUICK_MODE,
            "pretrain_epochs": PRETRAIN_EPOCHS_R4,
            "physics_warmup_epochs": PHYSICS_WARMUP_EPOCHS_R4,
            "adam_epochs": ADAM_EPOCHS_R4,
            "lbfgs_max_iter": LBFGS_MAX_ITER_R4,
            "collocation_points_per_interval": (
                COLLOCATION_POINTS_PER_INTERVAL_R4
            ),
        },
    }

    torch.save(checkpoint, model_path)

    print(
        f"{label} completed in {elapsed / 60.0:.2f} min; "
        f"best validation score={best_score:.3e}."
    )

    return TrainingResultR4(
        label=label,
        design_id=design_id,
        model=model,
        history=history,
        elapsed_seconds=elapsed,
        best_validation_score=float(best_score),
        seed=int(seed),
    )


In [ ]:
# =============================================================================
# ROUND 4 — CELL 12
# Dense evaluation, state/parameter metrics, constraint audit, and inference time
# =============================================================================


def predict_dense(model: AMRInverseModel, schedule_name: str,
                  dense: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    dense = DENSE_REFERENCE[schedule_name] if dense is None else dense
    t_values = dense['time'].to_numpy(float)
    t_tensor = tensor(t_values)
    model.eval()
    with torch.no_grad():
        prediction = model(t_tensor, schedule_name).detach().cpu().numpy()
    S = TRUE_PARAMETERS.carrying_capacity*prediction[:,0]
    R = TRUE_PARAMETERS.carrying_capacity*prediction[:,1]
    B = S+R
    q = np.divide(R,B,out=np.zeros_like(R),where=B>0)
    frame = dense.copy()
    frame['predicted_sensitive'] = S
    frame['predicted_resistant'] = R
    frame['predicted_total'] = B
    frame['predicted_resistant_fraction'] = q
    return frame


def relative_l2(predicted: np.ndarray, true: np.ndarray) -> float:
    return float(np.linalg.norm(predicted-true)/(np.linalg.norm(true)+1e-12))


def residual_rmse_dense(model: AMRInverseModel, schedule_name: str) -> float:
    schedule = SCHEDULES_R4[schedule_name]
    points = []
    for left,right,_ in schedule.intervals(T_START,T_FINAL):
        points.append(np.linspace(left+1e-5,right-1e-5,150))
    t_res = tensor(np.concatenate(points), requires_grad=True)
    if not model.physics_enabled:
        return np.nan
    residual_s,residual_r = physics_residual(model,t_res,schedule_name)
    value = torch.sqrt(torch.mean(residual_s**2+residual_r**2))
    return float(value.detach().cpu())


def inference_milliseconds(model: AMRInverseModel, schedule_name: str,
                           n_points: int = 1001, repeats: int = 20) -> float:
    t_eval = tensor(np.linspace(T_START,T_FINAL,n_points))
    with torch.no_grad():
        _ = model(t_eval,schedule_name)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        start=time.perf_counter()
        for _ in range(repeats):
            _=model(t_eval,schedule_name)
        if torch.cuda.is_available(): torch.cuda.synchronize()
    return 1000*(time.perf_counter()-start)/repeats


def evaluate_trained_model(result: TrainingResultR4,
                           schedule_name: str) -> Tuple[Dict[str,Any],pd.DataFrame]:
    frame = predict_dense(result.model,schedule_name)
    metrics: Dict[str,Any] = {
        'label':result.label, 'model_kind':result.model.model_kind,
        'schedule':schedule_name, 'training_seconds':result.elapsed_seconds,
        'best_validation_score':result.best_validation_score,
    }
    pairs = {
        'sensitive':('sensitive','predicted_sensitive'),
        'resistant':('resistant','predicted_resistant'),
        'total':('total','predicted_total'),
        'fraction':('resistant_fraction','predicted_resistant_fraction'),
    }
    for name,(true_col,pred_col) in pairs.items():
        true=frame[true_col].to_numpy(float); pred=frame[pred_col].to_numpy(float)
        metrics[f'rmse_{name}']=float(np.sqrt(mean_squared_error(true,pred)))
        metrics[f'mae_{name}']=float(mean_absolute_error(true,pred))
        metrics[f'r2_{name}']=float(r2_score(true,pred))
        metrics[f'relative_l2_{name}']=relative_l2(pred,true)

    S=frame['predicted_sensitive'].to_numpy(float)
    R=frame['predicted_resistant'].to_numpy(float)
    metrics.update({
        'maximum_negative_violation':float(max(0,-S.min(),-R.min())),
        'maximum_capacity_violation':float(max(0,(S+R).max()-TRUE_PARAMETERS.carrying_capacity)),
        'fraction_points_negative':float(np.mean((S<0)|(R<0))),
        'fraction_points_over_capacity':float(np.mean(S+R>TRUE_PARAMETERS.carrying_capacity+1e-8)),
        'physics_residual_rmse':residual_rmse_dense(result.model,schedule_name),
        'inference_ms_1001_points':inference_milliseconds(result.model,schedule_name),
        **result.model.inferred_parameters(),
    })
    if result.model.physics_enabled:
        metrics['relative_error_mu']=abs(metrics['mutation_probability']-TRUE_PARAMETERS.mutation_probability)/TRUE_PARAMETERS.mutation_probability
        metrics['relative_error_cost']=abs(metrics['resistance_fitness_cost']-TRUE_PARAMETERS.resistance_fitness_cost)/TRUE_PARAMETERS.resistance_fitness_cost
    else:
        metrics['relative_error_mu']=np.nan; metrics['relative_error_cost']=np.nan
    frame.to_csv(PREDICTION_DIR/f'{result.label}_{schedule_name}_predictions.csv',index=False)
    return metrics,frame


# -----------------------------------------------------------------------------
# Round 4 design-level aggregation
# -----------------------------------------------------------------------------

def evaluate_design_r4(
    result: TrainingResultR4,
    datasets: Dict[str, ObservationDataset],
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """
    Evaluate every schedule and return both schedule-level and design-level rows.
    Design-level RMSE values are calculated after concatenating dense trajectories,
    so every evaluated time point contributes directly.
    """
    schedule_rows: List[Dict[str, Any]] = []
    frames: List[pd.DataFrame] = []

    for schedule_name in datasets:
        metrics, frame = evaluate_trained_model(
            result,
            schedule_name,
        )
        metrics.update({
            "design_id": result.design_id,
            "seed": result.seed,
            "row_type": "schedule",
        })
        schedule_rows.append(metrics)

        frame = frame.copy()
        frame["schedule"] = schedule_name
        frame["design_id"] = result.design_id
        frame["seed"] = result.seed
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True)

    design_metrics: Dict[str, Any] = {
        "label": result.label,
        "model_kind": result.model.model_kind,
        "design_id": result.design_id,
        "seed": result.seed,
        "schedule": "ALL",
        "row_type": "design",
        "schedule_count": len(datasets),
        "observation_regime": ROUND4_DESIGNS[
            result.design_id
        ].observation_regime,
        "training_seconds": result.elapsed_seconds,
        "training_minutes": result.elapsed_seconds / 60.0,
        "best_validation_score": result.best_validation_score,
        **result.model.inferred_parameters(),
    }

    pairs = {
        "sensitive": (
            "sensitive",
            "predicted_sensitive",
        ),
        "resistant": (
            "resistant",
            "predicted_resistant",
        ),
        "total": (
            "total",
            "predicted_total",
        ),
        "fraction": (
            "resistant_fraction",
            "predicted_resistant_fraction",
        ),
    }

    for name, (true_column, predicted_column) in pairs.items():
        true_values = combined[true_column].to_numpy(float)
        predicted_values = combined[predicted_column].to_numpy(float)

        design_metrics[f"rmse_{name}"] = float(
            np.sqrt(
                mean_squared_error(
                    true_values,
                    predicted_values,
                )
            )
        )
        design_metrics[f"mae_{name}"] = float(
            mean_absolute_error(
                true_values,
                predicted_values,
            )
        )
        design_metrics[f"relative_l2_{name}"] = relative_l2(
            predicted_values,
            true_values,
        )

    sensitive_prediction = combined[
        "predicted_sensitive"
    ].to_numpy(float)
    resistant_prediction = combined[
        "predicted_resistant"
    ].to_numpy(float)

    design_metrics.update({
        "maximum_negative_violation": float(
            max(
                0.0,
                -sensitive_prediction.min(),
                -resistant_prediction.min(),
            )
        ),
        "maximum_capacity_violation": float(
            max(
                0.0,
                (
                    sensitive_prediction
                    + resistant_prediction
                ).max()
                - TRUE_PARAMETERS.carrying_capacity,
            )
        ),
        "fraction_points_negative": float(
            np.mean(
                (sensitive_prediction < 0.0)
                | (resistant_prediction < 0.0)
            )
        ),
        "fraction_points_over_capacity": float(
            np.mean(
                sensitive_prediction
                + resistant_prediction
                > TRUE_PARAMETERS.carrying_capacity
                + 1.0e-8
            )
        ),
        "physics_residual_rmse": float(
            np.sqrt(
                np.mean([
                    row["physics_residual_rmse"] ** 2
                    for row in schedule_rows
                ])
            )
        ),
        "relative_error_mu": abs(
            design_metrics["mutation_probability"]
            - TRUE_PARAMETERS.mutation_probability
        )
        / TRUE_PARAMETERS.mutation_probability,
        "relative_error_cost": abs(
            design_metrics["resistance_fitness_cost"]
            - TRUE_PARAMETERS.resistance_fitness_cost
        )
        / TRUE_PARAMETERS.resistance_fitness_cost,
    })

    all_rows = pd.DataFrame(
        schedule_rows + [design_metrics]
    )

    all_rows.to_csv(
        TABLE_DIR_R4
        / f"{result.label}_schedule_and_design_metrics.csv",
        index=False,
    )

    combined.to_csv(
        PREDICTION_DIR_R4
        / f"{result.label}_all_schedule_predictions.csv",
        index=False,
    )

    return design_metrics, all_rows


In [ ]:
# =============================================================================
# ROUND 4 — CELL 13
# Classical ODE nonlinear least-squares baseline with multiple starting points
# =============================================================================
# This baseline asks how well conventional parameter fitting performs when the
# states are generated by numerical integration rather than a neural network.
# It is essential because a PINN should not be compared only with other PINNs.


def ode_prediction_for_parameters(schedule_name: str, times: np.ndarray,
                                  log10_mu: float, cost: float) -> pd.DataFrame:
    parameters = replace(TRUE_PARAMETERS,
                         mutation_probability=10.0**log10_mu,
                         resistance_fitness_cost=float(cost))
    return solve_reference_r4(parameters,SCHEDULES_R4[schedule_name],times)


def ode_residual_vector(x: np.ndarray,
                       datasets: Dict[str,ObservationDataset]) -> np.ndarray:
    log10_mu,cost=x
    residuals=[]
    for schedule_name,dataset in datasets.items():
        all_times=np.unique(np.concatenate([dataset.total_time,dataset.fraction_time]))
        pred=ode_prediction_for_parameters(schedule_name,all_times,log10_mu,cost)
        pred_total=nearest_values(pred,dataset.total_time,'total')
        residuals.extend(np.log(pred_total+1e-8)-np.log(dataset.total_observed+1e-8))
        if len(dataset.fraction_time):
            pred_fraction=nearest_values(pred,dataset.fraction_time,'resistant_fraction')
            residuals.extend(2.0*(pred_fraction-dataset.fraction_observed))
    return np.asarray(residuals,float)


def fit_ode_baseline(datasets: Dict[str,ObservationDataset],
                     label: str, seed: int=SEED,
                     starts: int=12) -> Dict[str,Any]:
    rng=np.random.default_rng(seed)
    initial_points=[np.array([-4.0,0.10])]
    initial_points += [np.array([rng.uniform(-5.8,-2.2),rng.uniform(0.005,0.38)])
                       for _ in range(starts-1)]
    best=None
    start_time=time.time()
    for initial in initial_points:
        fit=least_squares(lambda x: ode_residual_vector(x,datasets),initial,
                          bounds=([-6.0,0.0],[-2.0,0.40]),
                          xtol=1e-11,ftol=1e-11,gtol=1e-11,max_nfev=400)
        if best is None or np.sum(fit.fun**2)<np.sum(best.fun**2):
            best=fit
    elapsed=time.time()-start_time
    result={
        'label':label,'model_kind':'ode_fit',
        'mutation_probability':10.0**best.x[0],
        'resistance_fitness_cost':best.x[1],
        'objective':float(np.sum(best.fun**2)),
        'nfev':int(best.nfev),'training_seconds':elapsed,
        'relative_error_mu':abs(10.0**best.x[0]-TRUE_PARAMETERS.mutation_probability)/TRUE_PARAMETERS.mutation_probability,
        'relative_error_cost':abs(best.x[1]-TRUE_PARAMETERS.resistance_fitness_cost)/TRUE_PARAMETERS.resistance_fitness_cost,
    }
    with open(REPORT_DIR_R4/f'{label}_ode_fit.json','w') as file:
        json.dump(result,file,indent=2)
    return result

print('ODE baseline functions are ready.')


# -----------------------------------------------------------------------------
# Round 4 matched ODE evaluation on a complete design
# -----------------------------------------------------------------------------

def evaluate_ode_design_r4(
    design_id: str,
    datasets: Dict[str, ObservationDataset],
    fit_result: Dict[str, Any],
) -> Dict[str, Any]:
    parameters = replace(
        TRUE_PARAMETERS,
        mutation_probability=fit_result["mutation_probability"],
        resistance_fitness_cost=fit_result["resistance_fitness_cost"],
    )

    frames = []
    for schedule_name in datasets:
        prediction = solve_reference_r4(
            parameters,
            SCHEDULES_R4[schedule_name],
            DENSE_REFERENCE[schedule_name]["time"].to_numpy(float),
        )
        true = DENSE_REFERENCE[schedule_name]
        frame = true.copy()
        frame["predicted_sensitive"] = prediction["sensitive"].to_numpy(float)
        frame["predicted_resistant"] = prediction["resistant"].to_numpy(float)
        frame["predicted_total"] = prediction["total"].to_numpy(float)
        frame["predicted_resistant_fraction"] = prediction[
            "resistant_fraction"
        ].to_numpy(float)
        frame["schedule"] = schedule_name
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True)

    row = {
        **fit_result,
        "design_id": design_id,
        "schedule_count": len(datasets),
        "observation_regime": ROUND4_DESIGNS[design_id].observation_regime,
    }

    for name, true_column, predicted_column in [
        ("sensitive", "sensitive", "predicted_sensitive"),
        ("resistant", "resistant", "predicted_resistant"),
        ("total", "total", "predicted_total"),
        (
            "fraction",
            "resistant_fraction",
            "predicted_resistant_fraction",
        ),
    ]:
        true_values = combined[true_column].to_numpy(float)
        predicted_values = combined[predicted_column].to_numpy(float)
        row[f"rmse_{name}"] = float(
            np.sqrt(
                mean_squared_error(
                    true_values,
                    predicted_values,
                )
            )
        )
        row[f"relative_l2_{name}"] = relative_l2(
            predicted_values,
            true_values,
        )

    return row

In [ ]:
# =============================================================================
# ROUND 4 — CELL 14
# Design-level sensitivity SVD, Fisher geometry, and confounding diagnostics
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# Neural optimization variability is not the same as information content.
# This cell differentiates the deterministic observation map with respect to
#
#     theta = [log10(mu), c_R]
#
# and analyzes its normalized sensitivity matrix. The smallest singular value
# measures the weakest observable parameter direction; the condition number
# measures confounding. This analysis depends on the sampling design, not on a
# trained neural-network checkpoint.


def observable_vector_from_design_r4(
    datasets: Dict[str, ObservationDataset],
    log10_mu: float,
    cost: float,
) -> np.ndarray:
    parameters = replace(
        TRUE_PARAMETERS,
        mutation_probability=10.0 ** float(log10_mu),
        resistance_fitness_cost=float(cost),
    )

    values: List[float] = []

    for schedule_name, dataset in datasets.items():
        all_times = np.unique(
            np.concatenate([
                dataset.total_time,
                dataset.fraction_time,
            ])
        )

        prediction = solve_reference_r4(
            parameters,
            SCHEDULES_R4[schedule_name],
            all_times,
        )

        predicted_total = nearest_values(
            prediction,
            dataset.total_time,
            "total",
        ) / TRUE_PARAMETERS.carrying_capacity

        # Log total load mirrors the training observation operator.
        values.extend(
            np.log(predicted_total + 1.0e-8)
        )

        if len(dataset.fraction_time):
            values.extend(
                nearest_values(
                    prediction,
                    dataset.fraction_time,
                    "resistant_fraction",
                )
            )

    return np.asarray(values, dtype=float)


def normalized_sensitivity_geometry_r4(
    datasets: Dict[str, ObservationDataset],
) -> Dict[str, Any]:
    theta = np.array([
        math.log10(TRUE_PARAMETERS.mutation_probability),
        TRUE_PARAMETERS.resistance_fitness_cost,
    ])

    steps = np.array([
        1.0e-3,
        1.0e-4,
    ])

    baseline = observable_vector_from_design_r4(
        datasets,
        theta[0],
        theta[1],
    )

    columns = []

    for parameter_index in range(2):
        plus = theta.copy()
        minus = theta.copy()
        plus[parameter_index] += steps[parameter_index]
        minus[parameter_index] -= steps[parameter_index]

        derivative = (
            observable_vector_from_design_r4(
                datasets,
                plus[0],
                plus[1],
            )
            - observable_vector_from_design_r4(
                datasets,
                minus[0],
                minus[1],
            )
        ) / (2.0 * steps[parameter_index])

        # Prevent near-zero fraction observations from receiving unbounded weight.
        output_scale = np.maximum(
            np.abs(baseline),
            1.0e-3,
        )
        columns.append(
            derivative / output_scale
        )

    sensitivity = np.column_stack(columns)
    _, singular_values, right_vectors = np.linalg.svd(
        sensitivity,
        full_matrices=False,
    )

    fisher = sensitivity.T @ sensitivity
    covariance_proxy = np.linalg.pinv(fisher)

    denominator = math.sqrt(
        max(covariance_proxy[0, 0], 0.0)
        * max(covariance_proxy[1, 1], 0.0)
    )
    correlation = (
        covariance_proxy[0, 1] / denominator
        if denominator > 0
        else np.nan
    )

    return {
        "matrix": sensitivity,
        "singular_values": singular_values,
        "right_vectors": right_vectors,
        "fisher": fisher,
        "covariance_proxy": covariance_proxy,
        "condition_number": float(
            singular_values[0]
            / max(singular_values[-1], 1.0e-15)
        ),
        "parameter_correlation": float(correlation),
    }


SENSITIVITY_RESULTS_R4: Dict[str, Dict[str, Any]] = {}
sensitivity_rows_r4 = []

for design_id in DESIGN_ORDER_R4:
    bundle = build_design_bundle_r4(
        design_id,
        seed=SEED,
        noise_level=0.0,
    )
    geometry = normalized_sensitivity_geometry_r4(bundle)
    SENSITIVITY_RESULTS_R4[design_id] = geometry

    sensitivity_rows_r4.append({
        "design_id": design_id,
        "schedule_count": ROUND4_DESIGNS[design_id].schedule_count,
        "observation_regime": ROUND4_DESIGNS[
            design_id
        ].observation_regime,
        "largest_singular_value": float(
            geometry["singular_values"][0]
        ),
        "smallest_singular_value": float(
            geometry["singular_values"][-1]
        ),
        "condition_number": geometry["condition_number"],
        "parameter_correlation": geometry[
            "parameter_correlation"
        ],
        "weak_direction_log10_mu": float(
            geometry["right_vectors"][-1, 0]
        ),
        "weak_direction_cost": float(
            geometry["right_vectors"][-1, 1]
        ),
    })

SENSITIVITY_SUMMARY_R4 = pd.DataFrame(
    sensitivity_rows_r4
)

SENSITIVITY_SUMMARY_R4.to_csv(
    TABLE_DIR_R4
    / "round4_design_identifiability_geometry.csv",
    index=False,
)

display(SENSITIVITY_SUMMARY_R4)

In [ ]:
# =============================================================================
# ROUND 4 — CELL 15
# Structural smoke test: hard constraints, shared parameters, loss, and training
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# This is a full pipeline check using a tiny M2 model. It verifies:
# - one shared mu/c_R pair across two schedule-specific state networks;
# - exact initial states and treatment-interface continuity;
# - positivity and carrying-capacity compliance;
# - finite observation and physics losses;
# - successful staged optimization and dense evaluation.
#
# It is not a manuscript result.

ROUND4_SMOKE_RESULT = None
ROUND4_SMOKE_METRICS = None

if RUN_STRUCTURAL_SMOKE_TEST:
    smoke_design_id = "M2"
    smoke_bundle = build_design_bundle_r4(
        smoke_design_id,
        seed=SEED + 700,
        noise_level=0.0,
    )

    smoke_schedules = {
        name: SCHEDULES_R4[name]
        for name in smoke_bundle
    }

    smoke_model = AMRInverseModel(
        smoke_schedules,
        model_kind="hard",
        width=16,
        depth=2,
        parameter_initialization=(3.0e-4, 0.15),
    ).to(DEVICE)

    dense_test = tensor(
        np.linspace(
            T_START,
            T_FINAL,
            1001,
        )
    )

    with torch.no_grad():
        for schedule_name in smoke_bundle:
            state = smoke_model(
                dense_test,
                schedule_name,
            ).detach().cpu().numpy()

            assert state.min() >= -1.0e-12
            assert state.sum(axis=1).max() <= 1.0 + 1.0e-12
            assert np.max(
                np.abs(
                    state[0]
                    - np.array([
                        INITIAL_S
                        / TRUE_PARAMETERS.carrying_capacity,
                        INITIAL_R
                        / TRUE_PARAMETERS.carrying_capacity,
                    ])
                )
            ) <= 1.0e-12

    smoke_collocation = make_collocation_bundle_r4(
        smoke_bundle,
        seed=SEED + 701,
        points_per_interval=16,
    )

    smoke_loss, smoke_components = complete_loss(
        smoke_model,
        smoke_bundle,
        smoke_collocation,
        physics_weight=1.0,
        split="train",
    )

    assert torch.isfinite(smoke_loss)

    smoke_loss.backward()

    gradient_norm = torch.sqrt(
        sum(
            torch.sum(parameter.grad ** 2)
            for parameter in smoke_model.parameters()
            if parameter.grad is not None
        )
    )

    assert torch.isfinite(gradient_norm)

    del smoke_model, smoke_collocation, smoke_loss
    clear_gpu_memory()

    ROUND4_SMOKE_RESULT = train_shared_parameter_model_r4(
        AMRInverseModel(
            smoke_schedules,
            model_kind="hard",
            width=16,
            depth=2,
            parameter_initialization=(3.0e-4, 0.15),
        ),
        smoke_bundle,
        design_id=smoke_design_id,
        label="round4_smoke_M2",
        seed=SEED + 702,
    )

    ROUND4_SMOKE_METRICS, smoke_rows = evaluate_design_r4(
        ROUND4_SMOKE_RESULT,
        smoke_bundle,
    )

    print("\nRound 4 structural smoke test passed.")
    display(
        pd.DataFrame([
            ROUND4_SMOKE_METRICS
        ])
    )
else:
    print(
        "Set RUN_STRUCTURAL_SMOKE_TEST=True for the reduced pipeline check."
    )


In [ ]:
# =============================================================================
# ROUND 4 — CELL 16
# Optional single-design demonstration: M4 with four schedules and shared parameters
# =============================================================================
# This cell is useful before committing to the complete 25-model paired-seed
# suite. It trains one M4 model, evaluates every schedule, and saves all dense
# predictions. Keep it disabled during the main primary experiment.

ROUND4_DEMO_RESULT = None
ROUND4_DEMO_METRICS = None

if RUN_SINGLE_DESIGN_DEMO:
    demo_design_id = "M4"
    demo_seed = SEED + 400_000
    demo_bundle = build_design_bundle_r4(
        demo_design_id,
        seed=demo_seed,
        noise_level=0.0,
    )

    demo_model = AMRInverseModel(
        {
            name: SCHEDULES_R4[name]
            for name in demo_bundle
        },
        model_kind="hard",
        width=NETWORK_WIDTH_R4,
        depth=NETWORK_DEPTH_R4,
        parameter_initialization=(3.0e-4, 0.15),
    )

    ROUND4_DEMO_RESULT = train_shared_parameter_model_r4(
        demo_model,
        demo_bundle,
        design_id=demo_design_id,
        label="round4_demo_M4",
        seed=demo_seed,
    )

    ROUND4_DEMO_METRICS, demo_rows = evaluate_design_r4(
        ROUND4_DEMO_RESULT,
        demo_bundle,
    )

    display(demo_rows)
else:
    print(
        "RUN_SINGLE_DESIGN_DEMO=False; the optional M4 demonstration is skipped."
    )


In [ ]:
# =============================================================================
# ROUND 4 — CELL 17
# Primary paired-seed nested-design experiment: M1--M5
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# This is the main Round 4 experiment.
#
# For every seed, the nested designs reuse the same schedule-specific synthetic
# observations and the same training seed. Therefore, differences between M1,
# M2, M3, and M4 are attributable primarily to added treatment environments.
# M5 uses O4 observations and is interpreted as an upper-information benchmark.
#
# RESUME SAFETY
# -------------
# One design/seed row is saved immediately after completion. Rerunning the cell
# skips completed pairs unless FORCE_RERUN_PRIMARY_R4=True.

PRIMARY_RAW_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_primary_multischedule_all_seeds_metrics.csv"
)

PRIMARY_SCHEDULE_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_primary_multischedule_schedule_metrics.csv"
)

FORCE_RERUN_PRIMARY_R4 = False

ROUND4_PRIMARY_RESULTS: Dict[str, TrainingResultR4] = {}
ROUND4_PRIMARY_ROWS: List[Dict[str, Any]] = []
ROUND4_PRIMARY_SCHEDULE_ROWS: List[Dict[str, Any]] = []


def save_primary_progress_r4() -> None:
    if ROUND4_PRIMARY_ROWS:
        frame = (
            pd.DataFrame(ROUND4_PRIMARY_ROWS)
            .drop_duplicates(
                subset=["design_id", "seed_id"],
                keep="last",
            )
        )
        order_map = {
            design_id: index
            for index, design_id in enumerate(
                DESIGN_ORDER_R4
            )
        }
        frame["_order"] = frame["design_id"].map(order_map)
        frame = (
            frame
            .sort_values(["_order", "seed_number"])
            .drop(columns="_order")
            .reset_index(drop=True)
        )
        frame.to_csv(
            PRIMARY_RAW_PATH_R4,
            index=False,
        )

    if ROUND4_PRIMARY_SCHEDULE_ROWS:
        schedule_frame = (
            pd.DataFrame(
                ROUND4_PRIMARY_SCHEDULE_ROWS
            )
            .drop_duplicates(
                subset=[
                    "design_id",
                    "seed_id",
                    "schedule",
                    "row_type",
                ],
                keep="last",
            )
        )
        schedule_frame.to_csv(
            PRIMARY_SCHEDULE_PATH_R4,
            index=False,
        )


if RUN_PRIMARY_MULTI_SCHEDULE_STUDY:
    if (
        PRIMARY_RAW_PATH_R4.exists()
        and not FORCE_RERUN_PRIMARY_R4
    ):
        existing = pd.read_csv(
            PRIMARY_RAW_PATH_R4
        )
        required_resume = {
            "design_id",
            "seed_id",
            "seed_number",
        }
        if required_resume.issubset(existing.columns):
            ROUND4_PRIMARY_ROWS = existing.to_dict(
                "records"
            )
            print(
                f"Resuming from {len(existing)} completed design-seed runs."
            )

    if (
        PRIMARY_SCHEDULE_PATH_R4.exists()
        and not FORCE_RERUN_PRIMARY_R4
    ):
        existing_schedule = pd.read_csv(
            PRIMARY_SCHEDULE_PATH_R4
        )
        ROUND4_PRIMARY_SCHEDULE_ROWS = (
            existing_schedule.to_dict("records")
        )

    completed_pairs = {
        (
            str(row["design_id"]),
            str(row["seed_id"]),
        )
        for row in ROUND4_PRIMARY_ROWS
    }

    total_runs = (
        len(DESIGN_ORDER_R4)
        * len(PRIMARY_SEED_IDS_R4)
    )

    print(
        f"Primary Round 4 suite: {len(DESIGN_ORDER_R4)} designs "
        f"x {len(PRIMARY_SEED_IDS_R4)} paired seeds = {total_runs} models."
    )

    for seed_number in PRIMARY_SEED_IDS_R4:
        seed_id = f"seed{seed_number}"

        # The same seeds are reused across every nested design.
        dataset_seed = (
            SEED
            + 10_000 * seed_number
        )
        training_seed = (
            SEED
            + 100_000
            + 10_000 * seed_number
        )

        print("\n" + "=" * 78)
        print(
            f"{seed_id}: dataset_seed={dataset_seed}, "
            f"training_seed={training_seed}"
        )
        print("=" * 78)

        for design_id in DESIGN_ORDER_R4:
            pair = (
                design_id,
                seed_id,
            )

            label = (
                f"round4_{design_id}_{seed_id}"
            )

            if (
                pair in completed_pairs
                and not FORCE_RERUN_PRIMARY_R4
            ):
                print(
                    f"Skipping completed run: {label}"
                )
                continue

            bundle = build_design_bundle_r4(
                design_id,
                seed=dataset_seed,
                noise_level=0.0,
            )

            model = AMRInverseModel(
                {
                    name: SCHEDULES_R4[name]
                    for name in bundle
                },
                model_kind="hard",
                width=NETWORK_WIDTH_R4,
                depth=NETWORK_DEPTH_R4,
                parameter_initialization=(
                    3.0e-4,
                    0.15,
                ),
            )

            try:
                result = train_shared_parameter_model_r4(
                    model,
                    bundle,
                    design_id=design_id,
                    label=label,
                    seed=training_seed,
                )

                design_metrics, all_rows = evaluate_design_r4(
                    result,
                    bundle,
                )

                design_metrics.update({
                    "seed_id": seed_id,
                    "seed_number": int(seed_number),
                    "dataset_seed": int(dataset_seed),
                    "training_seed": int(training_seed),
                    "mu_error_percent": (
                        100.0
                        * design_metrics[
                            "relative_error_mu"
                        ]
                    ),
                    "cost_error_percent": (
                        100.0
                        * design_metrics[
                            "relative_error_cost"
                        ]
                    ),
                })

                all_rows = all_rows.copy()
                all_rows["seed_id"] = seed_id
                all_rows["seed_number"] = int(
                    seed_number
                )
                all_rows["dataset_seed"] = int(
                    dataset_seed
                )
                all_rows["training_seed"] = int(
                    training_seed
                )

                ROUND4_PRIMARY_ROWS.append(
                    design_metrics
                )
                ROUND4_PRIMARY_SCHEDULE_ROWS.extend(
                    all_rows.to_dict("records")
                )
                completed_pairs.add(pair)

                ROUND4_PRIMARY_RESULTS[label] = result
                save_primary_progress_r4()

                completed_count = len(
                    {
                        (
                            str(row["design_id"]),
                            str(row["seed_id"]),
                        )
                        for row in ROUND4_PRIMARY_ROWS
                    }
                )

                print(
                    f"Saved {label}: "
                    f"{completed_count}/{total_runs} completed."
                )

                result.model.to("cpu")
                del model, bundle
                clear_gpu_memory()

            except Exception:
                save_primary_progress_r4()
                clear_gpu_memory()
                print(
                    f"\nRun failed at {label}. "
                    "All previously completed runs remain saved."
                )
                raise

    save_primary_progress_r4()

    ROUND4_PRIMARY_ALL_RUNS = pd.read_csv(
        PRIMARY_RAW_PATH_R4
    )

    print("\nPrimary Round 4 suite completed.")
    display(ROUND4_PRIMARY_ALL_RUNS)

else:
    print(
        "Set RUN_PRIMARY_MULTI_SCHEDULE_STUDY=True to run M1--M5 "
        "across the paired seed set."
    )

In [ ]:
# =============================================================================
# ROUND 4 — CELL 18
# Mean ± sample SD tables, paired effects, and manuscript-ready summaries
# =============================================================================
# This cell is plotting/training independent. It reloads the primary CSV when
# necessary and calculates mean ± sample SD using only observed design groups.

if (
    "ROUND4_PRIMARY_ALL_RUNS" not in globals()
    or not isinstance(
        ROUND4_PRIMARY_ALL_RUNS,
        pd.DataFrame,
    )
    or ROUND4_PRIMARY_ALL_RUNS.empty
):
    if PRIMARY_RAW_PATH_R4.exists():
        ROUND4_PRIMARY_ALL_RUNS = pd.read_csv(
            PRIMARY_RAW_PATH_R4
        )
    else:
        ROUND4_PRIMARY_ALL_RUNS = pd.DataFrame()

PRIMARY_SUMMARY_R4 = pd.DataFrame()
PRIMARY_MANUSCRIPT_TABLE_R4 = pd.DataFrame()
PRIMARY_PAIRED_EFFECTS_R4 = pd.DataFrame()

if not ROUND4_PRIMARY_ALL_RUNS.empty:
    raw = ROUND4_PRIMARY_ALL_RUNS.copy()

    numeric_metrics = [
        "rmse_sensitive",
        "rmse_resistant",
        "rmse_total",
        "rmse_fraction",
        "relative_l2_sensitive",
        "relative_l2_resistant",
        "mutation_probability",
        "mu_error_percent",
        "resistance_fitness_cost",
        "cost_error_percent",
        "physics_residual_rmse",
        "maximum_negative_violation",
        "maximum_capacity_violation",
        "training_minutes",
        "best_validation_score",
    ]

    grouped = raw.groupby(
        "design_id",
        observed=True,
        sort=False,
    )

    means = grouped[numeric_metrics].mean()
    standard_deviations = grouped[
        numeric_metrics
    ].std(ddof=1)
    counts = grouped.size().rename("n_seeds")

    PRIMARY_SUMMARY_R4 = pd.concat(
        [
            counts,
            means.add_prefix("mean_"),
            standard_deviations.add_prefix("std_"),
        ],
        axis=1,
    ).reset_index()

    order_map = {
        design_id: index
        for index, design_id in enumerate(
            DESIGN_ORDER_R4
        )
    }

    PRIMARY_SUMMARY_R4["_order"] = (
        PRIMARY_SUMMARY_R4["design_id"].map(
            order_map
        )
    )

    PRIMARY_SUMMARY_R4 = (
        PRIMARY_SUMMARY_R4
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    PRIMARY_SUMMARY_R4.to_csv(
        TABLE_DIR_R4
        / "round4_primary_summary_mean_sd.csv",
        index=False,
    )

    def format_mean_sd_r4(
        mean_value: float,
        std_value: float,
        metric: str,
    ) -> str:
        if pd.isna(mean_value):
            return "N/A"
        if pd.isna(std_value):
            return f"{mean_value:.4g} ± N/A"

        if metric == "mutation_probability":
            return (
                f"{mean_value:.3e} ± "
                f"{std_value:.2e}"
            )
        if metric in {
            "mu_error_percent",
            "cost_error_percent",
        }:
            return (
                f"{mean_value:.1f} ± "
                f"{std_value:.1f}"
            )
        if metric in {
            "rmse_fraction",
            "physics_residual_rmse",
        }:
            return (
                f"{mean_value:.3e} ± "
                f"{std_value:.2e}"
            )
        if metric == "training_minutes":
            return (
                f"{mean_value:.2f} ± "
                f"{std_value:.2f}"
            )
        return (
            f"{mean_value:.4f} ± "
            f"{std_value:.4f}"
        )

    manuscript_rows = []

    for _, summary_row in PRIMARY_SUMMARY_R4.iterrows():
        design_id = summary_row["design_id"]
        design = ROUND4_DESIGNS[design_id]

        output_row = {
            "Design": design_id,
            "Schedules": design.schedule_count,
            "Observation regime": design.observation_regime,
            "Seeds": int(summary_row["n_seeds"]),
        }

        for metric in [
            "rmse_sensitive",
            "rmse_resistant",
            "rmse_total",
            "rmse_fraction",
            "mutation_probability",
            "mu_error_percent",
            "resistance_fitness_cost",
            "cost_error_percent",
            "physics_residual_rmse",
            "training_minutes",
        ]:
            output_row[metric] = format_mean_sd_r4(
                summary_row[f"mean_{metric}"],
                summary_row[f"std_{metric}"],
                metric,
            )

        manuscript_rows.append(output_row)

    PRIMARY_MANUSCRIPT_TABLE_R4 = (
        pd.DataFrame(manuscript_rows)
        .rename(
            columns={
                "rmse_sensitive": "RMSE S",
                "rmse_resistant": "RMSE R",
                "rmse_total": "RMSE total",
                "rmse_fraction": "RMSE fraction",
                "mutation_probability": "Estimated mu",
                "mu_error_percent": "Mu error (%)",
                "resistance_fitness_cost": "Estimated c_R",
                "cost_error_percent": "c_R error (%)",
                "physics_residual_rmse": "Physics RMSE",
                "training_minutes": "Training time (min)",
            }
        )
    )

    PRIMARY_MANUSCRIPT_TABLE_R4.to_csv(
        TABLE_DIR_R4
        / "round4_primary_manuscript_mean_sd.csv",
        index=False,
    )

    # ---------------------------------------------------------------
    # Paired percentage effects relative to M1
    # ---------------------------------------------------------------
    paired_rows = []

    for metric in [
        "rmse_resistant",
        "rmse_fraction",
        "mu_error_percent",
        "cost_error_percent",
    ]:
        pivot = raw.pivot(
            index="seed_id",
            columns="design_id",
            values=metric,
        )

        for design_id in DESIGN_ORDER_R4[1:]:
            paired = pivot[["M1", design_id]].dropna()
            improvements = (
                100.0
                * (
                    paired["M1"]
                    - paired[design_id]
                )
                / paired["M1"]
            )

            paired_rows.append({
                "metric": metric,
                "comparison": f"{design_id} vs M1",
                "n_pairs": len(improvements),
                "mean_improvement_percent": float(
                    improvements.mean()
                ),
                "std_improvement_percent": float(
                    improvements.std(ddof=1)
                ),
                "median_improvement_percent": float(
                    improvements.median()
                ),
            })

    PRIMARY_PAIRED_EFFECTS_R4 = pd.DataFrame(
        paired_rows
    )

    PRIMARY_PAIRED_EFFECTS_R4.to_csv(
        TABLE_DIR_R4
        / "round4_primary_paired_effects.csv",
        index=False,
    )

    print("Primary mean ± SD table:")
    display(PRIMARY_MANUSCRIPT_TABLE_R4)

    print("\nPaired effects relative to M1:")
    display(PRIMARY_PAIRED_EFFECTS_R4)

else:
    print(
        "Primary results were not found. Run Cell 17 or place the saved "
        "round4_primary_multischedule_all_seeds_metrics.csv in the table folder."
    )

In [ ]:
# ROUND 4 — CELL 19
# Final publication figures for the completed five-seed M1–M5 experiment
# =============================================================================
# This cell performs NO TRAINING. It uses Cell 17 results and Cell 18 summaries.
# It generates four manuscript-oriented figures with consistent styling.
# =============================================================================

from typing import List, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullFormatter, LogLocator

# -----------------------------------------------------------------------------
# Reload completed data if the notebook was reopened.
# -----------------------------------------------------------------------------
if "PRIMARY_RAW_PATH_R4" not in globals():
    PRIMARY_RAW_PATH_R4 = TABLE_DIR_R4 / "round4_primary_multischedule_all_seeds_metrics.csv"

if (
    "ROUND4_PRIMARY_ALL_RUNS" not in globals()
    or not isinstance(ROUND4_PRIMARY_ALL_RUNS, pd.DataFrame)
    or ROUND4_PRIMARY_ALL_RUNS.empty
):
    if PRIMARY_RAW_PATH_R4.exists():
        ROUND4_PRIMARY_ALL_RUNS = pd.read_csv(PRIMARY_RAW_PATH_R4)
    else:
        raise FileNotFoundError(
            "Round 4 primary results were not found. Run Cell 17 first."
        )

if "DESIGN_ORDER_R4" not in globals():
    DESIGN_ORDER_R4 = ["M1", "M2", "M3", "M4", "M5"]

# Rebuild Cell 18 numerical summary only if necessary.
if (
    "PRIMARY_SUMMARY_R4" not in globals()
    or not isinstance(PRIMARY_SUMMARY_R4, pd.DataFrame)
    or PRIMARY_SUMMARY_R4.empty
):
    metrics = [
        "rmse_sensitive", "rmse_resistant", "rmse_total", "rmse_fraction",
        "mutation_probability", "mu_error_percent",
        "resistance_fitness_cost", "cost_error_percent",
        "physics_residual_rmse", "training_minutes",
    ]

    grouped = ROUND4_PRIMARY_ALL_RUNS.groupby(
        "design_id",
        observed=True,
        sort=False,
    )

    PRIMARY_SUMMARY_R4 = pd.concat(
        [
            grouped.size().rename("n_seeds"),
            grouped[metrics].mean().add_prefix("mean_"),
            grouped[metrics].std(ddof=1).add_prefix("std_"),
        ],
        axis=1,
    ).reset_index()

    order_map = {
        d: i
        for i, d in enumerate(DESIGN_ORDER_R4)
    }

    PRIMARY_SUMMARY_R4["_order"] = (
        PRIMARY_SUMMARY_R4["design_id"].map(order_map)
    )

    PRIMARY_SUMMARY_R4 = (
        PRIMARY_SUMMARY_R4
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

# -----------------------------------------------------------------------------
# Validate paired-seed counts.
# -----------------------------------------------------------------------------
seed_counts_r4 = (
    ROUND4_PRIMARY_ALL_RUNS
    .groupby("design_id", observed=True)["seed_id"]
    .nunique()
    .reindex(DESIGN_ORDER_R4, fill_value=0)
)

print("\nSeed counts used for Cell 19:")
print(seed_counts_r4.to_string())

if not (seed_counts_r4 == 5).all():
    print(
        "WARNING: at least one design does not contain five unique seeds. "
        "The plots will use the available completed runs."
    )

# -----------------------------------------------------------------------------
# Design metadata.
# -----------------------------------------------------------------------------
FALLBACK_META_R4 = {
    "M1": (1, "O3"),
    "M2": (2, "O3"),
    "M3": (3, "O3"),
    "M4": (4, "O3"),
    "M5": (4, "O4"),
}

def schedule_count_r4(design_id: str) -> int:
    if "ROUND4_DESIGNS" in globals() and design_id in ROUND4_DESIGNS:
        return int(
            ROUND4_DESIGNS[design_id].schedule_count
        )
    return FALLBACK_META_R4[design_id][0]

def observation_regime_r4(design_id: str) -> str:
    if "ROUND4_DESIGNS" in globals() and design_id in ROUND4_DESIGNS:
        return str(
            ROUND4_DESIGNS[design_id].observation_regime
        )
    return FALLBACK_META_R4[design_id][1]

def design_tick_labels_r4() -> List[str]:
    """
    Compact two-line labels used in the publication figures.

    The design ID remains visually dominant, while the second line reports
    schedule count and observation regime without crowding the x-axis.
    """
    labels = []

    for d in DESIGN_ORDER_R4:
        n = schedule_count_r4(d)
        schedule_text = "sched." if n == 1 else "scheds."

        labels.append(
            f"{d}\n{n} {schedule_text}, {observation_regime_r4(d)}"
        )

    return labels

# -----------------------------------------------------------------------------
# Publication style.
# -----------------------------------------------------------------------------
PUBLICATION_RC_R4 = {
    "font.family": "sans-serif",
    "font.size": 10.5,
    "axes.titlesize": 11.8,
    "axes.labelsize": 11.0,
    "xtick.labelsize": 9.0,
    "ytick.labelsize": 9.3,
    "legend.fontsize": 8.5,
    "axes.linewidth": 1.0,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
}

colors = plt.rcParams[
    "axes.prop_cycle"
].by_key()["color"]

DESIGN_COLORS_R4 = {
    d: colors[i % len(colors)]
    for i, d in enumerate(DESIGN_ORDER_R4)
}

DESIGN_MARKERS_R4 = {
    "M1": "o",
    "M2": "s",
    "M3": "^",
    "M4": "D",
    "M5": "P",
}

# -----------------------------------------------------------------------------
# Fallback figure saver.
# -----------------------------------------------------------------------------
if "save_figure_r4" not in globals():

    def save_figure_r4(fig, stem):

        FIGURE_DIR_R4.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.png",
            dpi=600,
            bbox_inches="tight",
        )

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.pdf",
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.svg",
            bbox_inches="tight",
            facecolor="white",
        )

# -----------------------------------------------------------------------------
# Helper functions.
# -----------------------------------------------------------------------------
def summary_value_r4(
    design_id: str,
    metric: str,
    statistic: str,
) -> float:

    row = PRIMARY_SUMMARY_R4[
        PRIMARY_SUMMARY_R4["design_id"] == design_id
    ]

    return float(
        row.iloc[0][f"{statistic}_{metric}"]
    )

def set_panel_frame_r4(ax):

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

def add_panel_label_r4(ax, label):

    ax.text(
        -0.10,
        1.045,
        label,
        transform=ax.transAxes,
        fontsize=12.5,
        fontweight="bold",
        ha="left",
        va="bottom",
    )

def distribution_panel_r4(
    ax,
    metric,
    title,
    ylabel,
    *,
    log_scale=False,
    true_value: Optional[float] = None,
):

    x = np.arange(
        len(DESIGN_ORDER_R4),
        dtype=float,
    )

    means = np.array([
        summary_value_r4(
            d,
            metric,
            "mean",
        )
        for d in DESIGN_ORDER_R4
    ])

    stds = np.array([
        summary_value_r4(
            d,
            metric,
            "std",
        )
        for d in DESIGN_ORDER_R4
    ])

    # -------------------------------------------------------------------------
    # Individual five-seed values.
    # -------------------------------------------------------------------------
    for i, d in enumerate(DESIGN_ORDER_R4):

        values = (
            ROUND4_PRIMARY_ALL_RUNS.loc[
                ROUND4_PRIMARY_ALL_RUNS[
                    "design_id"
                ] == d,
                metric,
            ]
            .astype(float)
            .dropna()
            .to_numpy()
        )

        jitter = (
            np.array([0.0])
            if len(values) <= 1
            else np.linspace(
                -0.115,
                0.115,
                len(values),
            )
        )

        ax.scatter(
            np.full(
                len(values),
                x[i],
            ) + jitter,
            values,
            s=29,
            marker=DESIGN_MARKERS_R4[d],
            facecolor=DESIGN_COLORS_R4[d],
            edgecolor="black",
            linewidth=0.34,
            alpha=0.27,
            zorder=2,
        )

    # -------------------------------------------------------------------------
    # Across-seed mean trajectory.
    # -------------------------------------------------------------------------
    ax.plot(
        x,
        means,
        color="black",
        linewidth=1.35,
        alpha=0.78,
        zorder=3,
    )

    # -------------------------------------------------------------------------
    # Mean ± sample SD.
    # -------------------------------------------------------------------------
    for i, d in enumerate(DESIGN_ORDER_R4):

        ax.errorbar(
            x[i],
            means[i],
            yerr=stds[i],
            fmt=DESIGN_MARKERS_R4[d],
            markersize=8.2,
            markerfacecolor=DESIGN_COLORS_R4[d],
            markeredgecolor="black",
            markeredgewidth=0.85,
            ecolor="black",
            elinewidth=1.18,
            capsize=4.2,
            capthick=1.0,
            zorder=5,
        )

    # -------------------------------------------------------------------------
    # True parameter value, when relevant.
    # -------------------------------------------------------------------------
    if true_value is not None:

        ax.axhline(
            true_value,
            linestyle="--",
            linewidth=1.35,
            color="black",
            label="True value",
            zorder=1,
        )

    # -------------------------------------------------------------------------
    # Log axis where required.
    # -------------------------------------------------------------------------
    if log_scale:

        ax.set_yscale("log")

        ax.yaxis.set_major_locator(
            LogLocator(base=10)
        )

        ax.yaxis.set_minor_formatter(
            NullFormatter()
        )

    ax.set_xticks(x)

    ax.set_xticklabels(
        design_tick_labels_r4()
    )

    ax.set_xlim(
        -0.38,
        len(DESIGN_ORDER_R4) - 0.62,
    )

    ax.set_title(
        title,
        loc="left",
        fontweight="semibold",
        pad=8,
    )

    ax.set_ylabel(
        ylabel
    )

    ax.grid(
        axis="y",
        which="major",
        alpha=0.20,
        linestyle="--",
    )

    ax.set_axisbelow(True)

    set_panel_frame_r4(ax)

# -----------------------------------------------------------------------------
# Common legend handles.
# -----------------------------------------------------------------------------
SEED_HANDLE_R4 = Line2D(
    [0],
    [0],
    marker="o",
    linestyle="None",
    markerfacecolor="0.65",
    markeredgecolor="black",
    alpha=0.30,
    markersize=5.8,
    label="Individual seed",
)

MEAN_HANDLE_R4 = Line2D(
    [0],
    [0],
    marker="D",
    linestyle="-",
    color="black",
    markerfacecolor="white",
    markeredgecolor="black",
    markersize=6.5,
    linewidth=1.2,
    label="Mean ± sample SD",
)

# =============================================================================
# FIGURE 1
# State reconstruction and physics consistency
# =============================================================================
with plt.rc_context(PUBLICATION_RC_R4):

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(12.6, 8.8),
    )

    panels = [
        (
            axes[0, 0],
            "rmse_sensitive",
            r"Sensitive-state reconstruction, $S(t)$",
            "Sensitive-state RMSE",
            False,
        ),
        (
            axes[0, 1],
            "rmse_resistant",
            r"Resistant-state reconstruction, $R(t)$",
            "Resistant-state RMSE",
            False,
        ),
        (
            axes[1, 0],
            "rmse_fraction",
            r"Phenotype-composition reconstruction, $q_R(t)$",
            "Resistant-fraction RMSE",
            True,
        ),
        (
            axes[1, 1],
            "physics_residual_rmse",
            "Dynamical consistency",
            "Physics-residual RMSE",
            True,
        ),
    ]

    for label, (
        ax,
        metric,
        title,
        ylabel,
        log_scale,
    ) in zip(
        ["A", "B", "C", "D"],
        panels,
    ):

        distribution_panel_r4(
            ax,
            metric,
            title,
            ylabel,
            log_scale=log_scale,
        )

        add_panel_label_r4(
            ax,
            label,
        )

    fig.suptitle(
        "Schedule diversity and hidden-state reconstruction",
        fontsize=15.7,
        fontweight="bold",
        y=0.985,
    )

    fig.legend(
        handles=[
            SEED_HANDLE_R4,
            MEAN_HANDLE_R4,
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.947),
        ncol=2,
        frameon=False,
    )

    fig.subplots_adjust(
        left=0.09,
        right=0.985,
        bottom=0.095,
        top=0.885,
        wspace=0.27,
        hspace=0.36,
    )

    save_figure_r4(
        fig,
        "round4_primary_state_and_physics_final",
    )

    plt.show()

# =============================================================================
# FIGURE 2
# FINAL MANUSCRIPT FIGURE:
# Shared-parameter recovery and practical identifiability
# =============================================================================
#
# Publication decisions:
#   - no large in-figure suptitle; the manuscript caption carries the message;
#   - x-axis shows only M1--M5; design details remain in the caption/table;
#   - individual seeds remain visible but visually secondary;
#   - mean ± sample SD remains the dominant statistical summary;
#   - dashed truth lines are identified by compact in-panel annotations rather
#     than redundant legend boxes;
#   - concise panel titles and aligned y-axis labels;
#   - no model training or numerical recomputation is performed here.
# =============================================================================

with plt.rc_context(PUBLICATION_RC_R4):

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(12.2, 8.15),
    )

    panels = [
        (
            axes[0, 0],
            "mutation_probability",
            "Mutation-probability recovery",
            r"Estimated mutation probability, $\hat{\mu}$",
            True,
            float(
                TRUE_PARAMETERS.mutation_probability
            ),
        ),
        (
            axes[0, 1],
            "resistance_fitness_cost",
            "Resistance-fitness-cost recovery",
            r"Estimated resistance fitness cost, $\hat{c}_R$",
            False,
            float(
                TRUE_PARAMETERS.resistance_fitness_cost
            ),
        ),
        (
            axes[1, 0],
            "mu_error_percent",
            "Mutation-probability error",
            r"Relative error in $\mu$ (%)",
            True,
            None,
        ),
        (
            axes[1, 1],
            "cost_error_percent",
            "Fitness-cost error",
            r"Relative error in $c_R$ (%)",
            False,
            None,
        ),
    ]

    for label, (
        ax,
        metric,
        title,
        ylabel,
        log_scale,
        true_value,
    ) in zip(
        ["A", "B", "C", "D"],
        panels,
    ):

        distribution_panel_r4(
            ax,
            metric,
            title,
            ylabel,
            log_scale=log_scale,
            true_value=true_value,
        )

        add_panel_label_r4(
            ax,
            label,
        )

        # Final manuscript refinement:
        # Keep the parameter-recovery figure visually compact by showing only
        # the formal nested-design identifiers on the x-axis. The schedule
        # count and O3/O4 definitions are reported in the experimental-design
        # figure/table and explained again in the manuscript caption.
        ax.set_xticks(
            np.arange(
                len(DESIGN_ORDER_R4)
            )
        )

        ax.set_xticklabels(
            DESIGN_ORDER_R4,
            fontsize=9.6,
        )

        # Slightly stronger panel-title hierarchy without an overall headline.
        ax.set_title(
            title,
            loc="left",
            fontsize=11.9,
            fontweight="semibold",
            pad=8,
        )

        ax.tick_params(
            axis="both",
            which="major",
            direction="out",
            width=0.85,
            length=3.8,
        )

    # -------------------------------------------------------------------------
    # Compact truth annotations.
    # -------------------------------------------------------------------------
    # The dashed horizontal lines are already drawn by distribution_panel_r4.
    # These text labels replace the two redundant "True value" legend boxes.
    # -------------------------------------------------------------------------

    axes[0, 0].text(
        0.975,
        0.955,
        r"Truth: $\mu^\star=10^{-4}$",
        transform=axes[0, 0].transAxes,
        ha="right",
        va="top",
        fontsize=8.9,
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="white",
            edgecolor="0.55",
            linewidth=0.55,
            alpha=0.92,
        ),
        zorder=8,
    )

    axes[0, 1].text(
        0.975,
        0.955,
        r"Truth: $c_R^\star=0.10$",
        transform=axes[0, 1].transAxes,
        ha="right",
        va="top",
        fontsize=8.9,
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="white",
            edgecolor="0.55",
            linewidth=0.55,
            alpha=0.92,
        ),
        zorder=8,
    )

    # -------------------------------------------------------------------------
    # Shared statistical legend.
    # -------------------------------------------------------------------------
    # Kept outside the data panels so that no observations or error bars are
    # obscured. The legend is intentionally compact for journal reduction.
    # -------------------------------------------------------------------------

    fig.legend(
        handles=[
            SEED_HANDLE_R4,
            MEAN_HANDLE_R4,
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.992),
        ncol=2,
        frameon=False,
        fontsize=9.0,
        handletextpad=0.55,
        columnspacing=1.6,
    )

    # Align labels for a cleaner 2 x 2 manuscript layout.
    fig.align_ylabels(
        axes[:, 0]
    )
    fig.align_ylabels(
        axes[:, 1]
    )

    # No figure-level title by design.
    fig.subplots_adjust(
        left=0.095,
        right=0.985,
        bottom=0.105,
        top=0.925,
        wspace=0.27,
        hspace=0.39,
    )

    save_figure_r4(
        fig,
        "round4_primary_parameter_recovery_final",
    )

    plt.show()

# =============================================================================
# FIGURE 3
# Paired information gain relative to M1
# =============================================================================
with plt.rc_context(PUBLICATION_RC_R4):

    effect_metrics = [
        "rmse_resistant",
        "rmse_fraction",
        "mu_error_percent",
        "cost_error_percent",
    ]

    effect_labels = [
        r"RMSE $R$",
        r"RMSE $q_R$",
        r"Error in $\mu$",
        r"Error in $c_R$",
    ]

    effect_matrix = np.full(
        (4, 4),
        np.nan,
    )

    for i, design_id in enumerate(
        DESIGN_ORDER_R4[1:]
    ):

        for j, metric in enumerate(
            effect_metrics
        ):

            pivot = ROUND4_PRIMARY_ALL_RUNS.pivot(
                index="seed_id",
                columns="design_id",
                values=metric,
            )

            paired = pivot[
                ["M1", design_id]
            ].dropna()

            effect_matrix[i, j] = float(
                (
                    100.0
                    * (
                        paired["M1"]
                        - paired[design_id]
                    )
                    / paired["M1"]
                ).mean()
            )

    fig, ax = plt.subplots(
        figsize=(9.3, 5.4),
    )

    limit = max(
        float(
            np.nanmax(
                np.abs(effect_matrix)
            )
        ),
        1.0,
    )

    image = ax.imshow(
        effect_matrix,
        aspect="auto",
        cmap="coolwarm",
        vmin=-limit,
        vmax=limit,
    )

    ax.set_xticks(
        np.arange(4)
    )

    ax.set_xticklabels(
        effect_labels
    )

    ax.set_yticks(
        np.arange(4)
    )

    ax.set_yticklabels([
        f"{d} vs M1"
        for d in DESIGN_ORDER_R4[1:]
    ])

    for i in range(4):

        for j in range(4):

            value = effect_matrix[
                i,
                j,
            ]

            ax.text(
                j,
                i,
                f"{value:.1f}%",
                ha="center",
                va="center",
                fontsize=9.5,
                fontweight="semibold",
                color=(
                    "white"
                    if abs(value) > 0.55 * limit
                    else "black"
                ),
            )

    cbar = fig.colorbar(
        image,
        ax=ax,
        pad=0.025,
    )

    cbar.set_label(
        "Mean paired improvement relative to M1 (%)"
    )

    ax.set_title(
        "Information gain from schedule diversity and phenotype observations",
        fontsize=14.2,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Evaluation quantity"
    )

    ax.set_ylabel(
        "Nested design comparison"
    )

    set_panel_frame_r4(ax)

    fig.tight_layout()

    save_figure_r4(
        fig,
        "round4_primary_information_gain_heatmap_final",
    )

    plt.show()

# =============================================================================
# FIGURE 4
# Accuracy-identifiability trade-off
# =============================================================================
with plt.rc_context(PUBLICATION_RC_R4):

    fig, ax = plt.subplots(
        figsize=(8.5, 5.9),
    )

    offsets = {
        "M1": (10, 9),
        "M2": (10, -14),
        "M3": (10, 9),
        "M4": (-10, 9),
        "M5": (-10, -15),
    }

    for d in DESIGN_ORDER_R4:

        row = PRIMARY_SUMMARY_R4[
            PRIMARY_SUMMARY_R4[
                "design_id"
            ] == d
        ].iloc[0]

        x = float(
            row["mean_rmse_resistant"]
        )

        y = float(
            row["mean_mu_error_percent"]
        )

        n = schedule_count_r4(d)

        ax.scatter(
            x,
            y,
            s=95 + 46 * n,
            marker=DESIGN_MARKERS_R4[d],
            facecolor=DESIGN_COLORS_R4[d],
            edgecolor="black",
            linewidth=0.85,
            zorder=4,
        )

        dx, dy = offsets[d]

        ax.annotate(
            d,
            (x, y),
            xytext=(dx, dy),
            textcoords="offset points",
            ha=(
                "right"
                if dx < 0
                else "left"
            ),
            va=(
                "bottom"
                if dy >= 0
                else "top"
            ),
            fontsize=9.7,
            fontweight="semibold",
        )

    ax.set_xscale(
        "log"
    )

    ax.set_yscale(
        "log"
    )

    xs = np.array([
        summary_value_r4(
            d,
            "rmse_resistant",
            "mean",
        )
        for d in DESIGN_ORDER_R4
    ])

    ys = np.array([
        summary_value_r4(
            d,
            "mu_error_percent",
            "mean",
        )
        for d in DESIGN_ORDER_R4
    ])

    ax.set_xlim(
        np.min(xs) / 1.35,
        np.max(xs) * 1.40,
    )

    ax.set_ylim(
        np.min(ys) / 1.45,
        np.max(ys) * 1.45,
    )

    ax.set_xlabel(
        "Mean resistant-state RMSE"
    )

    ax.set_ylabel(
        r"Mean relative error in $\mu$ (%)"
    )

    ax.set_title(
        "State accuracy versus mutation-parameter identifiability",
        fontsize=14.2,
        fontweight="bold",
        pad=12,
    )

    ax.grid(
        which="major",
        alpha=0.20,
        linestyle="--",
    )

    set_panel_frame_r4(
        ax
    )

    size_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=6 + 1.2 * n,
            label=(
                f"{n} schedule"
                if n == 1
                else f"{n} schedules"
            ),
        )
        for n in [1, 2, 3, 4]
    ]

    ax.legend(
        handles=size_handles,
        title="Schedule count",
        loc="upper right",
        frameon=True,
        fontsize=8.3,
        title_fontsize=8.6,
    )

    fig.tight_layout()

    save_figure_r4(
        fig,
        "round4_primary_accuracy_identifiability_tradeoff_final",
    )

    plt.show()

# =============================================================================
# FINAL CONFIRMATION
# =============================================================================
print(
    "\n" + "=" * 78
)

print(
    "ROUND 4 CELL 19 COMPLETED SUCCESSFULLY"
)

print(
    "=" * 78
)

print(
    "Generated figures:"
)

print(
    "  1. round4_primary_state_and_physics_final"
)

print(
    "  2. round4_primary_parameter_recovery_final"
)

print(
    "  3. round4_primary_information_gain_heatmap_final"
)

print(
    "  4. round4_primary_accuracy_identifiability_tradeoff_final"
)

print(
    "Figure directory:",
    FIGURE_DIR_R4.resolve(),
)

print(
    "No model was retrained."
)

In [ ]:
# =============================================================================
# ROUND 4 — CELL 20
# Publication-ready trajectory validation
#   Figure A: objectively selected representative M4 reconstruction
#   Figure B: five-seed M4-versus-M5 trajectory comparison (mean ± sample SD)
# =============================================================================
# This cell performs NO TRAINING. It reuses the completed metrics and dense
# prediction CSV files saved by Cell 17.
#
# M4 and M5 use the same four treatment schedules. M4 uses O3 observations,
# whereas M5 uses O4 phenotype-resolved observations. The second figure thus
# helps isolate the effect of richer phenotype information after schedule
# diversity has already been introduced.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import ScalarFormatter, MaxNLocator

# -----------------------------------------------------------------------------
# 1. Reload completed primary results if required
# -----------------------------------------------------------------------------
if (
    "ROUND4_PRIMARY_ALL_RUNS" not in globals()
    or not isinstance(ROUND4_PRIMARY_ALL_RUNS, pd.DataFrame)
    or ROUND4_PRIMARY_ALL_RUNS.empty
):
    primary_path_r4 = TABLE_DIR_R4 / "round4_primary_multischedule_all_seeds_metrics.csv"
    if not primary_path_r4.exists():
        raise FileNotFoundError("Round 4 primary results are unavailable. Run Cell 17 first.")
    ROUND4_PRIMARY_ALL_RUNS = pd.read_csv(primary_path_r4)
    print("Reloaded Round 4 primary results from:", primary_path_r4)

# -----------------------------------------------------------------------------
# 2. Fallback figure helpers
# -----------------------------------------------------------------------------
if "set_panel_frame_r4" not in globals():
    def set_panel_frame_r4(axis):
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1.0)

if "save_figure_r4" not in globals():
    def save_figure_r4(fig, stem):
        """Save a publication figure in raster and vector formats."""
        FIGURE_DIR_R4.mkdir(parents=True, exist_ok=True)

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.png",
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.pdf",
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.svg",
            bbox_inches="tight",
            facecolor="white",
        )

# -----------------------------------------------------------------------------
# 3. Consistent publication style
# -----------------------------------------------------------------------------
CELL20_RC_R4 = {
    "font.family": "sans-serif",
    "font.size": 10.2,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.0,
    "ytick.labelsize": 9.0,
    "legend.fontsize": 8.7,
    "axes.linewidth": 1.0,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
}

SCHEDULE_ORDER_R4_CELL20 = [
    "untreated",
    "early_continuous",
    "late_continuous",
    "pulsed",
]

SCHEDULE_DISPLAY_R4_CELL20 = {
    "untreated": "Untreated",
    "early_continuous": "Early continuous",
    "late_continuous": "Late continuous",
    "pulsed": "Pulsed",
}

ROW_PANEL_LABELS_R4 = ["A", "B", "C", "D"]

# -----------------------------------------------------------------------------
# 4. Axis formatting and treatment shading
# -----------------------------------------------------------------------------
def format_numeric_axis_r4(axis, *, resistant_fraction=False):
    """Compact numerical formatting with a limited number of y ticks."""

    formatter = ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits(
        (-3, 3) if resistant_fraction else (-4, 4)
    )

    axis.yaxis.set_major_formatter(formatter)

    # IMPORTANT FIX:
    # Matplotlib uses 'nbins' (plural), not 'nbin'.
    axis.yaxis.set_major_locator(
        MaxNLocator(nbins=5)
    )


def apply_common_axis_style_r4(axis, *, resistant_fraction=False):

    axis.grid(
        axis="both",
        alpha=0.16,
        linestyle="--",
        linewidth=0.65,
    )

    axis.set_axisbelow(True)

    set_panel_frame_r4(axis)

    format_numeric_axis_r4(
        axis,
        resistant_fraction=resistant_fraction,
    )


def shade_treatment_intervals_r4(axis, schedule_name):
    """Shade intervals for which prescribed antibiotic treatment is ON."""

    schedule = SCHEDULES_R4[schedule_name]

    for left, right, level in schedule.intervals(
        T_START,
        T_FINAL,
    ):
        if level > 0:
            axis.axvspan(
                left,
                right,
                facecolor="0.82",
                edgecolor="none",
                alpha=0.30,
                zorder=0,
            )

# -----------------------------------------------------------------------------
# 5. Objective representative M4 seed selection
# -----------------------------------------------------------------------------
m4_rows_r4 = (
    ROUND4_PRIMARY_ALL_RUNS[
        ROUND4_PRIMARY_ALL_RUNS["design_id"] == "M4"
    ]
    .copy()
    .reset_index(drop=True)
)

if m4_rows_r4.empty:
    raise RuntimeError("No completed M4 rows were found.")

selection_metrics_r4 = [
    "rmse_resistant",
    "rmse_fraction",
    "mu_error_percent",
    "cost_error_percent",
]

missing = [
    metric
    for metric in selection_metrics_r4
    if metric not in m4_rows_r4.columns
]

if missing:
    raise KeyError(
        f"Missing M4 columns needed for representative-seed selection: {missing}"
    )

representative_distance_r4 = np.zeros(
    len(m4_rows_r4),
    dtype=float,
)

for metric in selection_metrics_r4:

    values = (
        m4_rows_r4[metric]
        .astype(float)
        .to_numpy()
    )

    median = np.median(values)

    mad = np.median(
        np.abs(
            values - median
        )
    )

    scale = max(
        float(mad),
        1.0e-12,
    )

    representative_distance_r4 += (
        (values - median) / scale
    ) ** 2

representative_index_r4 = int(
    np.argmin(
        representative_distance_r4
    )
)

REPRESENTATIVE_M4_ROW_R4 = (
    m4_rows_r4.iloc[
        representative_index_r4
    ].copy()
)

representative_label_r4 = str(
    REPRESENTATIVE_M4_ROW_R4["label"]
)

representative_seed_r4 = str(
    REPRESENTATIVE_M4_ROW_R4["seed_id"]
)

representative_prediction_path_r4 = (
    PREDICTION_DIR_R4
    / f"{representative_label_r4}_all_schedule_predictions.csv"
)

if not representative_prediction_path_r4.exists():
    raise FileNotFoundError(
        "Representative M4 prediction file was not found:\n"
        f"{representative_prediction_path_r4}"
    )

representative_prediction_r4 = pd.read_csv(
    representative_prediction_path_r4
)

# Save an auditable record of why this seed was selected.
representative_audit_r4 = m4_rows_r4[
    [
        "label",
        "seed_id",
        "rmse_resistant",
        "rmse_fraction",
        "mu_error_percent",
        "cost_error_percent",
    ]
].copy()

representative_audit_r4[
    "median_distance_score"
] = representative_distance_r4

representative_audit_r4[
    "selected_representative"
] = (
    representative_audit_r4["label"]
    == representative_label_r4
)

representative_audit_path_r4 = (
    TABLE_DIR_R4
    / "round4_M4_representative_seed_selection.csv"
)

representative_audit_r4.to_csv(
    representative_audit_path_r4,
    index=False,
)

# =============================================================================
# FIGURE A — FINAL REPRESENTATIVE FOUR-SCHEDULE M4 RECONSTRUCTION
# =============================================================================
# Publication design:
#   - no large internal title; representative-seed information belongs in caption;
#   - one compact legend above the grid;
#   - column headings shown once, outside the axes;
#   - schedule names shown as clean row headers;
#   - "Resistant fraction" terminology used consistently throughout manuscript;
#   - shared column-level y labels reduce repetition;
#   - treatment shading retained;
#   - time label retained only on bottom row;
#   - all trajectories and numerical values are unchanged.
# =============================================================================

with plt.rc_context(CELL20_RC_R4):

    fig_a, axes_a = plt.subplots(
        4,
        3,
        figsize=(12.4, 9.35),
        sharex=True,
    )

    reference_color = None
    pinn_color = None

    for row_index, schedule_name in enumerate(
        SCHEDULE_ORDER_R4_CELL20
    ):

        frame = (
            representative_prediction_r4[
                representative_prediction_r4[
                    "schedule"
                ] == schedule_name
            ]
            .copy()
            .sort_values("time")
        )

        if frame.empty:
            raise RuntimeError(
                f"No representative trajectory data found for {schedule_name}."
            )

        # ---------------------------------------------------------------------
        # Sensitive population
        # ---------------------------------------------------------------------
        ref_s, = axes_a[
            row_index,
            0,
        ].plot(
            frame["time"],
            frame["sensitive"],
            linewidth=2.05,
            label="Reference",
            zorder=3,
        )

        pred_s, = axes_a[
            row_index,
            0,
        ].plot(
            frame["time"],
            frame["predicted_sensitive"],
            linestyle="--",
            linewidth=1.85,
            label="Hard PINN",
            zorder=4,
        )

        # ---------------------------------------------------------------------
        # Resistant population
        # ---------------------------------------------------------------------
        axes_a[
            row_index,
            1,
        ].plot(
            frame["time"],
            frame["resistant"],
            color=ref_s.get_color(),
            linewidth=2.05,
            zorder=3,
        )

        axes_a[
            row_index,
            1,
        ].plot(
            frame["time"],
            frame["predicted_resistant"],
            color=pred_s.get_color(),
            linestyle="--",
            linewidth=1.85,
            zorder=4,
        )

        # ---------------------------------------------------------------------
        # Resistant fraction
        # ---------------------------------------------------------------------
        axes_a[
            row_index,
            2,
        ].plot(
            frame["time"],
            frame["resistant_fraction"],
            color=ref_s.get_color(),
            linewidth=2.05,
            zorder=3,
        )

        axes_a[
            row_index,
            2,
        ].plot(
            frame["time"],
            frame["predicted_resistant_fraction"],
            color=pred_s.get_color(),
            linestyle="--",
            linewidth=1.85,
            zorder=4,
        )

        if row_index == 0:
            reference_color = ref_s.get_color()
            pinn_color = pred_s.get_color()

        # ---------------------------------------------------------------------
        # Common axis formatting and treatment shading
        # ---------------------------------------------------------------------
        for column_index in range(3):

            axis = axes_a[
                row_index,
                column_index,
            ]

            shade_treatment_intervals_r4(
                axis,
                schedule_name,
            )

            apply_common_axis_style_r4(
                axis,
                resistant_fraction=(
                    column_index == 2
                ),
            )

            # Per-panel y labels are intentionally removed. Clean shared
            # column-level labels are added below after final layout.
            axis.set_ylabel("")

    # Time is shown only on the bottom row.
    for axis in axes_a[-1, :]:
        axis.set_xlabel(
            "Time"
        )

    # -------------------------------------------------------------------------
    # Reserve clean space for legend, column headings, and row headers.
    # -------------------------------------------------------------------------
    fig_a.subplots_adjust(
        left=0.095,
        right=0.992,
        bottom=0.070,
        top=0.855,
        wspace=0.245,
        hspace=0.430,
    )

    # -------------------------------------------------------------------------
    # Column headings — shown once for the complete grid.
    # -------------------------------------------------------------------------
    column_titles = [
        r"Sensitive population, $S(t)$",
        r"Resistant population, $R(t)$",
        r"Resistant fraction, $q_R(t)$",
    ]

    for column_index, column_title in enumerate(
        column_titles
    ):

        position = axes_a[
            0,
            column_index,
        ].get_position()

        x_center = 0.5 * (
            position.x0 + position.x1
        )

        fig_a.text(
            x_center,
            0.895,
            column_title,
            ha="center",
            va="center",
            fontsize=11.7,
            fontweight="semibold",
        )

    # -------------------------------------------------------------------------
    # Row headers — deliberately separated from the column headings.
    # -------------------------------------------------------------------------
    for row_index, schedule_name in enumerate(
        SCHEDULE_ORDER_R4_CELL20
    ):

        position = axes_a[
            row_index,
            0,
        ].get_position()

        fig_a.text(
            position.x0,
            position.y1 + 0.010,
            (
                f"{ROW_PANEL_LABELS_R4[row_index]}  "
                f"{SCHEDULE_DISPLAY_R4_CELL20[schedule_name]}"
            ),
            ha="left",
            va="bottom",
            fontsize=10.9,
            fontweight="bold",
        )

    # -------------------------------------------------------------------------
    # Shared column-level y labels.
    # -------------------------------------------------------------------------
    shared_y_labels = [
        "Population",
        "Population",
        "Resistant fraction",
    ]

    for column_index, shared_label in enumerate(
        shared_y_labels
    ):

        column_position = axes_a[
            1,
            column_index,
        ].get_position()

        x_label = (
            column_position.x0 - 0.038
            if column_index == 0
            else column_position.x0 - 0.032
        )

        fig_a.text(
            x_label,
            0.465,
            shared_label,
            ha="center",
            va="center",
            rotation=90,
            fontsize=10.6,
        )

    # -------------------------------------------------------------------------
    # Compact shared legend.
    # -------------------------------------------------------------------------
    fig_a.legend(
        handles=[
            Line2D(
                [0],
                [0],
                color=reference_color,
                linewidth=2.05,
                label="Reference",
            ),
            Line2D(
                [0],
                [0],
                color=pinn_color,
                linestyle="--",
                linewidth=1.85,
                label="Hard PINN",
            ),
            Patch(
                facecolor="0.82",
                edgecolor="none",
                alpha=0.50,
                label="Treatment ON",
            ),
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.985),
        ncol=3,
        frameon=False,
        columnspacing=1.7,
        handlelength=2.7,
        fontsize=9.0,
    )

    # No figure-level title by design. The caption should state that this is
    # the objectively selected representative M4 run (seed3 in the final run).

    save_figure_r4(
        fig_a,
        "round4_representative_M4_trajectories_refined",
    )

    plt.show()

print(
    "\\nObjectively selected representative M4 run:"
)

print(
    "  Label:",
    representative_label_r4,
)

print(
    "  Seed:",
    representative_seed_r4,
)

print(
    "  Selection audit:",
    representative_audit_path_r4,
)

display(
    pd.DataFrame([
        REPRESENTATIVE_M4_ROW_R4
    ])
)

# -----------------------------------------------------------------------------
# 6. Load all saved prediction files for one design
# -----------------------------------------------------------------------------
def load_design_prediction_ensemble_r4(
    design_id
):

    design_rows = (
        ROUND4_PRIMARY_ALL_RUNS[
            ROUND4_PRIMARY_ALL_RUNS[
                "design_id"
            ] == design_id
        ]
        .copy()
        .sort_values("seed_id")
    )

    if design_rows.empty:
        raise RuntimeError(
            f"No completed rows found for design {design_id}."
        )

    frames = []

    for _, row in design_rows.iterrows():

        label = str(
            row["label"]
        )

        seed_id = str(
            row["seed_id"]
        )

        prediction_file = (
            PREDICTION_DIR_R4
            / f"{label}_all_schedule_predictions.csv"
        )

        if not prediction_file.exists():
            print(
                "WARNING: missing prediction file:",
                prediction_file,
            )
            continue

        frame = pd.read_csv(
            prediction_file
        )

        frame["design_id"] = design_id
        frame["seed_id"] = seed_id

        frames.append(
            frame
        )

    if not frames:
        raise RuntimeError(
            f"No usable prediction files were available for {design_id}."
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )

m4_prediction_ensemble_r4 = (
    load_design_prediction_ensemble_r4(
        "M4"
    )
)

m5_prediction_ensemble_r4 = (
    load_design_prediction_ensemble_r4(
        "M5"
    )
)

print(
    "\nPrediction ensemble sizes:"
)

print(
    "  M4 seeds:",
    m4_prediction_ensemble_r4[
        "seed_id"
    ].nunique(),
)

print(
    "  M5 seeds:",
    m5_prediction_ensemble_r4[
        "seed_id"
    ].nunique(),
)

# -----------------------------------------------------------------------------
# 7. Aggregate mean ± sample SD trajectories across seeds
# -----------------------------------------------------------------------------
def sample_sd_r4(values):

    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        float(
            np.std(
                values,
                ddof=1,
            )
        )
        if len(values) > 1
        else 0.0
    )


def aggregate_prediction_ensemble_r4(
    frame
):

    return (
        frame.groupby(
            [
                "schedule",
                "time",
            ],
            observed=True,
            sort=True,
        )
        .agg(
            resistant_reference=(
                "resistant",
                "mean",
            ),
            fraction_reference=(
                "resistant_fraction",
                "mean",
            ),
            resistant_mean=(
                "predicted_resistant",
                "mean",
            ),
            resistant_sd=(
                "predicted_resistant",
                sample_sd_r4,
            ),
            fraction_mean=(
                "predicted_resistant_fraction",
                "mean",
            ),
            fraction_sd=(
                "predicted_resistant_fraction",
                sample_sd_r4,
            ),
            n_seeds=(
                "seed_id",
                "nunique",
            ),
        )
        .reset_index()
    )

m4_trajectory_summary_r4 = (
    aggregate_prediction_ensemble_r4(
        m4_prediction_ensemble_r4
    )
)

m5_trajectory_summary_r4 = (
    aggregate_prediction_ensemble_r4(
        m5_prediction_ensemble_r4
    )
)

# =============================================================================
# FIGURE B — FINAL FIVE-SEED M4 VERSUS M5 TRAJECTORY COMPARISON
# =============================================================================
# This supporting figure isolates the effect of phenotype-rich O4 information
# after four-schedule diversity is already present. It is intended primarily
# for Supplementary Material.
# =============================================================================

with plt.rc_context(CELL20_RC_R4):

    fig_b, axes_b = plt.subplots(
        4,
        2,
        figsize=(10.9, 9.15),
        sharex=True,
    )

    legend_handles = None

    for row_index, schedule_name in enumerate(
        SCHEDULE_ORDER_R4_CELL20
    ):

        m4 = (
            m4_trajectory_summary_r4[
                m4_trajectory_summary_r4[
                    "schedule"
                ] == schedule_name
            ]
            .copy()
            .sort_values("time")
        )

        m5 = (
            m5_trajectory_summary_r4[
                m5_trajectory_summary_r4[
                    "schedule"
                ] == schedule_name
            ]
            .copy()
            .sort_values("time")
        )

        if m4.empty or m5.empty:
            raise RuntimeError(
                f"Missing M4 or M5 trajectory summary for {schedule_name}."
            )

        ax_r = axes_b[
            row_index,
            0,
        ]

        ax_q = axes_b[
            row_index,
            1,
        ]

        # ---------------------------------------------------------------------
        # Resistant population
        # ---------------------------------------------------------------------
        ref_line, = ax_r.plot(
            m4["time"],
            m4["resistant_reference"],
            linewidth=2.10,
            label="Reference",
            zorder=5,
        )

        m4_line, = ax_r.plot(
            m4["time"],
            m4["resistant_mean"],
            linestyle="--",
            linewidth=1.85,
            label="M4 mean",
            zorder=4,
        )

        m5_line, = ax_r.plot(
            m5["time"],
            m5["resistant_mean"],
            linestyle="-.",
            linewidth=1.85,
            label="M5 mean",
            zorder=4,
        )

        m4_mean = m4[
            "resistant_mean"
        ].to_numpy(float)

        m4_sd = m4[
            "resistant_sd"
        ].to_numpy(float)

        m5_mean = m5[
            "resistant_mean"
        ].to_numpy(float)

        m5_sd = m5[
            "resistant_sd"
        ].to_numpy(float)

        ax_r.fill_between(
            m4["time"].to_numpy(float),
            np.maximum(
                0.0,
                m4_mean - m4_sd,
            ),
            m4_mean + m4_sd,
            alpha=0.13,
            color=m4_line.get_color(),
            linewidth=0,
            zorder=2,
        )

        ax_r.fill_between(
            m5["time"].to_numpy(float),
            np.maximum(
                0.0,
                m5_mean - m5_sd,
            ),
            m5_mean + m5_sd,
            alpha=0.13,
            color=m5_line.get_color(),
            linewidth=0,
            zorder=2,
        )

        # ---------------------------------------------------------------------
        # Resistant fraction
        # ---------------------------------------------------------------------
        ax_q.plot(
            m4["time"],
            m4["fraction_reference"],
            color=ref_line.get_color(),
            linewidth=2.10,
            zorder=5,
        )

        m4_q_line, = ax_q.plot(
            m4["time"],
            m4["fraction_mean"],
            color=m4_line.get_color(),
            linestyle="--",
            linewidth=1.85,
            zorder=4,
        )

        m5_q_line, = ax_q.plot(
            m5["time"],
            m5["fraction_mean"],
            color=m5_line.get_color(),
            linestyle="-.",
            linewidth=1.85,
            zorder=4,
        )

        m4_q_mean = m4[
            "fraction_mean"
        ].to_numpy(float)

        m4_q_sd = m4[
            "fraction_sd"
        ].to_numpy(float)

        m5_q_mean = m5[
            "fraction_mean"
        ].to_numpy(float)

        m5_q_sd = m5[
            "fraction_sd"
        ].to_numpy(float)

        ax_q.fill_between(
            m4["time"].to_numpy(float),
            np.maximum(
                0.0,
                m4_q_mean - m4_q_sd,
            ),
            m4_q_mean + m4_q_sd,
            alpha=0.13,
            color=m4_q_line.get_color(),
            linewidth=0,
            zorder=2,
        )

        ax_q.fill_between(
            m5["time"].to_numpy(float),
            np.maximum(
                0.0,
                m5_q_mean - m5_q_sd,
            ),
            m5_q_mean + m5_q_sd,
            alpha=0.13,
            color=m5_q_line.get_color(),
            linewidth=0,
            zorder=2,
        )

        # Treatment shading + common style.
        for axis in (
            ax_r,
            ax_q,
        ):
            shade_treatment_intervals_r4(
                axis,
                schedule_name,
            )

        apply_common_axis_style_r4(
            ax_r,
            resistant_fraction=False,
        )

        apply_common_axis_style_r4(
            ax_q,
            resistant_fraction=True,
        )

        ax_r.set_ylabel("")
        ax_q.set_ylabel("")

        if row_index == 0:
            legend_handles = [
                ref_line,
                m4_line,
                m5_line,
                Patch(
                    facecolor="0.82",
                    edgecolor="none",
                    alpha=0.50,
                    label="Treatment ON",
                ),
            ]

    for axis in axes_b[-1, :]:
        axis.set_xlabel(
            "Time"
        )

    # Reserve explicit space for hierarchy outside the axes.
    fig_b.subplots_adjust(
        left=0.105,
        right=0.992,
        bottom=0.070,
        top=0.850,
        wspace=0.245,
        hspace=0.430,
    )

    # Column headings.
    comparison_titles = [
        r"Resistant population, $R(t)$",
        r"Resistant fraction, $q_R(t)$",
    ]

    for column_index, column_title in enumerate(
        comparison_titles
    ):

        position = axes_b[
            0,
            column_index,
        ].get_position()

        fig_b.text(
            0.5 * (
                position.x0 + position.x1
            ),
            0.895,
            column_title,
            ha="center",
            va="center",
            fontsize=11.7,
            fontweight="semibold",
        )

    # Row headers.
    for row_index, schedule_name in enumerate(
        SCHEDULE_ORDER_R4_CELL20
    ):

        position = axes_b[
            row_index,
            0,
        ].get_position()

        fig_b.text(
            position.x0,
            position.y1 + 0.010,
            (
                f"{ROW_PANEL_LABELS_R4[row_index]}  "
                f"{SCHEDULE_DISPLAY_R4_CELL20[schedule_name]}"
            ),
            ha="left",
            va="bottom",
            fontsize=10.9,
            fontweight="bold",
        )

    # Shared y labels.
    first_position = axes_b[
        1,
        0,
    ].get_position()

    second_position = axes_b[
        1,
        1,
    ].get_position()

    fig_b.text(
        first_position.x0 - 0.045,
        0.465,
        "Population",
        ha="center",
        va="center",
        rotation=90,
        fontsize=10.6,
    )

    fig_b.text(
        second_position.x0 - 0.038,
        0.465,
        "Resistant fraction",
        ha="center",
        va="center",
        rotation=90,
        fontsize=10.6,
    )

    # Compact shared legend.
    fig_b.legend(
        handles=legend_handles,
        labels=[
            "Reference",
            "M4 mean ± SD",
            "M5 mean ± SD",
            "Treatment ON",
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.985),
        ncol=4,
        frameon=False,
        columnspacing=1.45,
        handlelength=2.5,
        fontsize=8.9,
    )

    # No internal suptitle. The Supplementary caption should explain that
    # M4 and M5 share the four schedules and differ in O3 versus O4 information.

    save_figure_r4(
        fig_b,
        "round4_M4_vs_M5_five_seed_trajectory_comparison",
    )

    plt.show()

# -----------------------------------------------------------------------------
# 8. Save five-seed trajectory summaries
# -----------------------------------------------------------------------------
m4_trajectory_summary_path_r4 = (
    TABLE_DIR_R4
    / "round4_M4_five_seed_trajectory_summary.csv"
)

m5_trajectory_summary_path_r4 = (
    TABLE_DIR_R4
    / "round4_M5_five_seed_trajectory_summary.csv"
)

m4_trajectory_summary_r4.to_csv(
    m4_trajectory_summary_path_r4,
    index=False,
)

m5_trajectory_summary_r4.to_csv(
    m5_trajectory_summary_path_r4,
    index=False,
)

# -----------------------------------------------------------------------------
# 9. Final report
# -----------------------------------------------------------------------------
print(
    "\n" + "=" * 82
)

print(
    "ROUND 4 CELL 20 COMPLETED SUCCESSFULLY"
)

print(
    "=" * 82
)

print(
    "\nRepresentative M4 run:"
)

print(
    "  Label:",
    representative_label_r4,
)

print(
    "  Seed:",
    representative_seed_r4,
)

print(
    "\nGenerated Figure A:"
)

print(
    "  round4_representative_M4_trajectories_refined"
)

print(
    "\nGenerated Figure B:"
)

print(
    "  round4_M4_vs_M5_five_seed_trajectory_comparison"
)

print(
    "\nFive-seed trajectory summaries:"
)

print(
    " ",
    m4_trajectory_summary_path_r4
)

print(
    " ",
    m5_trajectory_summary_path_r4
)

print(
    "\nRepresentative-selection audit:"
)

print(
    " ",
    representative_audit_path_r4
)

print(
    "\nNo model was retrained."
)

print(
    "All figures were generated from prediction files already saved during Cell 17."
)

In [ ]:
# =============================================================================
# ROUND 4 — TEMPORARY SWITCH
# Run matched mechanistic ODE baselines only
# =============================================================================

RUN_STRUCTURAL_SMOKE_TEST = False
RUN_SINGLE_DESIGN_DEMO = False
RUN_PRIMARY_MULTI_SCHEDULE_STUDY = False

RUN_MATCHED_ODE_BASELINES = True

RUN_IDENTIFIABILITY_GEOMETRY = False
RUN_LEAVE_ONE_SCHEDULE_OUT = False
RUN_PROFILE_SURFACES = False
RUN_PARAMETER_TRANSFER_TEST = False
RUN_PLOTTING_ONLY = False

CURRENT_EXPERIMENT_R4 = "matched_ode_baselines"

EXPERIMENT_SWITCHES_R4 = {
    "structural_smoke_test": RUN_STRUCTURAL_SMOKE_TEST,
    "single_design_demo": RUN_SINGLE_DESIGN_DEMO,
    "primary_multi_schedule_study": RUN_PRIMARY_MULTI_SCHEDULE_STUDY,
    "matched_ode_baselines": RUN_MATCHED_ODE_BASELINES,
    "identifiability_geometry": RUN_IDENTIFIABILITY_GEOMETRY,
    "leave_one_schedule_out": RUN_LEAVE_ONE_SCHEDULE_OUT,
    "profile_surfaces": RUN_PROFILE_SURFACES,
    "parameter_transfer_test": RUN_PARAMETER_TRANSFER_TEST,
    "plotting_only": RUN_PLOTTING_ONLY,
}

active_experiments_r4 = [
    name
    for name, enabled in EXPERIMENT_SWITCHES_R4.items()
    if enabled
]

print("Current experiment:", CURRENT_EXPERIMENT_R4)
print("Active experiments:", active_experiments_r4)

assert RUN_MATCHED_ODE_BASELINES is True
assert RUN_PRIMARY_MULTI_SCHEDULE_STUDY is False
assert active_experiments_r4 == ["matched_ode_baselines"]

print("\nMatched ODE-baseline configuration is active.")
print("Run the ODE-baseline cell now.")

In [ ]:
# =============================================================================
# ROUND 4 — CELL 21
# Matched mechanistic ODE baselines for M1--M5
# =============================================================================
# The synthetic observations are generated from the same ODE family. Therefore,
# this baseline is expected to be extremely accurate in the noise-free matched-
# model setting. It is reported separately from the neural design comparison.

ODE_BASELINE_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_matched_ode_design_metrics.csv"
)

ODE_BASELINE_ROWS_R4: List[Dict[str, Any]] = []

if RUN_MATCHED_ODE_BASELINES:
    for design_index, design_id in enumerate(
        DESIGN_ORDER_R4
    ):
        bundle = build_design_bundle_r4(
            design_id,
            seed=SEED,
            noise_level=0.0,
        )

        fit = fit_ode_baseline(
            bundle,
            label=f"round4_ode_{design_id}",
            seed=SEED + 800_000 + design_index,
            starts=(5 if QUICK_MODE else 16),
        )

        row = evaluate_ode_design_r4(
            design_id,
            bundle,
            fit,
        )

        ODE_BASELINE_ROWS_R4.append(
            row
        )

    ODE_BASELINE_RESULTS_R4 = pd.DataFrame(
        ODE_BASELINE_ROWS_R4
    )

    ODE_BASELINE_RESULTS_R4.to_csv(
        ODE_BASELINE_PATH_R4,
        index=False,
    )

    display(ODE_BASELINE_RESULTS_R4)
else:
    if ODE_BASELINE_PATH_R4.exists():
        ODE_BASELINE_RESULTS_R4 = pd.read_csv(
            ODE_BASELINE_PATH_R4
        )
        print(
            "Loaded the existing matched ODE benchmark table."
        )
    else:
        ODE_BASELINE_RESULTS_R4 = pd.DataFrame()
        print(
            "Set RUN_MATCHED_ODE_BASELINES=True to fit the separate "
            "matched mechanistic benchmark."
        )

In [ ]:
# =============================================================================
# ROUND 4 — TEMPORARY SWITCH FOR CELL 22
# Identifiability-geometry / sensitivity analysis only
# =============================================================================

RUN_STRUCTURAL_SMOKE_TEST = False
RUN_SINGLE_DESIGN_DEMO = False
RUN_PRIMARY_MULTI_SCHEDULE_STUDY = False
RUN_MATCHED_ODE_BASELINES = False

RUN_IDENTIFIABILITY_GEOMETRY = True

RUN_LEAVE_ONE_SCHEDULE_OUT = False
RUN_PROFILE_SURFACES = False
RUN_PARAMETER_TRANSFER_TEST = False
RUN_PLOTTING_ONLY = False

CURRENT_EXPERIMENT_R4 = "identifiability_geometry"

EXPERIMENT_SWITCHES_R4 = {
    "structural_smoke_test": RUN_STRUCTURAL_SMOKE_TEST,
    "single_design_demo": RUN_SINGLE_DESIGN_DEMO,
    "primary_multi_schedule_study": RUN_PRIMARY_MULTI_SCHEDULE_STUDY,
    "matched_ode_baselines": RUN_MATCHED_ODE_BASELINES,
    "identifiability_geometry": RUN_IDENTIFIABILITY_GEOMETRY,
    "leave_one_schedule_out": RUN_LEAVE_ONE_SCHEDULE_OUT,
    "profile_surfaces": RUN_PROFILE_SURFACES,
    "parameter_transfer_test": RUN_PARAMETER_TRANSFER_TEST,
    "plotting_only": RUN_PLOTTING_ONLY,
}

active_experiments_r4 = [
    name
    for name, enabled in EXPERIMENT_SWITCHES_R4.items()
    if enabled
]

print("Current experiment:", CURRENT_EXPERIMENT_R4)
print("Active experiments:", active_experiments_r4)

assert RUN_IDENTIFIABILITY_GEOMETRY is True
assert RUN_PRIMARY_MULTI_SCHEDULE_STUDY is False
assert active_experiments_r4 == ["identifiability_geometry"]

print("\nIdentifiability-geometry configuration is active.")
print("Run Cell 22 now.")

In [ ]:
# ROUND 4 — CELL 22
# FINAL PUBLICATION FIGURES:
# Sensitivity SVD, local conditioning, parameter correlation,
# and weakest identifiable parameter direction
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# This cell performs NO PINN training.
#
# It uses SENSITIVITY_SUMMARY_R4 generated by the preceding Round 4
# identifiability-geometry computation and produces two final figures:
#
# FIGURE 22A
# ----------
# A. Sensitivity spectrum:
#       largest singular value, sigma_max
#       smallest singular value, sigma_min
#
# B. Sensitivity conditioning:
#       kappa = sigma_max / sigma_min
#
# C. Local parameter correlation:
#       |rho(log10(mu), c_R)|
#
# FIGURE 22B
# ----------
# Absolute components of the right singular vector associated with sigma_min.
# This vector describes the locally weakest identifiable parameter direction.
#
# IMPORTANT SCIENTIFIC POINT
# --------------------------
# A singular vector has arbitrary global sign:
#
#       v  and  -v
#
# describe exactly the same direction.
#
# Therefore the heatmap reports ABSOLUTE component magnitudes:
#
#       |v_log10(mu)|   and   |v_cR|
#
# rather than signed components.
#
# PUBLICATION REFINEMENTS
# -----------------------
# 1. Only M1-M5 appear on the x-axis.
# 2. Font sizes are increased for manuscript-width readability.
# 3. "Parameter confounding" is replaced with "Sensitivity conditioning".
# 4. sigma_max and sigma_min remain on a logarithmic axis.
# 5. kappa is shown on a linear axis because its observed range is narrow.
# 6. Correlation remains on the physically interpretable 0-1 scale.
# 7. Weak-direction magnitudes use a sequential color scale from 0 to 1.
# 8. Exact plotted values are exported to CSV for reproducibility.
#
# NORMALIZATION NOTE
# ------------------
# The singular values shown here use the same normalized sensitivity matrix
# constructed by the preceding identifiability-analysis computation.
# The exact normalization should be defined once in the Methods/caption.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import MaxNLocator


# =============================================================================
# 2. EXECUTION GUARD
# =============================================================================

if RUN_IDENTIFIABILITY_GEOMETRY or RUN_PLOTTING_ONLY:

    # =========================================================================
    # 3. SAFETY CHECK
    # =========================================================================

    if (
        "SENSITIVITY_SUMMARY_R4" not in globals()
        or not isinstance(SENSITIVITY_SUMMARY_R4, pd.DataFrame)
        or SENSITIVITY_SUMMARY_R4.empty
    ):

        print(
            "Sensitivity summary is unavailable. "
            "Run the Round 4 identifiability-geometry computation first."
        )

    else:

        # =====================================================================
        # 4. FALLBACK HELPERS
        # =====================================================================

        if "DESIGN_ORDER_R4" not in globals():
            DESIGN_ORDER_R4 = [
                "M1",
                "M2",
                "M3",
                "M4",
                "M5",
            ]


        if "set_panel_frame_r4" not in globals():

            def set_panel_frame_r4(axis):
                """
                Apply a consistent black frame around the plotting region.
                """

                for spine in axis.spines.values():
                    spine.set_visible(True)
                    spine.set_color("black")
                    spine.set_linewidth(1.0)


        if "save_figure_r4" not in globals():

            def save_figure_r4(fig, stem):
                """
                Save publication figures in high-resolution PNG and vector PDF.
                """

                FIGURE_DIR_R4.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                fig.savefig(
                    FIGURE_DIR_R4 / f"{stem}.png",
                    dpi=600,
                    bbox_inches="tight",
                    facecolor="white",
                )

                fig.savefig(
                    FIGURE_DIR_R4 / f"{stem}.pdf",
                    bbox_inches="tight",
                    facecolor="white",
                )

                fig.savefig(
                    FIGURE_DIR_R4 / f"{stem}.svg",
                    bbox_inches="tight",
                    facecolor="white",
                )


        # =====================================================================
        # 5. COPY AND VALIDATE THE SENSITIVITY SUMMARY
        # =====================================================================

        summary = SENSITIVITY_SUMMARY_R4.copy()


        required_columns_r4 = [
            "design_id",
            "largest_singular_value",
            "smallest_singular_value",
            "condition_number",
            "parameter_correlation",
            "weak_direction_log10_mu",
            "weak_direction_cost",
        ]


        missing_columns_r4 = [
            column
            for column in required_columns_r4
            if column not in summary.columns
        ]


        if missing_columns_r4:

            raise KeyError(
                "SENSITIVITY_SUMMARY_R4 is missing required columns: "
                f"{missing_columns_r4}"
            )


        # ---------------------------------------------------------------------
        # Force design labels to strings before categorical ordering.
        # ---------------------------------------------------------------------

        summary["design_id"] = (
            summary["design_id"]
            .astype(str)
        )


        unexpected_designs_r4 = sorted(
            set(summary["design_id"])
            - set(DESIGN_ORDER_R4)
        )


        if unexpected_designs_r4:

            raise ValueError(
                "Unexpected design IDs were found: "
                f"{unexpected_designs_r4}"
            )


        # ---------------------------------------------------------------------
        # Enforce manuscript design order M1 -> M5.
        # ---------------------------------------------------------------------

        summary["design_id"] = pd.Categorical(
            summary["design_id"],
            categories=DESIGN_ORDER_R4,
            ordered=True,
        )


        summary = (
            summary
            .sort_values("design_id")
            .reset_index(drop=True)
        )


        if len(summary) != len(DESIGN_ORDER_R4):

            raise ValueError(
                "Expected exactly one sensitivity-summary row for each "
                f"design M1-M5, but found {len(summary)} rows."
            )


        summary["design_id"] = (
            summary["design_id"]
            .astype(str)
        )


        # ---------------------------------------------------------------------
        # Convert every plotted numerical column explicitly.
        # ---------------------------------------------------------------------

        numeric_columns_r4 = [
            "largest_singular_value",
            "smallest_singular_value",
            "condition_number",
            "parameter_correlation",
            "weak_direction_log10_mu",
            "weak_direction_cost",
        ]


        for column in numeric_columns_r4:

            summary[column] = pd.to_numeric(
                summary[column],
                errors="raise",
            )


        # ---------------------------------------------------------------------
        # Mathematical integrity checks.
        # ---------------------------------------------------------------------

        if (
            summary["largest_singular_value"] <= 0
        ).any():

            raise ValueError(
                "largest_singular_value must be strictly positive."
            )


        if (
            summary["smallest_singular_value"] <= 0
        ).any():

            raise ValueError(
                "smallest_singular_value must be strictly positive."
            )


        if (
            summary["condition_number"] <= 0
        ).any():

            raise ValueError(
                "condition_number must be strictly positive."
            )


        # =====================================================================
        # 6. PREPARE EXACT PLOTTING VALUES
        # =====================================================================

        design_labels_r4 = (
            summary["design_id"]
            .tolist()
        )


        x_r4 = np.arange(
            len(summary),
            dtype=float,
        )


        largest_sv_r4 = (
            summary[
                "largest_singular_value"
            ]
            .to_numpy(float)
        )


        smallest_sv_r4 = (
            summary[
                "smallest_singular_value"
            ]
            .to_numpy(float)
        )


        condition_number_r4 = (
            summary[
                "condition_number"
            ]
            .to_numpy(float)
        )


        abs_correlation_r4 = np.abs(
            summary[
                "parameter_correlation"
            ]
            .to_numpy(float)
        )


        abs_weak_mu_r4 = np.abs(
            summary[
                "weak_direction_log10_mu"
            ]
            .to_numpy(float)
        )


        abs_weak_cost_r4 = np.abs(
            summary[
                "weak_direction_cost"
            ]
            .to_numpy(float)
        )


        # =====================================================================
        # 7. EXPORT EXACT VALUES USED IN THE FIGURES
        # =====================================================================

        plot_values_r4 = pd.DataFrame(
            {
                "design_id":
                    design_labels_r4,

                "largest_singular_value":
                    largest_sv_r4,

                "smallest_singular_value":
                    smallest_sv_r4,

                "condition_number":
                    condition_number_r4,

                "absolute_parameter_correlation":
                    abs_correlation_r4,

                "absolute_weak_direction_log10_mu":
                    abs_weak_mu_r4,

                "absolute_weak_direction_cost":
                    abs_weak_cost_r4,
            }
        )


        sensitivity_plot_values_path_r4 = (
            TABLE_DIR_R4
            / "round4_identifiability_geometry_plot_values.csv"
        )


        plot_values_r4.to_csv(
            sensitivity_plot_values_path_r4,
            index=False,
        )


        # =====================================================================
        # 8. FINAL PUBLICATION STYLE
        # =====================================================================

        CELL22_PUBLICATION_RC_R4 = {

            "font.family":
                "sans-serif",

            "font.size":
                11.0,

            "axes.titlesize":
                12.3,

            "axes.labelsize":
                11.3,

            "xtick.labelsize":
                10.0,

            "ytick.labelsize":
                10.0,

            "legend.fontsize":
                9.4,

            "axes.linewidth":
                1.0,

            "axes.edgecolor":
                "black",

            "axes.facecolor":
                "white",

            "figure.facecolor":
                "white",

            "savefig.facecolor":
                "white",
        }


        def style_identifiability_axis_r4(axis):
            """
            Apply a consistent, restrained journal-style appearance.
            """

            axis.grid(
                which="major",
                alpha=0.15,
                linestyle="--",
                linewidth=0.65,
            )

            axis.set_axisbelow(
                True
            )

            set_panel_frame_r4(
                axis
            )


        # =====================================================================
        # 9. FIGURE A
        # IDENTIFIABILITY GEOMETRY OF NESTED SCHEDULE DESIGNS
        # =====================================================================

        with plt.rc_context(
            CELL22_PUBLICATION_RC_R4
        ):

            fig_geometry_r4, axes_r4 = plt.subplots(
                1,
                3,
                figsize=(13.8, 4.9),
            )


            # =================================================================
            # PANEL A — SENSITIVITY SPECTRUM
            # =================================================================

            axes_r4[0].plot(
                x_r4,
                largest_sv_r4,
                marker="o",
                linewidth=2.15,
                markersize=6.2,
                label=r"$\sigma_{\max}$",
            )


            axes_r4[0].plot(
                x_r4,
                smallest_sv_r4,
                marker="s",
                linewidth=2.15,
                markersize=6.2,
                label=r"$\sigma_{\min}$",
            )


            axes_r4[0].set_yscale(
                "log"
            )


            axes_r4[0].set_ylabel(
                "Normalized sensitivity singular value"
            )


            axes_r4[0].set_title(
                "Sensitivity spectrum",
                loc="left",
                fontweight="semibold",
                pad=8,
            )


            axes_r4[0].legend(
                loc="upper left",
                frameon=False,
                handlelength=2.2,
                handletextpad=0.55,
            )


            # =================================================================
            # PANEL B — SENSITIVITY CONDITIONING
            # =================================================================
            # A linear axis is deliberately used because kappa occupies a
            # relatively narrow numerical range. This is easier to read than
            # displaying values around 5-9 on a logarithmic axis.
            # =================================================================

            axes_r4[1].plot(
                x_r4,
                condition_number_r4,
                marker="D",
                linewidth=2.15,
                markersize=6.0,
            )


            axes_r4[1].set_ylabel(
                r"Condition number, $\kappa$"
            )


            axes_r4[1].set_title(
                "Sensitivity conditioning",
                loc="left",
                fontweight="semibold",
                pad=8,
            )


            axes_r4[1].yaxis.set_major_locator(
                MaxNLocator(
                    nbins=5
                )
            )


            # Add modest vertical padding without exaggerating the range.
            condition_min_r4 = float(
                np.min(
                    condition_number_r4
                )
            )


            condition_max_r4 = float(
                np.max(
                    condition_number_r4
                )
            )


            condition_padding_r4 = max(
                0.20,
                0.08
                * (
                    condition_max_r4
                    - condition_min_r4
                ),
            )


            axes_r4[1].set_ylim(
                max(
                    0.0,
                    condition_min_r4
                    - condition_padding_r4,
                ),
                condition_max_r4
                + condition_padding_r4,
            )


            # =================================================================
            # PANEL C — LOCAL PARAMETER CORRELATION
            # =================================================================

            axes_r4[2].plot(
                x_r4,
                abs_correlation_r4,
                marker="^",
                linewidth=2.15,
                markersize=6.5,
            )


            # Keep the complete correlation range visible.
            # This avoids visually exaggerating the relatively small decline.
            axes_r4[2].set_ylim(
                0.0,
                1.02,
            )


            axes_r4[2].set_ylabel(
                r"Local correlation, $|\rho(\log_{10}\mu,\,c_R)|$"
            )


            axes_r4[2].set_title(
                "Local parameter coupling",
                loc="left",
                fontweight="semibold",
                pad=8,
            )


            axes_r4[2].yaxis.set_major_locator(
                MaxNLocator(
                    nbins=6
                )
            )


            # =================================================================
            # COMMON PANEL FORMATTING
            # =================================================================

            for panel_label_r4, axis_r4 in zip(
                [
                    "A",
                    "B",
                    "C",
                ],
                axes_r4,
            ):

                axis_r4.set_xticks(
                    x_r4
                )


                axis_r4.set_xticklabels(
                    design_labels_r4
                )


                # M1--M5 already define the x-axis categories; a repeated
                # "Design" label is unnecessary at manuscript scale.
                axis_r4.set_xlabel("")


                style_identifiability_axis_r4(
                    axis_r4
                )


                axis_r4.text(
                    -0.12,
                    1.06,
                    panel_label_r4,
                    transform=axis_r4.transAxes,
                    fontsize=12.8,
                    fontweight="bold",
                    ha="left",
                    va="bottom",
                )


            # =================================================================
            # FINAL MANUSCRIPT LAYOUT
            # =================================================================
            # No large figure-level title is used. The manuscript caption carries
            # the overall interpretation, leaving more space for the three panels.

            fig_geometry_r4.subplots_adjust(
                left=0.075,
                right=0.985,
                bottom=0.14,
                top=0.91,
                wspace=0.34,
            )


            save_figure_r4(
                fig_geometry_r4,
                "round4_identifiability_geometry_final",
            )


            plt.show()


        # =====================================================================
        # 10. FIGURE B
        # COMPOSITION OF THE WEAKEST IDENTIFIABLE DIRECTION
        # =====================================================================

        weak_vectors_abs_r4 = pd.DataFrame(
            {
                r"$|v_{\log_{10}\mu}|$":
                    abs_weak_mu_r4,

                r"$|v_{c_R}|$":
                    abs_weak_cost_r4,
            },
            index=design_labels_r4,
        )


        with plt.rc_context(
            CELL22_PUBLICATION_RC_R4
        ):

            fig_weak_r4, ax_weak_r4 = plt.subplots(
                figsize=(8.6, 5.0),
            )


            # -----------------------------------------------------------------
            # Sequential color scale:
            #
            # 0 = weak contribution
            # 1 = dominant contribution
            # -----------------------------------------------------------------

            image_r4 = ax_weak_r4.imshow(
                weak_vectors_abs_r4.to_numpy(
                    float
                ),
                aspect="auto",
                cmap="viridis",
                vmin=0.0,
                vmax=1.0,
            )


            # =================================================================
            # AXIS LABELS
            # =================================================================

            ax_weak_r4.set_xticks(
                np.arange(
                    weak_vectors_abs_r4.shape[1]
                )
            )


            ax_weak_r4.set_xticklabels(
                weak_vectors_abs_r4.columns.tolist(),
                fontsize=11.0,
            )


            ax_weak_r4.set_yticks(
                np.arange(
                    weak_vectors_abs_r4.shape[0]
                )
            )


            ax_weak_r4.set_yticklabels(
                weak_vectors_abs_r4.index.tolist(),
                fontsize=10.5,
            )


            # The two parameter-component labels are self-explanatory.
            ax_weak_r4.set_xlabel("")


            ax_weak_r4.set_ylabel(
                "Design"
            )


            # =================================================================
            # NUMERICAL CELL ANNOTATIONS
            # =================================================================

            for row_index_r4 in range(
                weak_vectors_abs_r4.shape[0]
            ):

                for column_index_r4 in range(
                    weak_vectors_abs_r4.shape[1]
                ):

                    value_r4 = float(
                        weak_vectors_abs_r4.iloc[
                            row_index_r4,
                            column_index_r4,
                        ]
                    )


                    ax_weak_r4.text(
                        column_index_r4,
                        row_index_r4,
                        f"{value_r4:.3f}",
                        ha="center",
                        va="center",
                        fontsize=11.3,
                        fontweight="semibold",
                        color=(
                            "white"
                            if value_r4 >= 0.60
                            else "black"
                        ),
                    )


            # =================================================================
            # COLORBAR
            # =================================================================

            colorbar_r4 = fig_weak_r4.colorbar(
                image_r4,
                ax=ax_weak_r4,
                pad=0.025,
                fraction=0.055,
            )


            colorbar_r4.set_label(
                r"Absolute weak-direction component, $|v|$",
                fontsize=11.0,
            )


            colorbar_r4.ax.tick_params(
                labelsize=9.5
            )


            # =================================================================
            # TITLE AND FRAME
            # =================================================================

            ax_weak_r4.set_title(
                "Weakest identifiable parameter direction",
                fontsize=13.6,
                fontweight="semibold",
                pad=10,
            )


            set_panel_frame_r4(
                ax_weak_r4
            )


            fig_weak_r4.tight_layout()


            save_figure_r4(
                fig_weak_r4,
                "round4_weak_parameter_direction_final",
            )


            plt.show()


        # =====================================================================
        # 11. FINAL CONSOLE REPORT
        # =====================================================================

        print(
            "\n"
            + "=" * 82
        )


        print(
            "ROUND 4 CELL 22 FINAL FIGURES GENERATED SUCCESSFULLY"
        )


        print(
            "=" * 82
        )


        print(
            "\nSaved Figure A:"
        )


        print(
            " ",
            FIGURE_DIR_R4
            / "round4_identifiability_geometry_final",
        )


        print(
            "\nSaved Figure B:"
        )


        print(
            " ",
            FIGURE_DIR_R4
            / "round4_weak_parameter_direction_final",
        )


        print(
            "\nExact values used for plotting:"
        )


        print(
            " ",
            sensitivity_plot_values_path_r4,
        )


        print(
            "\nScientific interpretation checks:"
        )


        print(
            "  1. Larger sigma_min means stronger sensitivity in the "
            "weakest local parameter direction."
        )


        print(
            "  2. Smaller kappa means improved local sensitivity conditioning."
        )


        print(
            "  3. Parameter correlation is displayed as an absolute magnitude."
        )


        print(
            "  4. Weak-direction entries are absolute magnitudes because "
            "the global sign of an SVD singular vector is arbitrary."
        )


        print(
            "  5. Singular-value normalization follows the normalized "
            "sensitivity-matrix construction used in the preceding "
            "identifiability-analysis computation."
        )


        # =====================================================================
        # 12. OPTIONAL NUMERICAL DISPLAY FOR MANUSCRIPT CHECKING
        # =====================================================================

        display(
            plot_values_r4
        )


else:

    print(
        "Set RUN_IDENTIFIABILITY_GEOMETRY=True to generate the "
        "final sensitivity-geometry figures."
    )

In [ ]:
# =============================================================================
# ROUND 4 — TEMPORARY SWITCH FOR CELL 23
# Leave-one-schedule-out contribution experiment only
# =============================================================================
# PURPOSE
# -------
# Activate only the leave-one-schedule-out experiment.
#
# The already completed:
#   - 25-model primary experiment,
#   - Cell 18 summaries,
#   - Cell 19 figures,
#   - Cell 20 trajectory analysis,
#   - Cell 22 identifiability geometry
#
# are NOT rerun.
# =============================================================================

RUN_STRUCTURAL_SMOKE_TEST = False
RUN_SINGLE_DESIGN_DEMO = False
RUN_PRIMARY_MULTI_SCHEDULE_STUDY = False
RUN_MATCHED_ODE_BASELINES = False
RUN_IDENTIFIABILITY_GEOMETRY = False

RUN_LEAVE_ONE_SCHEDULE_OUT = True

RUN_PROFILE_SURFACES = False
RUN_PARAMETER_TRANSFER_TEST = False
RUN_PLOTTING_ONLY = False


CURRENT_EXPERIMENT_R4 = "leave_one_schedule_out"


EXPERIMENT_SWITCHES_R4 = {
    "structural_smoke_test":
        RUN_STRUCTURAL_SMOKE_TEST,

    "single_design_demo":
        RUN_SINGLE_DESIGN_DEMO,

    "primary_multi_schedule_study":
        RUN_PRIMARY_MULTI_SCHEDULE_STUDY,

    "matched_ode_baselines":
        RUN_MATCHED_ODE_BASELINES,

    "identifiability_geometry":
        RUN_IDENTIFIABILITY_GEOMETRY,

    "leave_one_schedule_out":
        RUN_LEAVE_ONE_SCHEDULE_OUT,

    "profile_surfaces":
        RUN_PROFILE_SURFACES,

    "parameter_transfer_test":
        RUN_PARAMETER_TRANSFER_TEST,

    "plotting_only":
        RUN_PLOTTING_ONLY,
}


active_experiments_r4 = [
    name
    for name, enabled in EXPERIMENT_SWITCHES_R4.items()
    if enabled
]


print("=" * 72)
print("ROUND 4 — CELL 23 CONFIGURATION")
print("=" * 72)

print(
    "Current experiment:",
    CURRENT_EXPERIMENT_R4,
)

print(
    "Active experiments:",
    active_experiments_r4,
)


# -----------------------------------------------------------------------------
# Safety assertions
# -----------------------------------------------------------------------------

assert RUN_LEAVE_ONE_SCHEDULE_OUT is True

assert RUN_PRIMARY_MULTI_SCHEDULE_STUDY is False

assert RUN_IDENTIFIABILITY_GEOMETRY is False

assert RUN_MATCHED_ODE_BASELINES is False

assert active_experiments_r4 == [
    "leave_one_schedule_out"
]


print()
print(
    "Leave-one-schedule-out configuration is active."
)

print(
    "The completed 25-model primary experiment will NOT be rerun."
)

print(
    "You may now run Cell 23."
)

In [ ]:
# ROUND 4 — CELL 23
# Leave-one-schedule-out shared-parameter inference
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# M4 is refitted after omitting each schedule in turn. A large deterioration when
# one schedule is removed indicates that the omitted environment supplies unique
# information rather than merely more observations.
#
# This is an optional expensive suite. It is resume-safe and should normally use
# three or five paired seeds after the primary experiment is complete.

LOO_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_metrics.csv"
)

PAIRED_CHANGES_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_changes.csv"
)

PAIRED_SUMMARY_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_summary.csv"
)

LOO_CONFIGURATIONS_R4 = {
    "FULL": (
        "untreated",
        "early_continuous",
        "late_continuous",
        "pulsed",
    ),
    "NO_UNTREATED": (
        "early_continuous",
        "late_continuous",
        "pulsed",
    ),
    "NO_EARLY": (
        "untreated",
        "late_continuous",
        "pulsed",
    ),
    "NO_LATE": (
        "untreated",
        "early_continuous",
        "pulsed",
    ),
    "NO_PULSED": (
        "untreated",
        "early_continuous",
        "late_continuous",
    ),
}

OMITTED_SCHEDULE_LABELS_R4 = {
    "NO_UNTREATED": "Untreated",
    "NO_EARLY": "Early continuous",
    "NO_LATE": "Late continuous",
    "NO_PULSED": "Pulsed",
}

OMITTED_ORDER_R4 = [
    "Untreated",
    "Early continuous",
    "Late continuous",
    "Pulsed",
]

LOO_ROWS_R4: List[Dict[str, Any]] = []

if RUN_LEAVE_ONE_SCHEDULE_OUT:
    if LOO_PATH_R4.exists():
        LOO_ROWS_R4 = pd.read_csv(
            LOO_PATH_R4
        ).to_dict("records")

    completed = {
        (
            str(row["configuration"]),
            str(row["seed_id"]),
        )
        for row in LOO_ROWS_R4
    }

    loo_seeds = (
        PRIMARY_SEED_IDS_R4
        if QUICK_MODE
        else [1, 2, 3]
    )

    for seed_number in loo_seeds:
        seed_id = f"seed{seed_number}"
        dataset_seed = (
            SEED
            + 10_000 * seed_number
        )
        training_seed = (
            SEED
            + 500_000
            + 10_000 * seed_number
        )

        full_bundle = build_design_bundle_r4(
            "M4",
            seed=dataset_seed,
            noise_level=0.0,
        )

        for configuration, schedule_names in (
            LOO_CONFIGURATIONS_R4.items()
        ):
            if (
                configuration,
                seed_id,
            ) in completed:
                continue

            bundle = {
                name: full_bundle[name]
                for name in schedule_names
            }

            label = (
                f"round4_LOO_{configuration}_{seed_id}"
            )

            model = AMRInverseModel(
                {
                    name: SCHEDULES_R4[name]
                    for name in bundle
                },
                model_kind="hard",
                width=NETWORK_WIDTH_R4,
                depth=NETWORK_DEPTH_R4,
                parameter_initialization=(
                    3.0e-4,
                    0.15,
                ),
            )

            result = train_shared_parameter_model_r4(
                model,
                bundle,
                design_id="M4",
                label=label,
                seed=training_seed,
            )

            metrics, _ = evaluate_design_r4(
                result,
                bundle,
            )

            metrics.update({
                "configuration": configuration,
                "seed_id": seed_id,
                "seed_number": seed_number,
                "included_schedules": ", ".join(
                    schedule_names
                ),
                "mu_error_percent": (
                    100.0
                    * metrics["relative_error_mu"]
                ),
                "cost_error_percent": (
                    100.0
                    * metrics["relative_error_cost"]
                ),
            })

            LOO_ROWS_R4.append(metrics)
            completed.add(
                (
                    configuration,
                    seed_id,
                )
            )

            pd.DataFrame(
                LOO_ROWS_R4
            ).to_csv(
                LOO_PATH_R4,
                index=False,
            )

            result.model.to("cpu")
            del model, bundle
            clear_gpu_memory()

    LOO_RESULTS_R4 = pd.DataFrame(
        LOO_ROWS_R4
    )
    display(LOO_RESULTS_R4)

else:
    if LOO_PATH_R4.exists():
        LOO_RESULTS_R4 = pd.read_csv(
            LOO_PATH_R4
        )
    else:
        LOO_RESULTS_R4 = pd.DataFrame()

    print(
        "Set RUN_LEAVE_ONE_SCHEDULE_OUT=True to run the optional "
        "schedule-contribution experiment."
    )

# =============================================================================
# PAIRED ANALYSIS AGAINST FULL M4 (same seed)
# =============================================================================

if not LOO_RESULTS_R4.empty:
    loo_metrics = [
        "rmse_resistant",
        "rmse_fraction",
        "mu_error_percent",
        "cost_error_percent",
    ]

    required_columns = [
        "configuration",
        "seed_id",
        *loo_metrics,
    ]
    missing_columns = [
        column
        for column in required_columns
        if column not in LOO_RESULTS_R4.columns
    ]
    if missing_columns:
        raise KeyError(
            "LOO_RESULTS_R4 is missing required columns: "
            + ", ".join(missing_columns)
        )

    full_rows = (
        LOO_RESULTS_R4[
            LOO_RESULTS_R4["configuration"] == "FULL"
        ][["seed_id", *loo_metrics]]
        .rename(
            columns={
                metric: f"{metric}_full"
                for metric in loo_metrics
            }
        )
        .copy()
    )

    paired_rows = (
        LOO_RESULTS_R4[
            LOO_RESULTS_R4["configuration"] != "FULL"
        ]
        .merge(
            full_rows,
            on="seed_id",
            how="inner",
            validate="many_to_one",
        )
        .copy()
    )

    paired_rows["omitted_schedule"] = (
        paired_rows["configuration"]
        .map(OMITTED_SCHEDULE_LABELS_R4)
    )

    paired_change_columns = []
    for metric in loo_metrics:
        change_column = f"{metric}_change_percent"
        paired_rows[change_column] = (
            100.0
            * (
                paired_rows[metric]
                - paired_rows[f"{metric}_full"]
            )
            / paired_rows[f"{metric}_full"]
        )
        paired_change_columns.append(
            change_column
        )

    PAIRED_CHANGES_R4 = paired_rows[
        [
            "configuration",
            "omitted_schedule",
            "seed_id",
            "seed_number",
            *loo_metrics,
            *[f"{metric}_full" for metric in loo_metrics],
            *paired_change_columns,
        ]
    ].copy()

    PAIRED_CHANGES_R4.to_csv(
        PAIRED_CHANGES_PATH_R4,
        index=False,
    )

    PAIRED_SUMMARY_R4 = (
        PAIRED_CHANGES_R4
        .groupby(
            "omitted_schedule",
            observed=True,
        )[paired_change_columns]
        .mean()
        .reindex(OMITTED_ORDER_R4)
    )

    PAIRED_SUMMARY_R4.to_csv(
        PAIRED_SUMMARY_PATH_R4,
        index=True,
    )

    print("\nPaired leave-one-schedule-out changes relative to FULL M4:")
    display(PAIRED_CHANGES_R4)

    print("\nMean paired percentage change summary:")
    display(PAIRED_SUMMARY_R4)

    # =============================================================================
    # PUBLICATION SUMMARY FIGURE
    # =============================================================================

    plot_matrix = PAIRED_SUMMARY_R4.rename(
        columns={
            "rmse_resistant_change_percent": r"RMSE $R$",
            "rmse_fraction_change_percent": r"RMSE $q_R$",
            "mu_error_percent_change_percent": r"Error in $\mu$",
            "cost_error_percent_change_percent": r"Error in $c_R$",
        }
    )

    fig, ax = plt.subplots(
        figsize=(10.2, 5.8),
    )

    color_limit = 200.0

    image = ax.imshow(
        plot_matrix.to_numpy(float),
        aspect="auto",
        cmap="RdBu_r",
        vmin=-color_limit,
        vmax=color_limit,
    )

    ax.set_xticks(
        np.arange(plot_matrix.shape[1])
    )
    ax.set_xticklabels(
        plot_matrix.columns,
        fontsize=12.5,
    )

    ax.set_yticks(
        np.arange(plot_matrix.shape[0])
    )
    ax.set_yticklabels(
        plot_matrix.index,
        fontsize=12.5,
    )

    ax.set_ylabel(
        "Omitted schedule",
        fontsize=13.5,
    )

    # No large in-figure title is used in the final manuscript version.
    # The overall interpretation belongs in the figure caption.

    # Thin white grid between heatmap cells
    ax.set_xticks(
        np.arange(-0.5, plot_matrix.shape[1], 1),
        minor=True,
    )
    ax.set_yticks(
        np.arange(-0.5, plot_matrix.shape[0], 1),
        minor=True,
    )
    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=1.2,
        alpha=0.45,
    )
    ax.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    # Cell annotations
    for row_index in range(
        plot_matrix.shape[0]
    ):
        for column_index in range(
            plot_matrix.shape[1]
        ):
            value = plot_matrix.iloc[
                row_index,
                column_index,
            ]

            text_color = (
                "white"
                if abs(value) >= 95.0
                else "black"
            )

            ax.text(
                column_index,
                row_index,
                f"{value:+.1f}%",
                ha="center",
                va="center",
                color=text_color,
                fontsize=14,
                fontweight="bold",
            )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        pad=0.025,
        ticks=np.arange(-200, 201, 50),
    )
    colorbar.set_label(
        "Change after schedule omission vs. full M4 (%)",
        fontsize=13,
    )
    colorbar.ax.tick_params(
        labelsize=11.5,
    )

    # Sign interpretation is intentionally moved to the manuscript caption:
    # positive values = worse inference after omission;
    # negative values = improvement relative to full M4.

    fig.subplots_adjust(
        left=0.18,
        right=0.92,
        top=0.965,
        bottom=0.12,
    )

    save_figure_r4(
        fig,
        "round4_leave_one_schedule_out_contribution_final",
    )
    plt.show()

    print("\nSaved seed-level table:")
    print(" ", PAIRED_CHANGES_PATH_R4)

    print("\nSaved mean paired-summary table:")
    print(" ", PAIRED_SUMMARY_PATH_R4)

    print("\nSaved publication figure:")
    print(
        " ",
        FIGURE_DIR_R4
        / "round4_leave_one_schedule_out_contribution_final",
    )

else:
    print(
        "No leave-one-schedule-out results are available yet. "
        "Run the suite or load round4_leave_one_schedule_out_metrics.csv first."
    )

In [ ]:
# =============================================================================
# ROUND 4 — CELL 23B
# Final publication heatmap for leave-one-schedule-out analysis
# NO TRAINING — uses completed Cell 23 CSV only
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# -----------------------------------------------------------------------------
# 1. Load completed Cell 23 results
# -----------------------------------------------------------------------------
LOO_PATH_R4 = TABLE_DIR_R4 / "round4_leave_one_schedule_out_metrics.csv"

if LOO_PATH_R4.exists():
    LOO_RESULTS_R4 = pd.read_csv(LOO_PATH_R4)
else:
    raise FileNotFoundError(
        f"Completed Cell 23 results were not found:\n{LOO_PATH_R4}"
    )

print("Loaded completed Cell 23 results:", LOO_PATH_R4)

# -----------------------------------------------------------------------------
# 2. Validate expected configurations, seeds, and metrics
# -----------------------------------------------------------------------------
expected_configurations_r4 = [
    "FULL",
    "NO_UNTREATED",
    "NO_EARLY",
    "NO_LATE",
    "NO_PULSED",
]

expected_seeds_r4 = [
    "seed1",
    "seed2",
    "seed3",
]

required_metrics_r4 = [
    "rmse_resistant",
    "rmse_fraction",
    "mu_error_percent",
    "cost_error_percent",
]

required_columns_r4 = [
    "configuration",
    "seed_id",
] + required_metrics_r4

missing_columns_r4 = [
    column
    for column in required_columns_r4
    if column not in LOO_RESULTS_R4.columns
]

if missing_columns_r4:
    raise KeyError(
        f"Cell 23 CSV is missing required columns: {missing_columns_r4}"
    )

completed_pairs_r4 = set(
    zip(
        LOO_RESULTS_R4["configuration"].astype(str),
        LOO_RESULTS_R4["seed_id"].astype(str),
    )
)

expected_pairs_r4 = {
    (configuration, seed_id)
    for configuration in expected_configurations_r4
    for seed_id in expected_seeds_r4
}

missing_pairs_r4 = sorted(
    expected_pairs_r4 - completed_pairs_r4
)

if missing_pairs_r4:
    raise RuntimeError(
        "Cell 23 is incomplete. Missing pairs:\n"
        + "\n".join(map(str, missing_pairs_r4))
    )

print("Validated all 15/15 completed leave-one-schedule-out fits.")

# -----------------------------------------------------------------------------
# 3. Convert plotting metrics to numeric
# -----------------------------------------------------------------------------
loo_frame_r4 = LOO_RESULTS_R4.copy()

for metric in required_metrics_r4:
    loo_frame_r4[metric] = pd.to_numeric(
        loo_frame_r4[metric],
        errors="raise",
    )

# -----------------------------------------------------------------------------
# 4. Build paired FULL reference by seed
# -----------------------------------------------------------------------------
full_reference_r4 = (
    loo_frame_r4.loc[
        loo_frame_r4["configuration"] == "FULL",
        ["seed_id"] + required_metrics_r4,
    ]
    .drop_duplicates(subset=["seed_id"])
    .set_index("seed_id")
)

if set(full_reference_r4.index.astype(str)) != set(expected_seeds_r4):
    raise RuntimeError(
        "A unique FULL reference was not found for each of seed1, seed2, seed3."
    )

# -----------------------------------------------------------------------------
# 5. Calculate paired percentage changes relative to FULL
#
# Delta E (%) = 100 * (E_LOO - E_FULL) / E_FULL
#
# Positive -> omission worsened performance.
# Negative -> omission improved that metric.
# Zero     -> no change.
# -----------------------------------------------------------------------------
omission_order_r4 = [
    "NO_UNTREATED",
    "NO_EARLY",
    "NO_LATE",
    "NO_PULSED",
]

omission_display_r4 = {
    "NO_UNTREATED": "Omit untreated",
    "NO_EARLY": "Omit early continuous",
    "NO_LATE": "Omit late continuous",
    "NO_PULSED": "Omit pulsed",
}

metric_display_r4 = {
    "rmse_resistant": r"RMSE $R$",
    "rmse_fraction": r"RMSE $q_R$",
    "mu_error_percent": r"Error in $\mu$",
    "cost_error_percent": r"Error in $c_R$",
}

paired_rows_r4 = []

for configuration in omission_order_r4:

    configuration_frame = loo_frame_r4.loc[
        loo_frame_r4["configuration"] == configuration
    ].copy()

    for _, row in configuration_frame.iterrows():

        seed_id = str(
            row["seed_id"]
        )

        if seed_id not in full_reference_r4.index:
            raise RuntimeError(
                f"No paired FULL reference was found for {seed_id}."
            )

        for metric in required_metrics_r4:

            full_value = float(
                full_reference_r4.loc[
                    seed_id,
                    metric,
                ]
            )

            loo_value = float(
                row[metric]
            )

            if not np.isfinite(full_value) or full_value <= 0:
                raise ValueError(
                    f"Invalid FULL value for {seed_id}, {metric}: {full_value}"
                )

            paired_change_percent = (
                100.0
                * (loo_value - full_value)
                / full_value
            )

            paired_rows_r4.append(
                {
                    "configuration": configuration,
                    "seed_id": seed_id,
                    "metric": metric,
                    "full_value": full_value,
                    "loo_value": loo_value,
                    "paired_change_percent": paired_change_percent,
                }
            )

PAIRED_LOO_CHANGES_R4 = pd.DataFrame(
    paired_rows_r4
)

# -----------------------------------------------------------------------------
# 6. Mean ± sample SD across the three paired seeds
# -----------------------------------------------------------------------------
PAIRED_LOO_SUMMARY_R4 = (
    PAIRED_LOO_CHANGES_R4
    .groupby(
        [
            "configuration",
            "metric",
        ],
        observed=True,
    )[
        "paired_change_percent"
    ]
    .agg(
        mean_change_percent="mean",
        sd_change_percent="std",
        median_change_percent="median",
        n_pairs="count",
    )
    .reset_index()
)

# -----------------------------------------------------------------------------
# 7. Create mean and SD matrices in fixed manuscript order
# -----------------------------------------------------------------------------
mean_matrix_r4 = np.full(
    (
        len(omission_order_r4),
        len(required_metrics_r4),
    ),
    np.nan,
    dtype=float,
)

sd_matrix_r4 = np.full_like(
    mean_matrix_r4,
    np.nan,
)

for i, configuration in enumerate(
    omission_order_r4
):

    for j, metric in enumerate(
        required_metrics_r4
    ):

        subset = PAIRED_LOO_SUMMARY_R4.loc[
            (
                PAIRED_LOO_SUMMARY_R4["configuration"]
                == configuration
            )
            &
            (
                PAIRED_LOO_SUMMARY_R4["metric"]
                == metric
            )
        ]

        if len(subset) != 1:
            raise RuntimeError(
                f"Expected one summary row for {configuration}, {metric}; "
                f"found {len(subset)}."
            )

        mean_matrix_r4[i, j] = float(
            subset.iloc[0]["mean_change_percent"]
        )

        sd_matrix_r4[i, j] = float(
            subset.iloc[0]["sd_change_percent"]
        )

# -----------------------------------------------------------------------------
# 8. Save paired statistics
# -----------------------------------------------------------------------------
paired_change_path_r4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_changes.csv"
)

paired_summary_path_r4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_summary.csv"
)

PAIRED_LOO_CHANGES_R4.to_csv(
    paired_change_path_r4,
    index=False,
)

PAIRED_LOO_SUMMARY_R4.to_csv(
    paired_summary_path_r4,
    index=False,
)

# -----------------------------------------------------------------------------
# 9. Fallback figure saver
# -----------------------------------------------------------------------------
if "save_figure_r4" not in globals():

    def save_figure_r4(fig, stem):

        FIGURE_DIR_R4.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.png",
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )

        fig.savefig(
            FIGURE_DIR_R4 / f"{stem}.pdf",
            bbox_inches="tight",
            facecolor="white",
        )

# -----------------------------------------------------------------------------
# 10. Publication style
# -----------------------------------------------------------------------------
CELL23B_RC_R4 = {
    "font.family": "sans-serif",
    "font.size": 11.0,
    "axes.titlesize": 13.0,
    "axes.labelsize": 11.5,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "axes.linewidth": 1.0,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
}

# -----------------------------------------------------------------------------
# 11. Symmetric diverging color scale centered at zero
# -----------------------------------------------------------------------------
maximum_absolute_change_r4 = float(
    np.nanmax(
        np.abs(
            mean_matrix_r4
        )
    )
)

color_limit_r4 = max(
    25.0,
    np.ceil(
        maximum_absolute_change_r4
        / 10.0
    ) * 10.0,
)

color_norm_r4 = TwoSlopeNorm(
    vmin=-color_limit_r4,
    vcenter=0.0,
    vmax=color_limit_r4,
)

# -----------------------------------------------------------------------------
# 12. Final publication heatmap
# -----------------------------------------------------------------------------
with plt.rc_context(
    CELL23B_RC_R4
):

    fig_r4, ax_r4 = plt.subplots(
        figsize=(10.6, 5.8)
    )

    image_r4 = ax_r4.imshow(
        mean_matrix_r4,
        aspect="auto",
        cmap="RdBu_r",
        norm=color_norm_r4,
    )

    ax_r4.set_xticks(
        np.arange(
            len(required_metrics_r4)
        )
    )

    ax_r4.set_xticklabels(
        [
            metric_display_r4[metric]
            for metric in required_metrics_r4
        ]
    )

    ax_r4.set_yticks(
        np.arange(
            len(omission_order_r4)
        )
    )

    ax_r4.set_yticklabels(
        [
            omission_display_r4[configuration]
            for configuration in omission_order_r4
        ]
    )

    # -------------------------------------------------------------------------
    # Mean ± sample SD annotations
    # -------------------------------------------------------------------------
    for i in range(
        mean_matrix_r4.shape[0]
    ):

        for j in range(
            mean_matrix_r4.shape[1]
        ):

            mean_value = float(
                mean_matrix_r4[i, j]
            )

            sd_value = float(
                sd_matrix_r4[i, j]
            )

            text_color = (
                "white"
                if abs(mean_value)
                >= 0.50 * color_limit_r4
                else "black"
            )

            ax_r4.text(
                j,
                i,
                (
                    f"{mean_value:+.1f}%\n"
                    f"± {sd_value:.1f}%"
                ),
                ha="center",
                va="center",
                fontsize=10.3,
                fontweight="semibold",
                color=text_color,
                linespacing=1.15,
            )

    colorbar_r4 = fig_r4.colorbar(
        image_r4,
        ax=ax_r4,
        pad=0.025,
        fraction=0.055,
    )

    colorbar_r4.set_label(
        "Paired change relative to full M4 (%)",
        fontsize=11.0,
    )

    ax_r4.set_title(
        "Contribution of individual treatment schedules to M4 inference",
        fontsize=14.5,
        fontweight="bold",
        pad=13,
    )

    ax_r4.set_xlabel(
        "Evaluation quantity"
    )

    ax_r4.set_ylabel(
        "Removed schedule"
    )

    for spine in ax_r4.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

    fig_r4.tight_layout()

    save_figure_r4(
        fig_r4,
        "round4_leave_one_schedule_out_paired_heatmap_final",
    )

    plt.show()

# -----------------------------------------------------------------------------
# 13. Compact manuscript-check table
# -----------------------------------------------------------------------------
publication_rows_r4 = []

for i, configuration in enumerate(
    omission_order_r4
):

    output_row = {
        "Removed schedule":
            omission_display_r4[
                configuration
            ].replace(
                "Omit ",
                "",
            )
    }

    for j, metric in enumerate(
        required_metrics_r4
    ):

        output_row[
            metric_display_r4[
                metric
            ]
        ] = (
            f"{mean_matrix_r4[i, j]:+.1f} ± "
            f"{sd_matrix_r4[i, j]:.1f}%"
        )

    publication_rows_r4.append(
        output_row
    )

LOO_PUBLICATION_TABLE_R4 = pd.DataFrame(
    publication_rows_r4
)

print(
    "\nPaired leave-one-schedule-out effects "
    "(mean ± sample SD across three seeds):"
)

display(
    LOO_PUBLICATION_TABLE_R4
)

# -----------------------------------------------------------------------------
# 14. Final confirmation
# -----------------------------------------------------------------------------
print(
    "\n" + "=" * 84
)

print(
    "ROUND 4 CELL 23B COMPLETED SUCCESSFULLY"
)

print(
    "=" * 84
)

print(
    "\nNO MODELS WERE RETRAINED."
)

print(
    "\nSource results:"
)

print(
    " ",
    LOO_PATH_R4,
)

print(
    "\nSaved final figure:"
)

print(
    " ",
    FIGURE_DIR_R4
    / "round4_leave_one_schedule_out_paired_heatmap_final",
)

print(
    "\nSaved paired seed-level changes:"
)

print(
    " ",
    paired_change_path_r4,
)

print(
    "\nSaved paired mean ± SD summary:"
)

print(
    " ",
    paired_summary_path_r4,
)

print(
    "\nInterpretation:"
)

print(
    "  Positive percentage -> omission worsened performance."
)

print(
    "  Negative percentage -> omission improved that metric."
)

print(
    "  Zero percentage -> little or no change relative to paired FULL M4."
)

In [ ]:
# ROUND 4 — CELL 23C
# FINAL PUBLICATION REFINEMENT OF LEAVE-ONE-SCHEDULE-OUT ANALYSIS
# =============================================================================
# WHAT THIS CELL DOES
# -------------------
# NO TRAINING.
#
# This cell uses the paired seed-level changes already calculated by Cell 23B.
#
# It creates:
#
# FIGURE A
# --------
# A clean diverging heatmap of the MEAN paired percentage change relative to
# the full M4 design.
#
# The heatmap intentionally does NOT place SD values inside the cells because
# doing so makes the figure visually crowded.
#
# FIGURE B
# --------
# A four-panel paired-seed variability figure showing:
#
#     seed1
#     seed2
#     seed3
#     mean ± sample SD
#
# for every leave-one-schedule-out experiment.
#
# This second figure is especially important because mutation-probability
# recovery exhibits large between-seed variability.
#
# INTERPRETATION
# --------------
# Positive percentage:
#     omission increased error -> removed schedule contained useful in# ROUND 4 — CELL 23
# Leave-one-schedule-out shared-parameter inference
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# M4 is refitted after omitting each schedule in turn. A large deterioration when
# one schedule is removed indicates that the omitted environment supplies unique
# information rather than merely more observations.
#
# This is an optional expensive suite. It is resume-safe and should normally use
# three or five paired seeds after the primary experiment is complete.

LOO_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_metrics.csv"
)

PAIRED_CHANGES_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_changes.csv"
)

PAIRED_SUMMARY_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_summary.csv"
)

LOO_CONFIGURATIONS_R4 = {
    "FULL": (
        "untreated",
        "early_continuous",
        "late_continuous",
        "pulsed",
    ),
    "NO_UNTREATED": (
        "early_continuous",
        "late_continuous",
        "pulsed",
    ),
    "NO_EARLY": (
        "untreated",
        "late_continuous",
        "pulsed",
    ),
    "NO_LATE": (
        "untreated",
        "early_continuous",
        "pulsed",
    ),
    "NO_PULSED": (
        "untreated",
        "early_continuous",
        "late_continuous",
    ),
}

OMITTED_SCHEDULE_LABELS_R4 = {
    "NO_UNTREATED": "Untreated",
    "NO_EARLY": "Early continuous",
    "NO_LATE": "Late continuous",
    "NO_PULSED": "Pulsed",
}

OMITTED_ORDER_R4 = [
    "Untreated",
    "Early continuous",
    "Late continuous",
    "Pulsed",
]

LOO_ROWS_R4: List[Dict[str, Any]] = []

if RUN_LEAVE_ONE_SCHEDULE_OUT:
    if LOO_PATH_R4.exists():
        LOO_ROWS_R4 = pd.read_csv(
            LOO_PATH_R4
        ).to_dict("records")

    completed = {
        (
            str(row["configuration"]),
            str(row["seed_id"]),
        )
        for row in LOO_ROWS_R4
    }

    loo_seeds = (
        PRIMARY_SEED_IDS_R4
        if QUICK_MODE
        else [1, 2, 3]
    )

    for seed_number in loo_seeds:
        seed_id = f"seed{seed_number}"
        dataset_seed = (
            SEED
            + 10_000 * seed_number
        )
        training_seed = (
            SEED
            + 500_000
            + 10_000 * seed_number
        )

        full_bundle = build_design_bundle_r4(
            "M4",
            seed=dataset_seed,
            noise_level=0.0,
        )

        for configuration, schedule_names in (
            LOO_CONFIGURATIONS_R4.items()
        ):
            if (
                configuration,
                seed_id,
            ) in completed:
                continue

            bundle = {
                name: full_bundle[name]
                for name in schedule_names
            }

            label = (
                f"round4_LOO_{configuration}_{seed_id}"
            )

            model = AMRInverseModel(
                {
                    name: SCHEDULES_R4[name]
                    for name in bundle
                },
                model_kind="hard",
                width=NETWORK_WIDTH_R4,
                depth=NETWORK_DEPTH_R4,
                parameter_initialization=(
                    3.0e-4,
                    0.15,
                ),
            )

            result = train_shared_parameter_model_r4(
                model,
                bundle,
                design_id="M4",
                label=label,
                seed=training_seed,
            )

            metrics, _ = evaluate_design_r4(
                result,
                bundle,
            )

            metrics.update({
                "configuration": configuration,
                "seed_id": seed_id,
                "seed_number": seed_number,
                "included_schedules": ", ".join(
                    schedule_names
                ),
                "mu_error_percent": (
                    100.0
                    * metrics["relative_error_mu"]
                ),
                "cost_error_percent": (
                    100.0
                    * metrics["relative_error_cost"]
                ),
            })

            LOO_ROWS_R4.append(metrics)
            completed.add(
                (
                    configuration,
                    seed_id,
                )
            )

            pd.DataFrame(
                LOO_ROWS_R4
            ).to_csv(
                LOO_PATH_R4,
                index=False,
            )

            result.model.to("cpu")
            del model, bundle
            clear_gpu_memory()

    LOO_RESULTS_R4 = pd.DataFrame(
        LOO_ROWS_R4
    )
    display(LOO_RESULTS_R4)

else:
    if LOO_PATH_R4.exists():
        LOO_RESULTS_R4 = pd.read_csv(
            LOO_PATH_R4
        )
    else:
        LOO_RESULTS_R4 = pd.DataFrame()

    print(
        "Set RUN_LEAVE_ONE_SCHEDULE_OUT=True to run the optional "
        "schedule-contribution experiment."
    )

# =============================================================================
# PAIRED ANALYSIS AGAINST FULL M4 (same seed)
# =============================================================================

if not LOO_RESULTS_R4.empty:
    loo_metrics = [
        "rmse_resistant",
        "rmse_fraction",
        "mu_error_percent",
        "cost_error_percent",
    ]

    required_columns = [
        "configuration",
        "seed_id",
        *loo_metrics,
    ]
    missing_columns = [
        column
        for column in required_columns
        if column not in LOO_RESULTS_R4.columns
    ]
    if missing_columns:
        raise KeyError(
            "LOO_RESULTS_R4 is missing required columns: "
            + ", ".join(missing_columns)
        )

    full_rows = (
        LOO_RESULTS_R4[
            LOO_RESULTS_R4["configuration"] == "FULL"
        ][["seed_id", *loo_metrics]]
        .rename(
            columns={
                metric: f"{metric}_full"
                for metric in loo_metrics
            }
        )
        .copy()
    )

    paired_rows = (
        LOO_RESULTS_R4[
            LOO_RESULTS_R4["configuration"] != "FULL"
        ]
        .merge(
            full_rows,
            on="seed_id",
            how="inner",
            validate="many_to_one",
        )
        .copy()
    )

    paired_rows["omitted_schedule"] = (
        paired_rows["configuration"]
        .map(OMITTED_SCHEDULE_LABELS_R4)
    )

    paired_change_columns = []
    for metric in loo_metrics:
        change_column = f"{metric}_change_percent"
        paired_rows[change_column] = (
            100.0
            * (
                paired_rows[metric]
                - paired_rows[f"{metric}_full"]
            )
            / paired_rows[f"{metric}_full"]
        )
        paired_change_columns.append(
            change_column
        )

    PAIRED_CHANGES_R4 = paired_rows[
        [
            "configuration",
            "omitted_schedule",
            "seed_id",
            "seed_number",
            *loo_metrics,
            *[f"{metric}_full" for metric in loo_metrics],
            *paired_change_columns,
        ]
    ].copy()

    PAIRED_CHANGES_R4.to_csv(
        PAIRED_CHANGES_PATH_R4,
        index=False,
    )

    PAIRED_SUMMARY_R4 = (
        PAIRED_CHANGES_R4
        .groupby(
            "omitted_schedule",
            observed=True,
        )[paired_change_columns]
        .mean()
        .reindex(OMITTED_ORDER_R4)
    )

    PAIRED_SUMMARY_R4.to_csv(
        PAIRED_SUMMARY_PATH_R4,
        index=True,
    )

    print("\nPaired leave-one-schedule-out changes relative to FULL M4:")
    display(PAIRED_CHANGES_R4)

    print("\nMean paired percentage change summary:")
    display(PAIRED_SUMMARY_R4)

    # =============================================================================
    # PUBLICATION SUMMARY FIGURE
    # =============================================================================

    plot_matrix = PAIRED_SUMMARY_R4.rename(
        columns={
            "rmse_resistant_change_percent": r"RMSE $R$",
            "rmse_fraction_change_percent": r"RMSE $q_R$",
            "mu_error_percent_change_percent": r"Error in $\mu$",
            "cost_error_percent_change_percent": r"Error in $c_R$",
        }
    )

    fig, ax = plt.subplots(
        figsize=(10.2, 5.8),
    )

    color_limit = 200.0

    image = ax.imshow(
        plot_matrix.to_numpy(float),
        aspect="auto",
        cmap="RdBu_r",
        vmin=-color_limit,
        vmax=color_limit,
    )

    ax.set_xticks(
        np.arange(plot_matrix.shape[1])
    )
    ax.set_xticklabels(
        plot_matrix.columns,
        fontsize=12.5,
    )

    ax.set_yticks(
        np.arange(plot_matrix.shape[0])
    )
    ax.set_yticklabels(
        plot_matrix.index,
        fontsize=12.5,
    )

    ax.set_ylabel(
        "Omitted schedule",
        fontsize=13.5,
    )

    ax.set_title(
        "Schedule-specific contribution to multi-schedule inference",
        fontsize=18,
        fontweight="bold",
        pad=16,
    )

    # Thin white grid between heatmap cells
    ax.set_xticks(
        np.arange(-0.5, plot_matrix.shape[1], 1),
        minor=True,
    )
    ax.set_yticks(
        np.arange(-0.5, plot_matrix.shape[0], 1),
        minor=True,
    )
    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=1.2,
        alpha=0.45,
    )
    ax.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    # Cell annotations
    for row_index in range(
        plot_matrix.shape[0]
    ):
        for column_index in range(
            plot_matrix.shape[1]
        ):
            value = plot_matrix.iloc[
                row_index,
                column_index,
            ]

            text_color = (
                "white"
                if abs(value) >= 95.0
                else "black"
            )

            ax.text(
                column_index,
                row_index,
                f"{value:+.1f}%",
                ha="center",
                va="center",
                color=text_color,
                fontsize=14,
                fontweight="bold",
            )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        pad=0.025,
        ticks=np.arange(-200, 201, 50),
    )
    colorbar.set_label(
        "Change after schedule omission vs. full M4 (%)",
        fontsize=13,
    )
    colorbar.ax.tick_params(
        labelsize=11.5,
    )

    fig.text(
        0.5,
        0.035,
        (
            "Positive values indicate worse inference after omission; "
            "negative values indicate improvement relative to the full M4 design."
        ),
        ha="center",
        va="center",
        fontsize=11.2,
    )

    fig.subplots_adjust(
        left=0.18,
        right=0.92,
        top=0.87,
        bottom=0.18,
    )

    save_figure_r4(
        fig,
        "round4_leave_one_schedule_out_contribution_final",
    )
    plt.show()

    print("\nSaved seed-level table:")
    print(" ", PAIRED_CHANGES_PATH_R4)

    print("\nSaved mean paired-summary table:")
    print(" ", PAIRED_SUMMARY_PATH_R4)

    print("\nSaved publication figure:")
    print(
        " ",
        FIGURE_DIR_R4
        / "round4_leave_one_schedule_out_contribution_final",
    )

else:
    print(
        "No leave-one-schedule-out results are available yet. "
        "Run the suite or load round4_leave_one_schedule_out_metrics.csv first."
    )

# ---------------------------------------------------------------------------
# [Packaging note: this line was originally merged with the closing
#  parenthesis above in the source notebook, which broke Python parsing.
#  It has been restored here as a standalone comment. No code logic was
#  changed -- see CHANGELOG.md.]
# ...formation.
#
# Negative percentage:
#     omission reduced that particular error metric.
#
# Zero:
#     little change relative to the paired full-M4 fit.
#
# IMPORTANT
# ---------
# Negative effects should not automatically be interpreted as evidence that a
# schedule is biologically detrimental. They may reflect redundancy or a
# parameter-specific optimization/identifiability trade-off.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator


# =============================================================================
# 2. LOAD THE PAIRED CHANGES GENERATED BY CELL 23B
# =============================================================================

paired_change_path_r4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_changes.csv"
)


if paired_change_path_r4.exists():

    PAIRED_LOO_CHANGES_R4 = pd.read_csv(
        paired_change_path_r4
    )

    print(
        "Loaded paired Cell 23 results from:",
        paired_change_path_r4,
    )

else:

    raise FileNotFoundError(
        "The paired Cell 23B results were not found:\n"
        f"{paired_change_path_r4}\n\n"
        "Run Cell 23B first. No retraining is required."
    )


# =============================================================================
# 3. VALIDATE THE PAIRED RESULTS
# =============================================================================

required_columns_r4 = [
    "configuration",
    "seed_id",
    "metric",
    "paired_change_percent",
]


missing_columns_r4 = [
    column
    for column in required_columns_r4
    if column not in PAIRED_LOO_CHANGES_R4.columns
]


if missing_columns_r4:

    raise KeyError(
        "Paired LOO table is missing columns: "
        f"{missing_columns_r4}"
    )


PAIRED_LOO_CHANGES_R4[
    "paired_change_percent"
] = pd.to_numeric(
    PAIRED_LOO_CHANGES_R4[
        "paired_change_percent"
    ],
    errors="raise",
)


# =============================================================================
# 4. FIXED MANUSCRIPT ORDER
# =============================================================================

omission_order_r4 = [
    "NO_UNTREATED",
    "NO_EARLY",
    "NO_LATE",
    "NO_PULSED",
]


omission_display_r4 = {
    "NO_UNTREATED":
        "Untreated",

    "NO_EARLY":
        "Early continuous",

    "NO_LATE":
        "Late continuous",

    "NO_PULSED":
        "Pulsed",
}


metric_order_r4 = [
    "rmse_resistant",
    "rmse_fraction",
    "mu_error_percent",
    "cost_error_percent",
]


metric_heatmap_labels_r4 = {
    "rmse_resistant":
        r"RMSE $R$",

    "rmse_fraction":
        r"RMSE $q_R$",

    "mu_error_percent":
        r"Error in $\mu$",

    "cost_error_percent":
        r"Error in $c_R$",
}


metric_panel_titles_r4 = {
    "rmse_resistant":
        r"Resistant-state reconstruction, $R(t)$",

    "rmse_fraction":
        r"Phenotype-composition reconstruction, $q_R(t)$",

    "mu_error_percent":
        r"Mutation-probability identification, $\mu$",

    "cost_error_percent":
        r"Resistance-fitness-cost identification, $c_R$",
}


# =============================================================================
# 5. COMPUTE MEAN ± SAMPLE SD
# =============================================================================

LOO_FINAL_SUMMARY_R4 = (
    PAIRED_LOO_CHANGES_R4
    .groupby(
        [
            "configuration",
            "metric",
        ],
        observed=True,
    )[
        "paired_change_percent"
    ]
    .agg(
        mean_change_percent="mean",
        sd_change_percent="std",
        median_change_percent="median",
        n_pairs="count",
    )
    .reset_index()
)


# =============================================================================
# 6. BUILD MATRICES FOR THE HEATMAP
# =============================================================================

mean_matrix_r4 = np.full(
    (
        len(omission_order_r4),
        len(metric_order_r4),
    ),
    np.nan,
    dtype=float,
)


sd_matrix_r4 = np.full_like(
    mean_matrix_r4,
    np.nan,
)


for row_index, configuration in enumerate(
    omission_order_r4
):

    for column_index, metric in enumerate(
        metric_order_r4
    ):

        subset = LOO_FINAL_SUMMARY_R4.loc[
            (
                LOO_FINAL_SUMMARY_R4[
                    "configuration"
                ] == configuration
            )
            &
            (
                LOO_FINAL_SUMMARY_R4[
                    "metric"
                ] == metric
            )
        ]


        if len(subset) != 1:

            raise RuntimeError(
                "Expected exactly one summary result for "
                f"{configuration}, {metric}."
            )


        mean_matrix_r4[
            row_index,
            column_index,
        ] = float(
            subset.iloc[0][
                "mean_change_percent"
            ]
        )


        sd_matrix_r4[
            row_index,
            column_index,
        ] = float(
            subset.iloc[0][
                "sd_change_percent"
            ]
        )


# =============================================================================
# 7. FIGURE SAVING FALLBACK
# =============================================================================

if "save_figure_r4" not in globals():

    def save_figure_r4(
        figure,
        stem,
    ):

        FIGURE_DIR_R4.mkdir(
            parents=True,
            exist_ok=True,
        )


        figure.savefig(
            FIGURE_DIR_R4
            / f"{stem}.png",
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )


        figure.savefig(
            FIGURE_DIR_R4
            / f"{stem}.pdf",
            bbox_inches="tight",
            facecolor="white",
        )


# =============================================================================
# 8. PUBLICATION STYLE
# =============================================================================

CELL23C_RC_R4 = {

    "font.family":
        "sans-serif",

    "font.size":
        11.0,

    "axes.titlesize":
        12.0,

    "axes.labelsize":
        11.0,

    "xtick.labelsize":
        10.0,

    "ytick.labelsize":
        10.0,

    "legend.fontsize":
        9.0,

    "axes.linewidth":
        1.0,

    "axes.edgecolor":
        "black",

    "axes.facecolor":
        "white",

    "figure.facecolor":
        "white",

    "savefig.facecolor":
        "white",
}


# =============================================================================
# 9. FIGURE A — FINAL CLEAN CONTRIBUTION HEATMAP
# =============================================================================
# The color scale is symmetric around zero:
#
#     blue  -> omission reduced the metric
#     white -> little/no effect
#     red   -> omission increased the metric
# =============================================================================

maximum_effect_r4 = float(
    np.nanmax(
        np.abs(
            mean_matrix_r4
        )
    )
)


# Round upward to a visually clean 25% interval.
color_limit_r4 = max(
    50.0,
    np.ceil(
        maximum_effect_r4
        / 25.0
    )
    * 25.0,
)


heatmap_norm_r4 = TwoSlopeNorm(
    vmin=-color_limit_r4,
    vcenter=0.0,
    vmax=color_limit_r4,
)


with plt.rc_context(
    CELL23C_RC_R4
):

    fig_heat_r4, ax_heat_r4 = plt.subplots(
        figsize=(9.7, 5.3),
    )


    image_r4 = ax_heat_r4.imshow(
        mean_matrix_r4,
        aspect="auto",
        cmap="RdBu_r",
        norm=heatmap_norm_r4,
    )


    # -------------------------------------------------------------------------
    # Axis labels
    # -------------------------------------------------------------------------

    ax_heat_r4.set_xticks(
        np.arange(
            len(metric_order_r4)
        )
    )


    ax_heat_r4.set_xticklabels(
        [
            metric_heatmap_labels_r4[
                metric
            ]
            for metric in metric_order_r4
        ]
    )


    ax_heat_r4.set_yticks(
        np.arange(
            len(omission_order_r4)
        )
    )


    ax_heat_r4.set_yticklabels(
        [
            omission_display_r4[
                configuration
            ]
            for configuration in omission_order_r4
        ]
    )


    # -------------------------------------------------------------------------
    # Add only the MEAN effect to each cell.
    #
    # SD is deliberately moved to Figure B.
    # -------------------------------------------------------------------------

    for row_index in range(
        mean_matrix_r4.shape[0]
    ):

        for column_index in range(
            mean_matrix_r4.shape[1]
        ):

            value = float(
                mean_matrix_r4[
                    row_index,
                    column_index,
                ]
            )


            text_color = (
                "white"
                if abs(value)
                >= 0.48 * color_limit_r4
                else "black"
            )


            # -------------------------------------------------------------
            # Publication-quality annotation formatting
            # -------------------------------------------------------------
            # Preserve one decimal place for small effects so values such as
            # -0.5% are never displayed as the misleading "-0%".
            # Larger effects remain integer percentages for a clean heatmap.
            if abs(value) < 10.0:
                value_label = f"{value:+.1f}%"
            else:
                value_label = f"{value:+.0f}%"

            ax_heat_r4.text(
                column_index,
                row_index,
                value_label,
                ha="center",
                va="center",
                fontsize=11.7,
                fontweight="bold",
                color=text_color,
            )


    # -------------------------------------------------------------------------
    # Add thin boundaries between cells for cleaner separation.
    # -------------------------------------------------------------------------

    ax_heat_r4.set_xticks(
        np.arange(
            -0.5,
            len(metric_order_r4),
            1.0,
        ),
        minor=True,
    )


    ax_heat_r4.set_yticks(
        np.arange(
            -0.5,
            len(omission_order_r4),
            1.0,
        ),
        minor=True,
    )


    ax_heat_r4.grid(
        which="minor",
        linewidth=1.0,
        alpha=0.45,
    )


    ax_heat_r4.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )


    # -------------------------------------------------------------------------
    # Colorbar
    # -------------------------------------------------------------------------

    colorbar_r4 = fig_heat_r4.colorbar(
        image_r4,
        ax=ax_heat_r4,
        pad=0.025,
        fraction=0.055,
    )


    colorbar_r4.set_label(
        "Paired change relative to full M4 (%)"
    )


    # -------------------------------------------------------------------------
    # Labels and title
    # -------------------------------------------------------------------------

    ax_heat_r4.set_ylabel(
        "Omitted schedule"
    )


    ax_heat_r4.set_xlabel(
        "Evaluation quantity"
    )


    ax_heat_r4.set_title(
        "Schedule-specific contribution to multi-schedule inference",
        fontsize=14.5,
        fontweight="bold",
        pad=12,
    )


    for spine in ax_heat_r4.spines.values():

        spine.set_visible(
            True
        )

        spine.set_color(
            "black"
        )

        spine.set_linewidth(
            1.0
        )


    fig_heat_r4.tight_layout()


    save_figure_r4(
        fig_heat_r4,
        "round4_leave_one_schedule_out_heatmap_final",
    )


    plt.show()


# =============================================================================
# 10. FIGURE B — SEED-LEVEL VARIABILITY
# =============================================================================
# Why this figure matters:
#
# The mutation-probability effects have very large SD values. Showing the
# individual paired seeds prevents a reviewer from mistaking a large mean for
# a stable effect.
# =============================================================================

seed_order_r4 = [
    "seed1",
    "seed2",
    "seed3",
]


seed_markers_r4 = {
    "seed1":
        "o",

    "seed2":
        "s",

    "seed3":
        "^",
}


# Small fixed offsets prevent seed markers from sitting directly on top of one
# another without changing the actual schedule category.
seed_offsets_r4 = {
    "seed1":
        -0.10,

    "seed2":
        0.00,

    "seed3":
        0.10,
}


panel_labels_r4 = [
    "A",
    "B",
    "C",
    "D",
]


with plt.rc_context(
    CELL23C_RC_R4
):

    fig_seed_r4, axes_seed_r4 = plt.subplots(
        2,
        2,
        figsize=(11.7, 8.1),
    )


    axes_seed_r4 = axes_seed_r4.flatten()


    x_r4 = np.arange(
        len(omission_order_r4),
        dtype=float,
    )


    for panel_label, axis, metric in zip(
        panel_labels_r4,
        axes_seed_r4,
        metric_order_r4,
    ):

        metric_frame = (
            PAIRED_LOO_CHANGES_R4[
                PAIRED_LOO_CHANGES_R4[
                    "metric"
                ] == metric
            ]
            .copy()
        )


        # ---------------------------------------------------------------------
        # Individual paired seeds
        # ---------------------------------------------------------------------

        for seed_id in seed_order_r4:

            seed_values = []


            for configuration in omission_order_r4:

                subset = metric_frame.loc[
                    (
                        metric_frame[
                            "configuration"
                        ] == configuration
                    )
                    &
                    (
                        metric_frame[
                            "seed_id"
                        ].astype(str)
                        == seed_id
                    ),
                    "paired_change_percent",
                ]


                if len(subset) != 1:

                    raise RuntimeError(
                        "Expected one seed-level value for "
                        f"{metric}, {configuration}, {seed_id}."
                    )


                seed_values.append(
                    float(
                        subset.iloc[0]
                    )
                )


            axis.scatter(
                x_r4
                + seed_offsets_r4[
                    seed_id
                ],
                seed_values,
                marker=seed_markers_r4[
                    seed_id
                ],
                s=48,
                alpha=0.72,
                label=seed_id,
                zorder=3,
            )


        # ---------------------------------------------------------------------
        # Mean ± sample SD
        # ---------------------------------------------------------------------

        metric_means_r4 = []


        metric_sds_r4 = []


        for configuration in omission_order_r4:

            subset = LOO_FINAL_SUMMARY_R4.loc[
                (
                    LOO_FINAL_SUMMARY_R4[
                        "configuration"
                    ] == configuration
                )
                &
                (
                    LOO_FINAL_SUMMARY_R4[
                        "metric"
                    ] == metric
                )
            ]


            metric_means_r4.append(
                float(
                    subset.iloc[0][
                        "mean_change_percent"
                    ]
                )
            )


            metric_sds_r4.append(
                float(
                    subset.iloc[0][
                        "sd_change_percent"
                    ]
                )
            )


        axis.errorbar(
            x_r4,
            metric_means_r4,
            yerr=metric_sds_r4,
            fmt="D",
            markersize=6.5,
            linewidth=1.7,
            capsize=4.0,
            color="black",
            label="Mean ± SD",
            zorder=5,
        )


        # ---------------------------------------------------------------------
        # Neutral line
        # ---------------------------------------------------------------------

        axis.axhline(
            0.0,
            linestyle="--",
            linewidth=1.1,
            color="black",
            alpha=0.65,
            zorder=1,
        )


        # ---------------------------------------------------------------------
        # Formatting
        # ---------------------------------------------------------------------

        axis.set_xticks(
            x_r4
        )


        axis.set_xticklabels(
            [
                omission_display_r4[
                    configuration
                ]
                for configuration
                in omission_order_r4
            ],
            rotation=0,
        )


        axis.set_title(
            metric_panel_titles_r4[
                metric
            ],
            loc="left",
            fontweight="semibold",
            pad=8,
        )


        axis.set_ylabel(
            "Paired change relative to full M4 (%)"
        )


        axis.grid(
            axis="y",
            linestyle="--",
            alpha=0.20,
        )


        axis.set_axisbelow(
            True
        )


        axis.yaxis.set_major_locator(
            MaxNLocator(
                nbins=6
            )
        )


        for spine in axis.spines.values():

            spine.set_visible(
                True
            )

            spine.set_color(
                "black"
            )

            spine.set_linewidth(
                1.0
            )


        axis.text(
            -0.10,
            1.07,
            panel_label,
            transform=axis.transAxes,
            fontsize=12.5,
            fontweight="bold",
            ha="left",
            va="top",
        )


    # -------------------------------------------------------------------------
    # One shared legend outside the plotting panels.
    # -------------------------------------------------------------------------

    legend_handles_r4 = [
        Line2D(
            [0],
            [0],
            marker=seed_markers_r4[
                seed_id
            ],
            linestyle="None",
            markersize=6.5,
            label=seed_id,
        )
        for seed_id in seed_order_r4
    ]


    legend_handles_r4.append(
        Line2D(
            [0],
            [0],
            marker="D",
            color="black",
            linestyle="-",
            markersize=6.5,
            label="Mean ± SD",
        )
    )


    fig_seed_r4.legend(
        handles=legend_handles_r4,
        loc="upper center",
        bbox_to_anchor=(
            0.5,
            0.94,
        ),
        ncol=4,
        frameon=False,
    )


    fig_seed_r4.suptitle(
        "Paired-seed stability of schedule-specific contributions",
        fontsize=15.0,
        fontweight="bold",
        y=0.99,
    )


    fig_seed_r4.subplots_adjust(
        left=0.085,
        right=0.985,
        bottom=0.10,
        top=0.86,
        wspace=0.27,
        hspace=0.32,
    )


    save_figure_r4(
        fig_seed_r4,
        "round4_leave_one_schedule_out_seed_stability_final",
    )


    plt.show()


# =============================================================================
# 11. FINAL NUMERICAL TABLE
# =============================================================================

final_table_rows_r4 = []


for row_index, configuration in enumerate(
    omission_order_r4
):

    output_row = {
        "Omitted schedule":
            omission_display_r4[
                configuration
            ]
    }


    for column_index, metric in enumerate(
        metric_order_r4
    ):

        output_row[
            metric_heatmap_labels_r4[
                metric
            ]
        ] = (
            f"{mean_matrix_r4[row_index, column_index]:+.1f} "
            f"± {sd_matrix_r4[row_index, column_index]:.1f}%"
        )


    final_table_rows_r4.append(
        output_row
    )


LOO_FINAL_PUBLICATION_TABLE_R4 = pd.DataFrame(
    final_table_rows_r4
)


print(
    "\nFinal paired leave-one-schedule-out summary:"
)


display(
    LOO_FINAL_PUBLICATION_TABLE_R4
)


# =============================================================================
# 12. SAVE FINAL TABLE
# =============================================================================

final_table_path_r4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_final_publication_summary.csv"
)


LOO_FINAL_PUBLICATION_TABLE_R4.to_csv(
    final_table_path_r4,
    index=False,
)


# =============================================================================
# 13. FINAL REPORT
# =============================================================================

print(
    "\n" + "=" * 86
)


print(
    "ROUND 4 CELL 23C COMPLETED SUCCESSFULLY"
)


print(
    "=" * 86
)


print(
    "\nNO TRAINING WAS PERFORMED."
)


print(
    "\nFinal heatmap:"
)


print(
    " ",
    FIGURE_DIR_R4
    / "round4_leave_one_schedule_out_heatmap_final",
)


print(
    "\nSeed-stability figure:"
)


print(
    " ",
    FIGURE_DIR_R4
    / "round4_leave_one_schedule_out_seed_stability_final",
)


print(
    "\nFinal publication summary:"
)


print(
    " ",
    final_table_path_r4,
)

In [ ]:
# ROUND 4 — CELL 24
# FINAL PUBLICATION VERSION
# Parameter-objective geometry for M1 versus M4
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# This experiment visualizes practical parameter identifiability directly in
# the mechanistic ODE model, independently of PINN optimization.
#
# For each candidate parameter pair
#
#       theta = (log10(mu), c_R),
#
# the mechanistic observation residual is evaluated.
#
# The objective geometry is compared between:
#
#       M1 : early-continuous schedule only
#       M4 : untreated + early-continuous + late-continuous + pulsed
#
# SCIENTIFIC PURPOSE
# ------------------
# A long narrow low-objective valley indicates compensation between mu and c_R:
# many parameter combinations can explain the observations similarly well.
#
# A more localized objective region indicates improved practical
# identifiability.
#
# IMPORTANT INTERPRETATION
# ------------------------
# The objective surface is NOT expected to become perfectly isotropic under M4.
# Round 4 sensitivity analysis already showed that strong coupling between
# log10(mu) and c_R persists. Therefore, this figure is intended to determine
# whether multi-schedule observations sharpen the objective geometry rather
# than claim that parameter confounding disappears completely.
#
# FINAL PUBLICATION REFINEMENTS
# -----------------------------
# 1. Existing 41 x 41 surfaces are reused unless explicit recomputation is
#    requested.
#
# 2. Raw SSE is converted to residual-count-normalized MSE:
#
#       MSE = SSE / N_residuals
#
#    because M1 and M4 contain different numbers of schedules/observations.
#
# 3. The excess objective is referenced to the exact true parameter pair:
#
#       Delta MSE(theta)
#           = max[MSE(theta) - MSE(theta_true), 0].
#
# 4. M1 and M4 use the SAME contour levels and SAME color normalization.
#    Therefore colors are directly comparable between panels.
#
# 5. The true parameter pair and finite-grid minimum are shown separately.
#
# 6. The colorbar occupies a dedicated axis and cannot overlap Panel B.
#
# 7. A shared legend is placed outside the data panels.
#
# 8. Thin contour overlays emphasize the compensation-valley geometry.
#
# 9. Exact numerical grid-minimum information is exported to CSV.
#
# NO PINN TRAINING IS PERFORMED IN THIS CELL.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator


# =============================================================================
# 2. FILE PATHS
# =============================================================================

PROFILE_SURFACE_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_profile_surfaces_M1_M4.csv"
)

PROFILE_SURFACE_SUMMARY_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_profile_surfaces_M1_M4_summary.csv"
)


# =============================================================================
# 3. EXECUTION POLICY
# =============================================================================
# You have already completed the expensive 41 x 41 profile calculation.
#
# KEEP THIS FALSE.
#
# False:
#     reuse the existing profile-surface CSV.
#
# True:
#     recompute the full ODE profile surfaces.
# =============================================================================

RECOMPUTE_PROFILE_SURFACES_R4 = False


PROFILE_EPSILON_R4 = 1.0e-14


PROFILE_DESIGNS_R4 = [
    "M1",
    "M4",
]


TRUE_LOG10_MU_R4 = math.log10(
    TRUE_PARAMETERS.mutation_probability
)


TRUE_COST_R4 = float(
    TRUE_PARAMETERS.resistance_fitness_cost
)


# =============================================================================
# 4. FALLBACK PLOTTING HELPERS
# =============================================================================

if "set_panel_frame_r4" not in globals():

    def set_panel_frame_r4(axis):
        """
        Draw a consistent black frame around a publication panel.
        """

        for spine in axis.spines.values():

            spine.set_visible(
                True
            )

            spine.set_color(
                "black"
            )

            spine.set_linewidth(
                1.0
            )


if "save_figure_r4" not in globals():

    def save_figure_r4(
        figure,
        stem,
    ):
        """
        Save high-resolution raster and vector versions.
        """

        FIGURE_DIR_R4.mkdir(
            parents=True,
            exist_ok=True,
        )


        figure.savefig(
            FIGURE_DIR_R4
            / f"{stem}.png",
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )


        figure.savefig(
            FIGURE_DIR_R4
            / f"{stem}.pdf",
            bbox_inches="tight",
            facecolor="white",
        )

        figure.savefig(
            FIGURE_DIR_R4
            / f"{stem}.svg",
            bbox_inches="tight",
            facecolor="white",
        )


# =============================================================================
# 5. DETERMINE WHETHER RECOMPUTATION IS REQUIRED
# =============================================================================

need_recompute_r4 = (
    RUN_PROFILE_SURFACES
    and (
        RECOMPUTE_PROFILE_SURFACES_R4
        or not PROFILE_SURFACE_PATH_R4.exists()
    )
)


# =============================================================================
# 6. OPTIONAL FULL PROFILE-SURFACE COMPUTATION
# =============================================================================

if need_recompute_r4:

    grid_size_r4 = (
        13
        if QUICK_MODE
        else 41
    )


    log_mu_grid_r4 = np.linspace(
        -5.5,
        -2.5,
        grid_size_r4,
    )


    cost_grid_r4 = np.linspace(
        0.0,
        0.30,
        grid_size_r4,
    )


    profile_rows_r4 = []


    total_profile_points_r4 = (
        len(PROFILE_DESIGNS_R4)
        * grid_size_r4
        * grid_size_r4
    )


    completed_profile_points_r4 = 0


    print(
        f"Computing {total_profile_points_r4} profile points "
        f"({grid_size_r4} x {grid_size_r4} per design)."
    )


    for design_id_r4 in PROFILE_DESIGNS_R4:

        bundle_r4 = build_design_bundle_r4(
            design_id_r4,
            seed=SEED,
            noise_level=0.0,
        )


        print(
            f"\nComputing objective surface for {design_id_r4}..."
        )


        for row_number_r4, log_mu_r4 in enumerate(
            log_mu_grid_r4,
            start=1,
        ):

            for cost_r4 in cost_grid_r4:

                residual_r4 = np.asarray(
                    ode_residual_vector(
                        np.array(
                            [
                                log_mu_r4,
                                cost_r4,
                            ],
                            dtype=float,
                        ),
                        bundle_r4,
                    ),
                    dtype=float,
                ).reshape(-1)


                profile_rows_r4.append(
                    {
                        "design_id":
                            design_id_r4,

                        "log10_mu":
                            float(
                                log_mu_r4
                            ),

                        "cost":
                            float(
                                cost_r4
                            ),

                        "n_residuals":
                            int(
                                residual_r4.size
                            ),

                        "sse":
                            float(
                                np.sum(
                                    residual_r4 ** 2
                                )
                            ),

                        "mse":
                            float(
                                np.mean(
                                    residual_r4 ** 2
                                )
                            ),
                    }
                )


                completed_profile_points_r4 += 1


            print(
                f"  {design_id_r4}: "
                f"{row_number_r4}/{grid_size_r4} rows completed "
                f"({completed_profile_points_r4}/"
                f"{total_profile_points_r4} total points)"
            )


    PROFILE_SURFACE_R4 = pd.DataFrame(
        profile_rows_r4
    )


    PROFILE_SURFACE_R4.to_csv(
        PROFILE_SURFACE_PATH_R4,
        index=False,
    )


    print(
        "\nSaved newly calculated profile grid:"
    )

    print(
        PROFILE_SURFACE_PATH_R4
    )


# =============================================================================
# 7. LOAD EXISTING PROFILE SURFACE
# =============================================================================

elif PROFILE_SURFACE_PATH_R4.exists():

    PROFILE_SURFACE_R4 = pd.read_csv(
        PROFILE_SURFACE_PATH_R4
    )


    print(
        "Loaded existing profile grid:"
    )

    print(
        PROFILE_SURFACE_PATH_R4
    )


else:

    raise FileNotFoundError(
        "No completed profile-surface CSV was found.\n"
        "Set RUN_PROFILE_SURFACES=True and rerun Cell 24."
    )


# =============================================================================
# 8. VALIDATE PROFILE-SURFACE TABLE
# =============================================================================

required_profile_columns_r4 = [
    "design_id",
    "log10_mu",
    "cost",
    "sse",
]


missing_profile_columns_r4 = [
    column
    for column in required_profile_columns_r4
    if column not in PROFILE_SURFACE_R4.columns
]


if missing_profile_columns_r4:

    raise KeyError(
        "PROFILE_SURFACE_R4 is missing required columns: "
        f"{missing_profile_columns_r4}"
    )


PROFILE_SURFACE_R4 = (
    PROFILE_SURFACE_R4
    .copy()
)


PROFILE_SURFACE_R4[
    "design_id"
] = (
    PROFILE_SURFACE_R4[
        "design_id"
    ]
    .astype(str)
)


for column_r4 in [
    "log10_mu",
    "cost",
    "sse",
]:

    PROFILE_SURFACE_R4[
        column_r4
    ] = pd.to_numeric(
        PROFILE_SURFACE_R4[
            column_r4
        ],
        errors="raise",
    )


# =============================================================================
# 9. COMPUTE EXACT TRUE-PAIR REFERENCE AND RESIDUAL COUNTS
# =============================================================================
# M1 and M4 contain different numbers of observations.
#
# Comparing raw SSE would therefore partly reflect residual count.
#
# We instead use:
#
#       MSE = SSE / N_residuals.
#
# Only one exact ODE residual evaluation per design is needed here.
# =============================================================================

residual_count_r4 = {}

true_reference_mse_r4 = {}


for design_id_r4 in PROFILE_DESIGNS_R4:

    bundle_r4 = build_design_bundle_r4(
        design_id_r4,
        seed=SEED,
        noise_level=0.0,
    )


    true_residual_r4 = np.asarray(
        ode_residual_vector(
            np.array(
                [
                    TRUE_LOG10_MU_R4,
                    TRUE_COST_R4,
                ],
                dtype=float,
            ),
            bundle_r4,
        ),
        dtype=float,
    ).reshape(-1)


    residual_count_r4[
        design_id_r4
    ] = int(
        true_residual_r4.size
    )


    true_reference_mse_r4[
        design_id_r4
    ] = float(
        np.mean(
            true_residual_r4 ** 2
        )
    )


# =============================================================================
# 10. CONVERT RAW SSE TO RESIDUAL-NORMALIZED MSE
# =============================================================================

PROFILE_SURFACE_R4[
    "n_residuals"
] = (
    PROFILE_SURFACE_R4[
        "design_id"
    ]
    .map(
        residual_count_r4
    )
    .astype(int)
)


PROFILE_SURFACE_R4[
    "mse"
] = (
    PROFILE_SURFACE_R4[
        "sse"
    ]
    / PROFILE_SURFACE_R4[
        "n_residuals"
    ]
)


PROFILE_SURFACE_R4[
    "true_reference_mse"
] = (
    PROFILE_SURFACE_R4[
        "design_id"
    ]
    .map(
        true_reference_mse_r4
    )
    .astype(float)
)


# =============================================================================
# 11. EXCESS OBJECTIVE RELATIVE TO THE EXACT TRUE PARAMETER PAIR
# =============================================================================
# The synthetic observations were produced using the matched ODE model.
#
# Therefore the exact true pair provides the natural mechanistic reference.
# =============================================================================

PROFILE_SURFACE_R4[
    "delta_mse"
] = np.maximum(
    PROFILE_SURFACE_R4[
        "mse"
    ]
    - PROFILE_SURFACE_R4[
        "true_reference_mse"
    ],
    0.0,
)


PROFILE_SURFACE_R4[
    "log10_delta_mse"
] = np.log10(
    PROFILE_SURFACE_R4[
        "delta_mse"
    ]
    + PROFILE_EPSILON_R4
)


# Save the enriched table.
PROFILE_SURFACE_R4.to_csv(
    PROFILE_SURFACE_PATH_R4,
    index=False,
)


# =============================================================================
# 12. BUILD PIVOT MATRICES
# =============================================================================

profile_pivots_r4 = {}

profile_z_values_r4 = {}

all_profile_z_r4 = []


for design_id_r4 in PROFILE_DESIGNS_R4:

    frame_r4 = PROFILE_SURFACE_R4.loc[
        PROFILE_SURFACE_R4[
            "design_id"
        ] == design_id_r4
    ].copy()


    pivot_r4 = (
        frame_r4
        .pivot(
            index="cost",
            columns="log10_mu",
            values="log10_delta_mse",
        )
        .sort_index(
            axis=0
        )
        .sort_index(
            axis=1
        )
    )


    profile_pivots_r4[
        design_id_r4
    ] = pivot_r4


    z_values_r4 = pivot_r4.to_numpy(
        dtype=float
    )


    profile_z_values_r4[
        design_id_r4
    ] = z_values_r4


    finite_z_r4 = z_values_r4[
        np.isfinite(
            z_values_r4
        )
    ]


    all_profile_z_r4.extend(
        finite_z_r4.ravel().tolist()
    )


all_profile_z_r4 = np.asarray(
    all_profile_z_r4,
    dtype=float,
)


# =============================================================================
# 13. DEFINE ONE SHARED COLOR SCALE
# =============================================================================
# This is essential for valid visual comparison.
#
# M1 and M4 must not independently choose their contour ranges.
#
# The 2nd percentile is used as the lower display bound so isolated
# near-zero cells do not consume almost the entire dynamic range.
# =============================================================================

shared_vmin_r4 = float(
    np.nanpercentile(
        all_profile_z_r4,
        2.0,
    )
)


shared_vmax_r4 = float(
    np.nanmax(
        all_profile_z_r4
    )
)


if shared_vmax_r4 <= shared_vmin_r4:

    shared_vmax_r4 = (
        shared_vmin_r4
        + 1.0
    )


shared_contour_levels_r4 = np.linspace(
    shared_vmin_r4,
    shared_vmax_r4,
    30,
)


# Sparse contour-line levels for visual valley structure.
shared_line_levels_r4 = (
    shared_contour_levels_r4[
        3::5
    ]
)


# =============================================================================
# 14. NUMERICAL PROFILE-SURFACE SUMMARY
# =============================================================================

profile_summary_rows_r4 = []


for design_id_r4 in PROFILE_DESIGNS_R4:

    frame_r4 = PROFILE_SURFACE_R4.loc[
        PROFILE_SURFACE_R4[
            "design_id"
        ] == design_id_r4
    ].copy()


    minimum_row_r4 = frame_r4.loc[
        frame_r4[
            "mse"
        ].idxmin()
    ]


    profile_summary_rows_r4.append(
        {
            "design_id":
                design_id_r4,

            "schedule_count":
                (
                    1
                    if design_id_r4 == "M1"
                    else 4
                ),

            "grid_min_log10_mu":
                float(
                    minimum_row_r4[
                        "log10_mu"
                    ]
                ),

            "grid_min_mu":
                float(
                    10.0
                    ** float(
                        minimum_row_r4[
                            "log10_mu"
                        ]
                    )
                ),

            "grid_min_cost":
                float(
                    minimum_row_r4[
                        "cost"
                    ]
                ),

            "grid_min_mse":
                float(
                    minimum_row_r4[
                        "mse"
                    ]
                ),

            "true_reference_mse":
                float(
                    true_reference_mse_r4[
                        design_id_r4
                    ]
                ),

            "abs_grid_error_log10_mu":
                abs(
                    float(
                        minimum_row_r4[
                            "log10_mu"
                        ]
                    )
                    - TRUE_LOG10_MU_R4
                ),

            "abs_grid_error_cost":
                abs(
                    float(
                        minimum_row_r4[
                            "cost"
                        ]
                    )
                    - TRUE_COST_R4
                ),

            "n_residuals":
                int(
                    residual_count_r4[
                        design_id_r4
                    ]
                ),
        }
    )


PROFILE_SURFACE_SUMMARY_R4 = pd.DataFrame(
    profile_summary_rows_r4
)


PROFILE_SURFACE_SUMMARY_R4.to_csv(
    PROFILE_SURFACE_SUMMARY_PATH_R4,
    index=False,
)


# =============================================================================
# 15. FINAL PUBLICATION STYLE
# =============================================================================

CELL24_PUBLICATION_RC_R4 = {

    "font.family":
        "sans-serif",

    "font.size":
        11.0,

    "axes.titlesize":
        12.8,

    "axes.labelsize":
        11.5,

    "xtick.labelsize":
        10.0,

    "ytick.labelsize":
        10.0,

    "legend.fontsize":
        9.5,

    "axes.linewidth":
        1.0,

    "axes.edgecolor":
        "black",

    "axes.facecolor":
        "white",

    "figure.facecolor":
        "white",

    "savefig.facecolor":
        "white",
}


# =============================================================================
# 16. CREATE FINAL FIGURE
# =============================================================================

with plt.rc_context(
    CELL24_PUBLICATION_RC_R4
):

    fig_r4 = plt.figure(
        figsize=(
            12.4,
            5.15,
        )
    )


    # Dedicated third column prevents colorbar overlap.
    grid_spec_r4 = GridSpec(
        1,
        3,
        figure=fig_r4,
        width_ratios=[
            1.0,
            1.0,
            0.045,
        ],
        wspace=0.22,
    )


    ax_m1_r4 = fig_r4.add_subplot(
        grid_spec_r4[
            0,
            0,
        ]
    )


    ax_m4_r4 = fig_r4.add_subplot(
        grid_spec_r4[
            0,
            1,
        ],
        sharex=ax_m1_r4,
        sharey=ax_m1_r4,
    )


    colorbar_axis_r4 = fig_r4.add_subplot(
        grid_spec_r4[
            0,
            2,
        ]
    )


    axes_r4 = [
        ax_m1_r4,
        ax_m4_r4,
    ]


    panel_titles_r4 = {

        "M1":
            "M1 — single schedule",

        "M4":
            "M4 — four schedules",
    }


    contour_reference_r4 = None


    # =========================================================================
    # 17. DRAW M1 AND M4 USING IDENTICAL COLOR NORMALIZATION
    # =========================================================================

    for (
        axis_r4,
        design_id_r4,
        panel_label_r4,
    ) in zip(
        axes_r4,
        PROFILE_DESIGNS_R4,
        [
            "A",
            "B",
        ],
    ):

        frame_r4 = PROFILE_SURFACE_R4.loc[
            PROFILE_SURFACE_R4[
                "design_id"
            ] == design_id_r4
        ].copy()


        pivot_r4 = profile_pivots_r4[
            design_id_r4
        ]


        x_values_r4 = pivot_r4.columns.to_numpy(
            dtype=float
        )


        y_values_r4 = pivot_r4.index.to_numpy(
            dtype=float
        )


        z_values_r4 = profile_z_values_r4[
            design_id_r4
        ]


        # ---------------------------------------------------------------------
        # Filled objective contours
        # ---------------------------------------------------------------------

        contour_r4 = axis_r4.contourf(
            x_values_r4,
            y_values_r4,
            z_values_r4,
            levels=shared_contour_levels_r4,
            cmap="viridis",
            extend="both",
        )


        contour_reference_r4 = (
            contour_r4
        )


        # ---------------------------------------------------------------------
        # Thin contour overlays
        # ---------------------------------------------------------------------

        axis_r4.contour(
            x_values_r4,
            y_values_r4,
            z_values_r4,
            levels=shared_line_levels_r4,
            colors="black",
            linewidths=0.35,
            alpha=0.18,
        )


        # ---------------------------------------------------------------------
        # True-parameter reference lines
        # ---------------------------------------------------------------------

        axis_r4.axvline(
            TRUE_LOG10_MU_R4,
            linestyle=":",
            linewidth=1.0,
            color="black",
            alpha=0.48,
            zorder=3,
        )


        axis_r4.axhline(
            TRUE_COST_R4,
            linestyle=":",
            linewidth=1.0,
            color="black",
            alpha=0.48,
            zorder=3,
        )


        # ---------------------------------------------------------------------
        # Exact true parameter pair
        # ---------------------------------------------------------------------

        axis_r4.plot(
            TRUE_LOG10_MU_R4,
            TRUE_COST_R4,
            marker="*",
            markersize=13,
            markerfacecolor="white",
            markeredgecolor="black",
            markeredgewidth=1.2,
            linestyle="None",
            zorder=6,
        )


        # ---------------------------------------------------------------------
        # Best point available on finite 41 x 41 grid
        # ---------------------------------------------------------------------

        minimum_row_r4 = frame_r4.loc[
            frame_r4[
                "mse"
            ].idxmin()
        ]


        axis_r4.plot(
            float(
                minimum_row_r4[
                    "log10_mu"
                ]
            ),
            float(
                minimum_row_r4[
                    "cost"
                ]
            ),
            marker="X",
            markersize=8.5,
            color="tab:orange",
            linestyle="None",
            zorder=7,
        )


        # ---------------------------------------------------------------------
        # Titles and axis labels
        # ---------------------------------------------------------------------

        axis_r4.set_title(
            panel_titles_r4[
                design_id_r4
            ],
            loc="left",
            fontweight="semibold",
            pad=8,
        )


        axis_r4.set_xlabel(
            r"$\log_{10}(\mu)$"
        )


        axis_r4.set_xlim(
            -5.5,
            -2.5,
        )


        axis_r4.set_ylim(
            0.0,
            0.30,
        )


        axis_r4.xaxis.set_major_locator(
            MaxNLocator(
                nbins=7
            )
        )


        axis_r4.yaxis.set_major_locator(
            MaxNLocator(
                nbins=7
            )
        )


        axis_r4.grid(
            alpha=0.10,
            linewidth=0.6,
        )


        axis_r4.set_axisbelow(
            True
        )


        set_panel_frame_r4(
            axis_r4
        )


        # ---------------------------------------------------------------------
        # Panel labels
        # ---------------------------------------------------------------------

        axis_r4.text(
            -0.105,
            1.045,
            panel_label_r4,
            transform=axis_r4.transAxes,
            fontsize=12.8,
            fontweight="bold",
            ha="left",
            va="bottom",
        )


# =============================================================================
# 18. SHARED Y-AXIS FORMATTING
# =============================================================================

    ax_m1_r4.set_ylabel(
        r"Resistance fitness cost, $c_R$"
    )


    # Avoid duplicated labels because axes share the same y scale.
    ax_m4_r4.tick_params(
        axis="y",
        labelleft=False,
    )


# =============================================================================
# 19. SHARED COLORBAR
# =============================================================================
# Both panels use this SAME colorbar and SAME normalization.
# This should also be stated explicitly in the manuscript figure caption.
# =============================================================================

    colorbar_r4 = fig_r4.colorbar(
        contour_reference_r4,
        cax=colorbar_axis_r4,
    )


    colorbar_r4.set_label(
        r"$\log_{10}\!\left(\Delta\mathrm{MSE}+10^{-14}\right)$"
    )


    colorbar_r4.ax.tick_params(
        labelsize=9.5
    )


# =============================================================================
# 20. SHARED LEGEND OUTSIDE DATA REGION
# =============================================================================

    legend_handles_r4 = [

        Line2D(
            [0],
            [0],
            marker="*",
            linestyle="None",
            markersize=11,
            markerfacecolor="white",
            markeredgecolor="black",
            label="True parameter pair",
        ),

        Line2D(
            [0],
            [0],
            marker="X",
            linestyle="None",
            markersize=8,
            color="tab:orange",
            label="Finite-grid minimum",
        ),
    ]


    fig_r4.legend(
        handles=legend_handles_r4,
        loc="upper center",
        bbox_to_anchor=(
            0.50,
            0.972,
        ),
        ncol=2,
        frameon=False,
        fontsize=9.0,
        columnspacing=1.6,
        handletextpad=0.55,
    )


# =============================================================================
# 21. FINAL MANUSCRIPT TITLE
# =============================================================================
# No large in-figure title is used. The manuscript / Supplementary caption
# carries the overall interpretation, leaving more space for the data panels.


# =============================================================================
# 22. FINAL LAYOUT
# =============================================================================

    fig_r4.subplots_adjust(
        left=0.075,
        right=0.94,
        bottom=0.145,
        top=0.855,
    )


# =============================================================================
# 23. SAVE AND DISPLAY
# =============================================================================

    save_figure_r4(
        fig_r4,
        "round4_profile_surfaces_M1_vs_M4_final",
    )


    plt.show()


# =============================================================================
# 24. NUMERICAL MANUSCRIPT CHECK
# =============================================================================

print(
    "\nProfile-surface summary:"
)


display(
    PROFILE_SURFACE_SUMMARY_R4
)


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print(
    "\n"
    + "=" * 86
)


print(
    "ROUND 4 CELL 24 COMPLETED SUCCESSFULLY"
)


print(
    "=" * 86
)


if need_recompute_r4:

    print(
        "\nThe profile surfaces were recomputed."
    )

else:

    print(
        "\nThe existing profile grid was reused; "
        "the expensive ODE surfaces were NOT recomputed."
    )


print(
    "\nObjective comparison:"
)


print(
    "  M1 and M4 use residual-count-normalized MSE."
)


print(
    "  Both panels use exactly the same contour levels and color scale."
)


print(
    "  Delta MSE is referenced to the exact true parameter pair."
)


print(
    "\nScientific interpretation:"
)


print(
    "  Multi-schedule observations may sharpen the objective geometry, "
    "but a diagonal mu-c_R compensation direction can remain."
)


print(
    "  The figure should therefore be interpreted as evidence of improved, "
    "not necessarily complete, practical identifiability."
)


print(
    "\nFinal figure:"
)


print(
    " ",
    FIGURE_DIR_R4
    / "round4_profile_surfaces_M1_vs_M4_final",
)


print(
    "\nEnriched profile table:"
)


print(
    " ",
    PROFILE_SURFACE_PATH_R4,
)


print(
    "\nProfile summary table:"
)


print(
    " ",
    PROFILE_SURFACE_SUMMARY_PATH_R4,
)

In [ ]:
# CELL 25 — FINAL HEATMAP COLOR FIX
# EXACT VISUAL MATCH TO CELL 23C
# =============================================================================
# WHAT THIS BLOCK FIXES
# ---------------------
# Cell 23C uses:
#
#       cmap = "RdBu_r"
#
# with a symmetric signed scale around zero. However, the actual Cell 23
# observations occupy only part of that full color scale:
#
#       most negative observed effect  ~ -76%
#       most positive observed effect  ~ +187%
#       symmetric color limit          ~ ±200%
#
# Therefore, the visible Cell 23 palette does NOT extend to the darkest navy
# blue. Cell 25 previously normalized values directly from 0 to 1 over the
# entire RdBu_r palette, which incorrectly produced very dark blue cells.
#
# This block reconstructs the EXACT EFFECTIVE COLOR RANGE used by Cell 23C
# directly from the saved Cell 23 paired-change data.
#
# IMPORTANT
# ---------
# - Heatmap values remain unchanged.
# - Raw RMSE mean ± SD values remain unchanged.
# - Resistant-fraction RMSE is displayed in compact scientific notation.
# - No training is performed.
# - No transfer calculations are repeated.
# =============================================================================


# =============================================================================
# 1. EXTRA IMPORTS
# =============================================================================

from matplotlib.colors import (
    Normalize,
    TwoSlopeNorm,
    ListedColormap,
)


# =============================================================================
# 2. LOAD THE SAME CELL 23C DATA USED TO DEFINE ITS HEATMAP COLORS
# =============================================================================

CELL23_PAIRED_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_leave_one_schedule_out_paired_changes.csv"
)


# Defaults are provided only as a safety fallback.
cell23_low_color_position_r4 = 0.31
cell23_high_color_position_r4 = 0.97
cell23_color_limit_r4 = 200.0
cell23_min_effect_r4 = -76.0
cell23_max_effect_r4 = 187.0


if CELL23_PAIRED_PATH_R4.exists():

    cell23_paired_r4 = pd.read_csv(
        CELL23_PAIRED_PATH_R4
    )


    required_cell23_columns_r4 = [
        "configuration",
        "metric",
        "paired_change_percent",
    ]


    missing_cell23_columns_r4 = [
        column
        for column in required_cell23_columns_r4
        if column not in cell23_paired_r4.columns
    ]


    if not missing_cell23_columns_r4:

        cell23_metric_order_r4 = [
            "rmse_resistant",
            "rmse_fraction",
            "mu_error_percent",
            "cost_error_percent",
        ]


        cell23_configuration_order_r4 = [
            "NO_UNTREATED",
            "NO_EARLY",
            "NO_LATE",
            "NO_PULSED",
        ]


        # ---------------------------------------------------------------------
        # Reproduce the mean matrix from Cell 23C.
        # ---------------------------------------------------------------------

        cell23_mean_summary_r4 = (
            cell23_paired_r4
            .loc[
                cell23_paired_r4[
                    "configuration"
                ].isin(
                    cell23_configuration_order_r4
                )
                &
                cell23_paired_r4[
                    "metric"
                ].isin(
                    cell23_metric_order_r4
                )
            ]
            .groupby(
                [
                    "configuration",
                    "metric",
                ],
                observed=True,
            )[
                "paired_change_percent"
            ]
            .mean()
            .reset_index()
        )


        cell23_mean_matrix_r4 = np.full(
            (
                len(
                    cell23_configuration_order_r4
                ),
                len(
                    cell23_metric_order_r4
                ),
            ),
            np.nan,
            dtype=float,
        )


        for row_index_r4, configuration_r4 in enumerate(
            cell23_configuration_order_r4
        ):

            for column_index_r4, metric_r4 in enumerate(
                cell23_metric_order_r4
            ):

                subset_r4 = (
                    cell23_mean_summary_r4.loc[
                        (
                            cell23_mean_summary_r4[
                                "configuration"
                            ] == configuration_r4
                        )
                        &
                        (
                            cell23_mean_summary_r4[
                                "metric"
                            ] == metric_r4
                        ),
                        "paired_change_percent",
                    ]
                )


                if len(
                    subset_r4
                ) == 1:

                    cell23_mean_matrix_r4[
                        row_index_r4,
                        column_index_r4,
                    ] = float(
                        subset_r4.iloc[
                            0
                        ]
                    )


        # ---------------------------------------------------------------------
        # Reproduce Cell 23C's symmetric color limit.
        # ---------------------------------------------------------------------

        maximum_effect_r4 = float(
            np.nanmax(
                np.abs(
                    cell23_mean_matrix_r4
                )
            )
        )


        cell23_color_limit_r4 = max(
            50.0,
            np.ceil(
                maximum_effect_r4
                / 25.0
            )
            * 25.0,
        )


        cell23_min_effect_r4 = float(
            np.nanmin(
                cell23_mean_matrix_r4
            )
        )


        cell23_max_effect_r4 = float(
            np.nanmax(
                cell23_mean_matrix_r4
            )
        )


        # ---------------------------------------------------------------------
        # Reproduce the exact Cell 23C TwoSlopeNorm.
        # ---------------------------------------------------------------------

        cell23_norm_r4 = TwoSlopeNorm(
            vmin=-cell23_color_limit_r4,
            vcenter=0.0,
            vmax=cell23_color_limit_r4,
        )


        # ---------------------------------------------------------------------
        # Identify the ACTUAL part of RdBu_r visible in Cell 23C.
        #
        # For your current results this should be approximately:
        #
        #       low  ~ 0.31
        #       high ~ 0.97
        #
        # rather than the complete [0, 1] palette.
        # ---------------------------------------------------------------------

        cell23_low_color_position_r4 = float(
            cell23_norm_r4(
                cell23_min_effect_r4
            )
        )


        cell23_high_color_position_r4 = float(
            cell23_norm_r4(
                cell23_max_effect_r4
            )
        )


        print(
            "Cell 23C color range reproduced successfully."
        )


        print(
            "  Cell 23 symmetric color limit:",
            cell23_color_limit_r4,
        )


        print(
            "  Most negative Cell 23 mean:",
            f"{cell23_min_effect_r4:.1f}%",
        )


        print(
            "  Most positive Cell 23 mean:",
            f"{cell23_max_effect_r4:.1f}%",
        )


        print(
            "  Effective RdBu_r range:",
            f"{cell23_low_color_position_r4:.3f}",
            "to",
            f"{cell23_high_color_position_r4:.3f}",
        )


else:

    print(
        "Cell 23 paired-change CSV was not found; "
        "using the calibrated Cell 23C fallback palette."
    )


# =============================================================================
# 3. BUILD THE CELL-23-MATCHED COLORMAP
# =============================================================================
# IMPORTANT:
#
# Do NOT construct this with only four manually selected colors.
#
# Instead, densely sample the ORIGINAL RdBu_r colormap over exactly the
# portion that was visible in Cell 23C.
#
# This preserves the same smooth color progression as Cell 23.
# =============================================================================

base_cell23_cmap_r4 = plt.get_cmap(
    "RdBu_r"
)


cell23_color_samples_r4 = np.linspace(
    cell23_low_color_position_r4,
    cell23_high_color_position_r4,
    256,
)


CELL23_MATCHED_CMAP_R4 = ListedColormap(
    base_cell23_cmap_r4(
        cell23_color_samples_r4
    ),
    name="Cell23MatchedRdBu",
)


# =============================================================================
# 4. FINAL CELL 25 COLOR NORMALIZATION
# =============================================================================
# Cell 25 values are still:
#
#       0 = lowest relative error in that metric column
#       1 = largest relative error in that metric column
#
# But those values are now rendered using ONLY the color range actually visible
# in Cell 23C.
# =============================================================================

cell25_heatmap_norm_r4 = Normalize(
    vmin=0.0,
    vmax=1.0,
)


# =============================================================================
# 5. FINAL PUBLICATION FIGURE
# =============================================================================

with plt.rc_context(
    CELL25_PUBLICATION_RC_R4
):

    fig_r4, ax_r4 = plt.subplots(
        figsize=(
            11.2,
            5.35,
        )
    )


    # -------------------------------------------------------------------------
    # Heatmap
    # -------------------------------------------------------------------------

    image_r4 = ax_r4.imshow(
        normalized_matrix_r4,
        aspect="auto",
        cmap=CELL23_MATCHED_CMAP_R4,
        norm=cell25_heatmap_norm_r4,
    )


    # -------------------------------------------------------------------------
    # X-axis
    # -------------------------------------------------------------------------

    ax_r4.set_xticks(
        np.arange(
            len(
                transfer_metrics_r4
            )
        )
    )


    # Compact mathematical labels are easier to read after journal scaling.
    transfer_metric_labels_publication_r4 = {
        "rmse_sensitive":
            r"$\mathrm{RMSE}_{S}$",

        "rmse_resistant":
            r"$\mathrm{RMSE}_{R}$",

        "rmse_fraction":
            r"$\mathrm{RMSE}_{q_R}$",
    }

    ax_r4.set_xticklabels(
        [
            transfer_metric_labels_publication_r4.get(
                metric_r4,
                metric_display_r4[
                    metric_r4
                ],
            )
            for metric_r4
            in transfer_metrics_r4
        ]
    )


    # -------------------------------------------------------------------------
    # Y-axis
    # -------------------------------------------------------------------------

    ax_r4.set_yticks(
        np.arange(
            len(
                heldout_order_r4
            )
        )
    )


    ax_r4.set_yticklabels(
        [
            heldout_display_r4[
                schedule_r4
            ]
            for schedule_r4
            in heldout_order_r4
        ]
    )


    # -------------------------------------------------------------------------
    # Publication annotation formatter
    # -------------------------------------------------------------------------
    # State RMSE values retain the existing formatter. Resistant-fraction RMSE
    # is shown as (mean ± SD) x 10^k so very small values remain immediately
    # readable without long strings of leading zeros.
    def format_transfer_annotation_publication_r4(
        mean_value_r4,
        sd_value_r4,
        metric_r4,
    ):
        mean_value_r4 = float(
            mean_value_r4
        )

        sd_value_r4 = float(
            sd_value_r4
        )

        if (
            metric_r4 == "rmse_fraction"
            and mean_value_r4 != 0.0
        ):
            exponent_r4 = int(
                np.floor(
                    np.log10(
                        abs(
                            mean_value_r4
                        )
                    )
                )
            )

            scale_r4 = (
                10.0
                ** exponent_r4
            )

            mean_scaled_r4 = (
                mean_value_r4
                / scale_r4
            )

            sd_scaled_r4 = (
                sd_value_r4
                / scale_r4
            )

            return (
                rf"$({mean_scaled_r4:.3g}"
                rf"\,\pm\,{sd_scaled_r4:.3g})"
                rf"\times10^{{{exponent_r4}}}$"
            )

        return (
            f"{format_transfer_value_r4(mean_value_r4)} "
            f"± "
            f"{format_transfer_value_r4(sd_value_r4)}"
        )


    # -------------------------------------------------------------------------
    # Numerical annotations
    # -------------------------------------------------------------------------
    # Match Cell 23C:
    #
    # - NO background boxes.
    # - bold text directly on the heatmap.
    # - black on lighter colors.
    # - white only on sufficiently dark red regions.
    #
    # The synthetic visual effect below is ONLY used to reproduce Cell 23C's
    # text-color rule. It does not change any scientific value.
    # -------------------------------------------------------------------------

    for row_index_r4 in range(
        mean_matrix_r4.shape[
            0
        ]
    ):

        for column_index_r4 in range(
            mean_matrix_r4.shape[
                1
            ]
        ):

            mean_value_r4 = float(
                mean_matrix_r4[
                    row_index_r4,
                    column_index_r4,
                ]
            )


            sd_value_r4 = float(
                sd_matrix_r4[
                    row_index_r4,
                    column_index_r4,
                ]
            )


            normalized_value_r4 = float(
                normalized_matrix_r4[
                    row_index_r4,
                    column_index_r4,
                ]
            )


            # -----------------------------------------------------------------
            # Map Cell 25 normalized error onto the EFFECT RANGE actually seen
            # in Cell 23C. This is used only to choose black/white text.
            # -----------------------------------------------------------------

            cell23_equivalent_effect_r4 = (
                cell23_min_effect_r4
                +
                normalized_value_r4
                * (
                    cell23_max_effect_r4
                    - cell23_min_effect_r4
                )
            )


            # Same logic used by Cell 23C.
            text_color_r4 = (
                "white"
                if abs(
                    cell23_equivalent_effect_r4
                )
                >= (
                    0.48
                    * cell23_color_limit_r4
                )
                else "black"
            )


            metric_r4 = (
                transfer_metrics_r4[
                    column_index_r4
                ]
            )

            annotation_r4 = (
                format_transfer_annotation_publication_r4(
                    mean_value_r4,
                    sd_value_r4,
                    metric_r4,
                )
            )


            ax_r4.text(
                column_index_r4,
                row_index_r4,
                annotation_r4,

                ha="center",
                va="center",

                fontsize=10.5,
                fontweight="bold",

                color=text_color_r4,
            )


    # -------------------------------------------------------------------------
    # Thin cell boundaries — same style as Cell 23C
    # -------------------------------------------------------------------------

    ax_r4.set_xticks(
        np.arange(
            -0.5,
            len(
                transfer_metrics_r4
            ),
            1.0,
        ),
        minor=True,
    )


    ax_r4.set_yticks(
        np.arange(
            -0.5,
            len(
                heldout_order_r4
            ),
            1.0,
        ),
        minor=True,
    )


    ax_r4.grid(
        which="minor",
        linewidth=1.0,
        alpha=0.45,
    )


    ax_r4.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )


    # -------------------------------------------------------------------------
    # Colorbar
    # -------------------------------------------------------------------------

    colorbar_r4 = fig_r4.colorbar(
        image_r4,
        ax=ax_r4,
        pad=0.025,
        fraction=0.055,
    )


    colorbar_r4.set_label(
        "Within-metric normalized error"
    )


    colorbar_r4.set_ticks(
        [
            0.0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0,
        ]
    )


    # -------------------------------------------------------------------------
    # Labels
    # -------------------------------------------------------------------------

    ax_r4.set_ylabel(
        "Held-out treatment environment"
    )


    # The three metric labels already identify the evaluation quantities.
    ax_r4.set_xlabel("")


    # No large in-figure title is used in the final manuscript version.
    # The overall transfer interpretation belongs in the figure caption.


    # -------------------------------------------------------------------------
    # Frame — identical style to Cell 23C
    # -------------------------------------------------------------------------

    for spine_r4 in ax_r4.spines.values():

        spine_r4.set_visible(
            True
        )


        spine_r4.set_color(
            "black"
        )


        spine_r4.set_linewidth(
            1.0
        )


    # -------------------------------------------------------------------------
    # Final layout
    # -------------------------------------------------------------------------

    fig_r4.subplots_adjust(
        left=0.17,
        right=0.91,
        bottom=0.13,
        top=0.97,
    )


    # -------------------------------------------------------------------------
    # Save final publication figure
    # -------------------------------------------------------------------------

    save_figure_r4(
        fig_r4,
        "round4_heldout_schedule_parameter_transfer_final",
    )


    plt.show()

In [ ]:
# ROUND 4 — CELL 26
# FINAL PUBLICATION VERSION
# Optimization histories, convergence diagnostics, and CUDA-memory usage
# =============================================================================
# WHAT THIS CELL IS ABOUT
# -----------------------
# This cell summarizes the optimization behavior of one matched completed run
# per nested schedule design (M1-M5).
#
# The three panels show:
#
#   A. Validation convergence
#   B. Physics-residual convergence
#   C. CUDA reserved-memory history
#
# IMPORTANT SCIENTIFIC/PLOTTING DETAILS
# -------------------------------------
# 1. Histories are loaded from disk, so this cell can be executed after a
#    kernel restart.
#
# 2. The lowest-numbered paired seed is used for every design. This provides
#    a matched-seed convergence comparison rather than choosing the visually
#    best run.
#
# 3. Zero/non-positive physics values are NOT clipped to 1e-16.
#    They are replaced with NaN instead. Therefore, periods during which the
#    physics residual was not evaluated do not appear as artificially perfect
#    physics satisfaction.
#
# 4. Training-phase transitions are detected from:
#
#       stage / phase / training_stage / optimizer_stage
#
#    when available. If no explicit phase column exists, epoch resets are used
#    as a fallback.
#
# 5. The GPU panel reports CUDA RESERVED memory recorded at logging points.
#    It should not be interpreted as total GPU capacity or instantaneous peak
#    device utilization.
#
# 6. The legend is placed below the main title and outside all data panels.
#
# NO MODEL TRAINING IS PERFORMED IN THIS CELL.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter


# =============================================================================
# 2. LOAD PRIMARY RESULTS IF THE KERNEL WAS RESTARTED
# =============================================================================

PRIMARY_RESULTS_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_primary_multischedule_all_seeds_metrics.csv"
)


if (
    "ROUND4_PRIMARY_ALL_RUNS" not in globals()
    or ROUND4_PRIMARY_ALL_RUNS is None
    or ROUND4_PRIMARY_ALL_RUNS.empty
):

    if PRIMARY_RESULTS_PATH_R4.exists():

        ROUND4_PRIMARY_ALL_RUNS = pd.read_csv(
            PRIMARY_RESULTS_PATH_R4
        )

        print(
            "Loaded completed Round 4 primary results:"
        )

        print(
            " ",
            PRIMARY_RESULTS_PATH_R4,
        )

    else:

        raise FileNotFoundError(
            "The completed primary Round 4 results were not found:\n"
            f"{PRIMARY_RESULTS_PATH_R4}\n\n"
            "Complete the primary experiment before generating "
            "optimization diagnostics."
        )


# =============================================================================
# 3. VALIDATE PRIMARY RESULT COLUMNS
# =============================================================================

required_primary_columns_r4 = [
    "design_id",
    "label",
]


missing_primary_columns_r4 = [
    column
    for column in required_primary_columns_r4
    if column not in ROUND4_PRIMARY_ALL_RUNS.columns
]


if missing_primary_columns_r4:

    raise KeyError(
        "Primary results are missing required columns: "
        f"{missing_primary_columns_r4}"
    )


# =============================================================================
# 4. DESIGN ORDER
# =============================================================================

if "DESIGN_ORDER_R4" not in globals():

    DESIGN_ORDER_R4 = [
        "M1",
        "M2",
        "M3",
        "M4",
        "M5",
    ]


# =============================================================================
# 5. CONSISTENT DESIGN COLORS
# =============================================================================
# The same design uses the same color in Panels A-C.
# =============================================================================

default_colors_r4 = plt.rcParams[
    "axes.prop_cycle"
].by_key()[
    "color"
]


DESIGN_COLORS_R4 = {
    design_id_r4:
        default_colors_r4[
            index_r4
            % len(
                default_colors_r4
            )
        ]
    for index_r4, design_id_r4
    in enumerate(
        DESIGN_ORDER_R4
    )
}


# =============================================================================
# 6. LOAD ONE MATCHED HISTORY PER DESIGN
# =============================================================================
# We use the lowest-numbered seed for every design.
#
# This is preferable to selecting the run with the best validation score,
# because it avoids visual cherry-picking of convergence behavior.
# =============================================================================

history_frames_r4 = []

selected_history_rows_r4 = []


for design_id_r4 in DESIGN_ORDER_R4:

    matching_rows_r4 = (
        ROUND4_PRIMARY_ALL_RUNS.loc[
            ROUND4_PRIMARY_ALL_RUNS[
                "design_id"
            ].astype(str)
            == str(
                design_id_r4
            )
        ]
        .copy()
    )


    if matching_rows_r4.empty:

        print(
            f"Warning: no primary run found for {design_id_r4}."
        )

        continue


    # -------------------------------------------------------------------------
    # Select the same seed ordering rule for every design.
    # -------------------------------------------------------------------------

    if "seed_number" in matching_rows_r4.columns:

        matching_rows_r4 = (
            matching_rows_r4
            .sort_values(
                "seed_number"
            )
        )

    elif "seed_id" in matching_rows_r4.columns:

        matching_rows_r4 = (
            matching_rows_r4
            .sort_values(
                "seed_id"
            )
        )


    selected_row_r4 = (
        matching_rows_r4
        .iloc[0]
    )


    selected_label_r4 = str(
        selected_row_r4[
            "label"
        ]
    )


    history_path_r4 = (
        HISTORY_DIR_R4
        / f"{selected_label_r4}_history.csv"
    )


    if not history_path_r4.exists():

        print(
            f"Warning: history file not found for {design_id_r4}:"
        )

        print(
            " ",
            history_path_r4,
        )

        continue


    history_r4 = pd.read_csv(
        history_path_r4
    )


    history_r4[
        "design_id"
    ] = design_id_r4


    history_r4[
        "selected_label"
    ] = selected_label_r4


    history_r4[
        "history_index"
    ] = np.arange(
        len(
            history_r4
        ),
        dtype=int,
    )


    history_frames_r4.append(
        history_r4
    )


    selected_history_rows_r4.append(
        {
            "design_id":
                design_id_r4,

            "label":
                selected_label_r4,

            "history_file":
                str(
                    history_path_r4
                ),

            "n_logged_points":
                len(
                    history_r4
                ),

            "seed_number":
                (
                    selected_row_r4[
                        "seed_number"
                    ]
                    if "seed_number"
                    in selected_row_r4.index
                    else np.nan
                ),

            "seed_id":
                (
                    selected_row_r4[
                        "seed_id"
                    ]
                    if "seed_id"
                    in selected_row_r4.index
                    else ""
                ),
        }
    )


# =============================================================================
# 7. STOP IF NO HISTORIES WERE FOUND
# =============================================================================

if not history_frames_r4:

    raise FileNotFoundError(
        "No primary history files were found in:\n"
        f"{HISTORY_DIR_R4}"
    )


histories_r4 = pd.concat(
    history_frames_r4,
    ignore_index=True,
)


SELECTED_HISTORY_RUNS_R4 = pd.DataFrame(
    selected_history_rows_r4
)


print(
    "\nMatched convergence histories:"
)


display(
    SELECTED_HISTORY_RUNS_R4
)


# =============================================================================
# 8. VALIDATE HISTORY METRICS
# =============================================================================

required_history_columns_r4 = [
    "validation_score",
    "physics",
    "gpu_reserved_gb",
]


missing_history_columns_r4 = [
    column
    for column in required_history_columns_r4
    if column not in histories_r4.columns
]


if missing_history_columns_r4:

    raise KeyError(
        "History files are missing required diagnostic columns: "
        f"{missing_history_columns_r4}"
    )


# Convert required quantities safely to numeric.
for column_r4 in required_history_columns_r4:

    histories_r4[
        column_r4
    ] = pd.to_numeric(
        histories_r4[
            column_r4
        ],
        errors="coerce",
    )


# =============================================================================
# 9. CLEAN VALIDATION AND PHYSICS HISTORIES
# =============================================================================
# CRITICAL FIX:
#
# Previously:
#
#     physics.clip(lower=1e-16)
#
# converted zero or unavailable physics values into an apparently perfect
# residual of 1e-16.
#
# That is misleading on a logarithmic axis.
#
# Instead:
#
#     physics <= 0  -> NaN
#
# Matplotlib then leaves those portions blank.
# =============================================================================

histories_r4[
    "validation_plot"
] = histories_r4[
    "validation_score"
].where(
    np.isfinite(
        histories_r4[
            "validation_score"
        ]
    )
    &
    (
        histories_r4[
            "validation_score"
        ]
        > 0
    ),
    np.nan,
)


histories_r4[
    "physics_plot"
] = histories_r4[
    "physics"
].where(
    np.isfinite(
        histories_r4[
            "physics"
        ]
    )
    &
    (
        histories_r4[
            "physics"
        ]
        > 0
    ),
    np.nan,
)


histories_r4[
    "gpu_reserved_plot"
] = histories_r4[
    "gpu_reserved_gb"
].where(
    np.isfinite(
        histories_r4[
            "gpu_reserved_gb"
        ]
    ),
    np.nan,
)


# =============================================================================
# 10. DETECT TRAINING-PHASE BOUNDARIES
# =============================================================================
# Priority:
#
#   A. Explicit stage/phase column
#   B. Epoch reset locations
#   C. No phase markers if neither is available
#
# We derive the displayed boundaries from the first available matched history.
# =============================================================================

PHASE_COLUMN_CANDIDATES_R4 = [
    "stage",
    "phase",
    "training_stage",
    "optimizer_stage",
]


phase_column_r4 = next(
    (
        column
        for column in PHASE_COLUMN_CANDIDATES_R4
        if column in histories_r4.columns
    ),
    None,
)


phase_boundaries_r4 = []

phase_names_r4 = []


# Use M1 if available, otherwise the first design found.
reference_design_r4 = next(
    (
        design_id
        for design_id in DESIGN_ORDER_R4
        if design_id
        in histories_r4[
            "design_id"
        ].unique()
    ),
    histories_r4[
        "design_id"
    ].iloc[0],
)


reference_history_r4 = (
    histories_r4.loc[
        histories_r4[
            "design_id"
        ]
        == reference_design_r4
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# -----------------------------------------------------------------------------
# Option A: explicit phase/stage names
# -----------------------------------------------------------------------------

if phase_column_r4 is not None:

    phases_r4 = (
        reference_history_r4[
            phase_column_r4
        ]
        .astype(str)
        .fillna(
            ""
        )
        .tolist()
    )


    change_indices_r4 = [
        index_r4
        for index_r4 in range(
            1,
            len(
                phases_r4
            ),
        )
        if phases_r4[
            index_r4
        ]
        != phases_r4[
            index_r4 - 1
        ]
    ]


    phase_boundaries_r4 = (
        change_indices_r4
    )


    if len(
        phases_r4
    ) > 0:

        phase_names_r4 = [
            phases_r4[
                0
            ]
        ]


        for change_index_r4 in change_indices_r4:

            phase_names_r4.append(
                phases_r4[
                    change_index_r4
                ]
            )


# -----------------------------------------------------------------------------
# Option B: detect epoch resets
# -----------------------------------------------------------------------------

elif (
    "epoch"
    in reference_history_r4.columns
):

    epoch_values_r4 = pd.to_numeric(
        reference_history_r4[
            "epoch"
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    valid_epoch_mask_r4 = np.isfinite(
        epoch_values_r4
    )


    if np.sum(
        valid_epoch_mask_r4
    ) >= 2:

        epoch_difference_r4 = np.diff(
            epoch_values_r4
        )


        phase_boundaries_r4 = (
            np.where(
                epoch_difference_r4
                < 0
            )[0]
            + 1
        ).tolist()


        if len(
            phase_boundaries_r4
        ) == 2:

            phase_names_r4 = [
                "Pretraining",
                "Physics warm-up",
                "Joint optimization",
            ]

        elif len(
            phase_boundaries_r4
        ) > 0:

            phase_names_r4 = [
                f"Phase {index_r4 + 1}"
                for index_r4 in range(
                    len(
                        phase_boundaries_r4
                    )
                    + 1
                )
            ]


# =============================================================================
# 11. CLEAN PHASE NAMES FOR DISPLAY
# =============================================================================

def clean_phase_name_r4(name):
    """
    Convert internal stage identifiers into publication-friendly labels.
    """

    text = str(
        name
    ).strip()


    text = text.replace(
        "_",
        " "
    )


    lower_text = text.lower()


    if "pretrain" in lower_text:

        return "Pretraining"


    if (
        "warm" in lower_text
        and "phys" in lower_text
    ):

        return "Physics warm-up"


    if (
        "joint" in lower_text
        or "adam" in lower_text
    ):

        return "Joint optimization"


    if "lbfgs" in lower_text:

        return "L-BFGS"


    return text.title()


phase_names_r4 = [
    clean_phase_name_r4(
        name
    )
    for name in phase_names_r4
]


# =============================================================================
# 12. PUBLICATION STYLE
# =============================================================================

CELL26_PUBLICATION_RC_R4 = {

    "font.family":
        "sans-serif",

    "font.size":
        10.5,

    "axes.titlesize":
        12.5,

    "axes.labelsize":
        11.0,

    "xtick.labelsize":
        9.5,

    "ytick.labelsize":
        9.5,

    "legend.fontsize":
        9.5,

    "axes.linewidth":
        1.0,

    "axes.edgecolor":
        "black",

    "axes.facecolor":
        "white",

    "figure.facecolor":
        "white",

    "savefig.facecolor":
        "white",
}


# =============================================================================
# 13. CREATE FINAL THREE-PANEL FIGURE
# =============================================================================

with plt.rc_context(
    CELL26_PUBLICATION_RC_R4
):

    fig_r4, axes_r4 = plt.subplots(
        1,
        3,
        figsize=(
            14.2,
            5.2,
        ),
    )


    # =========================================================================
    # 14. PLOT EACH DESIGN
    # =========================================================================

    for design_id_r4 in DESIGN_ORDER_R4:

        frame_r4 = (
            histories_r4.loc[
                histories_r4[
                    "design_id"
                ]
                == design_id_r4
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        if frame_r4.empty:
            continue


        x_r4 = np.arange(
            len(
                frame_r4
            ),
            dtype=float,
        )


        color_r4 = DESIGN_COLORS_R4[
            design_id_r4
        ]


        # ---------------------------------------------------------------------
        # Panel A — validation convergence
        # ---------------------------------------------------------------------

        axes_r4[
            0
        ].plot(
            x_r4,
            frame_r4[
                "validation_plot"
            ],

            color=color_r4,

            marker="o",
            markersize=2.8,

            linewidth=1.6,

            label=design_id_r4,
        )


        # ---------------------------------------------------------------------
        # Panel B — physics residual
        # ---------------------------------------------------------------------

        axes_r4[
            1
        ].plot(
            x_r4,
            frame_r4[
                "physics_plot"
            ],

            color=color_r4,

            marker="o",
            markersize=2.8,

            linewidth=1.6,

            label=design_id_r4,
        )


        # ---------------------------------------------------------------------
        # Panel C — CUDA reserved memory
        # ---------------------------------------------------------------------

        axes_r4[
            2
        ].plot(
            x_r4,
            frame_r4[
                "gpu_reserved_plot"
            ],

            color=color_r4,

            marker="o",
            markersize=2.8,

            linewidth=1.6,

            label=design_id_r4,
        )


    # =========================================================================
    # 15. LOGARITHMIC Y-SCALES
    # =========================================================================

    axes_r4[
        0
    ].set_yscale(
        "log"
    )


    axes_r4[
        1
    ].set_yscale(
        "log"
    )


    # Minor log ticks improve readability without excessive labels.
    for axis_r4 in axes_r4[
        :2
    ]:

        axis_r4.yaxis.set_minor_locator(
            LogLocator(
                base=10.0,
                subs=np.arange(
                    2,
                    10,
                )
                * 0.1,
            )
        )


        axis_r4.yaxis.set_minor_formatter(
            NullFormatter()
        )


    # =========================================================================
    # 16. TRAINING-PHASE BOUNDARIES
    # =========================================================================
    # Draw phase changes on Panels A and B only.
    #
    # The GPU panel remains visually simpler because it is a resource diagnostic.
    # =========================================================================

    for boundary_r4 in phase_boundaries_r4:

        for axis_r4 in axes_r4[
            :2
        ]:

            axis_r4.axvline(
                boundary_r4
                - 0.5,

                color="0.35",

                linestyle="--",

                linewidth=0.85,

                alpha=0.55,

                zorder=0,
            )



    # =========================================================================
    # 17. PANEL TITLES AND LABELS
    # =========================================================================

    axes_r4[
        0
    ].set_title(
        "Validation convergence",
        loc="left",
        fontsize=12.5,
        fontweight="semibold",
        pad=8,
    )


    axes_r4[
        0
    ].set_ylabel(
        "Validation score"
    )


    axes_r4[
        1
    ].set_title(
        "Physics-residual convergence",
        loc="left",
        fontweight="semibold",
        pad=8,
    )


    axes_r4[
        1
    ].set_ylabel(
        "Physics-residual loss"
    )


    axes_r4[
        2
    ].set_title(
        "CUDA memory reservation",
        loc="left",
        fontweight="semibold",
        pad=8,
    )


    axes_r4[
        2
    ].set_ylabel(
        "Reserved CUDA memory (GB)"
    )


    axes_r4[
        2
    ].set_ylim(
        bottom=0.0
    )


    # =========================================================================
    # 18. COMMON AXIS FORMATTING
    # =========================================================================

    for (
        panel_label_r4,
        axis_r4,
    ) in zip(
        [
            "A",
            "B",
            "C",
        ],
        axes_r4,
    ):

        axis_r4.set_xlabel(
            "Logged optimization step"
        )


        # Light publication-style grid.
        axis_r4.grid(
            which="major",
            alpha=0.16,
            linewidth=0.7,
        )


        if axis_r4 in axes_r4[
            :2
        ]:

            axis_r4.grid(
                which="minor",
                alpha=0.055,
                linewidth=0.45,
            )


        axis_r4.set_axisbelow(
            True
        )


        set_panel_frame_r4(
            axis_r4
        )


        axis_r4.text(
            -0.13,
            1.055,
            panel_label_r4,

            transform=axis_r4.transAxes,

            fontsize=12.5,
            fontweight="bold",

            ha="left",
            va="bottom",
        )


    # =========================================================================
    # 19. CLEAN SHARED LEGEND
    # =========================================================================

    handles_r4, labels_r4 = (
        axes_r4[
            0
        ]
        .get_legend_handles_labels()
    )


    # Remove accidental duplicates while preserving order.
    legend_mapping_r4 = {}


    for handle_r4, label_r4 in zip(
        handles_r4,
        labels_r4,
    ):

        if label_r4 not in legend_mapping_r4:

            legend_mapping_r4[
                label_r4
            ] = handle_r4


    fig_r4.legend(
        list(
            legend_mapping_r4.values()
        ),

        list(
            legend_mapping_r4.keys()
        ),

        loc="upper center",

        bbox_to_anchor=(
            0.50,
            0.915,
        ),

        ncol=len(
            legend_mapping_r4
        ),

        frameon=False,

        handlelength=2.0,

        columnspacing=1.5,
    )


    # =========================================================================
    # 20. TRAINING-PHASE KEY
    # =========================================================================
    # Phase names are intentionally kept OUTSIDE Panel A. The vertical dashed
    # lines in Panels A and B still show the transition locations, while this
    # compact key explains the phase sequence without obscuring the data.
    # =========================================================================

    if phase_names_r4:

        phase_key_r4 = (
            "Training phases:   "
            + "   |   ".join(
                phase_names_r4
            )
        )

        fig_r4.text(
            0.50,
            0.835,
            phase_key_r4,
            ha="center",
            va="center",
            fontsize=8.6,
            color="0.30",
        )


    # =========================================================================
    # 21. MAIN TITLE
    # =========================================================================

    fig_r4.suptitle(
        "Optimization diagnostics across nested schedule designs",
        fontsize=15.0,
        fontweight="bold",
        y=0.985,
    )


    # =========================================================================
    # 22. FINAL LAYOUT
    # =========================================================================
    # Large top margin separates:
    #
    #       title
    #       legend
    #       panel titles
    #
    # and prevents the overlap present in the original figure.
    # =========================================================================

    fig_r4.subplots_adjust(
        left=0.065,
        right=0.985,
        bottom=0.15,
        top=0.70,
        wspace=0.30,
    )


    # =========================================================================
    # 23. SAVE AND DISPLAY
    # =========================================================================

    save_figure_r4(
        fig_r4,
        "round4_convergence_and_gpu_memory_final",
    )


    plt.show()


# =============================================================================
# 24. CREATE NUMERICAL DIAGNOSTIC SUMMARY
# =============================================================================

diagnostic_summary_rows_r4 = []


for design_id_r4 in DESIGN_ORDER_R4:

    frame_r4 = (
        histories_r4.loc[
            histories_r4[
                "design_id"
            ]
            == design_id_r4
        ]
        .copy()
    )


    if frame_r4.empty:
        continue


    valid_validation_r4 = frame_r4[
        "validation_plot"
    ].dropna()


    valid_physics_r4 = frame_r4[
        "physics_plot"
    ].dropna()


    valid_memory_r4 = frame_r4[
        "gpu_reserved_plot"
    ].dropna()


    diagnostic_summary_rows_r4.append(
        {
            "design_id":
                design_id_r4,

            "logged_points":
                int(
                    len(
                        frame_r4
                    )
                ),

            "minimum_validation_score":
                (
                    float(
                        valid_validation_r4.min()
                    )
                    if not valid_validation_r4.empty
                    else np.nan
                ),

            "final_validation_score":
                (
                    float(
                        valid_validation_r4.iloc[
                            -1
                        ]
                    )
                    if not valid_validation_r4.empty
                    else np.nan
                ),

            "minimum_physics_residual":
                (
                    float(
                        valid_physics_r4.min()
                    )
                    if not valid_physics_r4.empty
                    else np.nan
                ),

            "final_physics_residual":
                (
                    float(
                        valid_physics_r4.iloc[
                            -1
                        ]
                    )
                    if not valid_physics_r4.empty
                    else np.nan
                ),

            "maximum_logged_cuda_reserved_gb":
                (
                    float(
                        valid_memory_r4.max()
                    )
                    if not valid_memory_r4.empty
                    else np.nan
                ),
        }
    )


ROUND4_OPTIMIZATION_DIAGNOSTICS_R4 = pd.DataFrame(
    diagnostic_summary_rows_r4
)


OPTIMIZATION_DIAGNOSTICS_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_optimization_diagnostics_summary.csv"
)


ROUND4_OPTIMIZATION_DIAGNOSTICS_R4.to_csv(
    OPTIMIZATION_DIAGNOSTICS_PATH_R4,
    index=False,
)


print(
    "\nOptimization-diagnostic summary:"
)


display(
    ROUND4_OPTIMIZATION_DIAGNOSTICS_R4
)


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print(
    "\n"
    + "=" * 86
)


print(
    "ROUND 4 CELL 26 COMPLETED SUCCESSFULLY"
)


print(
    "=" * 86
)


print(
    "\nNO MODEL TRAINING WAS PERFORMED."
)


print(
    "\nMatched designs plotted:"
)


print(
    " ",
    [
        design_id_r4
        for design_id_r4
        in DESIGN_ORDER_R4
        if design_id_r4
        in histories_r4[
            "design_id"
        ].unique()
    ],
)


if phase_boundaries_r4:

    print(
        "\nDetected optimization phase boundaries:"
    )


    print(
        " ",
        phase_boundaries_r4,
    )


    if phase_names_r4:

        print(
            "Phase labels:"
        )

        print(
            " ",
            phase_names_r4,
        )

else:

    print(
        "\nNo explicit optimization phase boundaries were available "
        "in the saved history files."
    )


print(
    "\nPhysics-history correction:"
)


print(
    "  Non-positive/unavailable physics values were omitted rather than "
    "being clipped to 1e-16."
)


print(
    "\nGPU-memory interpretation:"
)


print(
    "  The figure reports CUDA reserved memory recorded at history log points; "
    "it is not a measure of total GPU capacity or instantaneous peak usage."
)


print(
    "\nSaved final figure:"
)


print(
    " ",
    FIGURE_DIR_R4
    / "round4_convergence_and_gpu_memory_final",
)


print(
    "\nSaved diagnostic summary:"
)


print(
    " ",
    OPTIMIZATION_DIAGNOSTICS_PATH_R4,
)

In [ ]:
# =============================================================================
# ROUND 4 — CELL 27
# FINAL MANUSCRIPT REPRODUCIBILITY AND HARD-CONSTRAINT AUDIT
# =============================================================================
# PURPOSE
# -------
# This is the final integrity/reproducibility check for Round 4.
#
# It does NOT train or optimize any model.
#
# It:
#   1. Reloads the completed 25-run primary table if necessary.
#   2. Identifies the exact primary M1-M5 runs.
#   3. Verifies that every expected all-schedule prediction file exists.
#   4. Audits every saved primary trajectory for:
#
#          S(t) >= 0
#         chedule prediction file exists.
#   4. Audits every saved primary trajectory for:
#
#          R(t) >= 0
#          S(t) + R(t) <= K
#
#      and, when t = T_START is present:
#
#          S(T_START) = INITIAL_S
#          R(T_START) = INITIAL_R
#
#   5. Saves:
#
#          round4_constraint_audit.csv
#          round4_manifest.json
#          round4_file_inventory.txt
#
# IMPORTANT
# ---------
# This final version intentionally does NOT use:
#
#     ROUND4_SMOKE_RESULT
#     ROUND4_DEMO_RESULT
#
# because those objects do not exist when the smoke/demo cells are skipped
# during the manuscript-scale final run.
#
# NO MODEL TRAINING IS PERFORMED.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path
from dataclasses import asdict, is_dataclass
import json
import math
import platform

import numpy as np
import pandas as pd


# =============================================================================
# 2. FINAL OUTPUT PATHS
# =============================================================================

PRIMARY_RESULTS_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_primary_multischedule_all_seeds_metrics.csv"
)

CONSTRAINT_AUDIT_PATH_R4 = (
    TABLE_DIR_R4
    / "round4_constraint_audit.csv"
)

MANIFEST_PATH_R4 = (
    REPORT_DIR_R4
    / "round4_manifest.json"
)

INVENTORY_PATH_R4 = (
    REPORT_DIR_R4
    / "round4_file_inventory.txt"
)


# =============================================================================
# 3. ENSURE OUTPUT DIRECTORIES EXIST
# =============================================================================

for directory_r4 in [
    TABLE_DIR_R4,
    REPORT_DIR_R4,
]:
    directory_r4.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# 4. JSON-SAFE CONVERSION HELPER
# =============================================================================

def json_safe_r4(value):
    """
    Recursively convert common Python, NumPy, Path and dataclass objects
    to JSON-serializable forms.
    """

    if is_dataclass(value):
        return json_safe_r4(
            asdict(value)
        )

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): json_safe_r4(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            json_safe_r4(item)
            for item in value
        ]

    if isinstance(value, float):
        if not math.isfinite(value):
            return None

    return value


# =============================================================================
# 5. LOAD THE COMPLETED PRIMARY RESULTS TABLE
# =============================================================================
# This makes Cell 27 safe after a kernel restart.
# =============================================================================

if (
    "ROUND4_PRIMARY_ALL_RUNS" not in globals()
    or ROUND4_PRIMARY_ALL_RUNS is None
    or ROUND4_PRIMARY_ALL_RUNS.empty
):

    if not PRIMARY_RESULTS_PATH_R4.exists():

        raise FileNotFoundError(
            "The completed Round 4 primary-results table was not found:\n"
            f"{PRIMARY_RESULTS_PATH_R4}\n\n"
            "Cell 27 requires the completed primary experiment."
        )

    ROUND4_PRIMARY_ALL_RUNS = pd.read_csv(
        PRIMARY_RESULTS_PATH_R4
    )

    print(
        "Loaded completed primary Round 4 results:"
    )
    print(
        " ",
        PRIMARY_RESULTS_PATH_R4
    )


# =============================================================================
# 6. IDENTIFY ONLY THE FINAL PRIMARY M1-M5 RUNS
# =============================================================================

if "DESIGN_ORDER_R4" not in globals():
    DESIGN_ORDER_R4 = [
        "M1",
        "M2",
        "M3",
        "M4",
        "M5",
    ]


required_primary_columns_r4 = [
    "label",
    "design_id",
]


missing_primary_columns_r4 = [
    column_r4
    for column_r4 in required_primary_columns_r4
    if column_r4 not in ROUND4_PRIMARY_ALL_RUNS.columns
]


if missing_primary_columns_r4:

    raise KeyError(
        "Primary results are missing required columns: "
        f"{missing_primary_columns_r4}"
    )


PRIMARY_AUDIT_RUNS_R4 = (
    ROUND4_PRIMARY_ALL_RUNS.loc[
        ROUND4_PRIMARY_ALL_RUNS[
            "design_id"
        ].astype(str).isin(
            DESIGN_ORDER_R4
        )
    ]
    .copy()
)


# Keep only the design-level rows if that column exists.
if "row_type" in PRIMARY_AUDIT_RUNS_R4.columns:

    design_rows_r4 = (
        PRIMARY_AUDIT_RUNS_R4[
            "row_type"
        ].astype(str)
        == "design"
    )

    if design_rows_r4.any():
        PRIMARY_AUDIT_RUNS_R4 = (
            PRIMARY_AUDIT_RUNS_R4.loc[
                design_rows_r4
            ]
            .copy()
        )


# Keep only ALL-schedule summary rows if schedule exists.
if "schedule" in PRIMARY_AUDIT_RUNS_R4.columns:

    all_rows_r4 = (
        PRIMARY_AUDIT_RUNS_R4[
            "schedule"
        ].astype(str)
        == "ALL"
    )

    if all_rows_r4.any():
        PRIMARY_AUDIT_RUNS_R4 = (
            PRIMARY_AUDIT_RUNS_R4.loc[
                all_rows_r4
            ]
            .copy()
        )


# Remove accidental duplicates.
PRIMARY_AUDIT_RUNS_R4 = (
    PRIMARY_AUDIT_RUNS_R4
    .drop_duplicates(
        subset=[
            "label"
        ]
    )
)


# Sort cleanly.
sort_columns_r4 = [
    column_r4
    for column_r4 in [
        "design_id",
        "seed_number",
        "seed_id",
    ]
    if column_r4
    in PRIMARY_AUDIT_RUNS_R4.columns
]


if sort_columns_r4:

    PRIMARY_AUDIT_RUNS_R4 = (
        PRIMARY_AUDIT_RUNS_R4
        .sort_values(
            sort_columns_r4
        )
        .reset_index(
            drop=True
        )
    )


EXPECTED_PRIMARY_LABELS_R4 = (
    PRIMARY_AUDIT_RUNS_R4[
        "label"
    ]
    .astype(str)
    .tolist()
)


print(
    "\nPrimary manuscript runs identified:",
    len(EXPECTED_PRIMARY_LABELS_R4),
)


# For your final experiment this should be 25.
if not QUICK_MODE:

    if len(EXPECTED_PRIMARY_LABELS_R4) != 25:

        print(
            "WARNING:"
        )
        print(
            "  Expected 25 final primary runs "
            "(5 designs x 5 seeds), but found",
            len(EXPECTED_PRIMARY_LABELS_R4),
        )


# =============================================================================
# 7. BUILD THE EXPECTED PRIMARY PREDICTION FILE LIST
# =============================================================================
# IMPORTANT:
#
# We construct each path directly from the primary-run labels.
# Therefore optional LOO/demo/smoke prediction files can never enter this audit.
# =============================================================================

expected_prediction_paths_r4 = {
    label_r4: (
        PREDICTION_DIR_R4
        / f"{label_r4}_all_schedule_predictions.csv"
    )
    for label_r4
    in EXPECTED_PRIMARY_LABELS_R4
}


missing_prediction_labels_r4 = [
    label_r4
    for label_r4, path_r4
    in expected_prediction_paths_r4.items()
    if not path_r4.exists()
]


found_prediction_labels_r4 = [
    label_r4
    for label_r4, path_r4
    in expected_prediction_paths_r4.items()
    if path_r4.exists()
]


print(
    "Saved primary prediction sets found:",
    len(found_prediction_labels_r4),
)


print(
    "Missing primary prediction sets:",
    len(missing_prediction_labels_r4),
)


if missing_prediction_labels_r4:

    print(
        "Missing labels:"
    )

    for label_r4 in missing_prediction_labels_r4:
        print(
            " ",
            label_r4,
        )


# =============================================================================
# 8. HARD-CONSTRAINT DEFINITIONS
# =============================================================================

K_R4 = float(
    TRUE_PARAMETERS.carrying_capacity
)


INITIAL_S_R4 = float(
    INITIAL_S
)


INITIAL_R_R4 = float(
    INITIAL_R
)


INITIAL_S_NORMALIZED_R4 = (
    INITIAL_S_R4
    / K_R4
)


INITIAL_R_NORMALIZED_R4 = (
    INITIAL_R_R4
    / K_R4
)


# Numerical tolerances after writing/reading CSV files.
CONSTRAINT_TOL_R4 = 1.0e-8
INITIAL_TOL_R4 = 1.0e-8


# =============================================================================
# 9. AUDIT EVERY SAVED PRIMARY TRAJECTORY
# =============================================================================

constraint_rows_r4 = []


required_prediction_columns_r4 = [
    "time",
    "predicted_sensitive",
    "predicted_resistant",
]


for label_r4 in EXPECTED_PRIMARY_LABELS_R4:

    prediction_path_r4 = (
        expected_prediction_paths_r4[
            label_r4
        ]
    )


    if not prediction_path_r4.exists():
        continue


    prediction_frame_r4 = pd.read_csv(
        prediction_path_r4
    )


    missing_columns_r4 = [
        column_r4
        for column_r4
        in required_prediction_columns_r4
        if column_r4
        not in prediction_frame_r4.columns
    ]


    if missing_columns_r4:

        constraint_rows_r4.append(
            {
                "label":
                    label_r4,

                "schedule":
                    "FILE_ERROR",

                "n_points":
                    0,

                "minimum_sensitive_normalized":
                    np.nan,

                "minimum_resistant_normalized":
                    np.nan,

                "maximum_total_normalized":
                    np.nan,

                "initial_state_error_normalized":
                    np.nan,

                "initial_state_checked":
                    False,

                "positivity_passed":
                    False,

                "capacity_passed":
                    False,

                "initial_condition_passed":
                    False,

                "all_constraints_passed":
                    False,

                "audit_note":
                    (
                        "Missing required columns: "
                        + ", ".join(
                            missing_columns_r4
                        )
                    ),
            }
        )

        continue


    # -------------------------------------------------------------------------
    # Separate schedules.
    # -------------------------------------------------------------------------

    if "schedule" in prediction_frame_r4.columns:

        schedule_groups_r4 = (
            prediction_frame_r4
            .groupby(
                "schedule",
                observed=True,
                sort=False,
            )
        )

    else:

        schedule_groups_r4 = [
            (
                "ALL",
                prediction_frame_r4,
            )
        ]


    for (
        schedule_name_r4,
        schedule_frame_r4,
    ) in schedule_groups_r4:

        schedule_frame_r4 = (
            schedule_frame_r4
            .copy()
            .sort_values(
                "time"
            )
            .reset_index(
                drop=True
            )
        )


        time_values_r4 = pd.to_numeric(
            schedule_frame_r4[
                "time"
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        sensitive_values_r4 = pd.to_numeric(
            schedule_frame_r4[
                "predicted_sensitive"
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        resistant_values_r4 = pd.to_numeric(
            schedule_frame_r4[
                "predicted_resistant"
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        finite_mask_r4 = (
            np.isfinite(
                time_values_r4
            )
            &
            np.isfinite(
                sensitive_values_r4
            )
            &
            np.isfinite(
                resistant_values_r4
            )
        )


        n_finite_r4 = int(
            finite_mask_r4.sum()
        )


        if n_finite_r4 == 0:

            constraint_rows_r4.append(
                {
                    "label":
                        label_r4,

                    "schedule":
                        str(
                            schedule_name_r4
                        ),

                    "n_points":
                        0,

                    "minimum_sensitive_normalized":
                        np.nan,

                    "minimum_resistant_normalized":
                        np.nan,

                    "maximum_total_normalized":
                        np.nan,

                    "initial_state_error_normalized":
                        np.nan,

                    "initial_state_checked":
                        False,

                    "positivity_passed":
                        False,

                    "capacity_passed":
                        False,

                    "initial_condition_passed":
                        False,

                    "all_constraints_passed":
                        False,

                    "audit_note":
                        "No finite prediction points.",
                }
            )

            continue


        time_valid_r4 = (
            time_values_r4[
                finite_mask_r4
            ]
        )


        sensitive_valid_r4 = (
            sensitive_values_r4[
                finite_mask_r4
            ]
        )


        resistant_valid_r4 = (
            resistant_values_r4[
                finite_mask_r4
            ]
        )


        # ---------------------------------------------------------------------
        # Normalize by carrying capacity.
        # ---------------------------------------------------------------------

        s_norm_r4 = (
            sensitive_valid_r4
            / K_R4
        )


        r_norm_r4 = (
            resistant_valid_r4
            / K_R4
        )


        total_norm_r4 = (
            s_norm_r4
            +
            r_norm_r4
        )


        minimum_sensitive_r4 = float(
            np.min(
                s_norm_r4
            )
        )


        minimum_resistant_r4 = float(
            np.min(
                r_norm_r4
            )
        )


        maximum_total_r4 = float(
            np.max(
                total_norm_r4
            )
        )


        # ---------------------------------------------------------------------
        # Positivity constraints.
        # ---------------------------------------------------------------------

        positivity_passed_r4 = bool(
            minimum_sensitive_r4
            >= -CONSTRAINT_TOL_R4

            and

            minimum_resistant_r4
            >= -CONSTRAINT_TOL_R4
        )


        # ---------------------------------------------------------------------
        # Carrying-capacity constraint.
        # ---------------------------------------------------------------------

        capacity_passed_r4 = bool(
            maximum_total_r4
            <= (
                1.0
                + CONSTRAINT_TOL_R4
            )
        )


        # ---------------------------------------------------------------------
        # Initial-condition audit.
        # ---------------------------------------------------------------------

        initial_state_checked_r4 = False
        initial_state_error_r4 = np.nan
        initial_condition_passed_r4 = True


        initial_candidates_r4 = np.where(
            np.isclose(
                time_valid_r4,
                float(
                    T_START
                ),
                atol=1.0e-10,
                rtol=0.0,
            )
        )[0]


        if len(
            initial_candidates_r4
        ) > 0:

            initial_index_r4 = int(
                initial_candidates_r4[
                    0
                ]
            )


            predicted_initial_s_norm_r4 = float(
                sensitive_valid_r4[
                    initial_index_r4
                ]
                / K_R4
            )


            predicted_initial_r_norm_r4 = float(
                resistant_valid_r4[
                    initial_index_r4
                ]
                / K_R4
            )


            initial_state_error_r4 = float(
                max(
                    abs(
                        predicted_initial_s_norm_r4
                        -
                        INITIAL_S_NORMALIZED_R4
                    ),
                    abs(
                        predicted_initial_r_norm_r4
                        -
                        INITIAL_R_NORMALIZED_R4
                    ),
                )
            )


            initial_state_checked_r4 = True


            initial_condition_passed_r4 = bool(
                initial_state_error_r4
                <= INITIAL_TOL_R4
            )


        # ---------------------------------------------------------------------
        # Final decision.
        # ---------------------------------------------------------------------

        all_constraints_passed_r4 = bool(
            positivity_passed_r4
            and
            capacity_passed_r4
            and
            initial_condition_passed_r4
        )


        constraint_rows_r4.append(
            {
                "label":
                    label_r4,

                "schedule":
                    str(
                        schedule_name_r4
                    ),

                "n_points":
                    n_finite_r4,

                "minimum_sensitive_normalized":
                    minimum_sensitive_r4,

                "minimum_resistant_normalized":
                    minimum_resistant_r4,

                "maximum_total_normalized":
                    maximum_total_r4,

                "initial_state_error_normalized":
                    initial_state_error_r4,

                "initial_state_checked":
                    initial_state_checked_r4,

                "positivity_passed":
                    positivity_passed_r4,

                "capacity_passed":
                    capacity_passed_r4,

                "initial_condition_passed":
                    initial_condition_passed_r4,

                "all_constraints_passed":
                    all_constraints_passed_r4,

                "audit_note":
                    "OK",
            }
        )


# =============================================================================
# 10. CREATE THE FINAL CONSTRAINT-AUDIT TABLE
# =============================================================================

ROUND4_CONSTRAINT_AUDIT = pd.DataFrame(
    constraint_rows_r4
)


if ROUND4_CONSTRAINT_AUDIT.empty:

    raise RuntimeError(
        "No primary prediction trajectory could be audited."
    )


ROUND4_CONSTRAINT_AUDIT.to_csv(
    CONSTRAINT_AUDIT_PATH_R4,
    index=False,
)


constraint_total_count_r4 = int(
    len(
        ROUND4_CONSTRAINT_AUDIT
    )
)


constraint_pass_count_r4 = int(
    ROUND4_CONSTRAINT_AUDIT[
        "all_constraints_passed"
    ]
    .astype(bool)
    .sum()
)


constraint_fail_count_r4 = (
    constraint_total_count_r4
    -
    constraint_pass_count_r4
)


all_constraints_passed_r4 = bool(
    ROUND4_CONSTRAINT_AUDIT[
        "all_constraints_passed"
    ]
    .astype(bool)
    .all()
)


# =============================================================================
# 11. INITIAL-CONDITION COVERAGE
# =============================================================================

initial_checked_count_r4 = int(
    ROUND4_CONSTRAINT_AUDIT[
        "initial_state_checked"
    ]
    .astype(bool)
    .sum()
)


# =============================================================================
# 12. MANUSCRIPT-SCALE COMPLETENESS CHECK
# =============================================================================

primary_prediction_complete_r4 = bool(
    len(
        missing_prediction_labels_r4
    )
    == 0
)


expected_primary_run_count_r4 = int(
    len(
        EXPECTED_PRIMARY_LABELS_R4
    )
)


found_primary_run_count_r4 = int(
    len(
        found_prediction_labels_r4
    )
)


# =============================================================================
# 13. RUNTIME SWITCHES
# =============================================================================

RUNTIME_SWITCHES_R4 = {

    "structural_smoke_test":
        bool(
            globals().get(
                "RUN_STRUCTURAL_SMOKE_TEST",
                False,
            )
        ),

    "single_design_demo":
        bool(
            globals().get(
                "RUN_SINGLE_DESIGN_DEMO",
                False,
            )
        ),

    "primary_multi_schedule_study":
        bool(
            globals().get(
                "RUN_PRIMARY_MULTI_SCHEDULE_STUDY",
                False,
            )
        ),

    "matched_ode_baselines":
        bool(
            globals().get(
                "RUN_MATCHED_ODE_BASELINES",
                False,
            )
        ),

    "identifiability_geometry":
        bool(
            globals().get(
                "RUN_IDENTIFIABILITY_GEOMETRY",
                False,
            )
        ),

    "leave_one_schedule_out":
        bool(
            globals().get(
                "RUN_LEAVE_ONE_SCHEDULE_OUT",
                False,
            )
        ),

    "profile_surfaces":
        bool(
            globals().get(
                "RUN_PROFILE_SURFACES",
                False,
            )
        ),

    "parameter_transfer_test":
        bool(
            globals().get(
                "RUN_PARAMETER_TRANSFER_TEST",
                False,
            )
        ),

    "plotting_only":
        bool(
            globals().get(
                "RUN_PLOTTING_ONLY",
                False,
            )
        ),
}


# =============================================================================
# 14. DESIGN METADATA
# =============================================================================

design_manifest_r4 = {}


if "ROUND4_DESIGNS" in globals():

    for design_id_r4 in DESIGN_ORDER_R4:

        if design_id_r4 not in ROUND4_DESIGNS:
            continue


        design_object_r4 = (
            ROUND4_DESIGNS[
                design_id_r4
            ]
        )


        if is_dataclass(
            design_object_r4
        ):

            design_manifest_r4[
                design_id_r4
            ] = asdict(
                design_object_r4
            )

        else:

            design_manifest_r4[
                design_id_r4
            ] = json_safe_r4(
                design_object_r4
            )


# =============================================================================
# 15. BUILD BASE REPRODUCIBILITY MANIFEST
# =============================================================================

ROUND4_AUDIT = {

    "round":
        4,

    "quick_mode":
        bool(
            QUICK_MODE
        ),

    "run_label":
        str(
            RUN_LABEL
        ),

    "project_root":
        str(
            PROJECT_ROOT.resolve()
        ),

    "output_root":
        str(
            ROUND4_ROOT.resolve()
        ),

    "python":
        platform.python_version(),

    "platform":
        platform.platform(),

    "seed":
        int(
            SEED
        ),

    "device":
        str(
            globals().get(
                "DEVICE",
                "unknown",
            )
        ),

    "dtype":
        str(
            globals().get(
                "DTYPE",
                "unknown",
            )
        ),

    "training_budgets": {

        "pretrain_epochs":
            int(
                PRETRAIN_EPOCHS_R4
            ),

        "physics_warmup_epochs":
            int(
                PHYSICS_WARMUP_EPOCHS_R4
            ),

        "adam_epochs":
            int(
                ADAM_EPOCHS_R4
            ),

        "lbfgs_max_iter":
            int(
                LBFGS_MAX_ITER_R4
            ),

        "collocation_points_per_interval":
            int(
                COLLOCATION_POINTS_PER_INTERVAL_R4
            ),

        "validation_points_per_interval":
            int(
                VALIDATION_POINTS_PER_INTERVAL_R4
            ),

        "lbfgs_points_per_interval":
            int(
                globals().get(
                    "LBFGS_POINTS_PER_INTERVAL_R4",
                    0,
                )
            ),

        "early_stopping_patience":
            int(
                globals().get(
                    "EARLY_STOPPING_PATIENCE_R4",
                    0,
                )
            ),

        "log_every":
            int(
                globals().get(
                    "LOG_EVERY_R4",
                    0,
                )
            ),

        "resample_every":
            int(
                globals().get(
                    "RESAMPLE_EVERY_R4",
                    0,
                )
            ),
    },

    "network": {

        "width":
            int(
                NETWORK_WIDTH_R4
            ),

        "depth":
            int(
                NETWORK_DEPTH_R4
            ),

        "state_learning_rate":
            float(
                STATE_LEARNING_RATE_R4
            ),

        "parameter_learning_rate":
            float(
                PARAMETER_LEARNING_RATE_R4
            ),

        "gradient_clip_norm":
            float(
                GRADIENT_CLIP_NORM_R4
            ),
    },

    "primary_seed_ids":
        json_safe_r4(
            PRIMARY_SEED_IDS_R4
        ),

    "primary_designs":
        json_safe_r4(
            design_manifest_r4
        ),

    "initial_experiment_switches":
        json_safe_r4(
            globals().get(
                "EXPERIMENT_SWITCHES_R4",
                {},
            )
        ),

    "runtime_switches_at_final_audit":
        json_safe_r4(
            RUNTIME_SWITCHES_R4
        ),

    "repository_provenance":
        json_safe_r4(
            globals().get(
                "REPOSITORY_PROVENANCE_R4",
                {},
            )
        ),

    "primary_prediction_completeness": {

        "expected_primary_runs":
            expected_primary_run_count_r4,

        "found_primary_prediction_sets":
            found_primary_run_count_r4,

        "missing_prediction_sets":
            len(
                missing_prediction_labels_r4
            ),

        "missing_labels":
            missing_prediction_labels_r4,

        "complete":
            primary_prediction_complete_r4,
    },

    "hard_constraint_audit": {

        "carrying_capacity":
            K_R4,

        "constraint_tolerance_normalized":
            CONSTRAINT_TOL_R4,

        "initial_condition_tolerance_normalized":
            INITIAL_TOL_R4,

        "audited_trajectory_records":
            constraint_total_count_r4,

        "passed_trajectory_records":
            constraint_pass_count_r4,

        "failed_trajectory_records":
            constraint_fail_count_r4,

        "initial_conditions_checked":
            initial_checked_count_r4,

        "all_constraints_passed":
            all_constraints_passed_r4,

        "constraint_table":
            str(
                CONSTRAINT_AUDIT_PATH_R4.resolve()
            ),
    },
}


# =============================================================================
# 16. FILE-COUNT HELPER
# =============================================================================

def count_files_r4(directory_r4):
    """
    Count files recursively inside a directory.
    """

    return len(
        [
            path_r4
            for path_r4
            in directory_r4.rglob("*")
            if path_r4.is_file()
        ]
    )


# =============================================================================
# 17. CREATE PLACEHOLDER REPORT FILES BEFORE FINAL INVENTORY
# =============================================================================
# This ensures that both report files themselves appear in the inventory/count.
# =============================================================================

MANIFEST_PATH_R4.touch(
    exist_ok=True
)


INVENTORY_PATH_R4.touch(
    exist_ok=True
)


# =============================================================================
# 18. FINAL FILE COUNTS
# =============================================================================

FINAL_FILE_COUNTS_R4 = {

    "model_files":
        count_files_r4(
            MODEL_DIR_R4
        ),

    "prediction_files":
        count_files_r4(
            PREDICTION_DIR_R4
        ),

    "history_files":
        count_files_r4(
            HISTORY_DIR_R4
        ),

    "figure_files":
        count_files_r4(
            FIGURE_DIR_R4
        ),

    "table_files":
        count_files_r4(
            TABLE_DIR_R4
        ),

    "report_files":
        count_files_r4(
            REPORT_DIR_R4
        ),
}


ROUND4_AUDIT[
    "file_counts"
] = FINAL_FILE_COUNTS_R4


# =============================================================================
# 19. CREATE FINAL FILE INVENTORY
# =============================================================================

inventory_r4 = sorted(
    str(
        path_r4.relative_to(
            PROJECT_ROOT
        )
    )
    for path_r4
    in ROUND4_ROOT.rglob("*")
    if path_r4.is_file()
)


FINAL_FILE_COUNTS_R4[
    "total_files"
] = len(
    inventory_r4
)


ROUND4_AUDIT[
    "file_counts"
][
    "total_files"
] = len(
    inventory_r4
)


INVENTORY_PATH_R4.write_text(
    "\n".join(
        inventory_r4
    ),
    encoding="utf-8",
)


# =============================================================================
# 20. WRITE FINAL JSON MANIFEST
# =============================================================================

with open(
    MANIFEST_PATH_R4,
    "w",
    encoding="utf-8",
) as file_r4:

    json.dump(
        json_safe_r4(
            ROUND4_AUDIT
        ),
        file_r4,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 21. DISPLAY THE CONSTRAINT-AUDIT SUMMARY
# =============================================================================

constraint_preview_columns_r4 = [
    "label",
    "schedule",
    "minimum_sensitive_normalized",
    "minimum_resistant_normalized",
    "maximum_total_normalized",
    "initial_state_error_normalized",
    "all_constraints_passed",
]


print(
    "\nConstraint-audit preview:"
)


display(
    ROUND4_CONSTRAINT_AUDIT[
        constraint_preview_columns_r4
    ].head(
        15
    )
)


# =============================================================================
# 22. OPTIONAL PER-DESIGN CONSTRAINT SUMMARY
# =============================================================================

audit_with_design_r4 = (
    ROUND4_CONSTRAINT_AUDIT
    .copy()
)


audit_with_design_r4[
    "design_id"
] = (
    audit_with_design_r4[
        "label"
    ]
    .str.extract(
        r"round4_(M\d)_"
    )[
        0
    ]
)


ROUND4_CONSTRAINT_SUMMARY_BY_DESIGN = (
    audit_with_design_r4
    .groupby(
        "design_id",
        observed=True,
    )
    .agg(
        audited_trajectories=(
            "all_constraints_passed",
            "size",
        ),

        passed_trajectories=(
            "all_constraints_passed",
            "sum",
        ),

        minimum_S_over_K=(
            "minimum_sensitive_normalized",
            "min",
        ),

        minimum_R_over_K=(
            "minimum_resistant_normalized",
            "min",
        ),

        maximum_total_over_K=(
            "maximum_total_normalized",
            "max",
        ),

        maximum_initial_error=(
            "initial_state_error_normalized",
            "max",
        ),
    )
    .reset_index()
)


print(
    "\nConstraint summary by design:"
)


display(
    ROUND4_CONSTRAINT_SUMMARY_BY_DESIGN
)


# =============================================================================
# 23. FINAL IMPLEMENTATION AUDIT REPORT
# =============================================================================

print(
    "\n"
    + "=" * 86
)


print(
    "ROUND 4 FINAL IMPLEMENTATION AUDIT"
)


print(
    "=" * 86
)


print(
    "\nRun configuration:"
)


print(
    "  QUICK_MODE:",
    QUICK_MODE,
)


print(
    "  Run label:",
    RUN_LABEL,
)


print(
    "  Output root:",
    ROUND4_ROOT.resolve(),
)


print(
    "\nPrimary prediction completeness:"
)


print(
    "  Expected primary runs:",
    expected_primary_run_count_r4,
)


print(
    "  Found primary prediction sets:",
    found_primary_run_count_r4,
)


print(
    "  Missing prediction sets:",
    len(
        missing_prediction_labels_r4
    ),
)


print(
    "  Complete:",
    primary_prediction_complete_r4,
)


print(
    "\nHard-constraint audit:"
)


print(
    "  Audited trajectory records:",
    constraint_total_count_r4,
)


print(
    "  Passed:",
    constraint_pass_count_r4,
)


print(
    "  Failed:",
    constraint_fail_count_r4,
)


print(
    "  Initial conditions checked:",
    initial_checked_count_r4,
)


print(
    "  All hard constraints passed:",
    all_constraints_passed_r4,
)


print(
    "\nRound 4 output inventory:"
)


print(
    "  Model files:",
    FINAL_FILE_COUNTS_R4[
        "model_files"
    ],
)


print(
    "  Prediction files:",
    FINAL_FILE_COUNTS_R4[
        "prediction_files"
    ],
)


print(
    "  History files:",
    FINAL_FILE_COUNTS_R4[
        "history_files"
    ],
)


print(
    "  Figure files:",
    FINAL_FILE_COUNTS_R4[
        "figure_files"
    ],
)


print(
    "  Table files:",
    FINAL_FILE_COUNTS_R4[
        "table_files"
    ],
)


print(
    "  Report files:",
    FINAL_FILE_COUNTS_R4[
        "report_files"
    ],
)


print(
    "  Total files:",
    FINAL_FILE_COUNTS_R4[
        "total_files"
    ],
)


print(
    "\nSaved constraint audit:"
)


print(
    " ",
    CONSTRAINT_AUDIT_PATH_R4.resolve(),
)


print(
    "\nSaved reproducibility manifest:"
)


print(
    " ",
    MANIFEST_PATH_R4.resolve(),
)


print(
    "\nSaved complete file inventory:"
)


print(
    " ",
    INVENTORY_PATH_R4.resolve(),
)


print(
    "\nNO MODEL TRAINING WAS PERFORMED."
)


# =============================================================================
# 24. FINAL MANUSCRIPT-SCALE STATUS
# =============================================================================

if QUICK_MODE:

    print(
        "\nQuick structural configuration detected."
    )

    print(
        "For manuscript-scale experiments, use QUICK_MODE=False."
    )

else:

    print(
        "\nManuscript-scale Round 4 configuration completed."
    )


# =============================================================================
# 25. FINAL ASSERTIONS
# =============================================================================
# These are intentionally placed AFTER writing the audit files.
#
# Therefore, even if an integrity check fails, the audit CSV and manifest
# remain available for diagnosis.
# =============================================================================

if missing_prediction_labels_r4:

    raise AssertionError(
        "Round 4 primary prediction set is incomplete. "
        f"{len(missing_prediction_labels_r4)} expected prediction "
        "file(s) are missing."
    )


if not all_constraints_passed_r4:

    failed_rows_r4 = (
        ROUND4_CONSTRAINT_AUDIT.loc[
            ~ROUND4_CONSTRAINT_AUDIT[
                "all_constraints_passed"
            ]
            .astype(bool)
        ]
    )

    print(
        "\nFAILED CONSTRAINT-AUDIT ROWS:"
    )

    display(
        failed_rows_r4
    )

    raise AssertionError(
        "At least one saved primary trajectory violated the "
        "hard feasibility constraints beyond numerical tolerance."
    )


# =============================================================================
# 26. SUCCESS MESSAGE
# =============================================================================

print(
    "\n"
    + "=" * 86
)


print(
    "ROUND 4 CELL 27 COMPLETED SUCCESSFULLY"
)


print(
    "=" * 86
)


print(
    "\nFinal audit status:"
)


print(
    "  Primary prediction completeness : PASS"
)


print(
    "  Positivity constraint           : PASS"
)


print(
    "  Carrying-capacity constraint    : PASS"
)


print(
    "  Initial-condition preservation  : PASS"
)


print(
    "  Reproducibility manifest        : SAVED"
)


print(
    "  Round 4 file inventory          : SAVED"
)